# A response-sensitive moisture window shapes soil moisture loss during global compound drought-heatwave events


## 0. Project paths and final-code policy

`SENSITIVE_CODE_DIR` is the final source for response-sensitive-region and SHAP code. Legacy sensitive-region/SHAP notebook cells are excluded.


In [ ]:
# -*- coding: utf-8 -*-
from pathlib import Path

PROJECT_ROOT = Path(r"D:\smq\paper2")
CODE_DIR = PROJECT_ROOT / "code"
DATA_DIR = PROJECT_ROOT / "data"
RESULT_DIR = PROJECT_ROOT / "result"
MODIFY_DIR = PROJECT_ROOT / "modify"
SENSITIVE_PACKAGE = MODIFY_DIR / "敏感区"
SENSITIVE_CODE_DIR = SENSITIVE_PACKAGE / "code"
SENSITIVE_DATA_DIR = SENSITIVE_PACKAGE / "data"
SENSITIVE_FIG_DIR = SENSITIVE_PACKAGE / "figures"

MANUSCRIPT_FINAL = MODIFY_DIR / "修改版20260627" / "Manuscript_r20260609.docx"
SUPPLEMENT_FINAL = MODIFY_DIR / "修改版20260627" / "supplement_r20260609.docx"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SENSITIVE_CODE_DIR:", SENSITIVE_CODE_DIR)
print("MANUSCRIPT_FINAL:", MANUSCRIPT_FINAL)
print("SUPPLEMENT_FINAL:", SUPPLEMENT_FINAL)


## 2. Identification of CDHW events

Main Methods: CDHW event detection, including daily SPEI, heatwave mask and CDHW mask/event detection.


### Daily SPEI calculation from precipitation and ETRef/PET




In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from climate_indices import indices
from climate_indices.compute import Periodicity
from climate_indices.indices import Distribution
import dask
from dask.distributed import Client, LocalCluster
import os

# ----------------------------
# 1. 首先配置 Dask 并行
# ----------------------------
cluster = LocalCluster(
    n_workers=20,           
    threads_per_worker=1,
    memory_limit="10GB"     
)
client = Client(cluster)

# print(client)
print("Dashboard 地址:", client.dashboard_link)


def spei_1d(precip, pet, scale=3,
            data_start_year=1915,
            calibration_year_initial=1915,
            calibration_year_final=2019):
    """计算一维时间序列的 SPEI"""
    if precip.size < scale or np.all(np.isnan(precip)) or np.all(np.isnan(pet)):
        return np.full(precip.shape, np.nan, dtype=float)


    spei_values = indices.spei(
        precips_mm=precip,
        pet_mm=pet,
        scale=scale,
        distribution=Distribution.gamma,
        periodicity=Periodicity.daily,
        data_start_year=data_start_year,
        calibration_year_initial=calibration_year_initial,
        calibration_year_final=calibration_year_final
    )
    return spei_values



et_his = xr.open_dataset(r"F:\paper4\GWSP3_W5E5\ETRef_daily.nc",chunks={"lat": 30, "lon": 30, "time": -1}).ETRef.astype(np.float32) 

pr_his = xr.open_dataset(r"F:\paper4\GWSP3_W5E5\Precipitation_daily.nc",chunks={"lat": 30, "lon": 30, "time": -1}).Precipitation.astype(np.float32)  


spei_da = xr.apply_ufunc(
    spei_1d,
    pr_his,
    et_his,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[["time"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
    dask_gufunc_kwargs={"allow_rechunk": True},
    kwargs={
        "scale": 60,
        "data_start_year": 1915,
        "calibration_year_initial": 1915,
        "calibration_year_final": 2019
    }
)


# ----------------------------
# 5. 保存结果（推荐：边算边写）
# ----------------------------
output_file = r"E:\smq\spei\SPEI_daily1.nc"
encoding = {"spei": {"zlib": True, "complevel": 4, "dtype": "float32"}}

# 如果文件已存在，先删除
if os.path.exists(output_file):
    os.remove(output_file)

# 使用 h5netcdf 引擎，支持 Dask 并行写
delayed_obj = spei_ds.to_netcdf(
    output_file,
    engine="h5netcdf",
    encoding=encoding,
    compute=False  # 边算边写，不一次性塞进内存
)

import dask
dask.compute(delayed_obj)  # 触发计算和写入

print(f"[Success] 文件已写入: {output_file}")


### Annual SPEI aggregation used for diagnostics




In [ ]:

import xarray as xr
import matplotlib.pyplot as plt

# ======================
# 1. 读取数据
# ======================
fp = r"E:\smq\spei\SPEI_daily1.nc"
spei_daily = xr.open_dataset(fp, engine="h5netcdf")['spei']

# ======================
# 2. 按年聚合（平均）
# ======================
spei_annual = spei_daily.resample(time="1Y").mean(dim="time")
spei_annual_mean = spei_annual.mean(dim=["lat", "lon"])

# ======================
# 3. 保存年聚合结果为 NetCDF
# ======================
output_file = r"E:\smq\spei\SPEI_annual_mean.nc"

# 保险起见，如已有旧文件则先删除
if os.path.exists(output_file):
    os.remove(output_file)

encoding = {"spei": {"zlib": True, "complevel": 4, "dtype": "float32"}}

# 创建Dataset并保存
spei_annual_ds = xr.Dataset({"spei": spei_annual})
spei_annual_ds.to_netcdf(output_file, engine="h5netcdf", encoding=encoding)

print(f"[✅ 成功] 年聚合 SPEI 文件已保存至：{output_file}")


### Heatwave-day mask from Tmax and calendar-day 95th percentile


In [ ]:
import xarray as xr
import numpy as np
from dask.distributed import Client
import os

# ======================
# 1. 启动 Dask
# ======================
client = Client(n_workers=10, threads_per_worker=2, memory_limit="100GB")
print("Dask dashboard:", client.dashboard_link)

# ======================
# 2. 参数设置
# ======================
min_days = 3
start_date, end_date = "1920-01-01", "2019-12-31"

# ======================
# 3. 读取数据
# ======================
tas95 = xr.open_dataset(r"E:\smq\tas95_1981_2010.nc",
                        chunks={"lat": 50, "lon": 50})["tas95"] - 273.15

tasmax = xr.open_dataset(r"E:\smq\Tmax\TMax_daily.nc",
                         engine="h5netcdf").sel(time=slice(start_date, end_date)) \
                         .chunk({"time": -1, "lat": 50, "lon": 50})["TMax"]

# ======================
# 4. 热浪检测（逐像元函数）
# ======================
def detect_events_1d(flag, min_days=3):
    mask = np.zeros_like(flag, dtype=bool)
    count = 0
    for i in range(len(flag)):
        if flag[i]:
            count += 1
        else:
            if count >= min_days:
                mask[i-count:i] = True
            count = 0
    if count >= min_days:
        mask[len(flag)-count:] = True
    return mask

# 热浪定义
hot_flag = tasmax > tas95

# 热浪mask（True 表示处于热浪）
hw_mask = xr.apply_ufunc(
    detect_events_1d,
    hot_flag,
    input_core_dims=[["time"]],
    output_core_dims=[["time"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[bool],
    kwargs={"min_days": min_days},
)

# ======================
# 5. 按年计算热浪天数占比
# ======================
# 每年热浪天数
hw_days = hw_mask.resample(time="1Y").sum(dim="time")

# 每年总天数（自动考虑闰年）
total_days = hw_mask["time"].resample(time="1Y").count()

# 每年占比（百分比）
hw_ratio = (hw_days / total_days) * 100
hw_ratio = hw_ratio.compute()
hw_ratio.name = "hw_days_ratio"
hw_ratio.attrs["units"] = "%"
hw_ratio.attrs["description"] = "每年热浪天数占全年天数比例（%）"

# ======================
# 6. 保存结果
# ======================
ds_out = hw_ratio.to_dataset(name="hw_days_ratio")

out_fp = r"D:\smq\paper2\data\hw_days_ratio_annual.nc"
if os.path.exists(out_fp):
    os.remove(out_fp)

encoding = {"hw_days_ratio": {"zlib": True, "complevel": 4, "dtype": "float32"}}
ds_out.to_netcdf(out_fp, engine="h5netcdf", encoding=encoding)

print(f"[✅ 成功] 文件已保存: {out_fp}")

# ======================
# 7. 关闭 Dask
# ======================
client.close()


### CDHW event mask: SPEI < -1 and Tmax > 95th percentile for >=3 consecutive days



In [ ]:
import xarray as xr
import numpy as np
from dask.distributed import Client

# ======================
# 配置 Dask
# ======================
client = Client(n_workers=10, threads_per_worker=2, memory_limit="100GB")
print("Dask dashboard:", client.dashboard_link)

# ======================
# 参数
# ======================
min_days = 3
spei_thresh = -1
start_date, end_date = "1920-01-01", "2019-12-31"

# ======================
# 打开数据
# ======================
tas95 = xr.open_dataset(r"E:\smq\tas95_1981_2010.nc",
                        chunks={"lat": 50, "lon": 50})["tas95"] - 273.15

tasmax = xr.open_dataset(r"E:\smq\Tmax\TMax_daily.nc",
                         engine="h5netcdf").sel(time=slice(start_date, end_date)) \
                         .chunk({"time": -1, "lat": 50, "lon": 50})["TMax"]

spei = xr.open_dataset(r"E:\smq\spei\SPEI_daily1.nc",
                       engine="h5netcdf").sel(time=slice(start_date, end_date)) \
                       .chunk({"time": -1, "lat": 50, "lon": 50})["spei"]

# ======================
# Step1: 干旱和热浪 flag
# ======================
drought_flag = spei < spei_thresh
hot_flag = tasmax > tas95

# ======================
# Step2: 定义逐像元函数，找到连续 >= min_days 的事件
# ======================
def detect_events_1d(flag, min_days=3):
    mask = np.zeros_like(flag, dtype=bool)
    count = 0
    for i in range(len(flag)):
        if flag[i]:
            count += 1
        else:
            if count >= min_days:
                mask[i-count:i] = True
            count = 0
    # 结尾再检查一次
    if count >= min_days:
        mask[len(flag)-count:] = True
    return mask

hw_mask = xr.apply_ufunc(
    detect_events_1d,
    hot_flag,
    input_core_dims=[["time"]],
    output_core_dims=[["time"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[bool],
    kwargs={"min_days": min_days},
)

# ======================
# Step3: CDHW = 热浪日 & 干旱日
# ======================
cdhw_mask = hw_mask & drought_flag

# ======================
# Step4: baseline = 非干旱 & 非热浪
# ======================
baseline_mask = (~drought_flag) & (~hw_mask)

# ======================
# Step5: 保存结果
# ======================
ds_out = xr.Dataset(
    {
        "drought_mask": drought_flag.astype(np.int8),
        "hw_mask": hw_mask.astype(np.int8),
        "cdhw_mask": cdhw_mask.astype(np.int8),
        "baseline_mask": baseline_mask.astype(np.int8),
    }
)

# 压缩设置
comp = dict(zlib=True, complevel=4)
encoding = {var: comp for var in ds_out.data_vars}

# 保存
out_fp = r"D:\smq\paper2\data\masks_cdhw_hw_di.nc"
ds_out.to_netcdf(out_fp, encoding=encoding)
print("保存完成:", out_fp)



## 3. CDHW characteristics and emergence from natural variability

Main Methods: signal emergence / TCIE; supports Fig. 1 and Table S14.


### Annual CDHW frequency




In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from dask.distributed import Client, LocalCluster
import os
import dask  # 提前导入dask，避免后续计算时导入延迟

# ======================
# 配置 Dask
# ======================
cluster = LocalCluster(
    n_workers=10,
    threads_per_worker=1,
    memory_limit="100GB"
)
client = Client(cluster)
print("Dashboard 地址:", client.dashboard_link)

# ======================
# 读掩膜数据（修复分块警告）
# ======================
mask_fp = r"D:\smq\paper2\data\cdhw\masks_cdhw_hw_di.nc"
ds_mask = xr.open_dataset(
    mask_fp,
    engine="h5netcdf"
).chunk({"time": -1, "lat": 50, "lon": 50})
cdhw_mask = ds_mask["cdhw_mask"]

# ======================
# 每个格点的事件→年度次数函数
# ======================
def process_point(mask_1d, time_vals):
    mask_np = mask_1d.astype(int)

    # Step1: 标记事件ID
    eid = 0
    in_event = False
    event_id = np.zeros_like(mask_np, dtype=int)
    for i, val in enumerate(mask_np):
        if val == 1 and not in_event:
            eid += 1
            in_event = True
        elif val == 0:
            in_event = False
        if in_event:
            event_id[i] = eid

    # Step2: 按年份统计事件数（取每个事件的开始日所在年份）
    years, counts = [], []
    if eid > 0:
        for e in np.unique(event_id):
            if e == 0:
                continue
            idx = np.where(event_id == e)[0]
            year = pd.to_datetime(time_vals[idx[0]]).year
            years.append(year)

    # Step3: 年度对齐
    years_all = np.arange(
        pd.to_datetime(time_vals[0]).year,
        pd.to_datetime(time_vals[-1]).year + 1
    )
    c = np.zeros(years_all.shape, dtype=np.float32)
    if len(years) > 0:
        df = pd.Series(years).value_counts()
        for y, n in df.items():
            c[np.searchsorted(years_all, y)] = n

    return c  # 输出 shape=(nyears,)

# ======================
# 计算完整年份范围（用于输出维度）
# ======================
years_all = np.arange(
    int(cdhw_mask.time.dt.year.min()),
    int(cdhw_mask.time.dt.year.max()) + 1
)

# ======================
# 并行调用函数
# ======================
annual_count = xr.apply_ufunc(
    process_point,
    cdhw_mask,
    cdhw_mask["time"],
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[["year"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[np.float32],
    dask_gufunc_kwargs={"output_sizes": {"year": len(years_all)}},
)

annual_count = annual_count.assign_coords(year=years_all)
annual_count.name = "annual_event_count"

# ======================
# 创建Dataset
# ======================
ds_out = xr.Dataset(
    data_vars={"annual_event_count": annual_count},
    coords={
        "year": years_all,
        "lat": cdhw_mask.lat,
        "lon": cdhw_mask.lon
    }
)

# ======================
# 保存结果
# ======================
output_file = r"D:\smq\paper2\data\cdhw\cdhw_event_count_yearly.nc"
encoding = {
    "annual_event_count": {"zlib": True, "complevel": 4, "dtype": "float32"}
}

if os.path.exists(output_file):
    os.remove(output_file)

print("保存结果...")
delayed = ds_out.to_netcdf(
    output_file,
    engine="h5netcdf",
    encoding=encoding,
    compute=False
)
dask.compute(delayed)

print(f"[Success] 文件已写入: {output_file}")
client.close()


### Annual CDHW duration




In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from dask.distributed import Client, LocalCluster
import dask
import os

# ======================
# 1. 启动 Dask 并行环境
# ======================
cluster = LocalCluster(
    n_workers=10,
    threads_per_worker=1,
    memory_limit="100GB"
)
client = Client(cluster)
print("Dashboard 地址:", client.dashboard_link)

# ======================
# 2. 读取掩膜数据（事件标记）
# ======================
mask_fp = r"D:\smq\paper2\data\cdhw\masks_cdhw_hw_di.nc"
ds_mask = xr.open_dataset(mask_fp, engine="h5netcdf").chunk({"time": -1, "lat": 20, "lon": 20})
cdhw_mask = ds_mask["cdhw_mask"]  # 值为 0/1，表示是否处于CDHW事件中

# ======================
# 3. 定义逐格点函数
# ======================
def calc_annual_duration(mask_1d, time_vals):
    """
    计算每个格点每年的平均事件持续时间（天）
    """
    times = pd.to_datetime(time_vals)
    years_all = np.arange(times[0].year, times[-1].year + 1)
    c = np.zeros(len(years_all), dtype=np.float32)

    in_event = False
    event_id = np.zeros(len(mask_1d), dtype=int)
    eid = 0

    for i, val in enumerate(mask_1d):
        if val > 0 and not in_event:
            eid += 1
            in_event = True
        elif val == 0:
            in_event = False
        if in_event:
            event_id[i] = eid

    # 收集每年事件持续时间
    yearly_durations = {year: [] for year in years_all}
    for e in np.unique(event_id):
        if e == 0:
            continue
        idx = np.where(event_id == e)[0]
        start_idx = idx[0]
        dur = len(idx)
        year = times[start_idx].year
        yearly_durations[year].append(dur)

    # 按年平均
    for i, y in enumerate(years_all):
        durs = yearly_durations[y]
        c[i] = np.mean(durs) if len(durs) > 0 else 0.0

    return c

# ======================
# 4. 年份范围
# ======================
years_all = np.arange(
    int(cdhw_mask.time.dt.year.min()),
    int(cdhw_mask.time.dt.year.max()) + 1
)

# ======================
# 5. 并行 apply_ufunc
# ======================
annual_duration = xr.apply_ufunc(
    calc_annual_duration,
    cdhw_mask,               # ✅ 正确输入：事件mask
    cdhw_mask["time"],
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[["year"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[np.float32],
    dask_gufunc_kwargs={"output_sizes": {"year": len(years_all)}},
)

annual_duration = annual_duration.assign_coords(year=years_all)
annual_duration.name = "annual_event_duration"

# ======================
# 6. 输出保存
# ======================
ds_out = xr.Dataset(
    {"annual_event_duration": annual_duration},
    coords={"year": years_all, "lat": cdhw_mask.lat, "lon": cdhw_mask.lon}
)

output_file = r"D:\smq\paper2\data\cdhw\cdhw_yearly_event_duration.nc"
encoding = {
    "annual_event_duration": {"zlib": True, "complevel": 4, "dtype": "float32"}
}

if os.path.exists(output_file):
    os.remove(output_file)

print("保存结果中...")
delayed = ds_out.to_netcdf(
    output_file, engine="h5netcdf", encoding=encoding, compute=False
)
dask.compute(delayed)

print(f"[✅ 成功] 文件已写入: {output_file}")
client.close()


### Daily CDHW intensity



In [ ]:
import xarray as xr
import numpy as np
from dask.distributed import Client, LocalCluster
import dask
import os

# ======================
# 配置 Dask
# ======================
cluster = LocalCluster(
    n_workers=10,           
    threads_per_worker=1,
    memory_limit="100GB"     
)
client = Client(cluster)
print("Dashboard 地址:", client.dashboard_link)

# ======================
# 参数
# ======================
start_date, end_date = "1920-01-01", "2019-12-31"

# ======================
# 打开数据
# ======================
tas25 = xr.open_dataset(r"D:\smq\paper2\data\tas25_1981_2010.nc")["tas25"] - 273.15
tas75 = xr.open_dataset(r"D:\smq\paper2\data\tas75_1981_2010.nc")["tas75"] - 273.15
tas95 = xr.open_dataset(r"E:\smq\tas95_1981_2010.nc")["tas95"] - 273.15

tasmax = xr.open_dataset(
    r"E:\smq\Tmax\TMax_daily.nc", engine="h5netcdf"
).sel(time=slice(start_date, end_date)).chunk({"time": -1, "lat": 50, "lon": 50})["TMax"]

spei = xr.open_dataset(
    r"E:\smq\spei\SPEI_daily1.nc", engine="h5netcdf"
).sel(time=slice(start_date, end_date)).chunk({"time": -1, "lat": 50, "lon": 50})["spei"]

# ✅ 直接读取现成的 CDHW 掩膜
cdhw_mask = xr.open_dataset(
    r"D:\smq\paper2\data\masks_cdhw_hw_di.nc", engine="h5netcdf"
)["cdhw_mask"].sel(time=slice(start_date, end_date)).chunk({"time": -1, "lat": 50, "lon": 50})

# ======================
# Step: 严重程度（逐日）
# ======================
daily_severity = xr.where(
    cdhw_mask,  # 直接用已有掩膜
    np.abs(spei) * (tasmax - tas25) / (tas75 - tas25),
    0
)

# ======================
# Step: 保存结果
# ======================
ds_out = xr.Dataset({"daily_severity": daily_severity})

output_file = r"D:\smq\paper2\data\cdhw_daily_severity.nc"
encoding = {"daily_severity": {"zlib": True, "complevel": 4, "dtype": "float32"}}

if os.path.exists(output_file):
    os.remove(output_file)

delayed_obj = ds_out.to_netcdf(
    output_file,
    engine="h5netcdf",
    encoding=encoding,
    compute=False
)

dask.compute(delayed_obj)
print(f"[Success] 文件已写入: {output_file}")


### Annual event-sum intensity




In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from dask.distributed import Client, LocalCluster
import os
import dask  # 提前导入dask，避免后续计算时导入延迟

# ======================
# 配置 Dask
# ======================
cluster = LocalCluster(
    n_workers=10,
    threads_per_worker=1,
    memory_limit="100GB"
)
client = Client(cluster)
print("Dashboard 地址:", client.dashboard_link)

# ======================
# 读数据（修复分块警告）
# ======================
mask_fp = r"D:\smq\paper2\data\cdhw\masks_cdhw_hw_di.nc"
ds_mask = xr.open_dataset(
    mask_fp,
    engine="h5netcdf"
).chunk({"time": -1, "lat": 20, "lon": 20})
cdhw_mask = ds_mask["cdhw_mask"]

severity_fp = r"D:\smq\paper2\data\cdhw\cdhw_severity_daily.nc"
ds_sev = xr.open_dataset(
    severity_fp,
    engine="h5netcdf"
).chunk({"time": -1, "lat": 20, "lon": 20})
daily_severity = ds_sev["daily_severity"]

# ======================
# 每个格点的事件→年度强度函数
# ======================
import numpy as np
import pandas as pd
import xarray as xr

def process_point_intensity(mask_1d, time_vals):
    """
    计算每个事件的平均强度，并按年份聚合（取每年所有事件平均强度的平均值）

    Parameters:
        mask_1d (np.ndarray): 一维数组，每日强度值（可以是 float）
        time_vals: 时间戳数组，对应 mask_1d 的时间维度

    Returns:
        c (np.ndarray): shape=(n_years,), 每年对应的事件平均强度的平均值
    """
    # 转换时间为 pandas DatetimeIndex
    times = pd.to_datetime(time_vals)

    # 定义年份范围
    years_all = np.arange(times[0].year, times[-1].year + 1)
    c = np.zeros(len(years_all), dtype=np.float32)  # 存储每年的结果

    # Step1: 提取事件（连续非零天为一个事件）
    in_event = False
    event_id = np.zeros(len(mask_1d), dtype=int)
    eid = 0

    for i, val in enumerate(mask_1d):
        if val > 0 and not in_event:
            eid += 1
            in_event = True
        elif val == 0:
            in_event = False
        if in_event:
            event_id[i] = eid

    # Step2: 遍历每个事件，计算其平均强度，并记录起始年份
    yearly_intensities = {year: [] for year in years_all}  # 每年收集事件的平均强度

    for e in np.unique(event_id):
        if e == 0:
            continue  # 跳过无事件区域

        indices = np.where(event_id == e)[0]
        start_idx = indices[0]
        event_values = mask_1d[indices]

        avg_intensity = np.sum(event_values)  # 该事件的平均强度
        year = times[start_idx].year

        if year in yearly_intensities:
            yearly_intensities[year].append(avg_intensity)

    # Step3: 对每一年，取该年所有事件平均强度的平均值
    for i, year in enumerate(years_all):
        intensities = yearly_intensities[year]
        if len(intensities) > 0:
            c[i] = np.mean(intensities)  # 可改为 np.sum 如果需要总和
        else:
            c[i] = 0.0  # 无事件则为0

    return c  # shape=(nyears,)
# ======================
# 计算完整年份范围（用于输出维度）
# ======================
years_all = np.arange(
    int(cdhw_mask.time.dt.year.min()),
    int(cdhw_mask.time.dt.year.max()) + 1
)

# ======================
# 并行调用函数
# ======================
annual_strength = xr.apply_ufunc(
    process_point_intensity,
    # cdhw_mask,
    daily_severity,
    cdhw_mask["time"],
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[["year"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[np.float32],
    dask_gufunc_kwargs={"output_sizes": {"year": len(years_all)}},
)

annual_strength = annual_strength.assign_coords(year=years_all)
annual_strength.name = "annual_event_severity"

# ======================
# 创建Dataset
# ======================
ds_out = xr.Dataset(
    data_vars={"annual_event_severity": annual_strength},
    coords={
        "year": years_all,
        "lat": cdhw_mask.lat,
        "lon": cdhw_mask.lon
    }
)

# ======================
# 保存结果
# ======================
output_file = r"D:\smq\paper2\data\cdhw\cdhw_yearly_event_severity.nc"
encoding = {
    "annual_event_severity": {"zlib": True, "complevel": 4, "dtype": "float32"}
}

if os.path.exists(output_file):
    os.remove(output_file)

print("保存结果...")
delayed = ds_out.to_netcdf(
    output_file,
    engine="h5netcdf",
    encoding=encoding,
    compute=False
)
dask.compute(delayed)

print(f"[Success] 文件已写入: {output_file}")
client.close()


### Annual event-mean intensity



In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from dask.distributed import Client, LocalCluster
import os
import dask  # 提前导入dask，避免后续计算时导入延迟

# ======================
# 配置 Dask
# ======================
cluster = LocalCluster(
    n_workers=10,
    threads_per_worker=1,
    memory_limit="100GB"
)
client = Client(cluster)
print("Dashboard 地址:", client.dashboard_link)

# ======================
# 读数据（修复分块警告）
# ======================
mask_fp = r"D:\smq\paper2\data\cdhw\masks_cdhw_hw_di.nc"
ds_mask = xr.open_dataset(
    mask_fp,
    engine="h5netcdf"
).chunk({"time": -1, "lat": 20, "lon": 20})
cdhw_mask = ds_mask["cdhw_mask"]

severity_fp = r"D:\smq\paper2\data\cdhw\cdhw_severity_daily.nc"
ds_sev = xr.open_dataset(
    severity_fp,
    engine="h5netcdf"
).chunk({"time": -1, "lat": 20, "lon": 20})
daily_severity = ds_sev["daily_severity"]

# ======================
# 每个格点的事件→年度强度函数
# ======================
import numpy as np
import pandas as pd
import xarray as xr

def process_point_intensity(mask_1d, time_vals):

    # 转换时间为 pandas DatetimeIndex
    times = pd.to_datetime(time_vals)

    # 定义年份范围
    years_all = np.arange(times[0].year, times[-1].year + 1)
    c = np.zeros(len(years_all), dtype=np.float32)  # 存储每年的结果

    # Step1: 提取事件（连续非零天为一个事件）
    in_event = False
    event_id = np.zeros(len(mask_1d), dtype=int)
    eid = 0

    for i, val in enumerate(mask_1d):
        if val > 0 and not in_event:
            eid += 1
            in_event = True
        elif val == 0:
            in_event = False
        if in_event:
            event_id[i] = eid

    # Step2: 遍历每个事件，计算其平均强度，并记录起始年份
    yearly_intensities = {year: [] for year in years_all}  # 每年收集事件的平均强度

    for e in np.unique(event_id):
        if e == 0:
            continue  # 跳过无事件区域

        indices = np.where(event_id == e)[0]
        start_idx = indices[0]
        event_values = mask_1d[indices]

        avg_intensity = np.mean(event_values)  # 该事件的平均强度
        year = times[start_idx].year

        if year in yearly_intensities:
            yearly_intensities[year].append(avg_intensity)

    # Step3: 对每一年，取该年所有事件平均强度的平均值
    for i, year in enumerate(years_all):
        intensities = yearly_intensities[year]
        if len(intensities) > 0:
            c[i] = np.mean(intensities)  # 可改为 np.sum 如果需要总和
        else:
            c[i] = 0.0  # 无事件则为0

    return c  # shape=(nyears,)
# ======================
# 计算完整年份范围（用于输出维度）
# ======================
years_all = np.arange(
    int(cdhw_mask.time.dt.year.min()),
    int(cdhw_mask.time.dt.year.max()) + 1
)

# ======================
# 并行调用函数
# ======================
annual_strength = xr.apply_ufunc(
    process_point_intensity,
    # cdhw_mask,
    daily_severity,
    cdhw_mask["time"],
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[["year"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[np.float32],
    dask_gufunc_kwargs={"output_sizes": {"year": len(years_all)}},
)

annual_strength = annual_strength.assign_coords(year=years_all)
annual_strength.name = "annual_severity"

# ======================
# 创建Dataset
# ======================
ds_out = xr.Dataset(
    data_vars={"annual_severity": annual_strength},
    coords={
        "year": years_all,
        "lat": cdhw_mask.lat,
        "lon": cdhw_mask.lon
    }
)

# ======================
# 保存结果
# ======================
output_file = r"D:\smq\paper2\data\cdhw\cdhw_meanevent_yearly_severity.nc"
encoding = {
    "annual_severity": {"zlib": True, "complevel": 4, "dtype": "float32"}
}

if os.path.exists(output_file):
    os.remove(output_file)

print("保存结果...")
delayed = ds_out.to_netcdf(
    output_file,
    engine="h5netcdf",
    encoding=encoding,
    compute=False
)
dask.compute(delayed)

print(f"[Success] 文件已写入: {output_file}")
client.close()


### Frequency-duration controls for intensity diagnostics



In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from scipy import stats
import statsmodels.api as sm

# ======================
# 1. 文件路径
# ======================
fp_y  = r"D:\smq\paper2\data\cdhw\cdhw_meanevent_yearly_severity.nc"
fp_x1 = r"D:\smq\paper2\data\cdhw\cdhw_event_count_yearly.nc"
fp_x2 = r"D:\smq\paper2\data\cdhw\cdhw_yearly_event_duration.nc"

# ======================
# 2. 读取数据
# ======================
y  = xr.open_dataset(fp_y,  engine="h5netcdf")["annual_severity"]
x1 = xr.open_dataset(fp_x1, engine="h5netcdf")["annual_event_count"]
x2 = xr.open_dataset(fp_x2, engine="h5netcdf")["annual_event_duration"]

# 统一时间维度
years = y["year"].values
x1 = x1.sel(year=years)
x2 = x2.sel(year=years)

# ======================
# ✅ 截取指定时间段（例如 1981–2010）
# ======================
start_year, end_year = 1920, 1980
y  = y.sel(year=slice(start_year, end_year))
x1 = x1.sel(year=slice(start_year, end_year))
x2 = x2.sel(year=slice(start_year, end_year))

print(f"已截取年份范围: {y.year.values[0]}–{y.year.values[-1]}")

# ======================
# 3. 定义逐格点多元回归函数
# ======================

def regress_pixel(y_arr, x1_arr, x2_arr):
    mask = np.isfinite(y_arr) & np.isfinite(x1_arr) & np.isfinite(x2_arr)
    if np.sum(mask) < 10:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    y_masked = y_arr[mask]
    x1_masked = x1_arr[mask]
    x2_masked = x2_arr[mask]

    if np.std(y_masked) == 0 or np.std(x1_masked) == 0 or np.std(x2_masked) == 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    # 标准化
    scaler = StandardScaler()
    X = np.stack([x1_masked, x2_masked], axis=1)
    X_scaled = scaler.fit_transform(X)
    y_scaled = (y_masked - np.mean(y_masked)) / np.std(y_masked)

    # 使用 statsmodels 回归以获得 p 值
    X_scaled = sm.add_constant(X_scaled)  # 添加常数项
    try:
        model = sm.OLS(y_scaled, X_scaled).fit()
    except Exception:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    a, b = model.params[1], model.params[2]
    p_a, p_b = model.pvalues[1], model.pvalues[2]
    r2 = model.rsquared

    return a, b, r2, p_a, p_b
    
# ======================
# 4. 逐格点回归
# ======================
lat = y.lat.values
lon = y.lon.values

coef_a = np.full((len(lat), len(lon)), np.nan)
coef_b = np.full((len(lat), len(lon)), np.nan)
r2_map = np.full((len(lat), len(lon)), np.nan)
p_a_map = np.full((len(lat), len(lon)), np.nan)
p_b_map = np.full((len(lat), len(lon)), np.nan)

for i in tqdm(range(len(lat)), desc="Latitude Loop"):
    for j in range(len(lon)):
        y_arr  = y[i, j, :].values
        x1_arr = x1[i, j, :].values
        x2_arr = x2[i, j, :].values
        a, b, r2, p_a, p_b = regress_pixel(y_arr, x1_arr, x2_arr)
        coef_a[i, j] = a
        coef_b[i, j] = b
        r2_map[i, j] = r2
        p_a_map[i, j] = p_a
        p_b_map[i, j] = p_b

# ======================
# 5. 保存结果为 xarray
# ======================
ds_out = xr.Dataset(
    {
        "coef_freq": (("lat", "lon"), coef_a),
        "coef_dur":  (("lat", "lon"), coef_b),
        "r2":        (("lat", "lon"), r2_map),
        "p_freq":    (("lat", "lon"), p_a_map),
        "p_dur":     (("lat", "lon"), p_b_map),
    },
    coords={"lat": lat, "lon": lon},
)

save_path = rf"D:\smq\paper2\data\cdhw\cdhw_regression_event_mean_coef_{start_year}_{end_year}.nc"
ds_out.to_netcdf(save_path, engine="h5netcdf")
print(f"✅ 回归完成并已保存 p 值：{save_path}")



### Fig. 1a frequency time series



In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import os
import matplotlib as mpl
from sklearn.linear_model import LinearRegression

mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"

# ======================
# 打开数据
# ======================
fre = xr.open_dataset(r"D:\smq\paper2\data\cdhw\cdhw_event_count_yearly.nc", engine="h5netcdf")["annual_event_count"]
eff = xr.open_dataset(r"D:\smq\paper2\data\modified_mask.nc")["mask_data"]
mask = eff > 1  # 掩膜

# ======================
# 掩膜并计算空间均值和标准差
# ======================
fre_masked = fre.where(mask)
fre_annual = fre_masked.mean(dim=["lat", "lon"])   # 年均

# ======================
# 使用滚动标准差替代RSD
# ======================
rolling_std = fre_annual.rolling(year=15, center=True, min_periods=1).std()
rolling_mean = fre_annual.rolling(year=15, center=True, min_periods=1).mean()
upper_band = rolling_mean + 2 * rolling_std
lower_band = rolling_mean - 2 * rolling_std

# ======================
# 基准期统计 (1920-1950)
# ======================
baseline = fre_annual.sel(year=slice(1920, 1950))
mean_val = baseline.mean().item()
std_val = baseline.std().item()
upper = mean_val + 1 * std_val
lower = mean_val - 1 * std_val

# ======================
# 找到 rolling 曲线第一次超出点
# ======================
rolling_arr = rolling_mean.values
years = rolling_mean.year.values
exceed_idx = np.where((rolling_arr > upper))[0]
if len(exceed_idx) > 0:
    first_exceed_idx = exceed_idx[0]
    exceed_year = years[first_exceed_idx]
    exceed_value = rolling_arr[first_exceed_idx]
else:
    exceed_year, exceed_value = None, None

# ======================
# 绘图
# ======================
fig, ax = plt.subplots(figsize=(6.5, 3.5))

# 阴影
ax.fill_between(
    fre_annual.year.values,
    lower_band.values,
    upper_band.values,
    color=(90/255, 180/255, 175/255),
    alpha=0.25,
    label="±2σ Rolling Std"
)

# 原始序列
fre_annual.plot(
    ax=ax,
    color=(90/255, 180/255, 175/255),
    linewidth=1.5,
    alpha=0.5,
    marker='o',
    markersize=3.5,
    markerfacecolor=(90/255, 180/255, 175/255),
    markeredgewidth=0.2,
    label="Frequency"
)

# 滑动平均
rolling_mean.plot(
    ax=ax,
    color=(90/255, 180/255, 175/255),
    linewidth=3,
    label="15-year Rolling"
)

# ======================
# 两段趋势线
# ======================
x = years
y = rolling_arr

# 1️⃣ 1970年前（1920-1979）
mask1 = (x >= 1920) & (x < 1980)
x1, y1 = x[mask1], y[mask1]
model1 = LinearRegression().fit(x1.reshape(-1, 1), y1)
y1_pred = model1.predict(x1.reshape(-1, 1))
ax.plot(x1, y1_pred, color='black', linestyle="--",alpha=0.65, linewidth=1.8, label="Pre-1980 Trend")

# 2️⃣ 1970年后（1980-2019）
mask2 = (x >= 1980) & (x <= 2019)
x2, y2 = x[mask2], y[mask2]
model2 = LinearRegression().fit(x2.reshape(-1, 1), y2)
y2_pred = model2.predict(x2.reshape(-1, 1))
ax.plot(x2, y2_pred, color='black', linestyle="--", alpha=0.85, linewidth=1.8, label="Post-1980 Trend")


# ======================
# 斜率文字标注（加粗 + 同色）
# ======================
slope1 = model1.coef_[0]
slope2 = model2.coef_[0]

# 文本位置（取线段中点附近）
x1_mid = x1[len(x1)//2]
y1_mid = y1_pred[len(y1)//2]
x2_mid = x2[len(x2)//2]
y2_mid = y2_pred[len(y2)//2]

ax.text(x1_mid, y1_mid + 0.3, f"Slope = {slope1:+.3f}",
        color='black',alpha=0.65, fontsize=16, fontweight="bold", ha="center")

ax.text(x2_mid+5, y2_mid-0.6, f"Slope = {slope2:+.3f}",
        color='black',alpha=0.85, fontsize=16, fontweight="bold", ha="center")

# ======================
# 基准期线与标注
# ======================
ax.axhline(y=mean_val, color="grey", linestyle="-", linewidth=1, alpha=0.8,label="Baseline Mean")#(1920–1950)
ax.axhline(y=upper, color="grey", linestyle="--", linewidth=1,  alpha=0.8,label="Baseline ±2σ")
ax.axhline(y=lower, color="grey", linestyle="--", linewidth=1 ,alpha=0.8)

# 首次超出点 + 垂直线 + 年份标签
if exceed_year is not None:
    point_color = 'grey'  # 与主曲线颜色一致，可改为"grey"等

    # 绘制超出点
    ax.scatter(exceed_year, exceed_value, color=point_color, s=90, zorder=6, alpha=0.9,
               label=f"TCIE")

    y_min, y_max = ax.get_ylim()
    # 计算超出点在y轴范围内的相对比例（用于ymax）
    ymax_ratio = (exceed_value - y_min) / (y_max - y_min)

    # 添加垂直虚线（从底部到超出点）
    ax.axvline(
        x=exceed_year,
        ymin=0,  # 从y轴底部开始
        ymax=ymax_ratio,  # 精确到超出点的y坐标
        color=point_color,
        linestyle="-",
        linewidth=1.5,
        alpha=0.6,
        zorder=5
    )

    # 年份文字（放在线底部右侧）
    ax.text(
        exceed_year + 1,        # 略偏右
        ax.get_ylim()[0] + 0.03, # 靠近底部
        f"{int(exceed_year)}",
        color=point_color,
        fontsize=16,
        va="bottom",
        ha="left",
        fontweight="bold"
    )

# ======================
# 坐标轴与美化
# ======================
ax.set_xlim(1920, 2020)
ax.set_ylim(0, 2.4)
ax.set_yticks(np.arange(0, 2.5, 0.6)) 
ax.set_title("")#Global Annual Mean CDHW Event Frequency
ax.set_ylabel("Frequency (events/year)",fontweight='bold',fontsize=19)
ax.set_xlabel("")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_position(('outward', 5))
ax.spines['bottom'].set_position(('outward', 5))
ax.tick_params(
    axis='both', which='both',
    direction='out',
    bottom=True, top=False,
    left=True, right=False,
    labelbottom=True, labelleft=True,
    length=6, width=1, color='black', labelsize=16
)



# 图例设置
ax.legend(frameon=False, ncol=2, loc='upper left', bbox_to_anchor=(-0.03, 1.2),
          fontsize=16, columnspacing=1.0)

plt.tight_layout()


# 保存
pdf_save_path = r"D:\smq\paper2\result\cdhw_event_frequency.pdf"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
# plt.savefig(pdf_save_path, format='pdf', bbox_inches='tight', dpi=300)
plt.show()
print(f"图形已保存为：{pdf_save_path}")



### Frequency TCIE sensitivity across sigma thresholds



In [ ]:
# ======================
# 多阈值 TCIE 计算
# ======================
sigma_list = [1.0, 1.5, 1.75, 1.96, 2.0, 2.5]

rolling_arr = rolling_mean.values
years = rolling_mean.year.values

print("TCIE estimates under different thresholds:")
print("-------------------------------------------")

for sigma in sigma_list:
    upper_thr = mean_val + sigma * std_val

    exceed_idx = np.where(rolling_arr > upper_thr)[0]

    if len(exceed_idx) > 0:
        tcie_year = int(years[exceed_idx[0]])
        print(f"{sigma:.2f}σ : {tcie_year}")
    else:
        print(f"{sigma:.2f}σ : Not detected")


### Fig. 1b duration time series



In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import os
import matplotlib as mpl
from sklearn.linear_model import LinearRegression

mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"

# ======================
# 打开数据
# ======================
dur = xr.open_dataset(r"D:\smq\paper2\data\cdhw\cdhw_yearly_event_duration.nc", engine="h5netcdf")["annual_event_duration"]
eff = xr.open_dataset(r"D:\\smq\\paper2\\data\\modified_mask.nc")["mask_data"]
mask = eff > 1  # 掩膜

# ======================
# 掩膜并计算空间均值和标准差
# ======================
fre_masked = dur.where(mask)
fre_annual = fre_masked.mean(dim=["lat", "lon"])   # 年均

# ======================
# 使用滚动标准差替代RSD
# ======================
rolling_std = fre_annual.rolling(year=15, center=True, min_periods=1).std()
rolling_mean = fre_annual.rolling(year=15, center=True, min_periods=1).mean()
upper_band = rolling_mean + 2 * rolling_std
lower_band = rolling_mean - 2 * rolling_std

# ======================
# 基准期统计 (1920-1950)
# ======================
baseline = fre_annual.sel(year=slice(1920, 1950))
mean_val = baseline.mean().item()
std_val = baseline.std().item()
upper = mean_val + 2 * std_val
lower = mean_val - 2 * std_val

# ======================
# 找到 rolling 曲线第一次超出 ±2σ 的点
# ======================
rolling_arr = rolling_mean.values
years = rolling_mean.year.values
exceed_idx = np.where((rolling_arr > upper) | (rolling_arr < lower))[0]

if len(exceed_idx) > 0:
    first_exceed_idx = exceed_idx[0]
    exceed_year = years[first_exceed_idx]
    exceed_value = rolling_arr[first_exceed_idx]
else:
    exceed_year, exceed_value = None, None

# ======================
# 绘图
# ======================
fig, ax = plt.subplots(figsize=(6.5, 3.5))

# 阴影：基于滚动标准差的置信区间
ax.fill_between(
    fre_annual.year.values,
    lower_band.values,
    upper_band.values,
    color=(141/255, 137/255, 220/255),
    alpha=0.25,
    label="±2σ Rolling Std"
)

# 原始年序列（加节点）
fre_annual.plot(
    ax=ax,
    color=(141/255, 137/255, 220/255),
    linewidth=1.5,
    alpha=0.5,
    marker='o',
    markersize=3,
    markerfacecolor=(141/255, 137/255, 220/255),
    markeredgewidth=0.2,
    label="Duration"
)

# 滑动平均
rolling_mean.plot(
    ax=ax,
    color=(141/255, 137/255, 220/255),
    linewidth=3,
    label="15-year Rolling"
)

# ======================
# 两段趋势线
# ======================
x = years
y = rolling_arr

# 1️⃣ 1970年前（1920-1979）
mask1 = (x >= 1920) & (x < 1980)
x1, y1 = x[mask1], y[mask1]
model1 = LinearRegression().fit(x1.reshape(-1, 1), y1)
y1_pred = model1.predict(x1.reshape(-1, 1))
ax.plot(x1, y1_pred, color='black', linestyle="--",alpha=0.65, linewidth=2.3, label="Pre-1980 trend")

# 2️⃣ 1970年后（1980-2019）
mask2 = (x >= 1980) & (x <= 2019)
x2, y2 = x[mask2], y[mask2]
model2 = LinearRegression().fit(x2.reshape(-1, 1), y2)
y2_pred = model2.predict(x2.reshape(-1, 1))
ax.plot(x2, y2_pred, color='black', linestyle="--", alpha=0.85, linewidth=2.3, label="Post-1980 trend")

# ======================
# 斜率文字标注（加粗 + 同色）
# ======================
slope1 = model1.coef_[0]
slope2 = model2.coef_[0]

# 文本位置（取线段中点附近）
x1_mid = x1[len(x1)//2]
y1_mid = y1_pred[len(y1)//2]
x2_mid = x2[len(x2)//2]
y2_mid = y2_pred[len(y2)//2]

ax.text(x1_mid, y1_mid - 0.8, f"Slope = {slope1:+.3f}",
        color='black',alpha=0.65, fontsize=16, fontweight="bold", ha="center")

ax.text(x2_mid, y2_mid -1.05, f"Slope = {slope2:+.3f}",
        color='black',alpha=0.85, fontsize=16, fontweight="bold", ha="center")

# ======================
# 基准期线与标注
# ======================
ax.axhline(y=mean_val, color="grey", linestyle="-", linewidth=1, alpha=0.8, label="Baseline Mean")
ax.axhline(y=upper, color="grey", linestyle="--", linewidth=1, alpha=0.8, label="Baseline ±2σ")
ax.axhline(y=lower, color="grey", linestyle="--", linewidth=1, alpha=0.8)

# 首次超出点 + 垂直线 + 年份标签
if exceed_year is not None:
    point_color = 'grey'  # 与主曲线颜色一致，可改为"grey"等

    # 绘制超出点
    ax.scatter(exceed_year, exceed_value, color=point_color, s=90, zorder=6, alpha=0.9,
               label=f"First Exceedance")

    # 添加垂直虚线（从底部到超出点）
    y_min, y_max = ax.get_ylim()
    # 计算超出点在y轴范围内的相对比例（用于ymax）
    ymax_ratio = (exceed_value - y_min) / (y_max - y_min)

    # 添加垂直虚线（从底部到超出点）
    ax.axvline(
        x=exceed_year,
        ymin=0,  # 从y轴底部开始
        ymax=ymax_ratio,  # 精确到超出点的y坐标
        color=point_color,
        linestyle="-",
        linewidth=1.5,
        alpha=0.6,
        zorder=5
    )

    # 年份文字（放在线底部右侧）
    ax.text(
        exceed_year + 1,        # 略偏右
        ax.get_ylim()[0] + 0.03, # 靠近底部
        f"{int(exceed_year)}",
        color=point_color,
        fontsize=16,
        va="bottom",
        ha="left",
        fontweight="bold"
    )
# ======================
# 坐标轴与美化
# ======================
ax.set_xlim(1920, 2020)
ax.set_ylim(0, 4)
ax.set_yticks(np.arange(0, 4.1, 1)) 
ax.set_title("")#Global Annual Mean CDHW Event Duration
ax.set_ylabel("Duration (days/event)",fontweight='bold',fontsize=19)
ax.set_xlabel("")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_position(('outward', 5))
ax.spines['bottom'].set_position(('outward', 5))
ax.tick_params(
    axis='both', which='both',
    direction='out',
    bottom=True, top=False,
    left=True, right=False,
    labelbottom=True, labelleft=True,
    length=6, width=1, color='black', labelsize=16
)

# 图例设置为两列
ax.legend(frameon=False, ncol=2, loc='upper left', bbox_to_anchor=(-0.03, 1.2),
          fontsize=16, columnspacing=1.0)

plt.tight_layout()

# 保存为 PDF
pdf_save_path = r"D:\smq\paper2\result\cdhw_event_duration.pdf"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
# plt.savefig(pdf_save_path, format='pdf', bbox_inches='tight', dpi=300)
plt.show()
print(f"图形已保存为：{pdf_save_path}")



### Duration TCIE sensitivity across sigma thresholds



In [ ]:
# ======================
# 多阈值 TCIE 计算
# ======================
sigma_list = [1.0, 1.5, 1.75, 1.96, 2.0, 2.5]

rolling_arr = rolling_mean.values
years = rolling_mean.year.values

print("TCIE estimates under different thresholds:")
print("-------------------------------------------")

for sigma in sigma_list:
    upper_thr = mean_val + sigma * std_val

    exceed_idx = np.where(rolling_arr > upper_thr)[0]

    if len(exceed_idx) > 0:
        tcie_year = int(years[exceed_idx[0]])
        print(f"{sigma:.2f}σ : {tcie_year}")
    else:
        print(f"{sigma:.2f}σ : Not detected")


### Fig. 1c intensity time series



In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import os
import matplotlib as mpl
from sklearn.linear_model import LinearRegression
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"
# ======================
# 打开数据
# ======================
sev = xr.open_dataset(r"D:\smq\paper2\data\cdhw\cdhw_meanevent_yearly_severity.nc", engine="h5netcdf")["annual_severity"]
eff = xr.open_dataset(r"D:\smq\paper2\data\modified_mask.nc")["mask_data"]
mask = eff > 1  # 掩膜

# ======================
# 掩膜并计算空间均值和标准差
# ======================
fre_masked = sev.where(mask)
fre_annual = fre_masked.mean(dim=["lat", "lon"])   # 年均

# ======================
# 使用滚动标准差替代RSD
# ======================
# 计算15年滚动标准差（与滚动平均窗口一致）
rolling_std = fre_annual.rolling(year=15, center=True, min_periods=1).std()

# 计算滚动均值
rolling_mean = fre_annual.rolling(year=15, center=True, min_periods=1).mean()

# 基于滚动标准差的上下边界（使用较小的系数获得更窄的阴影）
upper_band = rolling_mean + 2 * rolling_std
lower_band = rolling_mean - 2 * rolling_std

# ======================
# 基准期统计 (1920-1950)
# ======================
baseline = fre_annual.sel(year=slice(1920, 1950))
mean_val = baseline.mean().item()
std_val = baseline.std().item()
upper = mean_val + 2 * std_val
lower = mean_val - 2 * std_val

# ======================
# 找到 rolling 曲线第一次超出 ±1.75σ 的点
# ======================
rolling_arr = rolling_mean.values
years = rolling_mean.year.values
exceed_idx = np.where((rolling_arr > upper) | (rolling_arr < lower))[0]

if len(exceed_idx) > 0:
    first_exceed_idx = exceed_idx[0]
    exceed_year = years[first_exceed_idx]
    exceed_value = rolling_arr[first_exceed_idx]
else:
    exceed_year, exceed_value = None, None

# ======================
# 绘图
# ======================
fig, ax = plt.subplots(figsize=(6.5, 3.5))

# 阴影：基于滚动标准差的置信区间
ax.fill_between(
    fre_annual.year.values,
    lower_band.values,
    upper_band.values,
    color= (227/255, 99/255, 129/255),
    alpha=0.25,
    label="±2σ Rolling Std"
)

# 原始年序列（加节点）
fre_annual.plot(
    ax=ax,
    color= (227/255, 99/255, 129/255),
    linewidth=1.5,
    alpha=0.5,
    marker='o',          # 添加节点
    markersize=3,      # 节点大小
    markerfacecolor= (227/255, 99/255, 129/255),  # 与线条同色
    markeredgewidth=0.2,
    label="Intensity"
)

# 滑动平均
rolling_mean.plot(
    ax=ax,
    color= (227/255, 99/255, 129/255),
    linewidth=2.5,
    label="15-year Rolling"
)

# ======================
# 两段趋势线
# ======================
x = years
y = rolling_arr

# 1️⃣ 1970年前（1920-1979）
mask1 = (x >= 1920) & (x < 1980)
x1, y1 = x[mask1], y[mask1]
model1 = LinearRegression().fit(x1.reshape(-1, 1), y1)
y1_pred = model1.predict(x1.reshape(-1, 1))
ax.plot(x1, y1_pred, color='black', linestyle="--",alpha=0.65, linewidth=2.3, label="Pre-1980 trend")

# 2️⃣ 1970年后（1980-2019）
mask2 = (x >= 1980) & (x <= 2019)
x2, y2 = x[mask2], y[mask2]
model2 = LinearRegression().fit(x2.reshape(-1, 1), y2)
y2_pred = model2.predict(x2.reshape(-1, 1))
ax.plot(x2, y2_pred, color='black', linestyle="--", alpha=0.85, linewidth=2.3, label="Post-1980 trend")

# ======================
# 斜率文字标注（加粗 + 同色）
# ======================
slope1 = model1.coef_[0]
slope2 = model2.coef_[0]

# 文本位置（取线段中点附近）
x1_mid = x1[len(x1)//2]
y1_mid = y1_pred[len(y1)//2]
x2_mid = x2[len(x2)//2]
y2_mid = y2_pred[len(y2)//2]

ax.text(x1_mid, y1_mid-0.4 , f"Slope = {slope1:+.3f}",
        color='black',alpha=0.65, fontsize=16, fontweight="bold", ha="center")

ax.text(x2_mid, y2_mid-0.65 , f"Slope = {slope2:+.3f}",
        color='black',alpha=0.85, fontsize=16, fontweight="bold", ha="center")

# ======================
# 基准期线与标注
# ======================
ax.axhline(y=mean_val, color="grey", linestyle="-", linewidth=1, alpha=0.8, label="Baseline Mean")
ax.axhline(y=upper, color="grey", linestyle="--", linewidth=1, alpha=0.8, label="Baseline ±2σ")
ax.axhline(y=lower, color="grey", linestyle="--", linewidth=1, alpha=0.8)

if exceed_year is not None:
    point_color = 'grey'  # 与主曲线颜色一致，可改为"grey"等

    # 绘制超出点
    ax.scatter(exceed_year, exceed_value, color=point_color, s=90, zorder=6, alpha=0.9,
               label=f"First Exceedance")

    # 添加垂直虚线（从底部到超出点）
    y_min, y_max = ax.get_ylim()
    # 计算超出点在y轴范围内的相对比例（用于ymax）
    ymax_ratio = (exceed_value - y_min) / (y_max - y_min)

    # 添加垂直虚线（从底部到超出点）
    ax.axvline(
        x=exceed_year,
        ymin=0,  # 从y轴底部开始
        ymax=ymax_ratio,  # 精确到超出点的y坐标
        color=point_color,
        linestyle="-",
        linewidth=1.5,
        alpha=0.6,
        zorder=5
    )

    # 年份文字（放在线底部右侧）
    ax.text(
        exceed_year + 1,        # 略偏右
        ax.get_ylim()[0] + 0.03, # 靠近底部
        f"{int(exceed_year)}",
        color=point_color,
        fontsize=16,
        va="bottom",
        ha="left",
        fontweight="bold"
    )
# ======================
# 轴设置与美化
# ======================
ax.set_xlim(1920, 2020)
ax.set_ylim(0, 2.5)
ax.set_yticks(np.arange(0, 2.6, 0.5))
ax.set_title("")#Global Annual Mean CDHW Event Severity
ax.set_ylabel("Intensity(unit/year)",fontweight='bold',fontsize=19)
ax.set_xlabel("")

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_position(('outward', 5))
ax.spines['bottom'].set_position(('outward', 5))
ax.tick_params(
    axis='both', which='both',
    direction='out',
    bottom=True, top=False,
    left=True, right=False,
    labelbottom=True, labelleft=True,
    length=6, width=1, color='black', labelsize=16
)

# ======================
# 图例设置为两列
# ======================
ax.legend(frameon=False, ncol=2, loc='upper left', 
          bbox_to_anchor=(-0.03, 1.2), fontsize=16, columnspacing=1.0)

plt.tight_layout()

# ======================
# 保存为 PDF
# ======================
pdf_save_path = r"D:\smq\paper2\result\cdhw_daily_severity.pdf"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(pdf_save_path, format='pdf', bbox_inches='tight')
plt.show()
print(f"图形已保存为：{pdf_save_path}")



### Intensity TCIE sensitivity across sigma thresholds



In [ ]:
# ======================
# 多阈值 TCIE 计算
# ======================
sigma_list = [1.0, 1.5, 1.75, 1.96, 2.0, 2.5]

rolling_arr = rolling_mean.values
years = rolling_mean.year.values

print("TCIE estimates under different thresholds:")
print("-------------------------------------------")

for sigma in sigma_list:
    upper_thr = mean_val + sigma * std_val

    exceed_idx = np.where(rolling_arr > upper_thr)[0]

    if len(exceed_idx) > 0:
        tcie_year = int(years[exceed_idx[0]])
        print(f"{sigma:.2f}σ : {tcie_year}")
    else:
        print(f"{sigma:.2f}σ : Not detected")


### Fig. 1d frequency trend map



In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"
# ======================
# 文件路径
# ======================
fp = r"D:\smq\paper2\data\cdhw\cdhw_event_count_yearly.nc"
eff = xr.open_dataset(r"D:\smq\paper2\data\modified_mask.nc")["mask_data"]

start_year = 1920
end_year = 2019

# ======================
# 读取数据
# ======================
ds = xr.open_dataset(fp, engine="h5netcdf")
cdhw_freq_yearly = ds["annual_event_count"].sel(year=slice(start_year, end_year))
cdhw_freq_mean = cdhw_freq_yearly.mean(dim="year")
cdhw_freq_mean = cdhw_freq_mean.where(cdhw_freq_mean != 0)

# # 转换为百分比
# cdhw_freq_mean_percent = cdhw_freq_mean * 100


cdhw_freq_mean = cdhw_freq_mean.where(eff > 1)

# ======================
# 绘制空间分布
# ======================
# 创建带cartopy投影的图
fig = plt.figure(figsize=(6.5, 3.5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())
# 绘制数据
im = cdhw_freq_mean.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="GnBu",
    vmin=0,
    vmax=1.5,  # 因为数据乘以100了，所以最大值也相应调整
    add_labels=False,
    cbar_kwargs={
        "label": f"Frequency of CDHW ({start_year}–{end_year}) (events/year) ",
        "orientation": "horizontal",
        "pad": 0.05,
        "shrink": 0.8,
        "aspect": 30
    }
)

# 修改颜色条字体
cbar = im.colorbar
cbar.ax.tick_params(labelsize=16)  # 色带刻度字体大小
cbar.set_label(
    f"Frequency ({start_year}–{end_year}) (events/year) ",
    fontsize=19,
    fontweight='bold'
)

# 添加国界
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='black', alpha=0.8)
# 添加海岸线
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, alpha=0.8)

# 去掉图框
plt.box(False)
plt.tight_layout()

pdf_save_path = r"D:\smq\paper2\result\cdhw_event_frequency_Spatial.pdf"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(
    pdf_save_path,
    format='pdf',  # 指定输出 EPS 格式
    bbox_inches='tight',  # 紧凑布局
    dpi=300,  # 虽然 EPS 是矢量，但某些元素可能受 DPI 影响
)

plt.show()

### Fig. 1e duration trend map



In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np


mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"

# ======================
# 文件路径
# ======================
fp = r"D:\smq\paper2\data\cdhw\cdhw_yearly_event_duration.nc"
eff = xr.open_dataset(r"D:\smq\paper2\data\modified_mask.nc")["mask_data"]

start_year = 1920
end_year = 2019

# ======================
# 读取数据
# ======================
ds = xr.open_dataset(fp, engine="h5netcdf")
cdhw_dur_yearly = ds["annual_event_duration"].sel(year=slice(start_year, end_year))
cdhw_dur_mean = cdhw_dur_yearly.mean(dim="year")
cdhw_dur_mean = cdhw_dur_mean.where(cdhw_dur_mean != 0)


cdhw_dur_mean = cdhw_dur_mean.where(eff > 1)

# ======================
# 绘制空间分布
# ======================
# 创建带cartopy投影的图
fig = plt.figure(figsize=(6.5, 3.5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())
# 绘制数据
im = cdhw_dur_mean.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="Purples",
    vmin=0,
    vmax=3,  
    add_labels=False,
    cbar_kwargs={
        "label": f"Duration ({start_year}–{end_year}) (days/event)",
        "orientation": "horizontal",
        "pad": 0.05,
        "shrink": 0.8,
        "aspect": 30
    }
)
# 修改颜色条字体
cbar = im.colorbar
cbar.ax.tick_params(labelsize=16)  # 色带刻度字体大小
cbar.set_label(
    f"Duration ({start_year}–{end_year}) (days/event)",
    fontsize=19,
    fontweight='bold'
)

# 添加国界
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='black', alpha=0.8)
# 添加海岸线
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, alpha=0.8)

# 去掉图框
plt.box(False)
plt.tight_layout()

pdf_save_path = r"D:\smq\paper2\result\cdhw_event_duration_Spatial.pdf"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(
    pdf_save_path,
    format='pdf',  # 指定输出 EPS 格式
    bbox_inches='tight',  # 紧凑布局
    dpi=300,  # 虽然 EPS 是矢量，但某些元素可能受 DPI 影响
)


plt.show()

### Fig. 1f intensity trend map



In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"
# ======================
# 文件路径
# ======================
start_year = 1920
end_year = 2019

sev = xr.open_dataset(r"D:\smq\paper2\data\cdhw\cdhw_meanevent_yearly_severity.nc", engine="h5netcdf")["annual_severity"].sel(year=slice(start_year, end_year))
eff = xr.open_dataset(r"D:\smq\paper2\data\modified_mask.nc")["mask_data"]

sev_mean = sev.mean(dim="year")
sev_mean = sev_mean.where(sev_mean != 0)
sev_mean = sev_mean.where(eff >1)
# ======================
# 绘制空间分布
# ======================
# 创建带cartopy投影的图
fig = plt.figure(figsize=(6.5, 3.5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())
# 绘制数据
im = sev_mean.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdPu",
    vmin=0,
    vmax=2, 
    add_labels=False,
    cbar_kwargs={
        "label": f"Severity ({start_year}–{end_year})",
        "orientation": "horizontal",
        "pad": 0.05,
        "shrink": 0.8,
        "aspect": 30
    }
)

# 修改颜色条字体
cbar = im.colorbar
cbar.ax.tick_params(labelsize=16)  # 色带刻度字体大小
cbar.set_label(
    f"Intensity ({start_year}–{end_year}) (unit/year)",
    fontsize=19,
    fontweight='bold'
)

# 添加国界
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='black', alpha=0.8)
# 添加海岸线
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, alpha=0.8)

# 去掉图框
plt.box(False)
plt.tight_layout()
pdf_save_path = r"D:\smq\paper2\result\cdhw_daily_severity_Spatial.pdf"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(pdf_save_path, format='pdf', bbox_inches='tight')
plt.show()


### Frequency relative change map



In [ ]:
import matplotlib.colors as mcolors
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"
# ======================
# 文件路径
# ======================
fp = r"D:\smq\paper2\data\cdhw\cdhw_event_count_yearly.nc"
eff = xr.open_dataset(r"D:\smq\paper2\data\modified_mask.nc")["mask_data"]
ds = xr.open_dataset(fp, engine="h5netcdf")
cdhw_freq_yearly = ds["annual_event_count"]
# ======================
# 分别计算两个时期的均值
# ======================
base = cdhw_freq_yearly.sel(year=slice(1920, 1993)).mean(dim="year") 
recent = cdhw_freq_yearly.sel(year=slice(1993, 2019)).mean(dim="year") 

# 相对变化 (%) = (近期 - 基准) / 基准 * 100
# relative_change = ((recent - base) / base) * 100
relative_change = recent - base

# 掩膜
relative_change = relative_change.where(eff > 1)

# ======================
# 自定义色阶和颜色
# ======================
# 定义色阶断点
# levels = [-100, -50, 0, 50, 100, 200, 300, 400, 500]
levels = [-100, -50,0, 100, 200, 300, 400, 500]
levels=[i/200 for i in levels]

# 定义对应的颜色（可以使用颜色名称、十六进制代码或RGB元组）
# colors = ['lightgrey',(230/255, 246/255, 225/255),(186/255, 228/255, 199/255), (141/255, 211/255, 190/255), (89/255, 190/255, 204/255),(13/255, 109/255, 174/255),(8/255, 64/255, 129/255)]

# 定义第一个颜色（比如红色表示减少）
first_color ='lightgrey'
second_color= (230/255, 230/255, 230/255)
# 从GnBu色带中提取剩余颜色
cmap = plt.cm.GnBu
remaining_colors = [cmap(i) for i in np.linspace(0, 1, len(levels)-3)]  # 注意这里减2

# 组合颜色：第一个自定义颜色 + 剩余色带颜色
colors = [first_color] +[second_color] + remaining_colors

# 创建自定义colormap
cmap_custom = mcolors.ListedColormap(colors)
norm = mcolors.BoundaryNorm(levels, cmap_custom.N)

# ======================
# 绘制空间分布
# ======================
fig = plt.figure(figsize=(6.8, 3.8))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

# 绘制数据
im = relative_change.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap=cmap_custom,
    norm=norm,
    levels=levels,
    add_colorbar=False,  # 先不添加默认colorbar，后面自定义
    add_labels=False,
)

# 添加自定义colorbar
cbar = plt.colorbar(
    im, 
    ax=ax,
    orientation="horizontal",
    pad=0.05,
    shrink=0.8,
    aspect=30,
    boundaries=levels,
    ticks=levels,
    extend='both'  # 对于超出范围的值显示箭头
)
cbar.set_label(f"Change in Frequency (events/year)\n(1920–1993 vs 1993–2019)", fontsize=19,fontweight='bold')
cbar.ax.tick_params(labelsize=16)

# 添加国界
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='black', alpha=0.8)
# 添加海岸线
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, alpha=0.8)

# 去掉图框
plt.box(False)
plt.tight_layout()

# pdf_save_path = r"D:\smq\paper2\result\cdhw_event_frequency_relative_change.pdf"
pdf_save_path = r"D:\smq\paper2\result\cdhw_event_frequency_change.pdf"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(pdf_save_path,format='pdf', bbox_inches='tight', dpi=300)
plt.show()

### Duration relative change map



In [ ]:
import matplotlib.colors as mcolors
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"

eff = xr.open_dataset(r"D:\smq\paper2\data\modified_mask.nc")["mask_data"]
ds = xr.open_dataset(r"D:\smq\paper2\data\cdhw\cdhw_yearly_event_duration.nc", engine="h5netcdf")
cdhw_dur_yearly = ds["annual_event_duration"]
# ======================
# 分别计算两个时期的均值
# ======================
base = cdhw_dur_yearly.sel(year=slice(1920, 1996)).mean(dim="year") 
recent = cdhw_dur_yearly.sel(year=slice(1996, 2019)).mean(dim="year") 

# 相对变化 (%) = (近期 - 基准) / 基准 * 100
# relative_change = ((recent - base) / base) * 100
relative_change = recent - base


# 掩膜
relative_change = relative_change.where(eff > 1)

# # ======================
# # 自定义色阶和颜色：0之前一个色带，0之后另一个色带
# # ======================
# # 定义色阶断点
# levels = [-100, -80, -60, -40, -20, 0, 20, 40, 60, 80, 100]

# # 计算0之前和之后的区间数量
# n_negative = len([x for x in levels if x < 0])  # 负值区间数量：5个
# n_positive = len([x for x in levels if x > 0])  # 正值区间数量：5个

# # 0之前使用Purples色带（负值）
# cmap_negative = plt.cm.Blues_r
# negative_colors = [cmap_negative(i) for i in np.linspace(0, 0.7, n_negative)]  # 从0.3开始避免太浅

# # 0之后使用GnBu色带（正值）
# cmap_positive = plt.cm.Purples
# positive_colors = [cmap_positive(i) for i in np.linspace(0.3, 1, n_positive)]  # 从0.3开始避免太浅

# # 组合颜色：负值颜色 + 正值颜色
# colors = negative_colors + positive_colors

# # 创建自定义colormap
# cmap_custom = mcolors.ListedColormap(colors)
# norm = mcolors.BoundaryNorm(levels, cmap_custom.N)


# # ======================
# # 自定义色阶和颜色：0之前灰色，0之后另一个色带
# # ======================
# 定义色阶断点
# levels = [-100, -50, 0, 50, 100, 200, 300, 400, 500]
levels = [-100, -50, 0, 100, 200, 300, 400, 500]
levels=[i/100 for i in levels]

first_color ='lightgrey'
second_color= (230/255, 230/255, 230/255)
# 从GnBu色带中提取剩余颜色
cmap = plt.cm.Purples
remaining_colors = [cmap(i) for i in np.linspace(0.3, 1, len(levels)-3)]  # 注意这里减2

# 组合颜色：第一个自定义颜色 + 剩余色带颜色
colors = [first_color] +[second_color] + remaining_colors

# 创建自定义colormap
cmap_custom = mcolors.ListedColormap(colors)
norm = mcolors.BoundaryNorm(levels, cmap_custom.N)


# ======================
# 绘制空间分布
# ======================
fig = plt.figure(figsize=(6.8, 3.8))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

# 绘制数据
im = relative_change.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap=cmap_custom,
    norm=norm,
    levels=levels,
    add_colorbar=False,
    add_labels=False,
)

# 添加自定义colorbar
cbar = plt.colorbar(
    im, 
    ax=ax,
    orientation="horizontal",
    pad=0.05,
    shrink=0.8,
    aspect=30,
    boundaries=levels,
    ticks=levels,
    extend='both'
)
# cbar.set_label(f"Relative Change in Duration (%)\n(1920–1996 vs 1996–2019)", fontsize=16,fontweight='bold')  
cbar.set_label(f"Change in Duration (days/year)\n(1920–1996 vs 1996–2019)", fontsize=19,fontweight='bold')  
cbar.ax.tick_params(labelsize=16)

# 添加国界
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='black', alpha=0.8)
# 添加海岸线
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, alpha=0.8)

# 去掉图框
plt.box(False)
plt.tight_layout()

# pdf_save_path = r"D:\smq\paper2\result\cdhw_event_duration_relative_change.pdf"
pdf_save_path = r"D:\smq\paper2\result\cdhw_event_duration_change.pdf"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(pdf_save_path,format='pdf', bbox_inches='tight', dpi=300)
plt.show()

### Intensity relative change map



In [ ]:
import matplotlib.colors as mcolors
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"

fp =r"D:\smq\paper2\data\cdhw\cdhw_meanevent_yearly_severity.nc"
eff = xr.open_dataset(r"D:\smq\paper2\data\modified_mask.nc")["mask_data"]
sev = xr.open_dataset(fp, engine="h5netcdf")["annual_severity"]

# ======================
# 分别计算两个时期的均值
# ======================
base = sev.sel(year=slice(1920, 1997)).mean(dim="year") 
recent = sev.sel(year=slice(1997, 2020)).mean(dim="year") 

# 相对变化 (%) = (近期 - 基准) / 基准 * 100
# relative_change = ((recent - base) / base) * 100
relative_change = (recent - base) 

# 掩膜
relative_change = relative_change.where(eff > 1)
# # ======================
# # 自定义色阶和颜色：0之前灰色，0之后另一个色带
# # ======================
# 定义色阶断点
# levels = [-100, -50, 0, 50, 100, 200, 300, 400, 500]
levels = [-100, -50, 0, 50, 100, 200, 300, 400]
levels=[i/100 for i in levels]

first_color ='lightgrey'
second_color= (230/255, 230/255, 230/255)
# 从GnBu色带中提取剩余颜色
cmap = plt.cm.RdPu
remaining_colors = [cmap(i) for i in np.linspace(0.1, 0.9, len(levels)-3)]  # 注意这里减2

# 组合颜色：第一个自定义颜色 + 剩余色带颜色
colors = [first_color] +[second_color] + remaining_colors

# 创建自定义colormap
cmap_custom = mcolors.ListedColormap(colors)
norm = mcolors.BoundaryNorm(levels, cmap_custom.N)

# ======================
# 绘制空间分布
# ======================
fig = plt.figure(figsize=(6.8, 3.8))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

# 绘制数据
im = relative_change.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap=cmap_custom,
    norm=norm,
    levels=levels,
    add_colorbar=False,  # 先不添加默认colorbar，后面自定义
    add_labels=False,
)

# 添加自定义colorbar
cbar = plt.colorbar(
    im, 
    ax=ax,
    orientation="horizontal",
    pad=0.05,
    shrink=0.8,
    aspect=30,
    boundaries=levels,
    ticks=levels,
    extend='both'  # 对于超出范围的值显示箭头
)
# cbar.set_label(f"Relative Change in Intensity (%)\n(1920–1997 vs 1997–2019)", fontsize=16, fontweight='bold')
cbar.set_label(f"Change in Intensity (unit/year)\n(1920–1997 vs 1997–2019)", fontsize=19, fontweight='bold')
cbar.ax.tick_params(labelsize=16)

# 添加国界
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='black', alpha=0.8)
# 添加海岸线
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, alpha=0.8)

# 去掉图框
plt.box(False)
plt.tight_layout()
# pdf_save_path = r"D:\smq\paper2\result\cdhw_event_severity_relative_change.pdf"
pdf_save_path = r"D:\smq\paper2\result\cdhw_event_severity_change.pdf"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(pdf_save_path, format='pdf', bbox_inches='tight')
plt.show()


### Pre/post-1980 CDHW coefficient comparison map



In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib as mpl
import os

# ======================
# 1. 文件路径
# ======================
fp_early = r"D:\smq\paper2\data\cdhw\cdhw_regression_event_mean_coef_1920_1980.nc"
fp_late  = r"D:\smq\paper2\data\cdhw\cdhw_regression_event_mean_coef_1980_2019.nc"
mask_fp  = r"D:\smq\paper2\data\modified_mask.nc"

# ======================
# 2. 数据读取与显著掩膜
# ======================
mask_eff = xr.open_dataset(mask_fp)["mask_data"]
mask = mask_eff > 1  # True 表示保留区域

def load_and_process(fp):
    ds = xr.open_dataset(fp, engine='h5netcdf')
    RC_dur = ds["coef_dur"].where(mask)
    RC_hw  = ds["coef_freq"].where(mask)
    p_dur  = ds["p_dur"].where(mask)
    p_hw   = ds["p_freq"].where(mask)
    dominance = (RC_hw - RC_dur)
    sig = ((p_dur < 0.05) | (p_hw < 0.05))
    return dominance.where(sig)

dom_early = load_and_process(fp_early)
dom_late  = load_and_process(fp_late)

# ======================
# 3. 绘图设置
# ======================
fig, axes = plt.subplots(
    1, 2, figsize=(10, 4),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

titles = ["1920–1980", "1980–2019"]
data_list = [dom_early, dom_late]

# 色带配置
cmap = mpl.colors.LinearSegmentedColormap.from_list(
    'BlueRed', [(0, '#2166AC'), (0.5, 'white'), (1, '#FF931E')]
)
norm = mpl.colors.TwoSlopeNorm(vcenter=0, vmin=-1, vmax=1)

# ======================
# 4. 绘制地图
# ======================
for ax, data, title in zip(axes, data_list, titles):
    im = data.plot(
        ax=ax, transform=ccrs.PlateCarree(),
        cmap=cmap, norm=norm, add_colorbar=False
    )
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, color='gray')
    ax.add_feature(cfeature.BORDERS, linewidth=0.2, color='gray')
    ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

    # 隐藏边框
    for spine in ax.spines.values():
        spine.set_visible(False)

    # 时间标题
    ax.text(
        0.1, 1.1, title,
        transform=ax.transAxes,
        ha='center', va='top',
        fontsize=13,
        fontweight='bold'
    )
    ax.set_title("")

# ======================
# 5. 色带设置（使用自带标签）
# ======================
cbar = plt.colorbar(
    im, ax=axes, orientation='horizontal',
    pad=0.02,  # 调整色带与子图的距离
    fraction=0.035, 
    shrink=0.6
)

# 设置色带中心标签（替代原fig.text的"Normalized Contribution"）
cbar.set_label(
    "Normalized Contribution",
    fontsize=15,
    fontweight='bold',
    labelpad=10  # 标签与色带的距离
)

# 设置色带刻度
cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
cbar.ax.tick_params(labelsize=13, width=0.8)
cbar.outline.set_visible(False)

# 在色带两端添加"Dur"和"Freq"（基于色带的轴坐标）
# 获取色带轴的坐标范围（0到1，对应色带的左到右）
cbar_ax = cbar.ax
cbar_ax.text(
    -0.1, 0.5,  # 色带左侧（0位置），下方1.2个单位
    "Dur",
    ha='center',
    va='center',
    fontsize=13,
    fontweight='bold',
    color='#2166AC',
    transform=cbar_ax.transAxes  # 使用色带轴的坐标系统
)
cbar_ax.text(
    1.1, 0.5,  # 色带右侧（1位置），下方1.2个单位
    "Freq",
    ha='center',
    va='center',
    fontsize=13,
    fontweight='bold',
    color='#FF931E',
    transform=cbar_ax.transAxes
)

# ======================
# 保存与显示
# ======================
pdf_save_path = r"D:\smq\paper2\result\cdhw_Fre_Dur.pdf"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(pdf_save_path, format='pdf', bbox_inches='tight')  # 启用保存

plt.subplots_adjust(wspace=0.02, bottom=0.2)
plt.show()
print(f"图形已保存为：{pdf_save_path}")

### Representative CDHW event time-series example



In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import glob
import matplotlib.dates as mdates  # 用于日期格式化
from matplotlib.lines import Line2D  # 用于手动创建CDHW图例项
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
    # 2. 字体设置（Nature 推荐使用无衬线字体，如 Arial）
plt.rcParams["font.family"] = "Arial"
# =======================
# 1. 数据读取函数
# =======================
def load_data(lat_sel, lon_sel, start_date, end_date):

    spei_daily = xr.open_dataset(r"F:\smq\paper2\spei\SPEI_daily1.nc", engine="h5netcdf")['spei'].sel(time=slice(start_date, end_date)).sel(lat=lat_sel, lon=lon_sel, method="nearest")   
    tasmax = xr.open_dataset(r"F:\smq\paper2\data\TMax_daily.nc")['TMax'].sel(time=slice(start_date, end_date)).sel(lat=lat_sel, lon=lon_sel, method="nearest").load()
    tas95 = xr.open_dataset(r"D:\smq\paper2\data\tas95_1981_2010.nc")['tas95'].sel(lat=lat_sel, lon=lon_sel, method="nearest").load()
    cdhw_mask = xr.open_dataset(r"F:\smq\paper2\data\masks_cdhw_hw_di.nc")['cdhw_mask'].sel(time=slice(start_date, end_date)).sel(lat=lat_sel, lon=lon_sel, method="nearest").load()
    return {
        'spei': spei_daily,
        'tasmax': tasmax,
        'tas95': tas95,
        'cdhw_mask': cdhw_mask
    }

# =======================
# 2. 从掩膜构造事件（替代 detect_cdhw）
# =======================
def extract_events(time, mask):
    mask = mask.values.astype(bool)
    events = []
    start = None
    for i in range(len(mask)):
        if mask[i] and start is None:
            start = i
        elif not mask[i] and start is not None:
            events.append((time[start], time[i-1]))
            start = None
    if start is not None:
        events.append((time[start], time[-1]))
    return events

# =======================
# 3. 转DataFrame
# =======================
def build_dataframe(data):
    df = pd.DataFrame({
        'time': pd.to_datetime(data['spei'].time.values),
        'spei': data['spei'].values,   
        'temperature': data['tasmax'].values,
        'is_cdhw': data['cdhw_mask'].values.astype(bool)
    })
    df['year'] = df['time'].dt.year 
    df['month'] = df['time'].dt.month
    return df


# =======================
# 4. 可视化（修复框线问题）
# =======================
def plot_timeseries(df, events, tas95, data, lat, lon, start_date, end_date):
    fig, ax1 = plt.subplots(figsize=(3.5, 2.5))
    
    # ====== Temperature ======
    color_temp = (227/255, 81/255, 40/255)
    ax1.plot(df['time'], df['temperature'], color=color_temp, linewidth=2)
    tas95_value = tas95.values - 273.15
    ax1.axhline(tas95_value, color=color_temp, linestyle='--', linewidth=1.5)
    ax1.set_ylabel('Tmax (°C)', color=color_temp, fontsize=15, weight='bold')
    ax1.tick_params(axis='y', labelcolor=color_temp, color=color_temp, width=1.3)
    for tick in ax1.get_yticklabels():
        tick.set_color(color_temp)
    ax1.set_ylim(-10, 40)
    
    # 设置温度轴的框线
    ax1.spines['left'].set_color(color_temp)
    ax1.spines['left'].set_linewidth(1.5)
    # 保持其他框线为黑色
    ax1.spines['right'].set_visible(True)
    ax1.spines['top'].set_visible(True)
    

    # 设置x轴范围为截取的时间段
    ax1.set_xlim(pd.Timestamp(start_date), pd.Timestamp(end_date))
    
    # 设置x轴刻度 - 显示1st Jul, 15th Jul, 31st Jul
    ax1.xaxis.set_major_locator(mdates.DayLocator([1, 15, 31]))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    
    # 设置x轴月份标签
    for label in ax1.get_xticklabels():
        label.set_fontsize(13)

    # ====== SPEI ======
    ax2 = ax1.twinx()
    color_spei = (254/255, 185/255, 84/255)
    ax2.plot(df['time'], df['spei'], color=color_spei, linewidth=2)
    ax2.axhline(-1, color=color_spei, linestyle='--', linewidth=1.5, alpha=0.7)
    ax2.set_ylabel('SPEI', color=color_spei, fontsize=15, weight='bold')
    ax2.tick_params(axis='y', labelcolor=color_spei, color=color_spei, width=1.3)
    for tick in ax2.get_yticklabels():
        tick.set_color(color_spei)
    ax2.set_ylim(-3, 3)
    
    # 设置SPEI轴的框线
    ax2.spines['right'].set_color(color_spei)
    ax2.spines['right'].set_linewidth(1.5)
    # 隐藏SPEI轴的其他框线
    ax2.spines['top'].set_visible(False)
    ax2.spines['left'].set_visible(False)
    ax2.spines['bottom'].set_visible(False)


    # ====== CDHW 阴影 + 标签 ======
    for i, (start, end) in enumerate(events):
        label = "CDHW Event" if i == 0 else "_nolegend_"
        ax1.axvspan(start, end, color='gray', alpha=0.25, zorder=10, label=label)
        
        # 在阴影中间添加标签
        # mid = start + (end - start) / 2
        # ax1.text(mid, ax1.get_ylim()[1] - 5, "CDHW Event",
        #          color='black', fontsize=11, weight='bold',
        #          ha='center', va='top', rotation=0,
        #          bbox=dict(boxstyle="round,pad=0.2", facecolor='white', alpha=0.5, edgecolor='none'))

    # ====== 在阈值附近添加标签 ======
    # 温度阈值标签
    ax1.text(pd.Timestamp(start_date) + pd.Timedelta(days=2), tas95_value - 8, 
             f'Threshold: 95th_Tmax', 
             color=color_temp, fontsize=12, weight='bold',
             bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.5, edgecolor=color_temp))
    
    # SPEI阈值标签
    ax2.text(pd.Timestamp(start_date) + pd.Timedelta(days=2), -1 + 0.35, 
             'Threshold: Daily_SPEI', 
             color=color_spei, fontsize=12, weight='bold',
             bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.5, edgecolor=color_spei))
    ax1.tick_params(axis='y', labelsize=13)  # 温度轴字体大小
    ax2.tick_params(axis='y', labelsize=13)  # SPEI轴字体大小

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.10, right=0.85)
    plt.savefig(
        r'D:\smq\paper2\result\cdhw2.pdf',
        format='pdf',  # 指定输出 EPS 格式
        bbox_inches='tight',  # 紧凑布局
        dpi=300,  # 虽然 EPS 是矢量，但某些元素可能受 DPI 影响
    )
    plt.show()


# # =======================
# # 5. 主程序
# # =======================
# if __name__ == "__main__":
#     lat_sel = 45  # 地中海区域
#     lon_sel = 21
#     start_date, end_date = '2012-06-15', '2012-07-31'

#     data = load_data(lat_sel, lon_sel, start_date, end_date)
#     cdhw_mask, events = detect_cdhw(data)
#     df = build_dataframe(data, cdhw_mask)

#     print(f"共检测到 {len(events)} 个CDHW事件")
#     for i, e in enumerate(events, 1):
#         print(f"事件{i}: {e[0].strftime('%Y-%m-%d')} - {e[-1].strftime('%Y-%m-%d')}, 持续 {len(e)} 天")

#     # 移除了图例位置参数
#     plot_timeseries(df, events, data['tas95'], data, lat_sel, lon_sel, start_date, end_date)

# =======================
# 5. 主程序
# =======================
if __name__ == "__main__":
    lat_sel = 45
    lon_sel = 21
    start_date, end_date = '2012-06-15', '2012-07-31'

    data = load_data(lat_sel, lon_sel, start_date, end_date)
    df = build_dataframe(data)
    events = extract_events(df['time'].values, data['cdhw_mask'])

    plot_timeseries(df, events, data['tas95'], data, lat_sel, lon_sel, start_date, end_date)
    

## 4. ASM definition and soil moisture depletion under CDHWs

Main Methods: ASM definition and event-scale depletion/regression. This section keeps the final-relevant base ASM/depletion code before the revised sensitive-region framework.


### ASM state binning and CDHW intensity-response diagnostic



In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import xarray as xr
import numpy as np
import dask.array as da

mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"

fp_si  = r"F:\smq\paper2\data\cdhw\S_i_dask.nc"
fp_asm = r"F:\smq\paper2\w\w_daily.nc"

S_i = xr.open_dataset(fp_si, engine="h5netcdf", chunks="auto")["S_i"]
ASM = xr.open_dataset(fp_asm, engine="h5netcdf", chunks="auto")["w"]

period = slice("1920-01-01", "2019-12-31")
S_i = S_i.sel(time=period)
ASM = ASM.sel(time=period)

# ======================
# 事件识别
# ======================
S_prev = S_i.shift(time=1)
is_new_event = (S_i != S_prev)

ASM_event = ASM.where(is_new_event)

ASM_pre = ASM.shift(time=3).rolling(time=3, min_periods=1).mean()
delta_ASM = ASM_pre - ASM_event

delta_ASM = delta_ASM.where(delta_ASM > 0)

# ======================
# 手动bin计算（核心修改）
# ======================
bins = np.arange(0, 1.005, 0.01)
bin_centers = (bins[:-1] + bins[1:]) / 2

mean_list = []

for i in range(len(bins) - 1):
    lower = bins[i]
    upper = bins[i+1]
    
    mask = (ASM_event >= lower) & (ASM_event < upper)
    
    masked_delta = delta_ASM.where(mask)
    
    # 每个bin单独mean（仍然是lazy）
    mean_val = masked_delta.mean(dim=["time","lat","lon"])
    
    mean_list.append(mean_val)

# 合并成一个DataArray
mean_delta = xr.concat(mean_list, dim="asm_bin")
mean_delta = mean_delta.assign_coords(asm_bin=bin_centers)

# 现在才真正计算
mean_delta = mean_delta.compute()

y = mean_delta.values

# ======================
# 计算 ΔASM_max
# ======================
delta_ASM_max = np.nanmax(y)
print("ΔASM_max =", delta_ASM_max)

# ======================
# 多阈值敏感性
# ======================
alphas = [0.85, 0.9, 0.95,0.97,0.99]
cp_ranges = {}

for alpha in alphas:
    threshold = alpha * delta_ASM_max
    mask = y >= threshold
    
    cp_bins = bin_centers[mask]
    
    if len(cp_bins) > 0:
        cp_min = cp_bins.min()
        cp_max = cp_bins.max()
        cp_ranges[alpha] = (cp_min, cp_max)
        print(f"alpha={alpha}: {cp_min:.3f} - {cp_max:.3f}")
    else:
        cp_ranges[alpha] = None
        print(f"alpha={alpha}: no bins satisfy")


import os
# ======================
# 绘图
# ======================
fig, ax = plt.subplots(figsize=(4,2.8))

ax.plot(bin_centers, y, linewidth=2.5)

# ======================
# 计算 95% 阈值
# ======================
threshold_95 = 0.97 * delta_ASM_max
highlight_color = (254/255, 185/255, 84/255)

# ======================
# 坐标轴样式
# ======================
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_position(('outward', 5))
ax.spines['bottom'].set_position(('outward', 5))

ax.tick_params(
    axis='both', which='both',
    direction='out',
    bottom=True, top=False,
    left=True, right=False,
    labelbottom=True, labelleft=True,
    length=6, width=1, color='black', labelsize=13
)

# 加粗刻度字体
xticks = np.arange(0, 1.01, 0.2)
ax.set_xticks(xticks)
ax.set_xticklabels([f"{x:.1f}" for x in xticks])

ax.set_ylim(0.004, 0.02)
ax.set_yticks(np.arange(0.004, 0.024, 0.004)) 

# ======================
# 固定阴影 0.3–0.8
# ======================
ax.axvspan(
    0.3, 0.8,
    alpha=0.15,
    color="grey",
    zorder=0
)

# ======================
# 95% 横线
# ======================
ax.axhline(
    threshold_95,
    linestyle='--',
    linewidth=2,
    color=highlight_color
)

# ======================
# 竖线
# ======================
if cp_ranges[0.97] is not None:
    cp_min, cp_max = cp_ranges[0.97]
    ymin = ax.get_ylim()[0]

    ax.plot(
        [cp_min, cp_min],
        [threshold_95, ymin],
        linestyle='--',
        linewidth=1.5,
        color=highlight_color
    )

    ax.plot(
        [cp_max, cp_max],
        [threshold_95, ymin],
        linestyle='--',
        linewidth=1.5,
        color=highlight_color
    )

# ======================
# 横线标签（加粗）
# ======================
xmax = ax.get_xlim()[1]

ax.text(
    xmax,
    threshold_95,
    "95% threshold",
    color=highlight_color,
    fontsize=13,
    fontweight='bold',
    ha='right',
    va='bottom'
)

# ======================
# 坐标轴标签（加粗）
# ======================
ax.set_xlabel("ASM Bin", fontsize=15, fontweight='bold')
ax.set_ylabel("Mean ΔASM", fontsize=15, fontweight='bold')

plt.tight_layout()

pdf_save_path = r"D:\smq\paper2\result\ASM_0.3-0.8.pdf"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(pdf_save_path, format='pdf', bbox_inches='tight')

plt.show()

C:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\dask\array\core.py:5003: PerformanceWarning: Increasing number of chunks by factor of 57
  result = blockwise(


### Fig. 2 spatial map: CDHW-masked ASM loss



In [ ]:
import xarray as xr
import numpy as np
from scipy import stats
from dask.distributed import Client, LocalCluster
import dask
import os
import matplotlib.pyplot as plt

# ======================
# Dask 并行配置
# ======================
cluster = LocalCluster(n_workers=20, threads_per_worker=1, memory_limit="10GB")
client = Client(cluster)
print("Dashboard 地址:", client.dashboard_link)

# ======================
# 文件路径
# ======================
fp_w   = r"D:\smq\paper2\data\cdhw\w_by_masks_efficiency.nc"
fp_sev = r"D:\smq\paper2\data\cdhw\cdhw_severity_daily.nc"
fp_mask  = r"D:\smq\paper2\data\modified_mask.nc"
output_file = r"D:\smq\paper2\result\cdhw_w_event_regression.nc"

# ======================
# 读取数据（懒加载）
# ======================
ds_w   = xr.open_dataset(fp_w, engine="h5netcdf", chunks={"time": -1, "lat": 50, "lon": 50})["w_cdhw"]
ds_sev = xr.open_dataset(fp_sev, engine="h5netcdf", chunks={"time": -1, "lat": 50, "lon": 50})["daily_severity"]
mask   = xr.open_dataset(fp_mask, chunks={"lat": 50, "lon": 50})["mask_data"] > 1

# 对齐时间
ds_w, ds_sev = xr.align(ds_w, ds_sev, join="inner")

# ======================
# 定义逐格点的事件提取 + 回归函数
# ======================
def event_based_regression(w_1d, sev_1d):

    if np.all(np.isnan(w_1d)) or np.all(np.isnan(sev_1d)):
        return np.nan, np.nan, np.nan

    w_1d = np.asarray(w_1d, dtype=np.float32)
    sev_1d = np.asarray(sev_1d, dtype=np.float32)
    valid = np.isfinite(w_1d) & np.isfinite(sev_1d)
    if valid.sum() < 10:
        return np.nan, np.nan, np.nan

    # Step1: 提取事件（连续非零天为一个事件）
    in_event = False
    event_id = np.zeros(len(sev_1d), dtype=int)
    eid = 0

    for i, val in enumerate(sev_1d):
        if val > 0 and not in_event:
            eid += 1
            in_event = True
        elif val == 0:
            in_event = False
        if in_event:
            event_id[i] = eid

    # Step2: 提取每个事件的平均w和累计severity
    event_mean_w = []
    event_mean_sev = []
    for e in np.unique(event_id):
        if e == 0:
            continue
        idx = event_id == e
        event_mean_w.append(np.nansum(w_1d[idx]))
        event_mean_sev.append(np.nansum(sev_1d[idx]))

    event_mean_w = np.array(event_mean_w)
    event_mean_sev = np.array(event_mean_sev)

    # Step3: 线性回归（w_event ~ severity_event）
    if len(event_mean_w) < 3:
        return np.nan, np.nan, np.nan

    mask_valid = np.isfinite(event_mean_w) & np.isfinite(event_mean_sev)
    if mask_valid.sum() < 3:
        return np.nan, np.nan, np.nan

    slope, intercept, r, p, stderr = stats.linregress(event_mean_sev[mask_valid], event_mean_w[mask_valid])
    return slope, r, p


# ======================
# 并行 apply_ufunc
# ======================
slope, r, p = xr.apply_ufunc(
    event_based_regression,
    ds_w, ds_sev,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[[], [], []],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float, float, float]
)

# ======================
# 输出结果数据集
# ======================
ds_out = xr.Dataset({"slope": slope, "r": r, "p": p}).where(mask)
ds_out = ds_out.astype(np.float32)

encoding = {
    "slope": {"zlib": True, "complevel": 4, "dtype": "float32"},
    "r": {"zlib": True, "complevel": 4, "dtype": "float32"},
    "p": {"zlib": True, "complevel": 4, "dtype": "float32"},
}

if os.path.exists(output_file):
    os.remove(output_file)

print("⚙️ 正在计算并写入文件...")

# 延迟执行 + 并行写出
delayed_obj = ds_out.to_netcdf(
    output_file,
    engine="h5netcdf",
    encoding=encoding,
    compute=False
)
dask.compute(delayed_obj)

print(f"[✅ Success] 文件已写入: {output_file}")

# ======================
# 可视化验证
# ======================
# plt.figure(figsize=(6, 4))
# ds_out["slope"].plot(cmap="coolwarm", robust=True)
# plt.title("Slope (Event-mean W vs Event-sum Severity)")
# plt.show()


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import matplotlib as mpl
import regionmask
import os
from matplotlib.colors import LinearSegmentedColormap

mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"

# ======================
# 读取数据
# ======================
ds_out = xr.open_dataset(r"F:\smq\paper2\data\cdhw\cdhw_w_event_regression.nc", engine="h5netcdf")
delta = xr.open_dataset(r"F:\smq\paper2\su\wfc_daily.nc")["wfc"][0] - xr.open_dataset(r"F:\smq\paper2\su\wwp_daily.nc")["wwp"][0]
eff = xr.open_dataset(r"F:\smq\paper2\data\modified_mask.nc")["mask_data"]

slope = ds_out["slope"] * delta * 1000

p = ds_out["p"]

# ======================
# 限制有效区域（eff > 1）
# ======================
mask_eff = eff > 1
slope_eff = slope.where(mask_eff)
p_eff = p.where(mask_eff)

srex_regions = regionmask.defined_regions.srex

# ======================
# 仅使用有效区域
# ======================
slope_masked = slope.where(mask_eff)

p_masked = p.where(mask_eff)
# 显著性筛选
slope_sig = slope_masked.where(p_masked < 0.05)
slope_nonsig = xr.where((p_masked >= 0.05) & (~np.isnan(slope_masked)), 1, np.nan)

# ======================
# 绘图
# ======================
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())
ax.set_aspect(1.2)   # 值 <1 拉宽, 值 >1 拉高

base_cmap = plt.cm.Blues
custom_cmap = LinearSegmentedColormap.from_list(
    "trunc_RdYlBu_r",
    base_cmap(np.linspace(0.15, 1, 256))
)

# --- 显著区域（红蓝） ---
im = slope_sig.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap=custom_cmap,
    vmin=0, vmax=50,
    alpha=0.75,
    add_colorbar=False
)

# --- 非显著区域（灰色） ---
slope_nonsig.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="gray",
    alpha=0.2,
    add_colorbar=False
)

# ======================
# 绘制地图与 SREX 边界
# ======================
# ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor='black', alpha=0.7)
# ax.add_feature(cfeature.COASTLINE, linewidth=0.4, alpha=0.7)

# 文本样式
text_kws = dict(
    bbox=dict(color="none"),
    color='grey',
    weight='bold',
    fontsize=15,

)

# 线条样式
line_kws = dict(
    color='grey',
    linewidth=1.5,
    alpha=0.5,
)

# 绘制 SREX 区域边界和缩写标签
srex_regions.plot(
    ax=ax,
    projection=ccrs.PlateCarree(),
    label="abbrev",
    text_kws=text_kws,
    line_kws=line_kws
)

# ======================
# 添加色标
# ======================
cb = plt.colorbar(
    im,
    ax=ax,
    orientation="horizontal",
    fraction=0.05,
    pad=0.05,
    shrink=0.35,
    aspect=50
)

# cb.set_ticks(np.arange(0, 50, 1))

cb.ax.tick_params(labelsize=14)

cb.set_label(
    "Available-water response (mm per unit CDHW intensity)",
    fontsize=14,
    fontweight='bold'
)

plt.box(False)
plt.title("")
plt.tight_layout()

# ======================
# 保存结果
# ======================
pdf_save_path = r"D:\smq\paper2\result\Slope_sensitivity.pdf"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(pdf_save_path, format='pdf', bbox_inches='tight')
plt.show()


### Fig. 2b subgraph


In [ ]:
import os
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import gc

# ======================
# 文件路径
# ======================
fp_w = r"D:\smq\paper2\data\cdhw\w_by_masks_efficiency.nc"
fp_sev = r"D:\smq\paper2\data\cdhw\cdhw_severity_daily.nc"
fp_mask = r"D:\smq\paper2\data\modified_mask.nc"  # 若不需要掩膜可忽略
out_dir = r"D:\smq\paper2\data\cdhw"
save_fp = os.path.join(out_dir, "global_pre_binning_results.npz")

if os.path.exists(save_fp):
    print(f"✅ 已存在结果文件: {save_fp}")
    exit()

# ======================
# 1. 数据加载
# ======================
print("正在加载掩膜与主数据...")
chunks = {'time': -1, 'lat': 50, 'lon': 50}

region_ds = xr.open_dataset(fp_mask)["mask_data"]
region_mask_all = region_ds.where(region_ds>1)
valid_mask = ~np.isnan(region_mask_all)

ds_w = xr.open_dataset(fp_w, engine="h5netcdf", chunks=chunks)["w_cdhw"]
ds_sev = xr.open_dataset(fp_sev, engine="h5netcdf", chunks=chunks)["daily_severity"]

# 时间对齐
ds_w, ds_sev = xr.align(ds_w, ds_sev, join="inner")
time_vals = pd.DatetimeIndex(ds_w.time.values)
print(f"对齐后共有 {len(time_vals)} 天")

# ======================
# 2. 处理函数
# ======================
def process_batch(lat_batch, lon_batch, ds_w, ds_sev, region_mask, time_vals):
    SEV_batch, W_diff_pct_batch, W_anom_batch = [], [], []
    batch_points, batch_event_points = 0, 0
    
    for lat, lon in zip(lat_batch, lon_batch):
        mask_point = region_mask.sel(lat=lat, lon=lon, method="nearest")
        if mask_point.isnull() or not mask_point.item():
            continue
        batch_points += 1

        try:
            w_point = ds_w.sel(lat=lat, lon=lon, method="nearest").load()
            sev_point = ds_sev.sel(lat=lat, lon=lon, method="nearest").load()
        except:
            continue

        if w_point.isnull().all() or sev_point.isnull().all():
            continue

        w_series = pd.Series(w_point.values, index=time_vals)
        sev_vals = sev_point.values

        # 计算月均+滚动基线
        try:
            w_monthly = w_series.resample("ME").mean()
            w_baseline_monthly = w_monthly.rolling(window=3, center=True, min_periods=1).mean()
            w_baseline_daily = w_baseline_monthly.reindex(w_series.index, method='ffill')
        except:
            continue

        eps = 1e-6
        w_diff_percent = ((w_series - w_baseline_daily) / (w_baseline_daily + eps)) * 100
        w_min, w_max = w_series.min(), w_series.max()
        w_anomaly = (w_series - w_min) / (w_max - w_min) if w_max - w_min > eps else pd.Series(np.zeros(len(w_series)), index=w_series.index)

        mask_event = sev_vals > 0
        if not mask_event.any():
            continue

        batch_event_points += 1
        sev_ev = sev_vals[mask_event]
        w_diff_ev = w_diff_percent.values[mask_event]
        w_ano_ev = w_anomaly.values[mask_event]
        valid_mask = (~np.isnan(sev_ev)) & (~np.isnan(w_diff_ev)) & (~np.isnan(w_ano_ev))
        if valid_mask.sum() > 0:
            SEV_batch.append(sev_ev[valid_mask])
            W_diff_pct_batch.append(w_diff_ev[valid_mask])
            W_anom_batch.append(w_ano_ev[valid_mask])

        del w_point, sev_point, w_series, sev_vals
        gc.collect()

    return SEV_batch, W_diff_pct_batch, W_anom_batch, batch_points, batch_event_points


# ======================
# 3. 全局处理循环
# ======================
print("\n====== 开始处理全球数据 ======")
lat_vals = ds_w.lat.values
lon_vals = ds_w.lon.values
SEV_raw, W_diff_pct, W_anom = [], [], []
total_points, event_points = 0, 0

# 这里调整 batch_size 以控制内存
batch_size = 100
n_batches = int(np.ceil(len(lat_vals) * len(lon_vals) / batch_size))

for batch_idx in tqdm(range(n_batches), desc="Processing Global"):
    start_idx = batch_idx * batch_size
    end_idx = min((batch_idx + 1) * batch_size, len(lat_vals) * len(lon_vals))

    batch_lats, batch_lons = [], []
    for idx in range(start_idx, end_idx):
        i = idx // len(lon_vals)
        j = idx % len(lon_vals)
        if i < len(lat_vals):
            batch_lats.append(lat_vals[i])
            batch_lons.append(lon_vals[j])

    SEV_batch, W_pct_batch, W_ano_batch, batch_pts, batch_events = process_batch(
        batch_lats, batch_lons, ds_w, ds_sev, valid_mask, time_vals
    )
    total_points += batch_pts
    event_points += batch_events
    SEV_raw.extend(SEV_batch)
    W_diff_pct.extend(W_pct_batch)
    W_anom.extend(W_ano_batch)

    if batch_idx % 10 == 0:
        gc.collect()

# ======================
# 4. 保存结果
# ======================
if len(SEV_raw) == 0:
    print("❌ 全球无有效事件样本。")
else:
    SEV = np.concatenate(SEV_raw)
    W_PCT = np.concatenate(W_diff_pct)
    W_ANO = np.concatenate(W_anom)

    np.savez_compressed(save_fp, SEV=SEV, W_PCT=W_PCT, W_ANO=W_ANO,
                        total_points=total_points, event_points=event_points)
    print(f"✅ 全球结果保存到: {save_fp} ({len(SEV):,} events)")

    del SEV_raw, W_diff_pct, W_anom, SEV, W_PCT, W_ANO
    gc.collect()

print("\n🎯 全球数据处理完成！")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
import matplotlib as mpl
import os
# 设置全局字体为Arial
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.sans-serif'] = 'Arial'
mpl.rcParams['pdf.fonttype'] = 42

# ======================
# 数据读取
# ======================
data = np.load(r"F:\smq\paper2\data\cdhw\bin\global_pre_binning_results.npz")
SEV = data["SEV"]
W_PCT = data["W_PCT"]
W_ANO = data["W_ANO"]

# ====== 固定箱子数 ======
n_bins = 20  # 固定为10个箱子
sev_min, sev_max = np.nanmin(SEV), np.nanmax(SEV)
bins_wide = np.linspace(sev_min, sev_max, n_bins + 1)  # 等间距分箱

# 使用 digitize 并处理边界情况
sev_bins = np.digitize(SEV, bins_wide) - 1
# 将等于最大值的点放到最后一个箱子
sev_bins[sev_bins == len(bins_wide) - 1] = len(bins_wide) - 2

# 创建分箱标签和中点
bin_labels = [f"{bins_wide[i]:.1f}-{bins_wide[i+1]:.1f}" for i in range(len(bins_wide)-1)]
bin_midpoints = [(bins_wide[i] + bins_wide[i+1])/2 for i in range(len(bins_wide)-1)]

df = pd.DataFrame({
    "SEV_bin": sev_bins,
    "SEV_mid": [bin_midpoints[i] for i in sev_bins],  # 使用预计算的中点
    "W_PCT": W_PCT,
    "W_ANO": W_ANO
})
df = df[(df["SEV_bin"] >= 0) & (df["SEV_bin"] < len(bin_labels))]
df["SEV_label"] = df["SEV_bin"].map(lambda i: bin_labels[i])

# 样本数筛选
min_count = 30
count_per_bin = df["SEV_bin"].value_counts().sort_index()
valid_bins = count_per_bin[count_per_bin >= min_count].index
df = df[df["SEV_bin"].isin(valid_bins)]
df["SEV_label"] = df["SEV_label"].astype("category")

# 计算每个箱子的统计量
bin_stats = df.groupby("SEV_bin").agg({
    "SEV_mid": "first",
    "W_ANO": ["median", lambda x: x.quantile(0.05), lambda x: x.quantile(0.95)]
}).reset_index()
bin_stats.columns = ["SEV_bin", "SEV_mid", "W_ANO_median", "W_ANO_q05", "W_ANO_q95"]

# 拟合趋势线 + 置信区间
def get_regression_ci(X, y, X_pred, ci=95):
    X_sm = sm.add_constant(X)
    model = sm.OLS(y, X_sm).fit()
    X_pred_sm = sm.add_constant(X_pred)
    predictions = model.get_prediction(X_pred_sm)
    ci_result = predictions.conf_int(alpha=(100-ci)/100)
    return ci_result[:, 0], ci_result[:, 1], model

X = bin_stats["SEV_mid"].values
y_ano = bin_stats["W_ANO_median"].values
X_pred = np.linspace(X.min(), X.max(), 100)
ci_ano_lower, ci_ano_upper, model_ano = get_regression_ci(X, y_ano, X_pred, ci=95)
y_ano_fit = model_ano.predict(sm.add_constant(X_pred))

bin_stats["W_ANO_fit"] = model_ano.predict(sm.add_constant(bin_stats["SEV_mid"]))

# ======================
# 绘图部分（使用区间中位数作为横坐标）
# ======================
plt.figure(figsize=(2.7, 2.2))
ax = plt.gca()

# 应用您提供的坐标轴样式
ax.set_title("")
ax.set_ylabel("Normalized ASMA", fontsize=14, fontweight='bold')
ax.set_xlabel("CDHW Intensity", fontsize=14, fontweight='bold')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_position(('outward', 5))
ax.spines['bottom'].set_position(('outward', 5))
ax.tick_params(
    axis='both', which='both',
    direction='out',
    bottom=True, top=False,
    left=True, right=False,
    labelbottom=True, labelleft=True,
    length=6, width=1, color='black', labelsize=13
)

# 颜色映射
n_bins_plot = len(bin_stats)
colors_ano_edge = plt.cm.GnBu(np.linspace(0.3, 0.9, n_bins_plot))

# 绘制每个分箱的竖直线 + 中位圆点
for i, row in bin_stats.iterrows():
    color = colors_ano_edge[i]
    # 分布范围 (5%–95%)
    ax.plot([row["SEV_mid"], row["SEV_mid"]], [row["W_ANO_q05"], row["W_ANO_q95"]],
            color='lightgrey', linewidth=2, zorder=2)
    # 中位数圆点
    ax.scatter(row["SEV_mid"], row["W_ANO_median"],
               color='grey', s=30, zorder=3, edgecolors='white', linewidth=1.5)

# 拟合线及置信区间
plt.fill_between(bin_stats["SEV_mid"],
                 ci_ano_lower[np.linspace(0, len(X_pred)-1, len(bin_stats), dtype=int)],
                 ci_ano_upper[np.linspace(0, len(X_pred)-1, len(bin_stats), dtype=int)],
                 alpha=0.3, color='darkorange', label="95% CI of fit", zorder=4)
plt.plot(bin_stats["SEV_mid"], bin_stats["W_ANO_fit"],
         color='darkorange', linewidth=2.5, label="Median trend", zorder=5)

# 拟合结果文本
slope = model_ano.params[1]
p_value = model_ano.f_pvalue
significance = "p < 0.05" if p_value < 0.05 else f"p = {p_value:.3f}"
fit_text = f"Slope = {slope:.2f}"

# 取拟合线上中点位置，稍微往上移一点
x_text = np.median(bin_stats["SEV_mid"])
y_text = np.interp(x_text, bin_stats["SEV_mid"], bin_stats["W_ANO_fit"]) + 0.05

ax.text(
    0.05, 0.4,  
    fit_text,
    transform=ax.transAxes,  # 使用坐标轴比例定位
    fontsize=13,
    color='black',
    fontfamily='Arial',
    fontweight='bold',
    alpha=0.8,
    ha='left', va='bottom',
    zorder=10,
    # bbox=dict(
    #     boxstyle="round,pad=0.2",
    #     facecolor='white',
    #     edgecolor='white',
    #     alpha=0.8
    # )
)


# 坐标与标题 - 只显示两个端点和中间的label
x_ticks = bin_stats["SEV_mid"].values
if len(x_ticks) > 2:
    # 选择第一个、中间一个和最后一个
    selected_ticks = [x_ticks[0], x_ticks[len(x_ticks)//2], x_ticks[-1]]
    selected_labels = [f"{x_ticks[0]:.1f}", f"{x_ticks[len(x_ticks)//2]:.1f}", f"{x_ticks[-1]:.1f}"]
else:
    # 如果只有2个或更少，全部显示
    selected_ticks = x_ticks
    selected_labels = [f"{x:.1f}" for x in x_ticks]

plt.xticks(selected_ticks, selected_labels, fontsize=14, fontfamily='Arial')
plt.yticks(fontsize=9, fontfamily='Arial')
plt.text(0.3, 1, f"Global", transform=ax.transAxes, fontsize=15,fontfamily='Arial', fontweight='bold', verticalalignment='top')
plt.ylim(0, 1)
plt.yticks(np.arange(0, 1 + 0.01, 0.5), fontsize=14, fontfamily='Arial')
plt.xlim(bin_stats["SEV_mid"].min() - 0.1, bin_stats["SEV_mid"].max() + 0.1)
plt.tight_layout()

pdf_save_path = r"D:\smq\paper2\result\bins\Global.pdf"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(pdf_save_path, format='pdf', bbox_inches='tight')
plt.show()

## 5. Response-sensitive region framework

Main Methods and Supplementary C. 


### draw_framework_final_v10.py

Supplementary C Fig. S2 analytical framework drawing. Framework figure for the final sensitive-region method.


In [ ]:
# Source file: D:\smq\paper2\modify\敏感区\code\draw_framework_final_v10.py
# Staging copy used for notebook assembly: D:\software\codex\paper2\office_work\code_notebook_order_20260609\sensitive_code\draw_framework_final_v10.py
# Detected encoding: utf-8-sig
# Methods-order section: Supplementary C Fig. S2 analytical framework drawing
# NOTE: This is the final modified sensitive-region/SHAP script, not legacy notebook code.

from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle


OUT = Path(r"D:\Software\codex\sm_heat_drought\outputs\20260522_improved_framework_word")

COL = {
    "ink": "#202A33",
    "muted": "#65717B",
    "blue": "#2D6E9F",
    "blue_dark": "#23577F",
    "blue_light": "#EAF3FA",
    "grey": "#5E6972",
    "grey_light": "#F3F5F6",
    "amber": "#9A6B20",
    "amber_light": "#FAF1E2",
    "rule": "#C8D0D6",
    "white": "#FFFFFF",
}


def setup() -> None:
    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 6.0,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "savefig.transparent": False,
        "figure.facecolor": "white",
    })


def txt(ax, x, y, s, size=6.0, weight="normal", color=None, ha="left", va="center"):
    ax.text(x, y, s, fontsize=size, fontweight=weight, color=color or COL["ink"], ha=ha, va=va)


def rounded(ax, x, y, w, h, fc, ec, lw=0.7, r=0.010):
    p = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.004,rounding_size={r}",
        facecolor=fc,
        edgecolor=ec,
        linewidth=lw,
    )
    ax.add_patch(p)
    return p


def arrow(ax, x1, y1, x2, y2, color, lw=0.82):
    ax.add_patch(FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="-|>",
        mutation_scale=6.8,
        linewidth=lw,
        color=color,
        shrinkA=1.2,
        shrinkB=1.2,
    ))


def node(ax, x, y, w, h, no, title, line_a, line_b="", active=False):
    rounded(ax, x, y, w, h, "#DCECF6" if active else COL["white"], COL["blue"], lw=0.85, r=0.012)
    txt(ax, x + 0.016, y + h * 0.66, str(no), size=6.7, weight="bold", color=COL["blue"])
    txt(ax, x + 0.043, y + h * 0.66, title, size=6.15, weight="bold",
        color=COL["blue_dark"] if active else COL["ink"])
    txt(ax, x + 0.043, y + h * 0.36, line_a, size=4.55, color=COL["muted"])
    if line_b:
        txt(ax, x + 0.043, y + h * 0.18, line_b, size=4.55, color=COL["muted"])


def chip(ax, x, y, w, h, title, subtitle, ec, title_size=4.95):
    rounded(ax, x, y, w, h, COL["white"], ec, lw=0.62, r=0.009)
    txt(ax, x + w / 2, y + h * 0.63, title, size=title_size, weight="bold", ha="center")
    txt(ax, x + w / 2, y + h * 0.31, subtitle, size=4.10, color=COL["muted"], ha="center")


def draw() -> None:
    fig, ax = plt.subplots(figsize=(7.05, 2.65))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    ax.add_patch(Rectangle((0.035, 0.615), 0.930, 0.300, facecolor=COL["blue_light"], edgecolor="none"))
    txt(ax, 0.055, 0.875, "Primary A-mask construction", size=6.05, weight="bold", color=COL["blue_dark"])
    txt(ax, 0.835, 0.875, "solid path defines A", size=4.75, color=COL["muted"], ha="right")
    arrow(ax, 0.845, 0.875, 0.895, 0.875, COL["blue"], lw=0.72)

    y, h = 0.690, 0.105
    xs = [0.080, 0.305, 0.530, 0.765]
    ws = [0.145, 0.150, 0.160, 0.145]
    node(ax, xs[0], y, ws[0], h, 1, "Events", "SM, ASW", "CDHW / single events")
    node(ax, xs[1], y, ws[1], h, 2, "Response", "event-window loss", "positive-loss ratio")
    node(ax, xs[2], y, ws[2], h, 3, "Controls", "matched single events", "pixel-level regression")
    node(ax, xs[3], y - 0.005, ws[3], h + 0.010, 4, "A mask", "high loss", "ME > 0, RE > 0", active=True)
    for i in range(3):
        arrow(ax, xs[i] + ws[i] + 0.017, y + h / 2, xs[i + 1] - 0.017, y + h / 2, COL["blue"], lw=0.88)

    ax.plot([0.088, 0.912], [0.660, 0.660], color="#BDD2E2", lw=0.52)
    txt(
        ax, 0.50, 0.637,
        "A rule: high CDHW event-window loss + ME > 0 + RE > 0 + positive-loss consistency + event support",
        size=4.75, color=COL["blue_dark"], ha="center"
    )

    ax.add_patch(Rectangle((0.035, 0.350), 0.930, 0.170, facecolor=COL["grey_light"], edgecolor="none"))
    txt(ax, 0.055, 0.485, "Diagnostic checks", size=6.05, weight="bold", color=COL["grey"])
    txt(ax, 0.835, 0.485, "audit links only", size=4.75, color=COL["muted"], ha="right")
    ax.plot([0.845, 0.895], [0.485, 0.485], color=COL["rule"], lw=0.60, linestyle=(0, (2, 2)))

    for x, title, sub in [
        (0.115, "Metric audit", "absolute vs relative loss"),
        (0.325, "B/C layers", "preconditioning / recovery"),
        (0.535, "ASW window", "antecedent water state"),
        (0.745, "Robustness", "Q thresholds, PLR, support"),
    ]:
        chip(ax, x, 0.382, 0.140, 0.055, title, sub, COL["grey"])

    # Robustness audits the whole A-mask rule; the other diagnostics audit specific inputs/processes.
    for x in [0.185, 0.395, 0.605]:
        ax.plot([x, x], [0.520, 0.595], color=COL["rule"], lw=0.50, linestyle=(0, (2, 2)))
    ax.plot([0.500, 0.815], [0.600, 0.520], color=COL["rule"], lw=0.55, linestyle=(0, (2, 2)))

    txt(ax, 0.50, 0.322, "diagnostic only; not merged into the A-mask", size=4.85, color=COL["muted"], ha="center")
    ax.plot([0.230, 0.770], [0.306, 0.306], color=COL["rule"], lw=0.55)

    ax.add_patch(Rectangle((0.035, 0.105), 0.930, 0.130, facecolor=COL["amber_light"], edgecolor="none"))
    txt(ax, 0.055, 0.205, "Contextual interpretation", size=6.05, weight="bold", color=COL["amber"])
    for x, title, sub in [
        (0.205, "Trigger", "antecedent ASW effect"),
        (0.445, "Land cover", "composition"),
        (0.685, "Regions", "spatial summaries"),
    ]:
        chip(ax, x, 0.128, 0.135, 0.052, title, sub, COL["amber"], title_size=4.95)

    for ext, dpi in [("png", 600), ("tiff", 600), ("pdf", None), ("svg", None)]:
        fig.savefig(OUT / f"fig_Sx1_framework_final_v10.{ext}", dpi=dpi if dpi else None,
                    bbox_inches="tight", facecolor="white")
    plt.close(fig)


if __name__ == "__main__":
    setup()
    draw()


### run_event_window_specific_framework

Supplementary C S6-S7 event-window response metrics and A/B/C masks. Core script for CDHW/drought-only/heatwave-only separation, event-window loss, matched excess, regression excess, and A/strict-core/B/C masks.


In [ ]:
# Source file: D:\smq\paper2\modify\敏感区\code\run_event_window_specific_framework.py
# Staging copy used for notebook assembly: D:\software\codex\paper2\office_work\code_notebook_order_20260609\sensitive_code\run_event_window_specific_framework.py
# Detected encoding: utf-8-sig
# Methods-order section: Supplementary C S6-S7 event-window response metrics and A/B/C masks
# NOTE: This is the final modified sensitive-region/SHAP script, not legacy notebook code.

from __future__ import annotations

import json
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
from pandas.errors import EmptyDataError


ROOT = Path(r"D:\Software\codex\sm_heat_drought")
OUT = ROOT / "outputs" / "20260520_event_window_specific_framework"
PARTS = OUT / "parts"
OUT.mkdir(parents=True, exist_ok=True)
PARTS.mkdir(parents=True, exist_ok=True)

SOIL_DAILY = Path(r"E:\smq\sum_soil_daily.nc")
MASK_DAILY = Path(r"E:\smq\masks_cdhw_hw_di.nc")
DOMAIN_MASK = Path(r"E:\smq\modified_mask.nc")
IPCC_SOURCE = ROOT / "outputs" / "20260519_multi_method_sensitive_region" / "multi_method_pixels_with_flags.csv.gz"

LAT_BAND_SIZE = 4
WORKERS = 2
MIN_EVENTS = 20
MATCH_DISTANCE_MAX = 1.5
FORMATS = ("png", "pdf", "tiff")

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans", "sans-serif"],
        "font.size": 8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.linewidth": 0.75,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.facecolor": "white",
    }
)


def event_segments(mask: np.ndarray) -> list[tuple[int, int]]:
    if not mask.any():
        return []
    x = mask.astype(np.int8)
    changes = np.diff(np.concatenate(([0], x, [0])))
    starts = np.flatnonzero(changes == 1)
    ends = np.flatnonzero(changes == -1)
    return list(zip(starts.tolist(), ends.tolist()))


def circular_month_diff(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    d = np.abs(a[:, None] - b[None, :])
    return np.minimum(d, 12 - d)


def extract_event_table(soil: np.ndarray, mask: np.ndarray, months: np.ndarray, event_type: str) -> pd.DataFrame:
    rows = []
    ntime = len(soil)
    for start, end in event_segments(mask):
        if start < 30 or end <= start or end + 30 >= ntime:
            continue
        pre30 = float(np.nanmean(soil[start - 30 : start]))
        pre3 = float(np.nanmean(soil[start - 3 : start]))
        event_mean = float(np.nanmean(soil[start:end]))
        post30 = float(np.nanmean(soil[end : end + 30]))
        if not np.isfinite(pre30 + pre3 + event_mean + post30):
            continue
        duration = end - start
        event_window_loss = pre3 - event_mean
        preconditioning_loss = pre30 - pre3
        total_loss = pre30 - event_mean
        event_post = soil[start : end + 30]
        auc = float(np.nansum(np.maximum(pre30 - event_post, 0.0)))
        rows.append(
            {
                "event_type": event_type,
                "start": start,
                "duration": duration,
                "month": int(months[start]),
                "pre30": pre30,
                "pre3": pre3,
                "event_mean": event_mean,
                "post30": post30,
                "preconditioning_loss": preconditioning_loss,
                "event_window_loss": event_window_loss,
                "event_window_loss_rate": event_window_loss / duration,
                "total_loss_pre30_event": total_loss,
                "post30_deficit_pre30": pre30 - post30,
                "deficit_auc_pre30_event_post30": auc,
            }
        )
    return pd.DataFrame(rows)


def summarize_events(ev: pd.DataFrame, prefix: str) -> dict:
    if ev.empty:
        return {f"{prefix}_n": 0}
    out = {f"{prefix}_n": int(len(ev))}
    for col in [
        "duration",
        "pre30",
        "pre3",
        "preconditioning_loss",
        "event_window_loss",
        "event_window_loss_rate",
        "total_loss_pre30_event",
        "post30_deficit_pre30",
        "deficit_auc_pre30_event_post30",
    ]:
        out[f"{prefix}_{col}_mean"] = float(ev[col].mean())
        out[f"{prefix}_{col}_median"] = float(ev[col].median())
    out[f"{prefix}_positive_event_window_loss_ratio"] = float((ev["event_window_loss"] > 0).mean())
    out[f"{prefix}_positive_total_loss_ratio"] = float((ev["total_loss_pre30_event"] > 0).mean())
    out[f"{prefix}_preconditioning_share_mean"] = float(
        (ev["preconditioning_loss"] / (np.abs(ev["total_loss_pre30_event"]) + 1e-8)).replace([np.inf, -np.inf], np.nan).mean()
    )
    return out


def nearest_match_diffs(cdhw: pd.DataFrame, single: pd.DataFrame) -> dict:
    if len(cdhw) == 0 or len(single) == 0:
        return {"matched_n": 0}
    features = ["pre3", "preconditioning_loss", "duration"]
    all_feat = pd.concat([cdhw[features], single[features]], ignore_index=True)
    mu = all_feat.mean()
    sd = all_feat.std().replace(0, 1.0)
    c = ((cdhw[features] - mu) / sd).to_numpy(dtype="float64")
    s = ((single[features] - mu) / sd).to_numpy(dtype="float64")
    month_penalty = circular_month_diff(cdhw["month"].to_numpy(), single["month"].to_numpy()) / 3.0
    d = np.sqrt(((c[:, None, :] - s[None, :, :]) ** 2).sum(axis=2) + month_penalty**2)
    nearest = np.nanargmin(d, axis=1)
    nd = d[np.arange(len(cdhw)), nearest]
    ok = nd <= MATCH_DISTANCE_MAX
    if ok.sum() == 0:
        return {"matched_n": 0}
    c_ok = cdhw.iloc[np.flatnonzero(ok)]
    s_ok = single.iloc[nearest[ok]]

    def diff_stats(col: str, prefix: str) -> dict:
        diff = c_ok[col].to_numpy() - s_ok[col].to_numpy()
        return {
            f"matched_{prefix}_diff_mean": float(np.nanmean(diff)),
            f"matched_{prefix}_diff_median": float(np.nanmedian(diff)),
            f"matched_positive_{prefix}_diff_ratio": float(np.nanmean(diff > 0)),
        }

    out = {
        "matched_n": int(ok.sum()),
        "matched_share": float(ok.mean()),
        "matched_distance_median": float(np.median(nd[ok])),
    }
    out.update(diff_stats("event_window_loss", "event_window_loss"))
    out.update(diff_stats("event_window_loss_rate", "event_window_loss_rate"))
    out.update(diff_stats("total_loss_pre30_event", "total_loss"))
    out.update(diff_stats("post30_deficit_pre30", "post30"))
    out.update(diff_stats("deficit_auc_pre30_event_post30", "auc"))
    return out


def controlled_regression(events: pd.DataFrame) -> dict:
    if len(events) < 30 or events["event_type"].nunique() < 3:
        return {"reg_n": int(len(events)), "reg_ok": False}
    cols = [
        "event_type",
        "month",
        "pre3",
        "preconditioning_loss",
        "duration",
        "event_window_loss",
        "event_window_loss_rate",
        "total_loss_pre30_event",
        "post30_deficit_pre30",
        "deficit_auc_pre30_event_post30",
    ]
    df = events[cols].dropna()
    if len(df) < 30 or df["event_type"].nunique() < 3:
        return {"reg_n": int(len(df)), "reg_ok": False}
    counts = df["event_type"].value_counts()
    if any(counts.get(t, 0) < 10 for t in ["cdhw", "drought_only", "heat_only"]):
        return {"reg_n": int(len(df)), "reg_ok": False}

    x_cont = df[["pre3", "preconditioning_loss", "duration"]].copy()
    x_cont = (x_cont - x_cont.mean()) / x_cont.std().replace(0, 1.0)
    month = df["month"].to_numpy(dtype="float64")
    X = np.column_stack(
        [
            np.ones(len(df)),
            x_cont.to_numpy(dtype="float64"),
            np.sin(2 * np.pi * month / 12.0),
            np.cos(2 * np.pi * month / 12.0),
            (df["event_type"].to_numpy() == "cdhw").astype(float),
            (df["event_type"].to_numpy() == "heat_only").astype(float),
        ]
    )

    def fit(ycol: str, prefix: str) -> dict:
        y = df[ycol].to_numpy(dtype="float64")
        coef, *_ = np.linalg.lstsq(X, y, rcond=None)
        resid = y - X @ coef
        dof = len(y) - X.shape[1]
        if dof <= 0:
            return {}
        sigma2 = float((resid @ resid) / dof)
        xtx_inv = np.linalg.pinv(X.T @ X)
        se = np.sqrt(np.diag(xtx_inv) * sigma2)
        cdhw = coef[-2]
        heat = coef[-1]
        cdhw_t = cdhw / se[-2] if se[-2] > 0 else np.nan
        heat_t = heat / se[-1] if se[-1] > 0 else np.nan
        return {
            f"reg_{prefix}_cdhw_coef": float(cdhw),
            f"reg_{prefix}_cdhw_t": float(cdhw_t),
            f"reg_{prefix}_heat_coef": float(heat),
            f"reg_{prefix}_heat_t": float(heat_t),
            f"reg_{prefix}_cdhw_vs_best_single_coef": float(cdhw - max(0.0, heat)),
        }

    out = {"reg_n": int(len(df)), "reg_ok": True}
    out.update(fit("event_window_loss", "event_window"))
    out.update(fit("event_window_loss_rate", "event_window_rate"))
    out.update(fit("total_loss_pre30_event", "total"))
    out.update(fit("post30_deficit_pre30", "post30"))
    out.update(fit("deficit_auc_pre30_event_post30", "auc"))
    return out


def process_band(lat0: int, lat1: int) -> str:
    part = f"lat_{lat0:03d}_{lat1:03d}"
    out_csv = PARTS / f"{part}.csv.gz"
    done = PARTS / f"{part}.done"
    if done.exists() and out_csv.exists():
        return part

    soil_ds = xr.open_dataset(SOIL_DAILY, decode_times=False)
    mask_ds = xr.open_dataset(MASK_DAILY, decode_times=False)
    domain_ds = xr.open_dataset(DOMAIN_MASK, decode_times=False)
    mask_time = mask_ds["time"].values
    soil_time = soil_ds["time"].values
    start_idx = int(np.searchsorted(soil_time, mask_time[0]))
    end_idx = start_idx + len(mask_time)
    if not np.array_equal(soil_time[start_idx:end_idx], mask_time):
        raise RuntimeError("soil_daily and mask_daily time coordinates are not aligned")
    dates = pd.to_datetime(mask_time, unit="D", origin=pd.Timestamp("1901-01-01"))
    months = dates.month.to_numpy()

    lats = mask_ds["lat"].values[lat0:lat1]
    lons = mask_ds["lon"].values
    domain = domain_ds["mask_data"].isel(lat=slice(lat0, lat1)).values
    valid = domain == 2
    soil = soil_ds["sum_soil"].isel(time=slice(start_idx, end_idx), lat=slice(lat0, lat1)).values
    drought = mask_ds["drought_mask"].isel(lat=slice(lat0, lat1)).values.astype(bool)
    hw = mask_ds["hw_mask"].isel(lat=slice(lat0, lat1)).values.astype(bool)
    cdhw = mask_ds["cdhw_mask"].isel(lat=slice(lat0, lat1)).values.astype(bool)
    drought = np.moveaxis(drought, 2, 0)
    hw = np.moveaxis(hw, 2, 0)
    cdhw = np.moveaxis(cdhw, 2, 0)
    drought_only = drought & (~hw)
    heat_only = hw & (~drought)

    rows = []
    for ii in range(lat1 - lat0):
        for jj in range(len(lons)):
            if not valid[ii, jj]:
                continue
            s = soil[:, ii, jj]
            if not np.isfinite(s).any():
                continue
            cd = extract_event_table(s, cdhw[:, ii, jj], months, "cdhw")
            dr = extract_event_table(s, drought_only[:, ii, jj], months, "drought_only")
            he = extract_event_table(s, heat_only[:, ii, jj], months, "heat_only")
            if cd.empty and dr.empty and he.empty:
                continue
            single = pd.concat([dr, he], ignore_index=True)
            all_events = pd.concat([cd, dr, he], ignore_index=True)
            row = {
                "lat_idx_global": lat0 + ii,
                "lon_idx": jj,
                "lat": float(lats[ii]),
                "lon": float(lons[jj]),
            }
            row.update(summarize_events(cd, "cdhw"))
            row.update(summarize_events(dr, "drought_only"))
            row.update(summarize_events(he, "heat_only"))
            row.update(nearest_match_diffs(cd, single))
            row.update(controlled_regression(all_events))
            rows.append(row)

    if rows:
        pd.DataFrame(rows).to_csv(out_csv, index=False, compression="gzip")
    else:
        out_csv.write_bytes(b"")
    done.write_text(str(len(rows)), encoding="utf-8")
    soil_ds.close()
    mask_ds.close()
    domain_ds.close()
    return part


def add_mm_aliases(df: pd.DataFrame) -> pd.DataFrame:
    scale_cols = [
        col
        for col in df.columns
        if any(key in col for key in ["loss", "deficit", "auc", "coef", "rate"])
        and not col.endswith("_mm")
        and not col.endswith("_mm_day")
        and col not in ["reg_ok"]
    ]
    for col in scale_cols:
        if df[col].dtype.kind in "fiu":
            suffix = "_mm_day" if ("rate" in col or "auc" in col) else "_mm"
            df[f"{col}{suffix}"] = df[col] * 1000.0
    aliases = {
        "cdhw_event_window_loss_mean_mm": "cdhw_event_window_loss_mean_mm",
        "cdhw_event_window_loss_rate_mean_mm_day": "cdhw_event_window_loss_rate_mean_mm_day",
        "cdhw_total_loss_pre30_event_mean_mm": "cdhw_total_loss_mean_mm",
        "cdhw_preconditioning_loss_mean_mm": "cdhw_preconditioning_loss_mean_mm",
        "cdhw_post30_deficit_pre30_mean_mm": "cdhw_post30_deficit_mean_mm",
        "cdhw_deficit_auc_pre30_event_post30_mean_mm_day": "cdhw_deficit_auc_mean_mm_day",
        "matched_event_window_loss_diff_mean_mm": "matched_event_window_excess_mean_mm",
        "matched_event_window_loss_rate_diff_mean_mm_day": "matched_event_window_rate_excess_mean_mm_day",
        "matched_total_loss_diff_mean_mm": "matched_total_excess_mean_mm",
        "matched_post30_diff_mean_mm": "matched_post30_excess_mean_mm",
        "matched_auc_diff_mean_mm_day": "matched_auc_excess_mean_mm_day",
        "reg_event_window_cdhw_vs_best_single_coef_mm": "reg_event_window_excess_mm",
        "reg_event_window_rate_cdhw_vs_best_single_coef_mm_day": "reg_event_window_rate_excess_mm_day",
        "reg_total_cdhw_vs_best_single_coef_mm": "reg_total_excess_mm",
        "reg_post30_cdhw_vs_best_single_coef_mm": "reg_post30_excess_mm",
        "reg_auc_cdhw_vs_best_single_coef_mm_day": "reg_auc_excess_mm_day",
    }
    for src, dst in aliases.items():
        if src in df:
            df[dst] = df[src]
    return df


def grid(df: pd.DataFrame, col: str):
    lats = np.sort(df["lat"].unique())
    lons = np.sort(df["lon"].unique())
    arr = np.full((len(lats), len(lons)), np.nan, dtype="float32")
    lat_index = pd.Series(np.arange(len(lats)), index=lats)
    lon_index = pd.Series(np.arange(len(lons)), index=lons)
    arr[df["lat"].map(lat_index).to_numpy(), df["lon"].map(lon_index).to_numpy()] = df[col].to_numpy()
    return lons, lats, arr


def style_map(ax):
    ax.set_xlim(-180, 180)
    ax.set_ylim(-60, 85)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_xticks([-180, -120, -60, 0, 60, 120, 180])
    ax.set_yticks([-60, -30, 0, 30, 60])
    ax.grid(color="#d0d0d0", linewidth=0.3)


def savefig(fig, name):
    for ext in FORMATS:
        fig.savefig(OUT / f"{name}.{ext}", dpi=320, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def summarize_mask(df: pd.DataFrame, mask: pd.Series, name: str, role: str, denom: int) -> dict:
    sub = df.loc[mask]
    return {
        "class": name,
        "role": role,
        "n_pixels": int(mask.sum()),
        "share_pct_comparable": float(mask.sum() / denom * 100.0),
        "event_window_loss_median_mm": float(sub["cdhw_event_window_loss_mean_mm"].median()) if len(sub) else np.nan,
        "event_window_rate_median_mm_day": float(sub["cdhw_event_window_loss_rate_mean_mm_day"].median()) if len(sub) else np.nan,
        "matched_event_window_excess_median_mm": float(sub["matched_event_window_excess_mean_mm"].median()) if len(sub) else np.nan,
        "reg_event_window_excess_median_mm": float(sub["reg_event_window_excess_mm"].median()) if len(sub) else np.nan,
        "preconditioning_median_mm": float(sub["cdhw_preconditioning_loss_mean_mm"].median()) if len(sub) else np.nan,
        "total_depletion_median_mm": float(sub["cdhw_total_loss_mean_mm"].median()) if len(sub) else np.nan,
        "post30_deficit_median_mm": float(sub["cdhw_post30_deficit_mean_mm"].median()) if len(sub) else np.nan,
        "preconditioning_share_median": float(sub["cdhw_preconditioning_share_mean"].median()) if len(sub) else np.nan,
        "matched_positive_ratio_median": float(sub["matched_positive_event_window_loss_diff_ratio"].median()) if len(sub) else np.nan,
        "cdhw_positive_event_window_loss_ratio_median": float(sub["cdhw_positive_event_window_loss_ratio"].median()) if len(sub) else np.nan,
    }


def build_masks(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    comparable = (
        (df["cdhw_n"] >= MIN_EVENTS)
        & (df["drought_only_n"] >= MIN_EVENTS)
        & (df["heat_only_n"] >= MIN_EVENTS)
        & (df["matched_n"] >= 10)
        & df["reg_ok"].fillna(False).astype(bool)
        & df["cdhw_event_window_loss_mean_mm"].notna()
    )
    df["event_window_comparable"] = comparable
    base = df.loc[comparable].copy()
    thresholds = {
        "event_window_loss_q70_mm": float(base["cdhw_event_window_loss_mean_mm"].quantile(0.70)),
        "event_window_rate_q70_mm_day": float(base["cdhw_event_window_loss_rate_mean_mm_day"].quantile(0.70)),
        "matched_event_window_excess_q70_mm": float(base["matched_event_window_excess_mean_mm"].quantile(0.70)),
        "reg_event_window_excess_q70_mm": float(base["reg_event_window_excess_mm"].quantile(0.70)),
        "total_loss_q70_mm": float(base["cdhw_total_loss_mean_mm"].quantile(0.70)),
        "preconditioning_q70_mm": float(base["cdhw_preconditioning_loss_mean_mm"].quantile(0.70)),
        "post30_q70_mm": float(base["cdhw_post30_deficit_mean_mm"].quantile(0.70)),
        "auc_q70_mm_day": float(base["cdhw_deficit_auc_mean_mm_day"].quantile(0.70)),
    }
    df["A_event_window_specific"] = comparable & (
        (df["cdhw_event_window_loss_mean_mm"] >= thresholds["event_window_loss_q70_mm"])
        & (df["matched_event_window_excess_mean_mm"] > 0)
        & (df["reg_event_window_excess_mm"] > 0)
        & (df["cdhw_positive_event_window_loss_ratio"] >= 0.6)
    )
    df["A_event_window_specific_strict"] = comparable & (
        (df["cdhw_event_window_loss_mean_mm"] >= thresholds["event_window_loss_q70_mm"])
        & (df["cdhw_event_window_loss_rate_mean_mm_day"] >= thresholds["event_window_rate_q70_mm_day"])
        & (df["matched_event_window_excess_mean_mm"] >= thresholds["matched_event_window_excess_q70_mm"])
        & (df["reg_event_window_excess_mm"] >= thresholds["reg_event_window_excess_q70_mm"])
        & (df["cdhw_positive_event_window_loss_ratio"] >= 0.6)
    )
    df["B_preconditioned_cumulative_depletion"] = comparable & (
        (df["cdhw_total_loss_mean_mm"] >= thresholds["total_loss_q70_mm"])
        & (df["cdhw_preconditioning_loss_mean_mm"] >= thresholds["preconditioning_q70_mm"])
        & (df["cdhw_preconditioning_share_mean"] > 0.5)
    )
    df["C_recovery_vulnerable"] = comparable & (
        (df["cdhw_post30_deficit_mean_mm"] >= thresholds["post30_q70_mm"])
        | (df["cdhw_deficit_auc_mean_mm_day"] >= thresholds["auc_q70_mm_day"])
    )
    df["ABC_true_overlap_count"] = df[
        ["A_event_window_specific", "B_preconditioned_cumulative_depletion", "C_recovery_vulnerable"]
    ].fillna(False).astype(bool).sum(axis=1)
    df["ABC_true_type"] = 0
    df.loc[df["C_recovery_vulnerable"], "ABC_true_type"] = 3
    df.loc[df["B_preconditioned_cumulative_depletion"], "ABC_true_type"] = 2
    df.loc[df["A_event_window_specific"], "ABC_true_type"] = 1
    df.loc[df["ABC_true_overlap_count"] >= 2, "ABC_true_type"] = 4
    df.loc[df["ABC_true_overlap_count"] == 3, "ABC_true_type"] = 5
    labels = {
        0: "weak_or_other",
        1: "A_event_window_specific",
        2: "B_preconditioned_cumulative",
        3: "C_recovery_vulnerable",
        4: "multi_process_overlap",
        5: "A_B_C_overlap",
    }
    df["ABC_true_type_label"] = df["ABC_true_type"].map(labels)
    return df, thresholds


def plot_outputs(df: pd.DataFrame, summary: pd.DataFrame, ipcc: pd.DataFrame) -> None:
    fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.4), constrained_layout=True)
    specs = [
        ("cdhw_event_window_loss_mean_mm", "CDHW event-window loss (mm)", "viridis"),
        ("cdhw_event_window_loss_rate_mean_mm_day", "CDHW event-window loss rate (mm/day)", "viridis"),
        ("matched_event_window_excess_mean_mm", "Matched event-window excess (mm)", "coolwarm"),
        ("reg_event_window_excess_mm", "Regression-controlled event-window excess (mm)", "coolwarm"),
        ("cdhw_preconditioning_loss_mean_mm", "Preconditioning loss (mm)", "coolwarm"),
        ("cdhw_post30_deficit_mean_mm", "Post30 deficit from pre30 (mm)", "coolwarm"),
    ]
    for ax, (col, title, cmap) in zip(axes.ravel(), specs):
        lons, lats, arr = grid(df, col)
        vals = df[col].replace([np.inf, -np.inf], np.nan).dropna()
        lo, hi = np.nanpercentile(vals, [2, 98]) if len(vals) else (0, 1)
        if cmap == "coolwarm":
            m = max(abs(lo), abs(hi))
            im = ax.pcolormesh(lons, lats, arr, cmap=cmap, vmin=-m, vmax=m, shading="auto")
        else:
            im = ax.pcolormesh(lons, lats, arr, cmap=cmap, vmin=lo, vmax=hi, shading="auto")
        style_map(ax)
        ax.set_title(title)
        fig.colorbar(im, ax=ax, shrink=0.72)
    savefig(fig, "fig_event_window_specific_components")

    cmap = ListedColormap(["#e7e5df", "#7aa6c2", "#d8a25e", "#4d7c59", "#9b3a5a", "#5e4c8a"])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5], cmap.N)
    fig, ax = plt.subplots(figsize=(10.2, 5.0), constrained_layout=True)
    lons, lats, arr = grid(df, "ABC_true_type")
    ax.pcolormesh(lons, lats, arr, cmap=cmap, norm=norm, shading="auto")
    style_map(ax)
    ax.set_title("Event-window-specific A/B/C response classes")
    handles = [
        Patch(facecolor="#e7e5df", label="0 other"),
        Patch(facecolor="#7aa6c2", label="A event-window specific"),
        Patch(facecolor="#d8a25e", label="B preconditioned cumulative"),
        Patch(facecolor="#4d7c59", label="C recovery vulnerable"),
        Patch(facecolor="#9b3a5a", label="two-process overlap"),
        Patch(facecolor="#5e4c8a", label="A+B+C overlap"),
    ]
    ax.legend(handles=handles, loc="lower left", ncols=3, fontsize=7, frameon=True)
    savefig(fig, "fig_event_window_specific_ABC_map")

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), constrained_layout=True)
    axes[0].barh(summary["class"], summary["share_pct_comparable"], color="#4c78a8")
    axes[0].set_xlabel("Share of comparable pixels (%)")
    axes[0].set_title("Response-class size")
    axes[0].invert_yaxis()
    x = np.arange(len(summary))
    axes[1].bar(x - 0.24, summary["event_window_loss_median_mm"], width=0.24, label="event-window loss")
    axes[1].bar(x, summary["matched_event_window_excess_median_mm"], width=0.24, label="matched excess")
    axes[1].bar(x + 0.24, summary["reg_event_window_excess_median_mm"], width=0.24, label="reg. excess")
    axes[1].axhline(0, color="#777", linestyle="--", linewidth=0.8)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(summary["class"], rotation=30, ha="right")
    axes[1].set_ylabel("Median value (mm)")
    axes[1].set_title("Event-window evidence strength")
    axes[1].legend(fontsize=7)
    savefig(fig, "fig_event_window_specific_summary")

    sample = df.sample(min(35000, len(df)), random_state=20260520)
    core = sample["A_event_window_specific"].fillna(False).astype(bool)
    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.8), constrained_layout=True)
    axes[0].scatter(sample["matched_event_window_excess_mean_mm"], sample["reg_event_window_excess_mm"], s=3, c="#bdbdbd", alpha=0.35, linewidth=0)
    axes[0].scatter(sample.loc[core, "matched_event_window_excess_mean_mm"], sample.loc[core, "reg_event_window_excess_mm"], s=6, c="#5e4c8a", alpha=0.65, linewidth=0)
    axes[0].axvline(0, color="#777", linestyle="--", linewidth=0.8)
    axes[0].axhline(0, color="#777", linestyle="--", linewidth=0.8)
    axes[0].set_xlabel("Matched event-window excess (mm)")
    axes[0].set_ylabel("Regression-controlled event-window excess (mm)")
    axes[0].set_title("A-class evidence space")
    axes[1].scatter(sample["cdhw_preconditioning_loss_mean_mm"], sample["cdhw_event_window_loss_mean_mm"], s=3, c="#bdbdbd", alpha=0.35, linewidth=0)
    axes[1].scatter(sample.loc[core, "cdhw_preconditioning_loss_mean_mm"], sample.loc[core, "cdhw_event_window_loss_mean_mm"], s=6, c="#5e4c8a", alpha=0.65, linewidth=0)
    axes[1].axhline(0, color="#777", linestyle="--", linewidth=0.8)
    axes[1].axvline(0, color="#777", linestyle="--", linewidth=0.8)
    axes[1].set_xlabel("Preconditioning loss, pre30-pre3 (mm)")
    axes[1].set_ylabel("Event-window loss, pre3-event (mm)")
    axes[1].set_title("A class separates event loss from preconditioning")
    savefig(fig, "fig_event_window_specific_evidence_space")

    top = ipcc[ipcc["valid_pixels"] >= 200].sort_values("A_event_window_specific_pixels", ascending=False).head(18).iloc[::-1]
    fig, axes = plt.subplots(1, 2, figsize=(10.8, 6.2), constrained_layout=True)
    axes[0].barh(top["ipcc_acronym"], top["A_event_window_specific_pixels"], color="#5e4c8a")
    axes[0].set_xlabel("A event-window-specific pixels")
    axes[0].set_title("Regional contributions")
    sc = axes[1].scatter(
        top["A_event_window_specific_share_pct"],
        top["A_event_window_specific_event_window_loss_median_mm"],
        s=np.clip(top["A_event_window_specific_pixels"] / 3, 30, 250),
        c=top["A_event_window_specific_matched_event_window_excess_median_mm"],
        cmap="viridis",
        edgecolor="white",
        linewidth=0.5,
    )
    for _, r in top.iterrows():
        axes[1].text(
            r["A_event_window_specific_share_pct"] + 0.35,
            r["A_event_window_specific_event_window_loss_median_mm"],
            r["ipcc_acronym"],
            fontsize=6,
        )
    axes[1].set_xlabel("A share in region (%)")
    axes[1].set_ylabel("Median event-window loss in A (mm)")
    axes[1].set_title("Extent versus event-window loss")
    cb = fig.colorbar(sc, ax=axes[1])
    cb.set_label("Matched event-window excess (mm)")
    savefig(fig, "fig_event_window_specific_ipcc_patterns")


def main() -> None:
    tasks = [(i, min(i + LAT_BAND_SIZE, 360)) for i in range(0, 360, LAT_BAND_SIZE)]
    with ProcessPoolExecutor(max_workers=WORKERS) as ex:
        futures = [ex.submit(process_band, a, b) for a, b in tasks]
        for fut in as_completed(futures):
            print("finished", fut.result(), flush=True)

    frames = []
    for p in sorted(PARTS.glob("lat_*_*.csv.gz")):
        try:
            part = pd.read_csv(p)
        except EmptyDataError:
            continue
        if len(part):
            frames.append(part)
    if not frames:
        raise RuntimeError("No non-empty parts.")
    df = pd.concat(frames, ignore_index=True)
    df = add_mm_aliases(df)
    if IPCC_SOURCE.exists():
        ipcc = pd.read_csv(IPCC_SOURCE, usecols=["lat", "lon", "ipcc_acronym", "ipcc_name", "ipcc_type"])
        df = df.merge(ipcc.drop_duplicates(["lat", "lon"]), on=["lat", "lon"], how="left")
    else:
        df["ipcc_acronym"] = np.nan
        df["ipcc_name"] = np.nan
        df["ipcc_type"] = np.nan

    df, thresholds = build_masks(df)
    comparable_n = int(df["event_window_comparable"].sum())
    classes = [
        ("A_event_window_specific", "primary CDHW event-window response-sensitive candidate"),
        ("A_event_window_specific_strict", "strict event-window-specific core"),
        ("B_preconditioned_cumulative_depletion", "preconditioning-dominated cumulative depletion"),
        ("C_recovery_vulnerable", "post-event persistent deficit or high cumulative deficit"),
        ("ABC_true_overlap_count_ge2", "overlap of at least two A/B/C mechanisms"),
        ("ABC_true_overlap_count_eq3", "overlap of all three A/B/C mechanisms"),
    ]
    rows = []
    for name, role in classes:
        if name == "ABC_true_overlap_count_ge2":
            mask = df["ABC_true_overlap_count"] >= 2
        elif name == "ABC_true_overlap_count_eq3":
            mask = df["ABC_true_overlap_count"] == 3
        else:
            mask = df[name].fillna(False).astype(bool)
        rows.append(summarize_mask(df, mask, name, role, comparable_n))
    summary = pd.DataFrame(rows)
    summary.to_csv(OUT / "event_window_specific_class_summary.csv", index=False)

    ipcc_rows = []
    for keys, sub in df.groupby(["ipcc_acronym", "ipcc_name", "ipcc_type"], dropna=False):
        row = {"ipcc_acronym": keys[0], "ipcc_name": keys[1], "ipcc_type": keys[2], "valid_pixels": int(len(sub))}
        for col in ["A_event_window_specific", "A_event_window_specific_strict", "B_preconditioned_cumulative_depletion", "C_recovery_vulnerable"]:
            m = sub[col].fillna(False).astype(bool)
            row[f"{col}_pixels"] = int(m.sum())
            row[f"{col}_share_pct"] = float(m.mean() * 100.0)
            row[f"{col}_event_window_loss_median_mm"] = float(sub.loc[m, "cdhw_event_window_loss_mean_mm"].median()) if m.any() else np.nan
            row[f"{col}_matched_event_window_excess_median_mm"] = float(sub.loc[m, "matched_event_window_excess_mean_mm"].median()) if m.any() else np.nan
            row[f"{col}_reg_event_window_excess_median_mm"] = float(sub.loc[m, "reg_event_window_excess_mm"].median()) if m.any() else np.nan
        ipcc_rows.append(row)
    ipcc_summary = pd.DataFrame(ipcc_rows)
    ipcc_summary.to_csv(OUT / "event_window_specific_ipcc_summary.csv", index=False)

    df.to_csv(OUT / "event_window_specific_pixel_metrics.csv.gz", index=False, compression="gzip")

    ds_vars = {}
    for col in [
        "cdhw_event_window_loss_mean_mm",
        "cdhw_event_window_loss_rate_mean_mm_day",
        "matched_event_window_excess_mean_mm",
        "reg_event_window_excess_mm",
        "A_event_window_specific",
        "A_event_window_specific_strict",
        "B_preconditioned_cumulative_depletion",
        "C_recovery_vulnerable",
        "ABC_true_overlap_count",
        "ABC_true_type",
    ]:
        lons, lats, arr = grid(df, col)
        ds_vars[col] = xr.DataArray(arr, coords={"lat": lats, "lon": lons}, dims=("lat", "lon"))
    xr.Dataset(ds_vars, attrs={"title": "Event-window-specific CDHW response-sensitive framework"}).to_netcdf(
        OUT / "event_window_specific_products.nc"
    )

    with open(OUT / "event_window_specific_thresholds.json", "w", encoding="utf-8") as f:
        json.dump(thresholds, f, indent=2, ensure_ascii=False)

    plot_outputs(df, summary, ipcc_summary)

    report = f"""# Event-window-specific CDHW response-sensitive framework

## Purpose

This framework turns the provisional A class into a true event-window definition.
It asks whether CDHWs produce elevated soil-moisture losses during the event window
(`pre3 -> event`) after matching or controlling for antecedent water state,
preconditioning loss, event duration, and seasonality.

## Comparable sample

- Comparable pixels: {comparable_n:,}
- Minimum events per class: {MIN_EVENTS}
- Minimum matched CDHW events: 10

## Thresholds

{json.dumps(thresholds, indent=2)}

## Classes

- A: CDHW event-window response-sensitive candidate.
- A strict: conservative event-window-specific core.
- B: preconditioned cumulative-depletion region.
- C: recovery-vulnerable region.

## Summary

{summary.to_markdown(index=False)}

## Interpretation

Use `A_event_window_specific` as the main CDHW response-sensitive candidate if the
claim is event-window soil-moisture response to compound events. Use B and C as
separate mechanism classes. Do not merge A, B, and C into a single sensitive-region
definition unless the manuscript explicitly defines a broader cumulative-vulnerability
product.

## Outputs

- `event_window_specific_pixel_metrics.csv.gz`
- `event_window_specific_class_summary.csv`
- `event_window_specific_ipcc_summary.csv`
- `event_window_specific_products.nc`
- `event_window_specific_thresholds.json`
- `fig_event_window_specific_components.*`
- `fig_event_window_specific_ABC_map.*`
- `fig_event_window_specific_summary.*`
- `fig_event_window_specific_evidence_space.*`
- `fig_event_window_specific_ipcc_patterns.*`
"""
    (OUT / "EVENT_WINDOW_SPECIFIC_FRAMEWORK_REPORT.md").write_text(report, encoding="utf-8")
    print(json.dumps({"out": str(OUT), "comparable_pixels": comparable_n, "thresholds": thresholds}, indent=2))


if __name__ == "__main__":
    main()


### compute_A_mask_robustness

Supplementary C S8 robustness and threshold audit. Threshold perturbation, alternative definitions and spatial stability checks.




In [ ]:
# Source file: D:\smq\paper2\modify\敏感区\code\compute_A_mask_robustness.py
# Staging copy used for notebook assembly: D:\software\codex\paper2\office_work\code_notebook_order_20260609\sensitive_code\compute_A_mask_robustness.py
# Detected encoding: utf-8-sig
# Methods-order section: Supplementary C S8 robustness and threshold audit
# NOTE: This is the final modified sensitive-region/SHAP script, not legacy notebook code.

from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


ROOT = Path(r"D:\Software\codex\sm_heat_drought")
OUT = ROOT / "outputs" / "20260522_improved_framework_word"
PKG = Path(r"E:\smq\aa\敏感区")
EVT_PIX = ROOT / "outputs" / "20260520_event_window_specific_framework" / "event_window_specific_pixel_metrics.csv.gz"
FB_PIX = ROOT / "outputs" / "20260520_feedback_sensitive_framework" / "feedback_sensitive_pixel_metrics.csv.gz"


def ensure_dirs() -> None:
    OUT.mkdir(parents=True, exist_ok=True)
    for sub in ["data", "figures", "code", "reports"]:
        (PKG / sub).mkdir(parents=True, exist_ok=True)


def overlap_stats(mask: pd.Series | np.ndarray, ref: pd.Series | np.ndarray) -> dict:
    m = np.asarray(mask, dtype=bool)
    r = np.asarray(ref, dtype=bool)
    inter = int((m & r).sum())
    union = int((m | r).sum())
    n_m = int(m.sum())
    n_r = int(r.sum())
    return {
        "pixels": n_m,
        "intersection_with_main": inter,
        "jaccard_vs_main": inter / union if union else np.nan,
        "dice_vs_main": (2 * inter) / (n_m + n_r) if (n_m + n_r) else np.nan,
        "main_retention_pct": inter / n_r * 100 if n_r else np.nan,
        "candidate_overlap_pct": inter / n_m * 100 if n_m else np.nan,
    }


def build_base(df: pd.DataFrame) -> pd.Series:
    comparable = (
        df["event_window_comparable"].fillna(False).astype(bool)
        & df["reg_ok"].fillna(False).astype(bool)
        & df["cdhw_event_window_loss_mean_mm"].notna()
    )
    return comparable


def make_mask(
    df: pd.DataFrame,
    comparable: pd.Series,
    loss_q: float = 0.70,
    plr_thr: float = 0.60,
    me_rule: str = "positive",
    re_rule: str = "positive",
    rate_q: float | None = None,
) -> tuple[pd.Series, dict]:
    base = df.loc[comparable]
    thresholds = {
        "loss_threshold_mm": float(base["cdhw_event_window_loss_mean_mm"].quantile(loss_q)),
        "rate_threshold_mm_day": np.nan if rate_q is None else float(base["cdhw_event_window_loss_rate_mean_mm_day"].quantile(rate_q)),
        "me_q60_mm": float(base["matched_event_window_excess_mean_mm"].quantile(0.60)),
        "me_q70_mm": float(base["matched_event_window_excess_mean_mm"].quantile(0.70)),
        "re_q60_mm": float(base["reg_event_window_excess_mm"].quantile(0.60)),
        "re_q70_mm": float(base["reg_event_window_excess_mm"].quantile(0.70)),
    }
    mask = comparable.copy()
    mask &= df["cdhw_event_window_loss_mean_mm"] >= thresholds["loss_threshold_mm"]
    if rate_q is not None:
        mask &= df["cdhw_event_window_loss_rate_mean_mm_day"] >= thresholds["rate_threshold_mm_day"]
    if me_rule == "positive":
        mask &= df["matched_event_window_excess_mean_mm"] > 0
    elif me_rule == "q60":
        mask &= df["matched_event_window_excess_mean_mm"] >= thresholds["me_q60_mm"]
    elif me_rule == "q70":
        mask &= df["matched_event_window_excess_mean_mm"] >= thresholds["me_q70_mm"]
    else:
        raise ValueError(me_rule)
    if re_rule == "positive":
        mask &= df["reg_event_window_excess_mm"] > 0
    elif re_rule == "q60":
        mask &= df["reg_event_window_excess_mm"] >= thresholds["re_q60_mm"]
    elif re_rule == "q70":
        mask &= df["reg_event_window_excess_mm"] >= thresholds["re_q70_mm"]
    else:
        raise ValueError(re_rule)
    mask &= df["cdhw_positive_event_window_loss_ratio"] >= plr_thr
    return mask.fillna(False), thresholds


def classify_stability(df: pd.DataFrame, comparable: pd.Series, masks: dict[str, pd.Series]) -> pd.Series:
    # Conservative stability classes used only for visualization.
    count = np.zeros(len(df), dtype=np.int16)
    for m in masks.values():
        count += np.asarray(m, dtype=bool)
    cls = np.zeros(len(df), dtype=np.int8)
    cls[np.asarray(comparable, dtype=bool) & (count > 0)] = 1
    cls[count >= 3] = 2
    cls[count >= 8] = 3
    cls[np.asarray(masks["Q70_PLR0.6"], dtype=bool)] = 4
    return pd.Series(cls, index=df.index)


def grid_map(df: pd.DataFrame, values: pd.Series) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    lats = np.sort(df["lat"].dropna().unique())
    lons = np.sort(df["lon"].dropna().unique())
    arr = np.full((len(lats), len(lons)), np.nan, dtype=float)
    lat_idx = pd.Series(np.arange(len(lats)), index=lats)
    lon_idx = pd.Series(np.arange(len(lons)), index=lons)
    ok = df["lat"].notna() & df["lon"].notna()
    arr[df.loc[ok, "lat"].map(lat_idx).to_numpy(), df.loc[ok, "lon"].map(lon_idx).to_numpy()] = values.loc[ok].to_numpy()
    return lons, lats, arr


def draw_robustness_figure(grid_tbl: pd.DataFrame, variants: pd.DataFrame, df: pd.DataFrame, stability: pd.Series) -> None:
    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 7,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    })
    fig, axes = plt.subplots(2, 2, figsize=(10.6, 6.6), constrained_layout=True)

    # a: area heatmap
    piv_area = grid_tbl.pivot(index="plr_threshold", columns="loss_quantile", values="share_pct_comparable").sort_index(ascending=False)
    im = axes[0, 0].imshow(piv_area.values, cmap="Blues", vmin=0, vmax=max(35, np.nanmax(piv_area.values)))
    axes[0, 0].set_xticks(range(len(piv_area.columns)), [f"Q{int(c*100)}" for c in piv_area.columns])
    axes[0, 0].set_yticks(range(len(piv_area.index)), [f"{v:.1f}" for v in piv_area.index])
    axes[0, 0].set_xlabel("Loss quantile threshold")
    axes[0, 0].set_ylabel("Positive-loss ratio threshold")
    axes[0, 0].set_title("a  Mapped A area (%)")
    for i in range(piv_area.shape[0]):
        for j in range(piv_area.shape[1]):
            axes[0, 0].text(j, i, f"{piv_area.values[i, j]:.1f}", ha="center", va="center", fontsize=6)
    fig.colorbar(im, ax=axes[0, 0], fraction=0.046, pad=0.03, label="% comparable pixels")

    # b: Jaccard heatmap
    piv_j = grid_tbl.pivot(index="plr_threshold", columns="loss_quantile", values="jaccard_vs_main").sort_index(ascending=False)
    im2 = axes[0, 1].imshow(piv_j.values, cmap="YlGnBu", vmin=0, vmax=1)
    axes[0, 1].set_xticks(range(len(piv_j.columns)), [f"Q{int(c*100)}" for c in piv_j.columns])
    axes[0, 1].set_yticks(range(len(piv_j.index)), [f"{v:.1f}" for v in piv_j.index])
    axes[0, 1].set_xlabel("Loss quantile threshold")
    axes[0, 1].set_ylabel("Positive-loss ratio threshold")
    axes[0, 1].set_title("b  Spatial agreement with main A")
    for i in range(piv_j.shape[0]):
        for j in range(piv_j.shape[1]):
            axes[0, 1].text(j, i, f"{piv_j.values[i, j]:.2f}", ha="center", va="center", fontsize=6)
    fig.colorbar(im2, ax=axes[0, 1], fraction=0.046, pad=0.03, label="Jaccard")

    # c: alternative stricter controls
    v = variants.sort_values("pixels")
    axes[1, 0].barh(v["variant"], v["share_pct_comparable"], color="#6C8DB5")
    axes[1, 0].set_xlabel("% comparable pixels")
    axes[1, 0].set_title("c  Alternative A definitions")
    for y, x in enumerate(v["share_pct_comparable"]):
        axes[1, 0].text(x + 0.35, y, f"{x:.1f}%", va="center", fontsize=6)

    # d: spatial stability categories
    from matplotlib.colors import ListedColormap, BoundaryNorm
    lons, lats, arr = grid_map(df, stability)
    cmap = ListedColormap(["#FFFFFF", "#C9CDD1", "#A9CDE2", "#4D88B8", "#153E66"])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)
    axes[1, 1].pcolormesh(lons, lats, arr, cmap=cmap, norm=norm, shading="nearest")
    axes[1, 1].set_xlim(-180, 180)
    axes[1, 1].set_ylim(-60, 85)
    axes[1, 1].set_xlabel("Longitude")
    axes[1, 1].set_ylabel("Latitude")
    axes[1, 1].set_title("d  Threshold-stability footprint")
    axes[1, 1].grid(color="#dddddd", lw=0.25)
    handles = [
        plt.Line2D([0], [0], marker="s", color="none", markerfacecolor=c, markersize=7, label=l)
        for c, l in zip(cmap.colors[1:], ["selected by few alternatives", "selected by >=3 alternatives", "selected by >=8 alternatives", "main A"])
    ]
    axes[1, 1].legend(handles=handles, loc="lower left", fontsize=5.5, frameon=False)

    for ext, dpi in [("png", 600), ("tiff", 600), ("pdf", None), ("svg", None)]:
        fig.savefig(OUT / f"fig_A_mask_robustness_audit.{ext}", dpi=dpi if dpi else None, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def trigger_lag_robustness(main_a: pd.Series) -> pd.DataFrame:
    fb = pd.read_csv(FB_PIX)
    evt_keys = pd.read_csv(EVT_PIX, usecols=["lat_idx_global", "lon_idx", "A_event_window_specific"])
    df = fb.merge(evt_keys, on=["lat_idx_global", "lon_idx",], how="left")
    ref = df["A_event_window_specific"].fillna(False).astype(bool)
    rows = []
    for lag in [3, 7, 14, 30]:
        rr = df[f"asw_pre{lag}_rr"]
        rd_pct = df[f"asw_pre{lag}_rd_pct"]
        # For lag 7, use the bootstrap-supported primary flag when available.
        if lag == 7:
            m = df["feedback_sensitive_primary"].fillna(False).astype(bool) & (rd_pct > 0.5) & (rr > 1)
        else:
            m = df[f"asw_pre{lag}_rr_gt1"].fillna(False).astype(bool) & (rd_pct > 0.5) & (rr > 1)
        stats = overlap_stats(m, ref)
        stats.update({
            "lag_days": lag,
            "median_rr": float(rr[m].median()) if m.sum() else np.nan,
            "median_rd_pct": float(rd_pct[m].median()) if m.sum() else np.nan,
        })
        rows.append(stats)
    return pd.DataFrame(rows)


def main() -> None:
    ensure_dirs()
    cols = [
        "lat_idx_global", "lon_idx", "lat", "lon", "cdhw_n", "drought_only_n", "heat_only_n",
        "matched_n", "reg_ok", "event_window_comparable", "A_event_window_specific",
        "A_event_window_specific_strict", "cdhw_event_window_loss_mean_mm",
        "cdhw_event_window_loss_rate_mean_mm_day", "matched_event_window_excess_mean_mm",
        "reg_event_window_excess_mm", "cdhw_positive_event_window_loss_ratio",
    ]
    df = pd.read_csv(EVT_PIX, usecols=cols)
    comparable = build_base(df)
    main_mask, main_thr = make_mask(df, comparable, 0.70, 0.60)
    original = df["A_event_window_specific"].fillna(False).astype(bool)
    if int(main_mask.sum()) != int(original.sum()) or int((main_mask ^ original).sum()) != 0:
        raise RuntimeError(
            f"Main A reproduction failed: rebuilt={int(main_mask.sum())}, original={int(original.sum())}, diff={int((main_mask ^ original).sum())}"
        )
    comparable_n = int(comparable.sum())

    rows = []
    masks = {}
    for q in [0.60, 0.65, 0.70, 0.75, 0.80]:
        for plr in [0.50, 0.60, 0.70]:
            m, thr = make_mask(df, comparable, q, plr)
            name = f"Q{int(q*100)}_PLR{plr:.1f}"
            masks[name] = m
            row = {
                "setting": name,
                "loss_quantile": q,
                "plr_threshold": plr,
                "loss_threshold_mm": thr["loss_threshold_mm"],
                "share_pct_comparable": float(m.sum() / comparable_n * 100),
            }
            row.update(overlap_stats(m, main_mask))
            rows.append(row)
    grid_tbl = pd.DataFrame(rows)
    grid_tbl.to_csv(OUT / "A_mask_threshold_robustness_grid.csv", index=False, encoding="utf-8-sig")

    variant_specs = [
        ("main_Q70_PLR0.6", dict(loss_q=0.70, plr_thr=0.60, me_rule="positive", re_rule="positive", rate_q=None)),
        ("loose_Q60_PLR0.5", dict(loss_q=0.60, plr_thr=0.50, me_rule="positive", re_rule="positive", rate_q=None)),
        ("conservative_Q80_PLR0.7", dict(loss_q=0.80, plr_thr=0.70, me_rule="positive", re_rule="positive", rate_q=None)),
        ("add_rate_Q70", dict(loss_q=0.70, plr_thr=0.60, me_rule="positive", re_rule="positive", rate_q=0.70)),
        ("ME_RE_Q60", dict(loss_q=0.70, plr_thr=0.60, me_rule="q60", re_rule="q60", rate_q=None)),
        ("strict_core_like", dict(loss_q=0.70, plr_thr=0.60, me_rule="q70", re_rule="q70", rate_q=0.70)),
    ]
    variant_rows = []
    for name, kwargs in variant_specs:
        m, thr = make_mask(df, comparable, **kwargs)
        row = {
            "variant": name,
            "share_pct_comparable": float(m.sum() / comparable_n * 100),
            "loss_threshold_mm": thr["loss_threshold_mm"],
            "rate_threshold_mm_day": thr["rate_threshold_mm_day"],
            "me_rule": kwargs["me_rule"],
            "re_rule": kwargs["re_rule"],
        }
        row.update(overlap_stats(m, main_mask))
        variant_rows.append(row)
    variants = pd.DataFrame(variant_rows)
    variants.to_csv(OUT / "A_mask_alternative_definition_robustness.csv", index=False, encoding="utf-8-sig")

    stability = classify_stability(df, comparable, masks)
    stability_tbl = pd.DataFrame({
        "stability_class": [0, 1, 2, 3, 4],
        "description": ["not selected", "selected by few alternatives", "selected by >=3 alternatives", "selected by >=8 alternatives", "main A"],
        "pixels": [(stability == k).sum() for k in [0, 1, 2, 3, 4]],
    })
    stability_tbl["share_pct_comparable"] = stability_tbl["pixels"] / comparable_n * 100
    stability_tbl.to_csv(OUT / "A_mask_threshold_stability_classes.csv", index=False, encoding="utf-8-sig")

    trig = trigger_lag_robustness(main_mask)
    trig.to_csv(OUT / "trigger_lag_overlap_robustness_with_A.csv", index=False, encoding="utf-8-sig")

    draw_robustness_figure(grid_tbl, variants, df, stability)

    summary = {
        "comparable_pixels": comparable_n,
        "main_A_pixels": int(main_mask.sum()),
        "main_A_share_pct": float(main_mask.sum() / comparable_n * 100),
        "main_loss_threshold_q70_mm": main_thr["loss_threshold_mm"],
        "main_reproduced_original_A": True,
        "strict_core_like_pixels": int(variants.loc[variants["variant"] == "strict_core_like", "pixels"].iloc[0]),
        "strict_core_like_share_pct": float(variants.loc[variants["variant"] == "strict_core_like", "share_pct_comparable"].iloc[0]),
        "q60_plr05_pixels": int(grid_tbl.loc[grid_tbl["setting"] == "Q60_PLR0.5", "pixels"].iloc[0]),
        "q80_plr07_pixels": int(grid_tbl.loc[grid_tbl["setting"] == "Q80_PLR0.7", "pixels"].iloc[0]),
    }
    pd.Series(summary).to_csv(OUT / "A_mask_robustness_summary.csv", header=["value"], encoding="utf-8-sig")

    # Copy outputs into package.
    for fn in [
        "A_mask_threshold_robustness_grid.csv",
        "A_mask_alternative_definition_robustness.csv",
        "A_mask_threshold_stability_classes.csv",
        "trigger_lag_overlap_robustness_with_A.csv",
        "A_mask_robustness_summary.csv",
    ]:
        (PKG / "data" / fn).write_bytes((OUT / fn).read_bytes())
    for ext in ["png", "tiff", "pdf", "svg"]:
        (PKG / "figures" / f"fig_A_mask_robustness_audit.{ext}").write_bytes((OUT / f"fig_A_mask_robustness_audit.{ext}").read_bytes())
    (PKG / "code" / "compute_A_mask_robustness.py").write_bytes((OUT / "compute_A_mask_robustness.py").read_bytes())

    print(pd.Series(summary).to_string())
    print("\nThreshold grid:")
    print(grid_tbl[["setting", "pixels", "share_pct_comparable", "jaccard_vs_main", "main_retention_pct"]].to_string(index=False))
    print("\nVariants:")
    print(variants[["variant", "pixels", "share_pct_comparable", "jaccard_vs_main", "main_retention_pct"]].to_string(index=False))
    print("\nTrigger lag:")
    print(trig[["lag_days", "pixels", "jaccard_vs_main", "median_rr", "median_rd_pct"]].to_string(index=False))


if __name__ == "__main__":
    main()


### extract_asw_event_samples_for_metric_bias

Supplementary C S8 metric-bias event sample extraction. Extracts event samples for absolute/fractional soil moisture and ASM diagnostics.



In [ ]:
# Source file: D:\smq\paper2\modify\敏感区\code\extract_asw_event_samples_for_metric_bias.py
# Staging copy used for notebook assembly: D:\software\codex\paper2\office_work\code_notebook_order_20260609\sensitive_code\extract_asw_event_samples_for_metric_bias.py
# Detected encoding: utf-8-sig
# Methods-order section: Supplementary C S8 metric-bias event sample extraction
# NOTE: This is the final modified sensitive-region/SHAP script, not legacy notebook code.

from __future__ import annotations

import argparse
import concurrent.futures as cf
import json
import os
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from pandas.errors import EmptyDataError


@dataclass
class Config:
    w_daily: str = r"E:\smq\w_daily.nc"
    mask_daily: str = r"E:\smq\masks_cdhw_hw_di.nc"
    domain_mask: str = r"E:\smq\modified_mask.nc"
    out_dir: str = r"D:\Software\codex\sm_heat_drought\outputs\20260521_response_pattern_framework_redesign\asw_event_samples"
    start: str = "1920-01-01"
    end: str = "2019-12-31"
    pre_days: int = 3
    eps_asw: float = 0.05
    workers: int = 6
    lat_band_size: int = 8
    sample_per_band: int = 30000


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def contiguous_runs(flag: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    padded = np.r_[False, flag, False]
    starts = np.where((~padded[:-1]) & padded[1:])[0]
    ends = np.where(padded[:-1] & (~padded[1:]))[0] - 1
    return starts, ends


def pixel_asw_events(
    asw: np.ndarray,
    flag: np.ndarray,
    pre_days: int,
    eps_asw: float,
) -> list[tuple[float, float, float, float]]:
    valid = np.isfinite(asw)
    flag = (flag > 0) & valid
    if not flag.any():
        return []
    starts, ends = contiguous_runs(flag)
    rows: list[tuple[float, float, float, float]] = []
    for s, e in zip(starts, ends):
        if s < pre_days:
            continue
        pre = slice(s - pre_days, s)
        evt = slice(s, e + 1)
        if not valid[pre].all() or not valid[evt].any():
            continue
        asw_pre = float(np.nanmean(asw[pre]))
        asw_event = float(np.nanmean(asw[evt]))
        if not np.isfinite(asw_pre + asw_event):
            continue
        asw_loss_raw = asw_pre - asw_event
        asw_loss = max(asw_loss_raw, 0.0)
        asw_loss_frac = max(asw_loss_raw / max(abs(asw_pre), eps_asw), 0.0)
        rows.append((asw_pre, asw_event, asw_loss, asw_loss_frac))
    return rows


def process_band(task: tuple[int, int, Config]) -> dict:
    start_i, end_i, cfg = task
    out = Path(cfg.out_dir)
    part_dir = out / "parts"
    ensure_dir(part_dir)
    part_name = f"lat_{start_i:03d}_{end_i:03d}"
    out_csv = part_dir / f"{part_name}_asw_event_samples.csv.gz"
    done_file = part_dir / f"{part_name}.done"
    if done_file.exists() and out_csv.exists():
        return {"part": part_name, "skipped": True}

    w_ds = xr.open_dataset(cfg.w_daily, engine="h5netcdf", chunks={})
    mask_ds = xr.open_dataset(cfg.mask_daily, engine="h5netcdf", chunks={})
    domain_ds = xr.open_dataset(cfg.domain_mask, engine="h5netcdf", chunks={})

    w = w_ds["w"].isel(lat=slice(start_i, end_i)).sel(time=slice(cfg.start, cfg.end))
    cdhw = mask_ds["cdhw_mask"].isel(lat=slice(start_i, end_i)).sel(time=slice(cfg.start, cfg.end)).transpose("time", "lat", "lon")
    w, cdhw = xr.align(w, cdhw, join="inner")
    domain = domain_ds["mask_data"].isel(lat=slice(start_i, end_i)).interp_like(w.isel(time=0), method="nearest")

    asw_arr = w.load().to_numpy()
    flag_arr = cdhw.load().to_numpy()
    domain_arr = domain.load().to_numpy()

    sample_rows: list[tuple[float, float, float, float]] = []
    event_count = 0
    pixel_count = 0
    for yi in range(asw_arr.shape[1]):
        for xi in range(asw_arr.shape[2]):
            if not np.isfinite(domain_arr[yi, xi]) or domain_arr[yi, xi] <= 1:
                continue
            rows = pixel_asw_events(asw_arr[:, yi, xi], flag_arr[:, yi, xi], cfg.pre_days, cfg.eps_asw)
            if not rows:
                continue
            pixel_count += 1
            event_count += len(rows)
            sample_rows.extend(rows)

    if sample_rows:
        sample_df = pd.DataFrame(sample_rows, columns=["asw_pre3", "asw_event_mean", "asw_loss", "asw_loss_frac"])
        if len(sample_df) > cfg.sample_per_band:
            sample_df = sample_df.sample(cfg.sample_per_band, random_state=20260521)
        sample_df.to_csv(out_csv, index=False, compression="gzip")
    else:
        pd.DataFrame(columns=["asw_pre3", "asw_event_mean", "asw_loss", "asw_loss_frac"]).to_csv(out_csv, index=False, compression="gzip")

    done_file.write_text("done\n", encoding="utf-8")
    return {"part": part_name, "skipped": False, "pixels": pixel_count, "events": event_count, "samples": min(len(sample_rows), cfg.sample_per_band)}


def summarize(cfg: Config) -> None:
    out = Path(cfg.out_dir)
    files = sorted((out / "parts").glob("*_asw_event_samples.csv.gz"))
    frames = []
    for f in files:
        try:
            frames.append(pd.read_csv(f))
        except EmptyDataError:
            continue
    samples = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["asw_pre3", "asw_event_mean", "asw_loss", "asw_loss_frac"])
    samples.to_csv(out / "asw_event_samples_for_metric_bias.csv.gz", index=False, compression="gzip")
    summary = {
        "n_samples": int(len(samples)),
        "asw_pre3_min": float(samples["asw_pre3"].min()) if len(samples) else None,
        "asw_pre3_max": float(samples["asw_pre3"].max()) if len(samples) else None,
        "asw_loss_mean": float(samples["asw_loss"].mean()) if len(samples) else None,
        "asw_loss_frac_mean": float(samples["asw_loss_frac"].mean()) if len(samples) else None,
    }
    (out / "asw_event_samples_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False, indent=2))


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--summarize-only", action="store_true")
    parser.add_argument("--workers", type=int, default=None)
    args = parser.parse_args()

    cfg = Config()
    if args.workers:
        cfg.workers = args.workers

    out = Path(cfg.out_dir)
    ensure_dir(out)
    ensure_dir(out / "parts")
    (out / "config.json").write_text(json.dumps(asdict(cfg), ensure_ascii=False, indent=2), encoding="utf-8")

    if not args.summarize_only:
        with xr.open_dataset(cfg.w_daily, engine="h5netcdf", chunks={}) as ds:
            n_lat = ds.sizes["lat"]
        tasks = [(i, min(i + cfg.lat_band_size, n_lat), cfg) for i in range(0, n_lat, cfg.lat_band_size)]
        results = []
        with cf.ProcessPoolExecutor(max_workers=cfg.workers) as ex:
            futs = [ex.submit(process_band, t) for t in tasks]
            for fut in cf.as_completed(futs):
                res = fut.result()
                results.append(res)
                print(json.dumps(res), flush=True)
        pd.DataFrame(results).to_csv(out / "asw_sample_part_run_log.csv", index=False, encoding="utf-8-sig")
    summarize(cfg)


if __name__ == "__main__":
    os.environ.setdefault("OMP_NUM_THREADS", "1")
    os.environ.setdefault("MKL_NUM_THREADS", "1")
    main()


### analyze_event_level_critical_window

Supplementary C S8 Fig. S6 antecedent water-state diagnostic. Event-level antecedent ASM/soil moisture versus event-window loss.



In [ ]:
# Source file: D:\smq\paper2\modify\敏感区\code\analyze_event_level_critical_window.py
# Staging copy used for notebook assembly: D:\software\codex\paper2\office_work\code_notebook_order_20260609\sensitive_code\analyze_event_level_critical_window.py
# Detected encoding: utf-8-sig
# Methods-order section: Supplementary C S8 Fig. S6 antecedent water-state diagnostic
# NOTE: This is the final modified sensitive-region/SHAP script, not legacy notebook code.

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt


ROOT = Path(r"D:\Software\codex\sm_heat_drought")
OUT = ROOT / "outputs" / "20260521_response_pattern_framework_redesign"
EVENT_FILE = ROOT / "outputs" / "20260512_event_metrics_global" / "soil_event_samples_for_curve.csv.gz"
ASW_LOSS_CURVE_FILE = ROOT / "outputs" / "20260512_event_metrics_global" / "global_curve_asm_abs.csv"


mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "sans-serif"],
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.transparent": False,
    "image.composite_image": False,
    "pdf.compression": 0,
    "font.size": 7,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.linewidth": 0.65,
    "axes.labelsize": 7,
    "xtick.labelsize": 6.5,
    "ytick.labelsize": 6.5,
    "legend.fontsize": 6.5,
})


COLORS = {
    "dry": "#6e90b8",
    "critical": "#d8843b",
    "wet": "#6b9b74",
    "line": "#252525",
    "soil": "#5f5f5f",
    "shade": "#f2c185",
    "grid": "#d9d9d9",
}


def save_all(fig: plt.Figure, stem: str) -> None:
    for ext, dpi in [("png", 600), ("tiff", 600), ("pdf", None), ("svg", None)]:
        path = OUT / f"{stem}.{ext}"
        if dpi is None:
            fig.savefig(path, bbox_inches="tight", facecolor="white")
        else:
            fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")


def binned_curve(df: pd.DataFrame, x: str, y: str, bins: np.ndarray) -> pd.DataFrame:
    d = df[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    d = d[(d[x] >= bins[0]) & (d[x] <= bins[-1])].copy()
    d["bin"] = pd.cut(d[x], bins=bins, include_lowest=True)
    g = (
        d.groupby("bin", observed=True)
        .agg(
            x_mean=(x, "mean"),
            x_min=(x, "min"),
            x_max=(x, "max"),
            n=(x, "size"),
            y_mean=(y, "mean"),
            y_median=(y, "median"),
            y_q25=(y, lambda s: s.quantile(0.25)),
            y_q75=(y, lambda s: s.quantile(0.75)),
            y_q10=(y, lambda s: s.quantile(0.10)),
            y_q90=(y, lambda s: s.quantile(0.90)),
        )
        .reset_index(drop=True)
    )
    return g


def regime_summary(df: pd.DataFrame) -> pd.DataFrame:
    regimes = [
        ("dry edge", "ASW <= 0.20", df["asm_pre3"].between(0.0, 0.20, inclusive="both")),
        ("middle state", "0.30 <= ASW <= 0.50", df["asm_pre3"].between(0.30, 0.50, inclusive="both")),
        ("wet state", "ASW >= 0.80 and <= 1.00", df["asm_pre3"].between(0.80, 1.00, inclusive="both")),
        ("very wet tail", "ASW > 1.00", df["asm_pre3"] > 1.00),
    ]
    rows = []
    for name, rule, mask in regimes:
        sub = df.loc[mask]
        rows.append({
            "regime": name,
            "rule": rule,
            "n_events": int(len(sub)),
            "soil_loss_median_mm": float(sub["soil_loss"].median() * 1000),
            "soil_loss_mean_mm": float(sub["soil_loss"].mean() * 1000),
            "soil_loss_q25_mm": float(sub["soil_loss"].quantile(0.25) * 1000),
            "soil_loss_q75_mm": float(sub["soil_loss"].quantile(0.75) * 1000),
            "soil_loss_frac_median_pct": float(sub["soil_loss_frac"].median() * 100),
            "soil_pre3_median_m": float(sub["soil_pre3"].median()),
            "asw_pre3_median": float(sub["asm_pre3"].median()),
        })
    out = pd.DataFrame(rows)
    mid = out.loc[out["regime"].eq("middle state"), "soil_loss_median_mm"].iloc[0]
    out["relative_to_middle_median"] = out["soil_loss_median_mm"] / mid
    return out


def main() -> None:
    OUT.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(EVENT_FILE)
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    df["soil_loss_mm"] = df["soil_loss"] * 1000.0
    df["soil_loss_frac_pct"] = df["soil_loss_frac"] * 100.0

    asw_bins = np.arange(0.0, 1.0001, 0.05)
    soil_lo = df["soil_pre3"].quantile(0.01)
    soil_hi = df["soil_pre3"].quantile(0.99)
    soil_bins = np.linspace(soil_lo, soil_hi, 21)

    asw_curve = binned_curve(df, "asm_pre3", "soil_loss_mm", asw_bins)
    frac_curve = binned_curve(df, "asm_pre3", "soil_loss_frac_pct", asw_bins)
    soil_curve = binned_curve(df, "soil_pre3", "soil_loss_mm", soil_bins)

    asw_curve.to_csv(OUT / "event_level_asw_pre3_loss_curve.csv", index=False, encoding="utf-8-sig")
    frac_curve.to_csv(OUT / "event_level_asw_pre3_fractional_loss_curve.csv", index=False, encoding="utf-8-sig")
    soil_curve.to_csv(OUT / "event_level_soil_pre3_loss_curve.csv", index=False, encoding="utf-8-sig")

    summary = regime_summary(df)
    peak = asw_curve.loc[asw_curve["y_median"].idxmax()]
    summary.attrs["peak_asw_bin_center"] = float(peak["x_mean"])
    summary.attrs["peak_median_loss_mm"] = float(peak["y_median"])
    summary.to_csv(OUT / "event_level_critical_window_regime_summary.csv", index=False, encoding="utf-8-sig")

    asw_loss_curve = pd.read_csv(ASW_LOSS_CURVE_FILE)
    asw_loss_curve = asw_loss_curve[(asw_loss_curve["bin_center"] >= 0) & (asw_loss_curve["bin_center"] <= 1)].copy()

    fig = plt.figure(figsize=(7.25, 7.2), constrained_layout=False)
    gs = fig.add_gridspec(
        3, 2,
        left=0.075, right=0.985, top=0.94, bottom=0.105,
        wspace=0.26, hspace=0.46,
    )
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])
    ax5 = fig.add_subplot(gs[2, 0])
    ax6 = fig.add_subplot(gs[2, 1])

    rng = np.random.default_rng(20260521)
    jitter_asw = rng.normal(0, 0.0025, size=len(df))
    jitter_soil = rng.normal(0, 0.0008, size=len(df))

    # Panel a: event-level ASW response curve.
    ax1.axvspan(0.30, 0.50, color=COLORS["shade"], alpha=0.45, lw=0, label="middle-state window")
    p1 = df["asm_pre3"].between(0, 1) & df["soil_loss_mm"].between(0, 12)
    ax1.scatter(
        np.clip(df.loc[p1, "asm_pre3"].to_numpy() + jitter_asw[p1], 0, 1),
        df.loc[p1, "soil_loss_mm"],
        s=0.25, color="#8c8c8c", alpha=0.025, linewidth=0, rasterized=True, zorder=0,
    )
    ax1.fill_between(
        asw_curve["x_mean"], asw_curve["y_q25"], asw_curve["y_q75"],
        color="#bdbdbd", alpha=0.28, lw=0, label="IQR"
    )
    ax1.plot(asw_curve["x_mean"], asw_curve["y_median"], color=COLORS["line"], lw=1.6, label="median")
    ax1.scatter(asw_curve["x_mean"], asw_curve["y_median"], s=10, color=COLORS["line"], zorder=3)
    ax1.set_xlim(0, 1)
    ax1.set_ylim(-0.25, 10.5)
    ax1.set_xlabel("Antecedent ASW, 3-day mean")
    ax1.set_ylabel("CDHW event-window soil-water loss (mm)")
    ax1.set_title("a  Event-level response peaks away from the dry edge", loc="left", fontsize=8, fontweight="bold")
    ax1.grid(axis="y", color=COLORS["grid"], lw=0.35)
    ax1.text(
        0.03, 0.94,
        f"Peak bin: ASW ~ {peak['x_mean']:.2f}\n"
        f"grey points: {int(p1.sum()):,} events",
        transform=ax1.transAxes, va="top", ha="left", fontsize=6.2,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#bdbdbd", lw=0.4),
    )

    # Panel b: actual soil-water amount response curve.
    p2 = df["soil_pre3"].between(soil_bins[0], soil_bins[-1]) & df["soil_loss_mm"].between(0, 18)
    ax2.scatter(
        df.loc[p2, "soil_pre3"].to_numpy() + jitter_soil[p2],
        df.loc[p2, "soil_loss_mm"],
        s=0.25, color="#8c8c8c", alpha=0.02, linewidth=0, rasterized=True, zorder=0,
    )
    ax2.fill_between(
        soil_curve["x_mean"], soil_curve["y_q25"], soil_curve["y_q75"],
        color="#bdbdbd", alpha=0.28, lw=0
    )
    ax2.plot(soil_curve["x_mean"], soil_curve["y_median"], color=COLORS["soil"], lw=1.6)
    ax2.scatter(soil_curve["x_mean"], soil_curve["y_median"], s=10, color=COLORS["soil"], zorder=3)
    ax2.set_xlabel("Antecedent soil moisture, 3-day mean (m)")
    ax2.set_ylabel("CDHW event-window soil-water loss (mm)")
    ax2.set_ylim(-0.5, 15.5)
    ax2.set_title("b  Absolute storage gives a mostly monotonic relation", loc="left", fontsize=8, fontweight="bold")
    ax2.grid(axis="y", color=COLORS["grid"], lw=0.35)
    ax2.text(
        0.03, 0.94,
        "Absolute storage: slight low-end dip,\nthen a strong increase at high storage.",
        transform=ax2.transAxes, va="top", ha="left", fontsize=6.2,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#bdbdbd", lw=0.4),
    )

    # Panel c: ASW loss curve from the aggregated event-level histogram.
    ax3.axvspan(0.30, 0.50, color=COLORS["shade"], alpha=0.45, lw=0)
    count_scaled = asw_loss_curve["count"] / asw_loss_curve["count"].max()
    ax3.bar(
        asw_loss_curve["bin_center"],
        count_scaled * asw_loss_curve["response_mean"].max() * 0.22,
        width=0.018,
        color="#d9d9d9",
        edgecolor="none",
        align="center",
        zorder=0,
        label="relative event count",
    )
    ax3.plot(
        asw_loss_curve["bin_center"],
        asw_loss_curve["response_mean"],
        color="#2f6f73",
        lw=1.6,
        label="mean ASW loss",
    )
    ax3.scatter(
        asw_loss_curve["bin_center"],
        asw_loss_curve["response_mean"],
        s=8,
        color="#2f6f73",
        zorder=3,
    )
    asw_loss_peak = asw_loss_curve.loc[asw_loss_curve["response_mean"].idxmax()]
    ax3.set_xlim(0, 1)
    ax3.set_xlabel("Antecedent ASW, 3-day mean")
    ax3.set_ylabel("ASW loss during CDHW")
    ax3.set_title("c  ASW loss increases toward wetter antecedent states", loc="left", fontsize=8, fontweight="bold")
    ax3.grid(axis="y", color=COLORS["grid"], lw=0.35)
    ax3.text(
        0.03, 0.94,
        f"Peak bin: ASW ~ {asw_loss_peak['bin_center']:.2f}\n"
        "bars: relative event count",
        transform=ax3.transAxes, va="top", ha="left", fontsize=6.2,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#bdbdbd", lw=0.4),
    )

    # Panel d: regime bars.
    plot_sum = summary.loc[summary["regime"].isin(["dry edge", "middle state", "wet state", "very wet tail"])].copy()
    x = np.arange(len(plot_sum))
    colors = [COLORS["dry"], COLORS["critical"], COLORS["wet"], "#8f7f6d"]
    yerr = np.vstack([
        plot_sum["soil_loss_median_mm"] - plot_sum["soil_loss_q25_mm"],
        plot_sum["soil_loss_q75_mm"] - plot_sum["soil_loss_median_mm"],
    ])
    ax4.bar(x, plot_sum["soil_loss_median_mm"], color=colors, width=0.68, edgecolor="white", linewidth=0.4)
    ax4.errorbar(x, plot_sum["soil_loss_median_mm"], yerr=yerr, fmt="none", ecolor="#333333", lw=0.7, capsize=2)
    ax4.set_xticks(x)
    ax4.set_xticklabels(["dry\n<=0.2", "middle\n0.3-0.5", "wet\n0.8-1.0", "very wet\n>1.0"])
    ax4.set_ylabel("Median soil-water loss (mm)")
    ax4.set_title("d  Dry edge is weak; wet tail is not consistently weak", loc="left", fontsize=8, fontweight="bold")
    ax4.grid(axis="y", color=COLORS["grid"], lw=0.35)
    for i, row in enumerate(plot_sum.itertuples(index=False)):
        ax4.text(i, row.soil_loss_median_mm + yerr[1, i] + 0.15, f"n={row.n_events/1000:.0f}k", ha="center", va="bottom", fontsize=5.8)

    # Panel e: fractional loss diagnostic.
    ax5.axvspan(0.30, 0.50, color=COLORS["shade"], alpha=0.45, lw=0)
    p4 = df["asm_pre3"].between(0, 1) & df["soil_loss_frac_pct"].between(0, 2.5)
    ax5.scatter(
        np.clip(df.loc[p4, "asm_pre3"].to_numpy() + jitter_asw[p4], 0, 1),
        df.loc[p4, "soil_loss_frac_pct"],
        s=0.25, color="#8c8c8c", alpha=0.025, linewidth=0, rasterized=True, zorder=0,
    )
    ax5.fill_between(
        frac_curve["x_mean"], frac_curve["y_q25"], frac_curve["y_q75"],
        color="#bdbdbd", alpha=0.28, lw=0
    )
    ax5.plot(frac_curve["x_mean"], frac_curve["y_median"], color="#7a4e7c", lw=1.6)
    ax5.scatter(frac_curve["x_mean"], frac_curve["y_median"], s=10, color="#7a4e7c", zorder=3)
    ax5.set_xlim(0, 1)
    ax5.set_ylim(-0.08, 2.05)
    ax5.set_xlabel("Antecedent ASW, 3-day mean")
    ax5.set_ylabel("Fractional soil-water loss (%)")
    ax5.set_title("e  Fractional response is diagnostic, not the primary mask", loc="left", fontsize=8, fontweight="bold")
    ax5.grid(axis="y", color=COLORS["grid"], lw=0.35)

    # Panel f: interpretation box.
    ax6.axis("off")
    ax6.set_title("f  Interpretation for sensitive-region wording", loc="left", fontsize=8, fontweight="bold")
    interpretation_text = (
        "Definition used in the manuscript\n"
        "Response-sensitive pixels are locations where CDHWs produce enhanced\n"
        "event-window soil-water depletion after accounting for antecedent\n"
        "soil-water state, pre-event depletion, event duration and season.\n\n"
        "Event-level diagnostic\n"
        "ASW-based curves identify a water-state response window: loss rises\n"
        "rapidly from the dry edge, peaks near ASW 0.3-0.5, and then declines\n"
        "slightly before increasing again in very wet/non-standard states.\n\n"
        "Variable distinction\n"
        "Absolute soil moisture mainly reflects available storage; ASW is the\n"
        "state variable used to discuss dry, middle and wet regimes."
    )
    ax6.text(
        0.02, 0.96, interpretation_text,
        transform=ax6.transAxes, va="top", ha="left", fontsize=6.6, linespacing=1.35,
        bbox=dict(boxstyle="round,pad=0.45", fc="#f7f7f7", ec="#bdbdbd", lw=0.6),
    )

    fig.suptitle(
        "Event-level diagnosis of the antecedent water-state response window",
        x=0.075, y=0.985, ha="left", fontsize=9.5, fontweight="bold"
    )
    fig.text(
        0.075, 0.035,
        "Source: CDHW event-level samples. Grey points are event samples clipped to the displayed y-range; "
        "curves show bin-level medians and IQR. ASW is the normalized antecedent water state; "
        "soil moisture is absolute soil-water storage.",
        fontsize=6.2, color="#4d4d4d"
    )
    save_all(fig, "fig_event_level_critical_response_window")
    plt.close(fig)

    # Machine-readable diagnosis.
    dry_med = summary.loc[summary["regime"].eq("dry edge"), "soil_loss_median_mm"].iloc[0]
    mid_med = summary.loc[summary["regime"].eq("middle state"), "soil_loss_median_mm"].iloc[0]
    wet_med = summary.loc[summary["regime"].eq("wet state"), "soil_loss_median_mm"].iloc[0]
    very_wet_med = summary.loc[summary["regime"].eq("very wet tail"), "soil_loss_median_mm"].iloc[0]
    diagnosis = pd.DataFrame([{
        "event_count": int(len(df)),
        "asw_0_1_event_count": int(df["asm_pre3"].between(0, 1).sum()),
        "peak_asw_bin_center": float(peak["x_mean"]),
        "peak_median_loss_mm": float(peak["y_median"]),
        "dry_edge_median_loss_mm": float(dry_med),
        "middle_state_median_loss_mm": float(mid_med),
        "wet_state_median_loss_mm": float(wet_med),
        "very_wet_tail_median_loss_mm": float(very_wet_med),
        "middle_over_dry": float(mid_med / dry_med),
        "middle_over_wet": float(mid_med / wet_med),
        "middle_over_very_wet": float(mid_med / very_wet_med),
        "interpretation": (
            "ASW-based event-level curve supports weak dry-edge response and a local middle-state maximum; "
            "the wet end is not uniformly weak under raw absolute loss, so a strict inverted-U claim requires "
            "event-level excess or severity-normalized metrics."
        ),
    }])
    diagnosis.to_csv(OUT / "event_level_critical_window_diagnosis.csv", index=False, encoding="utf-8-sig")

    asw_first = asw_curve.iloc[0]
    asw_mid = asw_curve.loc[asw_curve["x_mean"].between(0.30, 0.50), "y_median"].max()
    asw_post_min_row = asw_curve.loc[asw_curve["x_mean"].between(0.55, 0.80)].sort_values("y_median").iloc[0]
    soil_trough = soil_curve.sort_values("y_median").iloc[0]
    soil_peak = soil_curve.sort_values("y_median").iloc[-1]
    key_values = pd.DataFrame([
        {
            "curve": "ASW_pre3_vs_absolute_loss",
            "feature": "dry edge bin",
            "x_value": float(asw_first["x_mean"]),
            "median_loss_mm": float(asw_first["y_median"]),
            "n_events": int(asw_first["n"]),
            "interpretation": "very dry antecedent ASW has weak event-window loss",
        },
        {
            "curve": "ASW_pre3_vs_absolute_loss",
            "feature": "middle-state peak",
            "x_value": float(peak["x_mean"]),
            "median_loss_mm": float(peak["y_median"]),
            "n_events": int(peak["n"]),
            "interpretation": "response reaches a local maximum in the intermediate ASW state",
        },
        {
            "curve": "ASW_pre3_vs_absolute_loss",
            "feature": "post-peak local minimum",
            "x_value": float(asw_post_min_row["x_mean"]),
            "median_loss_mm": float(asw_post_min_row["y_median"]),
            "n_events": int(asw_post_min_row["n"]),
            "interpretation": "median loss slightly declines after the middle-state peak",
        },
        {
            "curve": "soil_pre3_vs_absolute_loss",
            "feature": "low-storage trough",
            "x_value": float(soil_trough["x_mean"]),
            "median_loss_mm": float(soil_trough["y_median"]),
            "n_events": int(soil_trough["n"]),
            "interpretation": "absolute soil-water storage shows a low-end trough before increasing",
        },
        {
            "curve": "soil_pre3_vs_absolute_loss",
            "feature": "high-storage maximum",
            "x_value": float(soil_peak["x_mean"]),
            "median_loss_mm": float(soil_peak["y_median"]),
            "n_events": int(soil_peak["n"]),
            "interpretation": "absolute loss becomes largest where antecedent storage is high",
        },
        {
            "curve": "ASW_pre3_vs_ASW_loss",
            "feature": "high-ASW maximum",
            "x_value": float(asw_loss_peak["bin_center"]),
            "median_loss_mm": np.nan,
            "n_events": int(asw_loss_peak["count"]),
            "asw_loss_mean": float(asw_loss_peak["response_mean"]),
            "interpretation": "ASW loss itself increases toward wetter antecedent ASW states in the available binned product",
        },
    ])
    key_values["relative_to_asw_middle_peak"] = key_values["median_loss_mm"] / float(peak["y_median"])
    key_values.to_csv(OUT / "event_level_curve_key_values.csv", index=False, encoding="utf-8-sig")

    print(diagnosis.to_string(index=False))
    print(key_values.to_string(index=False))
    print(f"Saved outputs to {OUT}")


if __name__ == "__main__":
    main()


### make_landcover_metric_trigger_figures

Supplementary C S8-S9 metric, trigger-sensitive and land-cover diagnostics. Generates Fig. S5, S7, S8 and related tables.



In [ ]:
# Source file: D:\smq\paper2\modify\敏感区\code\make_landcover_metric_trigger_figures.py
# Staging copy used for notebook assembly: D:\software\codex\paper2\office_work\code_notebook_order_20260609\sensitive_code\make_landcover_metric_trigger_figures.py
# Detected encoding: utf-8-sig
# Methods-order section: Supplementary C S8-S9 metric, trigger-sensitive and land-cover diagnostics
# NOTE: This is the final modified sensitive-region/SHAP script, not legacy notebook code.

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt


ROOT = Path(r"D:\Software\codex\sm_heat_drought")
OUT = ROOT / "outputs" / "20260521_response_pattern_framework_redesign"
EVW = ROOT / "outputs" / "20260520_event_window_specific_framework" / "event_window_specific_pixel_metrics.csv.gz"
LC = ROOT / "outputs" / "20260512_event_metrics_global" / "global_pixel_metrics_all_frameworks_landcover.csv.gz"
FB = ROOT / "outputs" / "20260520_feedback_sensitive_framework" / "feedback_sensitive_pixel_metrics.csv.gz"
EVENTS = ROOT / "outputs" / "20260512_event_metrics_global" / "soil_event_samples_for_curve.csv.gz"
ASW_EVENTS = OUT / "asw_event_samples" / "asw_event_samples_for_metric_bias.csv.gz"


mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "sans-serif"],
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.transparent": False,
    "image.composite_image": False,
    "pdf.compression": 0,
    "font.size": 7,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.linewidth": 0.65,
    "axes.labelsize": 7,
    "xtick.labelsize": 6.4,
    "ytick.labelsize": 6.4,
    "legend.fontsize": 6.2,
})

COL = {
    "all": "#8b8b8b",
    "A": "#d8843b",
    "FB": "#4d79a8",
    "both": "#7a4e7c",
    "nonA": "#bdbdbd",
    "forest": "#4f8a5b",
    "grassland": "#b9a44c",
    "line_abs": "#2b2b2b",
    "line_frac": "#7a4e7c",
    "grid": "#d9d9d9",
}


def save(fig, stem):
    for ext, dpi in [("png", 600), ("tiff", 600), ("pdf", None), ("svg", None)]:
        path = OUT / f"{stem}.{ext}"
        if dpi is None:
            fig.savefig(path, bbox_inches="tight", facecolor="white")
        else:
            fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")


def binned(df, x, y, bins):
    d = df[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    d = d[(d[x] >= bins[0]) & (d[x] <= bins[-1])].copy()
    d["bin"] = pd.cut(d[x], bins=bins, include_lowest=True)
    return (
        d.groupby("bin", observed=True)
        .agg(
            x=(x, "mean"),
            n=(x, "size"),
            y_med=(y, "median"),
            y_mean=(y, "mean"),
            y_q25=(y, lambda s: s.quantile(0.25)),
            y_q75=(y, lambda s: s.quantile(0.75)),
        )
        .reset_index(drop=True)
    )


def load_pixel_table():
    evw_cols = [
        "lat_idx_global", "lon_idx", "lat", "lon", "event_window_comparable",
        "A_event_window_specific", "A_event_window_specific_strict",
        "cdhw_event_window_loss_mean_mm", "matched_event_window_excess_mean_mm",
        "reg_event_window_excess_mm", "cdhw_preconditioning_loss_mean_mm",
    ]
    lc_cols = [
        "lat_idx_global", "lon_idx", "dominant_landcover", "dominant_landcover_frac",
        "forest_frac_mean_2000_2010", "grassland_frac_mean_2000_2010",
        "irrigated_non_paddy_frac_mean_2000_2010", "irrigated_paddy_frac_mean_2000_2010",
        "sealed_frac_mean_2000_2010", "water_frac_mean_2000_2010",
    ]
    fb_cols = [
        "lat_idx_global", "lon_idx", "asw_pre7_rr", "asw_pre7_rd_pct",
        "asw_pre7_onset_n", "feedback_sensitive_primary",
    ]
    evw = pd.read_csv(EVW, usecols=evw_cols)
    lc = pd.read_csv(LC, usecols=lc_cols)
    fb = pd.read_csv(FB, usecols=fb_cols)
    df = evw.merge(lc, on=["lat_idx_global", "lon_idx"], how="left")
    df = df.merge(fb, on=["lat_idx_global", "lon_idx"], how="left")
    df = df[df["event_window_comparable"].fillna(False)].copy()
    df["A"] = df["A_event_window_specific"].fillna(False).astype(bool)
    df["A_strict"] = df["A_event_window_specific_strict"].fillna(False).astype(bool)
    df["feedback_strong_rd05"] = (
        df["feedback_sensitive_primary"].fillna(False).astype(bool)
        & (df["asw_pre7_rd_pct"] > 0.5)
        & (df["asw_pre7_rr"] > 1)
    )
    df["both_A_feedback"] = df["A"] & df["feedback_strong_rd05"]
    df["A_only"] = df["A"] & ~df["feedback_strong_rd05"]
    df["feedback_only"] = df["feedback_strong_rd05"] & ~df["A"]
    df["neither"] = ~df["A"] & ~df["feedback_strong_rd05"]
    return df


def summarize_landcover(df):
    groups = {
        "Global comparable": np.ones(len(df), dtype=bool),
        "A response-sensitive": df["A"].values,
        "Strong feedback": df["feedback_strong_rd05"].values,
        "Both": df["both_A_feedback"].values,
        "Non-A comparable": (~df["A"]).values,
    }
    rows = []
    for group, mask in groups.items():
        sub = df.loc[mask]
        if len(sub) == 0:
            continue
        counts = sub["dominant_landcover"].value_counts(dropna=False)
        for landcover, n in counts.items():
            rows.append({
                "group": group,
                "landcover": landcover,
                "n": int(n),
                "share_pct": float(n / len(sub) * 100),
                "group_n": int(len(sub)),
            })
    comp = pd.DataFrame(rows)
    base = comp[comp["group"].eq("Global comparable")][["landcover", "share_pct"]].rename(columns={"share_pct": "global_share_pct"})
    comp = comp.merge(base, on="landcover", how="left")
    comp["enrichment_vs_global"] = comp["share_pct"] / comp["global_share_pct"]

    frac_cols = [
        "forest_frac_mean_2000_2010", "grassland_frac_mean_2000_2010",
        "irrigated_non_paddy_frac_mean_2000_2010", "irrigated_paddy_frac_mean_2000_2010",
        "sealed_frac_mean_2000_2010", "water_frac_mean_2000_2010",
    ]
    frac_rows = []
    for group, mask in groups.items():
        sub = df.loc[mask]
        if len(sub) == 0:
            continue
        row = {"group": group, "n": int(len(sub))}
        for col in frac_cols:
            row[col + "_mean"] = float(sub[col].mean())
            row[col + "_median"] = float(sub[col].median())
        frac_rows.append(row)
    frac = pd.DataFrame(frac_rows)

    landcover_total = df["dominant_landcover"].value_counts().rename("all_n").reset_index().rename(columns={"index": "landcover"})
    a_by_lc = df[df["A"]]["dominant_landcover"].value_counts().rename("A_n").reset_index().rename(columns={"index": "landcover"})
    fb_by_lc = df[df["feedback_strong_rd05"]]["dominant_landcover"].value_counts().rename("feedback_n").reset_index().rename(columns={"index": "landcover"})
    rate = landcover_total.merge(a_by_lc, on="dominant_landcover", how="left").merge(fb_by_lc, on="dominant_landcover", how="left").fillna(0)
    rate["A_selection_rate_pct"] = rate["A_n"] / rate["all_n"] * 100
    rate["feedback_selection_rate_pct"] = rate["feedback_n"] / rate["all_n"] * 100
    return comp, frac, rate


def plot_landcover(df, comp, frac, rate):
    classes = ["forest", "grassland", "irrigated_paddy", "irrigated_non_paddy", "water", "sealed"]
    fig = plt.figure(figsize=(7.25, 5.5))
    gs = fig.add_gridspec(2, 2, left=0.08, right=0.985, bottom=0.10, top=0.92, wspace=0.32, hspace=0.46)
    ax1, ax2, ax3, ax4 = [fig.add_subplot(gs[i, j]) for i in range(2) for j in range(2)]

    sub = comp[comp["group"].isin(["Global comparable", "A response-sensitive", "Strong feedback"])].copy()
    pivot = sub.pivot_table(index="landcover", columns="group", values="share_pct", fill_value=0).reindex(classes).fillna(0)
    x = np.arange(len(classes))
    w = 0.25
    ax1.bar(x - w, pivot["Global comparable"], width=w, color=COL["all"], label="global comparable")
    ax1.bar(x, pivot["A response-sensitive"], width=w, color=COL["A"], label="A response")
    ax1.bar(x + w, pivot["Strong feedback"], width=w, color=COL["FB"], label="strong feedback")
    ax1.set_xticks(x)
    ax1.set_xticklabels(["forest", "grass", "irr.\npaddy", "irr.\nnon-paddy", "water", "sealed"], rotation=0)
    ax1.set_ylabel("Dominant-class composition (%)")
    ax1.set_title("a  Composition must be compared with the global background", loc="left", fontsize=8, fontweight="bold")
    ax1.legend(ncol=1, loc="upper right")
    ax1.grid(axis="y", color=COL["grid"], lw=0.35)

    rate = rate.set_index("dominant_landcover").reindex(classes).fillna(0)
    ax2.bar(x - 0.16, rate["A_selection_rate_pct"], width=0.32, color=COL["A"], label="A response")
    ax2.bar(x + 0.16, rate["feedback_selection_rate_pct"], width=0.32, color=COL["FB"], label="strong feedback")
    ax2.set_xticks(x)
    ax2.set_xticklabels(["forest", "grass", "irr.\npaddy", "irr.\nnon-paddy", "water", "sealed"])
    ax2.set_ylabel("Within-class selected pixels (%)")
    ax2.set_title("b  Selection rate within each land-cover class", loc="left", fontsize=8, fontweight="bold")
    ax2.legend()
    ax2.grid(axis="y", color=COL["grid"], lw=0.35)

    frac_long = frac.set_index("group")
    groups = ["Global comparable", "A response-sensitive", "Strong feedback", "Both", "Non-A comparable"]
    y = np.arange(len(groups))
    forest_med = [frac_long.loc[g, "forest_frac_mean_2000_2010_median"] if g in frac_long.index else np.nan for g in groups]
    grass_med = [frac_long.loc[g, "grassland_frac_mean_2000_2010_median"] if g in frac_long.index else np.nan for g in groups]
    ax3.barh(y - 0.16, forest_med, height=0.30, color=COL["forest"], label="forest fraction")
    ax3.barh(y + 0.16, grass_med, height=0.30, color=COL["grassland"], label="grassland fraction")
    ax3.set_yticks(y)
    ax3.set_yticklabels(["global", "A response", "feedback", "both", "non-A"])
    ax3.set_xlim(0, 1)
    ax3.set_xlabel("Median pixel fraction")
    ax3.set_title("c  Continuous fractions avoid hard-classification bias", loc="left", fontsize=8, fontweight="bold")
    ax3.legend(loc="lower right")
    ax3.grid(axis="x", color=COL["grid"], lw=0.35)

    sizes = {
        "A only": int(df["A_only"].sum()),
        "Feedback only": int(df["feedback_only"].sum()),
        "Both": int(df["both_A_feedback"].sum()),
        "Neither": int(df["neither"].sum()),
    }
    total = len(df)
    labels = list(sizes)
    vals = np.array([sizes[k] for k in labels])
    colors = [COL["A"], COL["FB"], COL["both"], "#d0d0d0"]
    ax4.bar(labels, vals / total * 100, color=colors)
    for i, v in enumerate(vals):
        ax4.text(i, v / total * 100 + 0.7, f"{v:,}", ha="center", va="bottom", fontsize=6.2)
    ax4.set_ylabel("Share of comparable pixels (%)")
    ax4.set_title("d  Trigger-sensitive and response-sensitive regions separate", loc="left", fontsize=8, fontweight="bold")
    ax4.tick_params(axis="x", rotation=20)
    ax4.grid(axis="y", color=COL["grid"], lw=0.35)

    fig.suptitle("Land-cover context and feedback-response separation", x=0.08, y=0.985, ha="left", fontsize=9.5, fontweight="bold")
    save(fig, "fig_landcover_global_feedback_response_comparison")
    plt.close(fig)


def plot_abs_relative_metric_bias():
    events = pd.read_csv(EVENTS)
    events = events.replace([np.inf, -np.inf], np.nan).dropna()
    events["soil_loss_mm"] = events["soil_loss"] * 1000
    events["soil_loss_frac_pct"] = events["soil_loss_frac"] * 100
    lo, hi = events["soil_pre3"].quantile([0.01, 0.99])
    bins = np.linspace(lo, hi, 24)
    abs_curve = binned(events, "soil_pre3", "soil_loss_mm", bins)
    frac_curve = binned(events, "soil_pre3", "soil_loss_frac_pct", bins)
    asw_events = pd.read_csv(ASW_EVENTS)
    asw_events = asw_events.replace([np.inf, -np.inf], np.nan).dropna()
    asw_events["asw_loss_pct_points"] = asw_events["asw_loss"] * 100.0
    asw_events["asw_loss_frac_pct"] = asw_events["asw_loss_frac"] * 100.0
    asw_sub = asw_events[(asw_events["asw_pre3"] >= 0) & (asw_events["asw_pre3"] <= 1)].copy()
    asw_bins = np.linspace(0, 1, 24)
    asw_abs_curve = binned(asw_sub, "asw_pre3", "asw_loss_pct_points", asw_bins)
    asw_frac_curve = binned(asw_sub, "asw_pre3", "asw_loss_frac_pct", asw_bins)
    abs_curve.to_csv(OUT / "metric_bias_abs_loss_by_soil_pre3.csv", index=False, encoding="utf-8-sig")
    frac_curve.to_csv(OUT / "metric_bias_fractional_loss_by_soil_pre3.csv", index=False, encoding="utf-8-sig")
    asw_abs_curve.to_csv(OUT / "metric_bias_asw_absolute_loss_by_asw_pre3.csv", index=False, encoding="utf-8-sig")
    asw_frac_curve.to_csv(OUT / "metric_bias_asw_fractional_loss_by_asw_pre3.csv", index=False, encoding="utf-8-sig")

    rng = np.random.default_rng(20260521)
    fig = plt.figure(figsize=(7.25, 6.2))
    gs = fig.add_gridspec(2, 2, left=0.08, right=0.985, bottom=0.10, top=0.90, wspace=0.30, hspace=0.42)
    ax1, ax2, ax3, ax4 = [fig.add_subplot(gs[i, j]) for i in range(2) for j in range(2)]

    p1 = events["soil_loss_mm"].between(0, 18) & events["soil_pre3"].between(lo, hi)
    ax1.scatter(
        events.loc[p1, "soil_pre3"].to_numpy() + rng.normal(0, 0.0008, p1.sum()),
        events.loc[p1, "soil_loss_mm"],
        s=0.22, color="#8c8c8c", alpha=0.018, linewidth=0, rasterized=True,
    )
    ax1.fill_between(abs_curve["x"], abs_curve["y_q25"], abs_curve["y_q75"], color="#bdbdbd", alpha=0.28, lw=0)
    ax1.plot(abs_curve["x"], abs_curve["y_med"], color=COL["line_abs"], lw=1.7)
    ax1.scatter(abs_curve["x"], abs_curve["y_med"], s=10, color=COL["line_abs"])
    ax1.set_xlabel("Antecedent soil moisture, 3-day mean (m)")
    ax1.set_ylabel("Absolute soil-water loss (mm)")
    ax1.set_title("a  Absolute soil-water loss favours water-rich events", loc="left", fontsize=8, fontweight="bold")
    ax1.set_ylim(-0.5, 15.5)
    ax1.grid(axis="y", color=COL["grid"], lw=0.35)
    ax1.text(0.03, 0.94, "More antecedent storage\nmeans more water can be lost.", transform=ax1.transAxes, va="top", fontsize=6.2,
             bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#bdbdbd", lw=0.4))

    p2 = events["soil_loss_frac_pct"].between(0, 2.5) & events["soil_pre3"].between(lo, hi)
    ax2.scatter(
        events.loc[p2, "soil_pre3"].to_numpy() + rng.normal(0, 0.0008, p2.sum()),
        events.loc[p2, "soil_loss_frac_pct"],
        s=0.22, color="#8c8c8c", alpha=0.018, linewidth=0, rasterized=True,
    )
    ax2.fill_between(frac_curve["x"], frac_curve["y_q25"], frac_curve["y_q75"], color="#bdbdbd", alpha=0.28, lw=0)
    ax2.plot(frac_curve["x"], frac_curve["y_med"], color=COL["line_frac"], lw=1.7)
    ax2.scatter(frac_curve["x"], frac_curve["y_med"], s=10, color=COL["line_frac"])
    ax2.set_xlabel("Antecedent soil moisture, 3-day mean (m)")
    ax2.set_ylabel("Fractional soil-water loss (%)")
    ax2.set_title("b  Fractional soil-water loss is denominator-sensitive", loc="left", fontsize=8, fontweight="bold")
    ax2.set_ylim(-0.08, 2.1)
    ax2.grid(axis="y", color=COL["grid"], lw=0.35)
    ax2.text(0.03, 0.94, "Dry pixels can look strong\nbecause SM_pre3 is small.", transform=ax2.transAxes, va="top", fontsize=6.2,
             bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#bdbdbd", lw=0.4))

    p3 = asw_sub["asw_loss_pct_points"].between(0, 12)
    ax3.scatter(
        np.clip(asw_sub.loc[p3, "asw_pre3"].to_numpy() + rng.normal(0, 0.0025, p3.sum()), 0, 1),
        asw_sub.loc[p3, "asw_loss_pct_points"],
        s=0.22, color="#8c8c8c", alpha=0.018, linewidth=0, rasterized=True,
    )
    ax3.fill_between(asw_abs_curve["x"], asw_abs_curve["y_q25"], asw_abs_curve["y_q75"], color="#bdbdbd", alpha=0.28, lw=0)
    ax3.plot(asw_abs_curve["x"], asw_abs_curve["y_med"], color="#2f6f73", lw=1.7)
    ax3.scatter(asw_abs_curve["x"], asw_abs_curve["y_med"], s=10, color="#2f6f73")
    ax3.set_xlabel("Antecedent ASW, 3-day mean")
    ax3.set_ylabel("Absolute ASW loss (percentage points)")
    ax3.set_title("c  Absolute ASW loss rises away from the dry edge", loc="left", fontsize=8, fontweight="bold")
    ax3.set_xlim(0, 1)
    ax3.set_ylim(-0.3, 10.5)
    ax3.grid(axis="y", color=COL["grid"], lw=0.35)
    ax3.text(0.03, 0.94, "Low ASW has little state\nleft to lose.", transform=ax3.transAxes, va="top", fontsize=6.2,
             bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#bdbdbd", lw=0.4))

    p4 = asw_sub["asw_loss_frac_pct"].between(0, 60)
    ax4.scatter(
        np.clip(asw_sub.loc[p4, "asw_pre3"].to_numpy() + rng.normal(0, 0.0025, p4.sum()), 0, 1),
        asw_sub.loc[p4, "asw_loss_frac_pct"],
        s=0.22, color="#8c8c8c", alpha=0.018, linewidth=0, rasterized=True,
    )
    ax4.fill_between(asw_frac_curve["x"], asw_frac_curve["y_q25"], asw_frac_curve["y_q75"], color="#bdbdbd", alpha=0.28, lw=0)
    ax4.plot(asw_frac_curve["x"], asw_frac_curve["y_med"], color="#7a4e7c", lw=1.7)
    ax4.scatter(asw_frac_curve["x"], asw_frac_curve["y_med"], s=10, color="#7a4e7c")
    ax4.set_xlabel("Antecedent ASW, 3-day mean")
    ax4.set_ylabel("Fractional ASW loss (%)")
    ax4.set_title("d  Fractional ASW loss emphasizes dry-state denominators", loc="left", fontsize=8, fontweight="bold")
    ax4.set_xlim(0, 1)
    ax4.grid(axis="y", color=COL["grid"], lw=0.35)
    ax4.text(0.03, 0.94, "Diagnostic only; this can\nover-weight dry states.", transform=ax4.transAxes, va="top", fontsize=6.2,
             bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#bdbdbd", lw=0.4))

    fig.suptitle("Metric dependence of CDHW event-window soil-water response", x=0.08, y=0.975, ha="left", fontsize=9.5, fontweight="bold")
    save(fig, "fig_absolute_vs_fractional_loss_metric_bias")
    plt.close(fig)


def summarize_overlap(df, comp, frac):
    total = len(df)
    a = int(df["A"].sum())
    fb = int(df["feedback_strong_rd05"].sum())
    both = int(df["both_A_feedback"].sum())
    out = pd.DataFrame([{
        "comparable_pixels": total,
        "A_response_pixels": a,
        "A_response_share_pct": a / total * 100,
        "strong_feedback_pixels": fb,
        "strong_feedback_share_pct": fb / total * 100,
        "overlap_pixels": both,
        "overlap_share_of_A_pct": both / a * 100,
        "overlap_share_of_feedback_pct": both / fb * 100,
        "jaccard": both / (a + fb - both),
        "A_only_pixels": int(df["A_only"].sum()),
        "feedback_only_pixels": int(df["feedback_only"].sum()),
    }])
    out.to_csv(OUT / "true_A_vs_strong_feedback_overlap_summary.csv", index=False, encoding="utf-8-sig")
    comp.to_csv(OUT / "landcover_global_A_feedback_composition_enrichment.csv", index=False, encoding="utf-8-sig")
    frac.to_csv(OUT / "landcover_global_A_feedback_fraction_summary.csv", index=False, encoding="utf-8-sig")
    return out


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    df = load_pixel_table()
    comp, frac, rate = summarize_landcover(df)
    overlap = summarize_overlap(df, comp, frac)
    rate.to_csv(OUT / "landcover_within_class_selection_rates.csv", index=False, encoding="utf-8-sig")
    plot_landcover(df, comp, frac, rate)
    plot_abs_relative_metric_bias()
    print(overlap.to_string(index=False))
    print("Saved figures and summaries to", OUT)


if __name__ == "__main__":
    main()


## 6. Pre-onset hydrothermal pathway types

Main Methods: triggering-phase hydrothermal pathways and Fig. 5.


### Extract pre-onset ETa and Tmax event tables




In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
from dask.distributed import Client, LocalCluster, as_completed 
from tqdm.notebook import tqdm
import os, warnings
warnings.filterwarnings("ignore")

# =======================
# 1. 初始化 Dask 集群
# =======================
cluster = LocalCluster(
    n_workers=20,
    threads_per_worker=1,
    memory_limit="8GB"
)
client = Client(cluster)
print("✅ Dask Dashboard:", client.dashboard_link)


# =======================
# 2. 提取事件函数
# =======================
def extract_events(time, mask):
    mask = mask.values.astype(bool)
    events = []
    start = None
    for i in range(len(mask)):
        if mask[i] and start is None:
            start = i
        elif not mask[i] and start is not None:
            events.append((time[start], time[i - 1]))
            start = None
    if start is not None:
        events.append((time[start], time[-1]))
    return events


# =======================
# 3. 时间范围与区域掩膜
# =======================
start_date, end_date = '1920-01-01', '2019-12-31'
ref_ds = xr.open_dataset(r"F:\smq\paper2\data\masks_cdhw_hw_di.nc")['cdhw_mask']
band_ds = xr.open_dataset(r"D:\smq\paper2\data\SREX_MASK.nc")['band_data']

region_ids = np.unique(band_ds.values[~np.isnan(band_ds.values)]).astype(int)
print(f"📍 区域数量: {len(region_ids)} 个")

# =======================
# 4. 加载数据（只加载一次）
# =======================
print("📦 加载数据中...")
data_all = {
    'RSM': xr.open_dataset(r"E:\smq\w\w_daily.nc")['w'].sel(time=slice(start_date, end_date)),
    'spei': xr.open_dataset(r"E:\smq\spei\SPEI_daily1.nc", engine="h5netcdf")['spei'].sel(time=slice(start_date, end_date)),
    'tasmax': xr.open_dataset(r"F:\smq\paper2\data\TMax_daily.nc")['TMax'].sel(time=slice(start_date, end_date)),
    'pr': xr.open_dataset(r"E:\smq\su\Precipitation_daily.nc")['Precipitation'].sel(time=slice(start_date, end_date)),
    'tas95': xr.open_dataset(r"D:\smq\paper2\data\tas95_1981_2010.nc")['tas95'],
    'cdhw_mask': ref_ds.sel(time=slice(start_date, end_date)),
}
print("✅ 数据加载完成。")


# =======================
# 5. 单格点处理函数
# =======================
def process_point(lat_sel, lon_sel):
    records = []
    try:
        data = {k: v.sel(lat=lat_sel, lon=lon_sel, method="nearest") for k, v in data_all.items()}
        events = extract_events(data['cdhw_mask'].time.values, data['cdhw_mask'])
        if len(events) == 0:
            return []

        etref = xr.open_dataset(r"E:\smq\su\ETRef_daily.nc")['ETRef'].sel(lat=lat_sel, lon=lon_sel, method="nearest")
        evapt = xr.open_dataset(r"E:\smq\evp\evapt_daily.nc")['evapt'].sel(lat=lat_sel, lon=lon_sel, method="nearest")
        trans = xr.open_dataset(r"E:\smq\evp\trans_daily.nc")['trans'].sel(lat=lat_sel, lon=lon_sel, method="nearest")

        for idx, (start, end) in enumerate(events):
            pre4_dates = pd.date_range(end=pd.Timestamp(start), periods=4)
            pre4_dates = pre4_dates[
                (pre4_dates >= data['pr'].time.min().values) &
                (pre4_dates <= data['pr'].time.max().values)
            ]
            if len(pre4_dates) < 4:
                continue

            pr_sel = data['pr'].sel(time=pre4_dates)
            tas_sel = data['tasmax'].sel(time=pre4_dates)
            etref_sel = etref.sel(time=pre4_dates)
            evapt_sel = evapt.sel(time=pre4_dates)
            trans_sel = trans.sel(time=pre4_dates)

            for t, p, tas, et, ev, tr in zip(
                pre4_dates, pr_sel.values, tas_sel.values,
                etref_sel.values, evapt_sel.values, trans_sel.values
            ):
                records.append({
                    'lat': float(lat_sel),
                    'lon': float(lon_sel),
                    'event_index': idx + 1,
                    'event_start': pd.to_datetime(start),
                    'time': pd.to_datetime(t),
                    'precipitation': float(p),
                    'temperature': float(tas),
                    'ETRef': float(et),
                    'Evap': float(ev),
                    'Trans': float(tr)
                })
    except Exception as e:
        return [{'lat': float(lat_sel), 'lon': float(lon_sel), 'error': str(e)}]
    return records


# =======================
# 6. 逐区域循环
# =======================
save_root = r"F:\smq\paper2\CDHW_pre3days_by_band"
os.makedirs(save_root, exist_ok=True)

for rid in region_ids:
    print(f"\n🌍 开始处理区域 ID = {rid}")
    mask = band_ds.where(band_ds == rid)
    lats = mask.lat.values
    lons = mask.lon.values

    valid_points = np.argwhere(~np.isnan(mask.values))
    if valid_points.size == 0:
        print(f"⚠️ 区域 {rid} 无有效格点，跳过。")
        continue

    lat_idx, lon_idx = valid_points[:, 0], valid_points[:, 1]
    lat_list = lats[lat_idx]
    lon_list = lons[lon_idx]
    print(f"📍 区域格点数: {len(lat_list)}")

    futures = [client.submit(process_point, lat, lon) for lat, lon in zip(lat_list, lon_list)]

    results = []
    count = 0
    batch_size = 3000
    save_dir = os.path.join(save_root, f"region_{rid}")
    os.makedirs(save_dir, exist_ok=True)
5
    for future in tqdm(as_completed(futures), total=len(futures), desc=f"⏳ Region {rid}"):
        res = future.result()
        if len(res) > 0:
            results.extend(res)
        count += 1

        if count % batch_size == 0 and len(results) > 0:
            df_tmp = pd.DataFrame(results)
            out_path = os.path.join(save_dir, f"region_{rid}_part_{count//batch_size:03d}.csv")
            df_tmp.to_csv(out_path, index=False)
            print(f"💾 已保存中间结果: {out_path}")
            results = []

    if len(results) > 0:
        df_tmp = pd.DataFrame(results)
        out_path = os.path.join(save_dir, f"region_{rid}_final.csv")
        df_tmp.to_csv(out_path, index=False)
        print(f"💾 已保存最终结果: {out_path}")

    print(f"✅ 区域 {rid} 完成！")

print("\n🎯 所有区域计算完成！")


### Regional DeltaET-DeltaT pathway analysis




In [ ]:
# -*- coding: utf-8 -*-
"""
区域批量 CDHW 事件级 ΔET–ΔT 反馈分析
自动遍历每个 region 文件夹（region_1, region_2, ...），
合并该区域所有 CSV 文件并执行完整反馈分析。
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.stats import linregress

# ==================== 全局设置 ====================
BASE_DIR = r"F:\smq\paper2\CDHW_pre3days_by_band"   # 根目录
OUT_ROOT = r"F:\smq\paper2\CDHW_feedback_results"    # 输出目录
os.makedirs(OUT_ROOT, exist_ok=True)

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False


# ==================== 定义分析函数 ====================
def analyze_region(region_name, df):
    """对一个区域 DataFrame 执行 ΔET–ΔT 分析"""
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values(["lat", "lon", "event_index", "time"]).reset_index(drop=True)

    # 单位转换 (m → mm/day)
    for col in ["Evap", "Trans", "ETRef", "precipitation"]:
        df[col] = df[col] * 1000

    df["ET"] = df["Evap"] + df["Trans"]

    records = []
    eid_global = 0

    for (lat, lon, eid), g in df.groupby(["lat", "lon", "event_index"]):
        g = g.sort_values("time")
        if len(g) < 4:
            continue
        ET1, ET4 = g["ET"].iloc[0], g["ET"].iloc[-1]
        T1, T4 = g["temperature"].iloc[0], g["temperature"].iloc[-1]
        ΔET = ET4 - ET1
        ΔT = T4 - T1
        coeff = ΔT / abs(ΔET) if abs(ΔET) > 1e-8 else np.nan
        eid_global += 1
        records.append({
            "event_id": eid_global,
            "lat": lat,
            "lon": lon,
            "ΔET(mm/day)": ΔET,
            "ΔT(°C)": ΔT,
            "反馈强度系数(°C_per_mm)": coeff,
            "ET_day1": ET1,
            "ET_day4": ET4,
            "T_day1": T1,
            "T_day4": T4
        })

    if len(records) == 0:
        print(f"⚠️ {region_name} 无有效事件，跳过。")
        return

    dfΔ = pd.DataFrame(records)

    # 四象限分类
    def classify(row):
        if row["ΔET(mm/day)"] < 0 and row["ΔT(°C)"] > 0:
            return "干旱驱动升温"
        elif row["ΔET(mm/day)"] > 0 and row["ΔT(°C)"] < 0:
            return "热浪驱动蒸散下降"
        elif row["ΔET(mm/day)"] < 0 and row["ΔT(°C)"] < 0:
            return "同步降温干化"
        elif row["ΔET(mm/day)"] > 0 and row["ΔT(°C)"] > 0:
            return "同步升温湿化"
        else:
            return "不确定"

    dfΔ["机制分类"] = dfΔ.apply(classify, axis=1)

    # 回归分析
    slope, intercept, r, p, _ = linregress(dfΔ["ΔET(mm/day)"], dfΔ["ΔT(°C)"])
    r2 = r**2
    counts = dfΔ["机制分类"].value_counts()
    ratios = (counts / len(dfΔ) * 100).round(1)
    dominant = ratios.idxmax()

    # 输出路径
    out_dir = os.path.join(OUT_ROOT, region_name)
    os.makedirs(out_dir, exist_ok=True)

    # ===== 图1：ΔET–ΔT 散点 =====
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(dfΔ["ΔET(mm/day)"], dfΔ["ΔT(°C)"], c='teal', alpha=0.5, s=10)
    xg = np.linspace(dfΔ["ΔET(mm/day)"].min(), dfΔ["ΔET(mm/day)"].max(), 100)
    ax.plot(xg, intercept + slope * xg, 'r-', lw=2,
            label=f"ΔT = {slope:.2f}·ΔET + {intercept:.2f}\nR²={r2:.2f}, p={p:.3f}")
    ax.axhline(0, color='k', ls='--', alpha=0.5)
    ax.axvline(0, color='k', ls='--', alpha=0.5)
    ax.set_xlabel("ΔET (mm/day)")
    ax.set_ylabel("ΔT (°C)")
    ax.set_title(f"{region_name} 区域 ΔET–ΔT 散点与回归")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f"{region_name}_scatter.png"), dpi=180)
    plt.close(fig)

    # ===== 图2：反馈强度直方图 =====
    fig2, ax2 = plt.subplots(figsize=(7, 5))
    ax2.hist(dfΔ["反馈强度系数(°C_per_mm)"].dropna(), bins=25, color='orange', alpha=0.7)
    ax2.set_xlabel("反馈强度系数 (°C/mm)")
    ax2.set_ylabel("事件数")
    ax2.set_title(f"{region_name} 区域反馈强度系数分布")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(os.path.join(out_dir, f"{region_name}_feedback_hist.png"), dpi=180)
    plt.close(fig2)

    # ===== 报告输出 =====
    report = []
    report.append("="*70)
    report.append(f"           {region_name} 区域 土壤–气温 反馈分析报告")
    report.append("="*70)
    report.append(f"事件总数: {len(dfΔ)}")
    report.append(f"空间覆盖: {dfΔ['lat'].nunique()} × {dfΔ['lon'].nunique()} 格点")
    report.append("")
    report.append("一、整体 ΔET–ΔT 回归结果")
    report.append(f"  斜率 β = {slope:.3f} °C/mm/day")
    report.append(f"  截距 α = {intercept:.3f}")
    report.append(f"  R² = {r2:.3f}, p值 = {p:.3f}")
    report.append("")
    report.append("二、机制分布（四象限分类）")
    for mech, ratio in ratios.items():
        report.append(f"  {mech}: {counts[mech]} 个 ({ratio}%)")
    report.append("")
    report.append(f"三、主导机制：{dominant}")
    if "干旱驱动" in dominant:
        report.append("→ 干旱先行、升温响应明显，存在正反馈。")
    elif "热浪驱动" in dominant:
        report.append("→ 升温在先、蒸散被动下降，为反向驱动。")
    else:
        report.append("→ 无显著单向主导反馈。")
    report.append("")
    report.append(f"平均反馈强度系数: {dfΔ['反馈强度系数(°C_per_mm)'].mean():.3f} °C/mm")
    report.append(f"图表与结果已保存至 {out_dir}/")
    report.append("="*70)

    with open(os.path.join(out_dir, f"{region_name}_report.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(report))

    dfΔ.to_csv(os.path.join(out_dir, f"{region_name}_event_feedback.csv"),
               index=False, encoding="utf-8-sig")

    print(f"✅ {region_name} 分析完成，共 {len(dfΔ)} 个事件。")


# ==================== 主循环：遍历所有 region 文件夹 ====================
for region_name in sorted(os.listdir(BASE_DIR)):
    region_path = os.path.join(BASE_DIR, region_name)
    if not os.path.isdir(region_path):
        continue

    # 搜索该 region 下的所有 csv 文件
    csv_files = [os.path.join(region_path, f)
                 for f in os.listdir(region_path) if f.endswith(".csv")]
    if len(csv_files) == 0:
        print(f"⚠️ {region_name} 无 CSV 文件，跳过。")
        continue

    print(f"\n📂 正在处理 {region_name} ({len(csv_files)} 个文件)...")

    # 合并多个 part/final 文件
    df_list = []
    for fpath in csv_files:
        try:
            df_tmp = pd.read_csv(fpath)
            df_list.append(df_tmp)
        except Exception as e:
            print(f"❌ 文件 {fpath} 读取失败: {e}")
    if len(df_list) == 0:
        continue
    df_region = pd.concat(df_list, ignore_index=True)

    # 分析
    analyze_region(region_name, df_region)

print("\n🎯 全部区域处理完成！结果保存在:", OUT_ROOT)


### Merge regional event CSVs into global DeltaET-DeltaT table

Source: `D:\smq\paper2\code\code.ipynb` cell `39`.


In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import os

BASE_DIR = r"F:\smq\paper2\CDHW_pre3days_by_band"
OUT_DIR = r"D:\smq\paper2\data"
os.makedirs(OUT_DIR, exist_ok=True)

records = []

for region_name in sorted(os.listdir(BASE_DIR)):
    region_path = os.path.join(BASE_DIR, region_name)
    if not os.path.isdir(region_path):
        continue
    csv_files = [os.path.join(region_path, f)
                 for f in os.listdir(region_path) if f.endswith(".csv")]
    for f in csv_files:
        try:
            df = pd.read_csv(f)
            df["time"] = pd.to_datetime(df["time"])
            df = df.sort_values(["lat", "lon", "event_index", "time"]).reset_index(drop=True)
            df["Evap"] *= 1000
            df["Trans"] *= 1000
            df["ET"] = df["Evap"] + df["Trans"]

            for (lat, lon, eid), g in df.groupby(["lat", "lon", "event_index"]):
                if len(g) < 4:
                    continue
                g = g.sort_values("time")
                ET1, ET4 = g["ET"].iloc[0], g["ET"].iloc[-1]
                T1, T4 = g["temperature"].iloc[0], g["temperature"].iloc[-1]
                records.append({"lat": lat, "lon": lon, "ΔET(mm/day)": ET4 - ET1, "ΔT(°C)": T4 - T1})
        except Exception as e:
            print(f"❌ 文件 {f} 处理失败: {e}")

dfΔ = pd.DataFrame(records)
dfΔ.to_csv(os.path.join(OUT_DIR, "delta_ET_T_all.csv"), index=False)
print(f"✅ 已保存 ΔET–ΔT 数据，共 {len(dfΔ)} 条记录。")


### Global DeltaET-DeltaT pathway analysis




In [ ]:
# -*- coding: utf-8 -*-
"""
全区域合并 CDHW 事件级 ΔET–ΔT 反馈分析
遍历所有 region 文件夹（region_1, region_2, ...），
合并所有 CSV 文件后统一执行反馈分析。
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.stats import linregress

# ==================== 全局设置 ====================
BASE_DIR = r"F:\smq\paper2\CDHW_pre3days_by_band"   # 根目录
OUT_DIR = r"F:\smq\paper2\CDHW_feedback_results_all"  # 输出目录（单一）
os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False


# ==================== 定义分析函数 ====================
def analyze_all(df):
    """对全区域合并 DataFrame 执行 ΔET–ΔT 分析"""
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values(["lat", "lon", "event_index", "time"]).reset_index(drop=True)

    # 单位转换 (m → mm/day)
    for col in ["Evap", "Trans", "ETRef", "precipitation"]:
        if col in df.columns:
            df[col] = df[col] * 1000

    df["ET"] = df["Evap"] + df["Trans"]

    records = []
    eid_global = 0

    for (lat, lon, eid), g in df.groupby(["lat", "lon", "event_index"]):
        g = g.sort_values("time")
        if len(g) < 4:
            continue
        ET1, ET4 = g["ET"].iloc[0], g["ET"].iloc[-1]
        T1, T4 = g["temperature"].iloc[0], g["temperature"].iloc[-1]
        ΔET = ET4 - ET1
        ΔT = T4 - T1
        coeff = ΔT / abs(ΔET) if abs(ΔET) > 1e-8 else np.nan
        eid_global += 1
        records.append({
            "event_id": eid_global,
            "lat": lat,
            "lon": lon,
            "ΔET(mm/day)": ΔET,
            "ΔT(°C)": ΔT,
            "反馈强度系数(°C_per_mm)": coeff,
            "ET_day1": ET1,
            "ET_day4": ET4,
            "T_day1": T1,
            "T_day4": T4
        })

    if len(records) == 0:
        print("⚠️ 无有效事件，结束分析。")
        return

    dfΔ = pd.DataFrame(records)

    # 四象限分类
    def classify(row):
        if row["ΔET(mm/day)"] < 0 and row["ΔT(°C)"] > 0:
            return "干旱驱动升温"
        elif row["ΔET(mm/day)"] > 0 and row["ΔT(°C)"] < 0:
            return "热浪驱动蒸散下降"
        elif row["ΔET(mm/day)"] < 0 and row["ΔT(°C)"] < 0:
            return "同步降温干化"
        elif row["ΔET(mm/day)"] > 0 and row["ΔT(°C)"] > 0:
            return "同步升温湿化"
        else:
            return "不确定"

    dfΔ["机制分类"] = dfΔ.apply(classify, axis=1)

    # 回归分析
    slope, intercept, r, p, _ = linregress(dfΔ["ΔET(mm/day)"], dfΔ["ΔT(°C)"])
    r2 = r**2
    counts = dfΔ["机制分类"].value_counts()
    ratios = (counts / len(dfΔ) * 100).round(1)
    dominant = ratios.idxmax()

    # ===== 图1：ΔET–ΔT 散点 =====
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(dfΔ["ΔET(mm/day)"], dfΔ["ΔT(°C)"], c='teal', alpha=0.5, s=10)
    xg = np.linspace(dfΔ["ΔET(mm/day)"].min(), dfΔ["ΔET(mm/day)"].max(), 100)
    ax.plot(xg, intercept + slope * xg, 'r-', lw=2,
            label=f"ΔT = {slope:.2f}·ΔET + {intercept:.2f}\nR²={r2:.2f}, p={p:.3f}")
    ax.axhline(0, color='k', ls='--', alpha=0.5)
    ax.axvline(0, color='k', ls='--', alpha=0.5)
    ax.set_xlabel("ΔET (mm/day)")
    ax.set_ylabel("ΔT (°C)")
    ax.set_title("全区域 ΔET–ΔT 散点与回归")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "Global_scatter.png"), dpi=180)
    plt.close(fig)

    # ===== 图2：反馈强度直方图 =====
    fig2, ax2 = plt.subplots(figsize=(7, 5))
    ax2.hist(dfΔ["反馈强度系数(°C_per_mm)"].dropna(), bins=25, color='orange', alpha=0.7)
    ax2.set_xlabel("反馈强度系数 (°C/mm)")
    ax2.set_ylabel("事件数")
    ax2.set_title("全区域反馈强度系数分布")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(os.path.join(OUT_DIR, "Global_feedback_hist.png"), dpi=180)
    plt.close(fig2)

    # ===== 报告输出 =====
    report = []
    report.append("="*70)
    report.append("           全区域 土壤–气温 反馈分析报告")
    report.append("="*70)
    report.append(f"事件总数: {len(dfΔ)}")
    report.append(f"空间覆盖: {dfΔ['lat'].nunique()} × {dfΔ['lon'].nunique()} 格点")
    report.append("")
    report.append("一、整体 ΔET–ΔT 回归结果")
    report.append(f"  斜率 β = {slope:.3f} °C/mm/day")
    report.append(f"  截距 α = {intercept:.3f}")
    report.append(f"  R² = {r2:.3f}, p值 = {p:.3f}")
    report.append("")
    report.append("二、机制分布（四象限分类）")
    for mech, ratio in ratios.items():
        report.append(f"  {mech}: {counts[mech]} 个 ({ratio}%)")
    report.append("")
    report.append(f"三、主导机制：{dominant}")
    if "干旱驱动" in dominant:
        report.append("→ 干旱先行、升温响应明显，存在正反馈。")
    elif "热浪驱动" in dominant:
        report.append("→ 升温在先、蒸散被动下降，为反向驱动。")
    else:
        report.append("→ 无显著单向主导反馈。")
    report.append("")
    report.append(f"平均反馈强度系数: {dfΔ['反馈强度系数(°C_per_mm)'].mean():.3f} °C/mm")
    report.append(f"图表与结果已保存至 {OUT_DIR}/")
    report.append("="*70)

    with open(os.path.join(OUT_DIR, "Global_feedback_report.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(report))

    dfΔ.to_csv(os.path.join(OUT_DIR, "Global_event_feedback.csv"),
               index=False, encoding="utf-8-sig")

    print(f"\n✅ 全区域分析完成，共 {len(dfΔ)} 个事件。结果保存在: {OUT_DIR}")


# ==================== 主循环：读取并合并所有 CSV ====================
all_dfs = []
for region_name in sorted(os.listdir(BASE_DIR)):
    region_path = os.path.join(BASE_DIR, region_name)
    if not os.path.isdir(region_path):
        continue
    csv_files = [os.path.join(region_path, f)
                 for f in os.listdir(region_path) if f.endswith(".csv")]
    if not csv_files:
        continue
    for f in csv_files:
        try:
            df_tmp = pd.read_csv(f)
            df_tmp["region"] = region_name
            all_dfs.append(df_tmp)
        except Exception as e:
            print(f"❌ 文件 {f} 读取失败: {e}")

if not all_dfs:
    print("❌ 未找到任何 CSV 文件。")
else:
    df_all = pd.concat(all_dfs, ignore_index=True)
    print(f"📦 合并完成，共 {len(df_all)} 条记录，来自 {len(all_dfs)} 个文件。")
    analyze_all(df_all)


### Global hydrothermal pathway composition

Source: `D:\smq\paper2\code\fig.ipynb` cell `26`.


In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import matplotlib.pyplot as plt
import os
import matplotlib as mpl

# 字体设置
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"

# ===============================
# 1. 读取数据
# ===============================
IN_FILE = r"F:\smq\paper2\CDHW_feedback_results_all\Global_event_feedback.csv"
OUT_DIR = r"D:\smq\paper2\result\summary_pie"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(IN_FILE)
print(f"📂 Loaded {len(df)} feedback events.")

# ===============================
# 2. 分类逻辑
# ===============================
cond_dry_hot = (df["ΔET(mm/day)"] < 0) & (df["ΔT(°C)"] > 0)
cond_wet_hot = (df["ΔET(mm/day)"] > 0) & (df["ΔT(°C)"] > 0)
cond_other   = ~(cond_dry_hot | cond_wet_hot)

counts = [
    cond_dry_hot.sum(),
    cond_wet_hot.sum(),
    cond_other.sum()
]
labels = [
    "Drought-induced \nwarming",      # 干旱驱动升温  
    "Synchronous \nwarming–wetting",  # 同步升温湿化  
    "Others \n(undetected)"           # 其他（未检出）
]
# colors = ['#6182cc','#b9d8f7', '#e0eff2' ]
colors = ['#f03c20', '#feb14c', '#ffeeaa']

# ===============================
# 3. 绘制饼状图（带黑边）
# ===============================
fig, ax = plt.subplots(figsize=(3, 3))
wedges, texts, autotexts = ax.pie(
    counts, labels=labels, colors=colors,
    autopct=lambda p: f'{p:.1f}%',  # 显示百分比
    startangle=90, counterclock=False,
    textprops={'fontsize': 9, 'color': 'black'},
    wedgeprops={'edgecolor': 'black', 'linewidth': 1.2}  # ✅ 黑边
)
plt.text(0.5, 1.1, "Global Proportion of Feedback \nMechanisms in CDHW Events", transform=ax.transAxes,
         fontsize=12.5, fontfamily='Arial', fontweight='bold',
         ha='center', va='bottom')

# 调整文字样式
for t in texts + autotexts:
    t.set_fontweight('bold')

plt.tight_layout()

# ===============================
# 4. 保存图像
# ===============================
save_path = os.path.join(OUT_DIR, "Global_feedback_pie.pdf")
plt.savefig(save_path, format='pdf', bbox_inches='tight', dpi=300)
plt.show()

print(f"✅ 饼图已保存至：{save_path}")


### Regional hydrothermal pathway composition




In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import os
import re

mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"

# 区域代码到名称映射
region_code_to_name = {
    1:  "ALA", 2:  "AMZ", 3:  "CAM", 4:  "CAR", 5:  "CAS",
    6:  "CEU", 7:  "CGI", 8:  "CNA", 9:  "EAF", 10: "EAS",
    11: "ENA", 12: "MED", 13: "NAS", 14: "NAU", 15: "NEB",
    16: "NEU", 17: "SAF", 18: "SAH", 19: "SAS", 20: "SAU",
    21: "SEA", 22: "SSA", 23: "TIB", 24: "WAF", 25: "WAS",
    26: "WNA", 27: "WSA"
}

# ==================== 路径设置 ====================
RESULT_DIR = r"F:\smq\paper2\CDHW_feedback_results"
OUT_DIR = os.path.join(r"D:\smq\paper2\result", "summary_pie")
os.makedirs(OUT_DIR, exist_ok=True)

# ==================== 汇总与绘图 ====================
summary_records = []

for region_folder in sorted(os.listdir(RESULT_DIR)):
    region_path = os.path.join(RESULT_DIR, region_folder)
    if not os.path.isdir(region_path):
        continue

    csv_file = os.path.join(region_path, f"{region_folder}_event_feedback.csv")
    if not os.path.exists(csv_file):
        continue

    try:
        # ===== 提取 region_数字 → 英文简称 =====
        match = re.search(r"region_(\d+)", region_folder)
        if match:
            region_code = int(match.group(1))
            region_name = region_code_to_name.get(region_code, f"R{region_code}")
        else:
            region_name = region_folder

        # ===== 数据读取 =====
        df = pd.read_csv(csv_file)
        counts = df["机制分类"].value_counts()
        total = counts.sum()

        dry = counts.get("干旱驱动升温", 0)
        wet = counts.get("同步升温湿化", 0)
        other = total - dry - wet

        p_dry = dry / total * 100 if total > 0 else 0
        p_wet = wet / total * 100 if total > 0 else 0
        p_other = other / total * 100 if total > 0 else 0

        summary_records.append({
            "region": region_name,
            "干旱驱动升温(%)": p_dry,
            "同步升温湿化(%)": p_wet,
            "未检出(%)": p_other,
            "事件总数": total
        })

        # ===============================
        # 绘制无标签饼状图
        # ===============================
        colors = ['#f03c20', '#feb14c', '#ffeeaa']
        values = [dry, wet, other]

        fig, ax = plt.subplots(figsize=(1, 1))
        ax.pie(
            values,
            colors=colors,
            startangle=90,
            counterclock=False,
            wedgeprops={'edgecolor': 'black', 'linewidth': 1.0}
        )

        # 关闭所有标签和比例文字
        plt.axis('off')
        
        plt.text(0.5, 1, f"{region_name}", transform=ax.transAxes,
                 fontsize=11, fontfamily='Arial', fontweight='bold',
                 ha='center', va='bottom')
        
        # 保存文件
        out_path = os.path.join(OUT_DIR, f"{region_name}_pie.pdf")
        plt.tight_layout()
        plt.savefig(out_path, dpi=300, bbox_inches="tight", format="pdf")
        plt.close()

        print(f"✅ {region_name} 饼图完成。")

    except Exception as e:
        print(f"❌ {region_folder} 处理失败: {e}")

print("\n🎯 全部饼图绘制完成！")
print(f"📁 结果目录: {OUT_DIR}")


### Fig. 5a global DeltaET-DeltaT scatter/regression




In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import os
import matplotlib as mpl

mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"

IN_FILE = r"D:\smq\paper2\data\delta_ET_T_all.csv"
OUT_DIR = r"D:\smq\paper2\result"
os.makedirs(OUT_DIR, exist_ok=True)

dfΔ = pd.read_csv(IN_FILE)
print(f"📂 Loaded {len(dfΔ)} ΔET–ΔT data points.")

# 数据筛选（可选）
# dfΔ = dfΔ[(dfΔ["ΔET(mm/day)"].between(-5,5)) & (dfΔ["ΔT(°C)"].between(-5,5))]

# 回归分析
slope, intercept, r, p, _ = linregress(dfΔ["ΔET(mm/day)"], dfΔ["ΔT(°C)"])
r2 = r**2

# 绘图
fig, ax = plt.subplots(figsize=(2.5, 2))
ax.scatter(dfΔ["ΔET(mm/day)"], dfΔ["ΔT(°C)"], c='lightgrey', alpha=0.4, s=10, edgecolors='none')

# 绘制回归线
xg = np.linspace(dfΔ["ΔET(mm/day)"].min(), dfΔ["ΔET(mm/day)"].max(), 200)
ax.plot(xg, intercept + slope * xg, 'darkorange', lw=2, 
        label=f"R²={r2:.2f}**")

# 美化坐标轴
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_position(('outward', 5))
ax.spines['bottom'].set_position(('outward', 5))

ax.set_xlim(-10, 15)
ax.set_xticks(np.arange(-10, 16, 10)) 
ax.set_ylim(-20, 30)
ax.set_yticks(np.arange(-20, 30, 20)) 

# 坐标轴标签（保持英文）
ax.set_xlabel("ΔET (mm/day)", fontweight='bold', fontsize=15)
ax.set_ylabel("ΔT (°C)", fontweight='bold', fontsize=16)
ax.axhline(0, color='k', ls='--', alpha=0.3)
ax.axvline(0, color='k', ls='--', alpha=0.3)
ax.tick_params(
    axis='both', which='both',
    direction='out',
    bottom=True, top=False,
    left=True, right=False,
    labelbottom=True, labelleft=True,
    length=6, width=1, color='black', labelsize=14
)

# 显式指定图例位置，设置字体大小为14并移除边框
ax.legend(
    loc='upper left',
    frameon=False,
    prop={'size': 14, 'weight': 'bold'}  # 在这里同时指定字体大小和加粗
)


fig.tight_layout()

# 保存为 PDF
# pdf_save_path = r"D:\smq\paper2\result\global_scatter.pdf"
# os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
# # plt.savefig(pdf_save_path, format='pdf', bbox_inches='tight', dpi=300)
# plt.show()
# print(f"图形已保存为：{pdf_save_path}")

# pdf_save_path = r"D:\smq\paper2\result\global_scatter.tif"
# os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
# plt.savefig(pdf_save_path, format='tif', bbox_inches='tight', dpi=150)
# plt.show()
# print(f"图形已保存为：{pdf_save_path}")

pdf_save_path = r"D:\smq\paper2\result\global_scatter.png"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(pdf_save_path, format='png', bbox_inches='tight', transparent=True,dpi=300)
plt.show()
print(f"图形已保存为：{pdf_save_path}")

# 输出结果（英文）
print(f"\nSlope β = {slope:.3f} °C/mm/day, R² = {r2:.3f}, p = {p:.3f}")

## 7. Event-based factor processing and ExtraTrees-SHAP - final modified code

Main Methods and Supplementary D. Legacy old-notebook SHAP cells are excluded

### run_event_level_original_consistent_m0_m1_m2_shap_parallel

Supplementary D M0/M1/M2 ExtraTrees and SHAP fitting. For paper-facing reproduction, explicitly set the row cap, tree count and SHAP sample size used in the manuscript.




In [ ]:
# Source file: D:\smq\paper2\modify\敏感区\code\run_event_level_original_consistent_m0_m1_m2_shap_parallel.py
# Staging copy used for notebook assembly: D:\software\codex\paper2\office_work\code_notebook_order_20260609\sensitive_code\run_event_level_original_consistent_m0_m1_m2_shap_parallel.py
# Detected encoding: utf-8-sig
# Methods-order section: Supplementary D M0/M1/M2 ExtraTrees and SHAP fitting
# NOTE: This is the final modified sensitive-region/SHAP script, not legacy notebook code.

from __future__ import annotations

import argparse
import json
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from matplotlib.gridspec import GridSpec
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler


ROOT = Path(r"D:\Software\codex\sm_heat_drought")
BASE = ROOT / "outputs" / "20260522_shap_method_rebuild"
EVENT_DIR = BASE / "event_level_original_consistent_by_latblock"
PARTS_DIR = EVENT_DIR / "parts"
OUT = BASE / "event_level_original_consistent_m0_m1_m2_shap_parallel"
DATA_DIR = OUT / "data"
FIG_DIR = OUT / "figures"
REPORT_DIR = OUT / "reports"
CODE_DIR = OUT / "code"

TARGET = "w_sum"
M0_FEATURES = ["spei_sum", "tmax_sum", "precip_sum", "vpd_sum", "et0_sum"]
LAND_FEATURES = ["fracforest", "fracgrassland", "fracirrNonPaddy", "fracirrPaddy", "fracsealed", "fracwater"]
M1_FEATURES = M0_FEATURES + LAND_FEATURES
M2_FEATURES = M1_FEATURES + ["duration"]
MODEL_SPECS = {"M0": M0_FEATURES, "M1": M1_FEATURES, "M2": M2_FEATURES}

FEATURE_LABEL = {
    "spei_sum": "SPEI",
    "tmax_sum": "Tmax",
    "precip_sum": "Pr",
    "vpd_sum": "VPD",
    "et0_sum": "ET0",
    "fracforest": "Forest",
    "fracgrassland": "Grass",
    "fracirrNonPaddy": "IrrNonPaddy",
    "fracirrPaddy": "IrrPaddy",
    "fracsealed": "Sealed",
    "fracwater": "Water",
    "duration": "Duration",
}
FEATURE_BLOCK = {f: "Meteorology" for f in M0_FEATURES}
FEATURE_BLOCK.update({f: "Land cover" for f in LAND_FEATURES})
FEATURE_BLOCK["duration"] = "Event structure"

SREX_ORDER_ORIGINAL = [
    "WSA", "WNA", "WAF", "SSA", "SEA", "SAU", "SAS", "SAF", "NEU", "NEB",
    "NAU", "NAS", "MED", "ENA", "EAS", "EAF", "CNA", "CEU", "CAM", "AMZ",
    "ALA", "CGI", "TIB", "EUS", "CAS", "WAS", "SAH",
]

COLORS = {
    "M0": "#7895B2",
    "M1": "#5FA777",
    "M2": "#C07A4A",
    "Meteorology": "#4E79A7",
    "Land cover": "#59A14F",
    "Event structure": "#9C755F",
    "bar": "#E9E9E9",
}

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "font.size": 8,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "axes.linewidth": 0.8,
        "savefig.transparent": False,
        "image.composite_image": False,
        "pdf.compression": 0,
    }
)


def ensure_dirs() -> None:
    for d in [OUT, DATA_DIR, FIG_DIR, REPORT_DIR, CODE_DIR]:
        d.mkdir(parents=True, exist_ok=True)


def save_all(fig: plt.Figure, stem: Path, dpi: int = 600) -> None:
    fig.savefig(stem.with_suffix(".png"), dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(stem.with_suffix(".tiff"), dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")
    fig.savefig(stem.with_suffix(".svg"), bbox_inches="tight", facecolor="white")


def feature_label(feature: str) -> str:
    return FEATURE_LABEL.get(feature, feature)


def sample_event_table(max_model_rows: int, seed: int) -> Path:
    out_path = DATA_DIR / f"event_level_model_sample_n{max_model_rows}_seed{seed}.parquet"
    if out_path.exists():
        return out_path
    manifest_path = EVENT_DIR / "event_level_table_manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(manifest_path)
    cols = sorted(set([TARGET, "srex_abbr", "srex_num", "lat", "lon"] + M2_FEATURES))
    frames = []
    for p in sorted(PARTS_DIR.glob("*.parquet")):
        frames.append(pd.read_parquet(p, columns=cols))
    df = pd.concat(frames, ignore_index=True)
    df = df.dropna(subset=[TARGET] + M2_FEATURES).copy()
    for col in [TARGET] + M2_FEATURES:
        df[col] = df[col].astype("float32")
    if "srex_abbr" in df.columns:
        df["srex_abbr"] = df["srex_abbr"].astype("string")
    # Preserve the original 80% sampling step, then cap the model sample.
    df = df.sample(frac=0.80, random_state=seed)
    if len(df) > max_model_rows:
        df = df.sample(n=max_model_rows, random_state=seed)
    df = df.reset_index(drop=True)
    df.to_parquet(out_path, index=False, compression="zstd")
    meta = {
        "source_manifest": str(manifest_path),
        "sample_path": str(out_path),
        "rows": int(len(df)),
        "max_model_rows": int(max_model_rows),
        "seed": int(seed),
        "note": "Event-level random sample after the original 80% sampling step.",
    }
    (REPORT_DIR / "event_level_model_sample_manifest.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
    return out_path


def fit_model(
    model_name: str,
    sample_path: Path,
    n_jobs: int,
    shap_sample_size: int,
    seed: int,
    n_estimators: int,
    features: list[str] | None = None,
    output_prefix: str | None = None,
) -> None:
    features = MODEL_SPECS[model_name] if features is None else features
    output_prefix = model_name if output_prefix is None else output_prefix
    df = pd.read_parquet(sample_path, columns=sorted(set([TARGET] + features)))
    X_raw = df[features].astype("float32").copy()
    y = df[TARGET].astype("float32").to_numpy()
    scaler = MinMaxScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=features, index=X_raw.index)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.30, random_state=seed)
    model = ExtraTreesRegressor(
        n_estimators=n_estimators,
        max_depth=25,
        max_features="sqrt",
        bootstrap=False,
        random_state=0,
        n_jobs=n_jobs,
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    perf = {
        "model": model_name,
        "target": TARGET,
        "features": ";".join(features),
        "n_features": len(features),
        "n_model_rows_after_80pct_sample": int(len(df)),
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "r2": float(r2_score(y_test, pred)),
        "rmse": float(mean_squared_error(y_test, pred) ** 0.5),
        "mae": float(mean_absolute_error(y_test, pred)),
        "n_estimators": int(n_estimators),
        "n_jobs": int(n_jobs),
    }
    shap_n = min(shap_sample_size, len(X_train))
    X_shap = X_train.sample(n=shap_n, random_state=seed)
    raw_shap = X_raw.loc[X_shap.index].copy()
    y_shap = pd.Series(y, index=X_scaled.index).loc[X_shap.index].to_numpy()
    explainer = shap.TreeExplainer(model)
    shap_values = np.asarray(explainer.shap_values(X_shap))
    expected_value = np.asarray(explainer.expected_value).reshape(-1)
    mean_abs = np.abs(shap_values).mean(axis=0)
    denom = float(mean_abs.sum())
    imp_rows = []
    for j, feature in enumerate(features):
        corr = np.nan
        if np.std(X_shap[feature].to_numpy()) > 0 and np.std(shap_values[:, j]) > 0:
            corr = float(np.corrcoef(X_shap[feature].to_numpy(), shap_values[:, j])[0, 1])
        imp_rows.append(
            {
                "model": model_name,
                "feature": feature,
                "feature_label": feature_label(feature),
                "block": FEATURE_BLOCK[feature],
                "mean_abs_shap": float(mean_abs[j]),
                "relative_importance": float(mean_abs[j] / denom) if denom else np.nan,
                "shap_feature_corr": corr,
            }
        )
    pd.DataFrame([perf]).to_csv(DATA_DIR / f"global_{output_prefix}_performance.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(imp_rows).to_csv(DATA_DIR / f"global_{output_prefix}_importance.csv", index=False, encoding="utf-8-sig")
    np.savez_compressed(
        DATA_DIR / f"global_{output_prefix}_shap_values.npz",
        shap_values=shap_values.astype("float32"),
        X_shap_scaled=X_shap.to_numpy(dtype="float32"),
        X_shap_raw=raw_shap.to_numpy(dtype="float32"),
        y_shap=y_shap.astype("float32"),
        feature_names=np.array(features, dtype=object),
        feature_labels=np.array([feature_label(f) for f in features], dtype=object),
        expected_value=expected_value.astype("float32"),
        target=np.array([TARGET], dtype=object),
    )


def run_worker(args: argparse.Namespace) -> None:
    ensure_dirs()
    fit_model(args.worker_model, Path(args.sample_path), args.n_jobs, args.shap_sample_size, args.seed, args.n_estimators)


def launch_global_workers(sample_path: Path, args: argparse.Namespace) -> None:
    procs = []
    for model in ["M0", "M1", "M2"]:
        cmd = [
            sys.executable,
            str(Path(__file__)),
            "--worker-model",
            model,
            "--sample-path",
            str(sample_path),
            "--n-jobs",
            str(args.n_jobs_per_model),
            "--shap-sample-size",
            str(args.shap_sample_size),
            "--seed",
            str(args.seed),
            "--n-estimators",
            str(args.n_estimators),
        ]
        log = REPORT_DIR / f"worker_{model}.log"
        with log.open("w", encoding="utf-8") as fh:
            procs.append((model, subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT)))
    failures = []
    for model, proc in procs:
        rc = proc.wait()
        if rc != 0:
            failures.append((model, rc, str(REPORT_DIR / f"worker_{model}.log")))
    if failures:
        raise RuntimeError(f"Global worker failures: {failures}")


def combine_global_results() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    perf = pd.concat([pd.read_csv(DATA_DIR / f"global_{m}_performance.csv") for m in ["M0", "M1", "M2"]], ignore_index=True)
    imp = pd.concat([pd.read_csv(DATA_DIR / f"global_{m}_importance.csv") for m in ["M0", "M1", "M2"]], ignore_index=True)
    block = (
        imp.groupby(["model", "block"], as_index=False)["relative_importance"]
        .sum()
        .pivot(index="model", columns="block", values="relative_importance")
        .fillna(0)
    )
    for col in ["Meteorology", "Land cover", "Event structure"]:
        if col not in block.columns:
            block[col] = 0.0
    comp = perf.copy().set_index("model")
    comp["delta_r2_vs_previous"] = comp["r2"].diff()
    comp["relative_shap_meteorology"] = block["Meteorology"]
    comp["relative_shap_land_cover"] = block["Land cover"]
    comp["relative_shap_event_structure"] = block["Event structure"]
    comp = comp.reset_index()
    perf.to_csv(DATA_DIR / "event_level_model_performance.csv", index=False, encoding="utf-8-sig")
    imp.to_csv(DATA_DIR / "event_level_global_shap_importance.csv", index=False, encoding="utf-8-sig")
    comp.to_csv(DATA_DIR / "event_level_M0_M1_M2_comparison.csv", index=False, encoding="utf-8-sig")
    return perf, imp, comp


def fit_srex_summary(sample_path: Path, min_region_rows: int, max_region_rows: int, seed: int, n_jobs: int, n_estimators: int, shap_sample_size: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = pd.read_parquet(sample_path)
    perfs = []
    imps = []
    regions = [r for r in SREX_ORDER_ORIGINAL if r in set(df["srex_abbr"].dropna())]
    regions += sorted(set(df["srex_abbr"].dropna()) - set(regions))
    for region in regions:
        sub = df[df["srex_abbr"] == region].dropna(subset=[TARGET] + M2_FEATURES).copy()
        if len(sub) < min_region_rows:
            continue
        if len(sub) > max_region_rows:
            sub = sub.sample(n=max_region_rows, random_state=seed)
        tmp_path = DATA_DIR / f"_tmp_srex_{region}.parquet"
        sub.to_parquet(tmp_path, index=False, compression="zstd")
        fit_model(
            "M2",
            tmp_path,
            n_jobs=n_jobs,
            shap_sample_size=min(shap_sample_size, 500),
            seed=seed,
            n_estimators=max(200, min(n_estimators, 400)),
            features=M2_FEATURES,
            output_prefix=f"srex_{region}",
        )
        p = pd.read_csv(DATA_DIR / f"global_srex_{region}_performance.csv")
        p["srex_abbr"] = region
        i = pd.read_csv(DATA_DIR / f"global_srex_{region}_importance.csv")
        i["srex_abbr"] = region
        i["model"] = "M2"
        p["model"] = "M2"
        perfs.append(p)
        imps.append(i)
        tmp_path.unlink(missing_ok=True)
    perf_df = pd.concat(perfs, ignore_index=True) if perfs else pd.DataFrame()
    imp_df = pd.concat(imps, ignore_index=True) if imps else pd.DataFrame()
    if not perf_df.empty:
        perf_df.to_csv(DATA_DIR / "event_level_srex_M2_model_performance.csv", index=False, encoding="utf-8-sig")
    if not imp_df.empty:
        imp_df.to_csv(DATA_DIR / "event_level_srex_M2_shap_importance.csv", index=False, encoding="utf-8-sig")
    return perf_df, imp_df


def beeswarm_y(x: np.ndarray, base_y: float, bin_count: int = 55, step: float = 0.045, max_spread: float = 0.30) -> np.ndarray:
    x = np.asarray(x)
    if len(x) == 0 or not np.isfinite(x).any() or np.nanmin(x) == np.nanmax(x):
        return np.full_like(x, base_y, dtype=float)
    bins = np.linspace(np.nanmin(x), np.nanmax(x), bin_count + 1)
    inds = np.digitize(x, bins) - 1
    y = np.empty_like(x, dtype=float)
    for b in np.unique(inds):
        loc = np.where(inds == b)[0]
        offsets = []
        for k in range(len(loc)):
            if k == 0:
                offsets.append(0)
            else:
                mag = (k + 1) // 2
                sign = 1 if k % 2 else -1
                offsets.append(sign * mag * step)
        y[loc] = base_y + np.clip(np.asarray(offsets), -max_spread, max_spread)
    return y


def load_global_shap_for_plot(model_name: str = "M2") -> tuple[np.ndarray, np.ndarray, list[str], list[str]]:
    z = np.load(DATA_DIR / f"global_{model_name}_shap_values.npz", allow_pickle=True)
    return z["shap_values"], z["X_shap_scaled"], [str(v) for v in z["feature_names"]], [str(v) for v in z["feature_labels"]]


def plot_fig4(global_imp: pd.DataFrame, srex_imp: pd.DataFrame, perf: pd.DataFrame) -> Path:
    shap_values, X_scaled, features, labels = load_global_shap_for_plot("M2")
    imp_m2 = global_imp[global_imp["model"] == "M2"].copy()
    order = imp_m2.sort_values("mean_abs_shap", ascending=False)["feature"].tolist()
    y_positions = np.arange(len(order))[::-1]
    order_idx = [features.index(f) for f in order]
    label_map = dict(zip(features, labels))
    imp_map = dict(zip(imp_m2["feature"], imp_m2["mean_abs_shap"]))
    fig = plt.figure(figsize=(9.2, 13.0))
    gs = GridSpec(2, 1, height_ratios=[0.95, 2.25], hspace=0.35, figure=fig)
    ax = fig.add_subplot(gs[0])
    ax_bar = ax.twiny()
    ax.set_zorder(2)
    ax_bar.set_zorder(1)
    ax.patch.set_alpha(0)
    max_imp = max(imp_map.values()) * 1.22 if imp_map else 1.0
    ax_bar.set_xlim(0, max_imp)
    ax_bar.barh(y_positions, [imp_map[f] for f in order], height=0.78, color=COLORS["bar"], edgecolor="none")
    ax_bar.set_yticks([])
    ax_bar.set_xlabel("Global importance (mean |SHAP|)", fontsize=12, fontweight="bold", labelpad=8, color="#777777")
    ax_bar.tick_params(axis="x", colors="#777777", labelsize=10, width=1.0, length=6)
    ax_bar.spines["top"].set_color("#999999")
    ax_bar.spines["bottom"].set_visible(False)
    ax_bar.spines["left"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)
    cmap = mpl.colormaps["RdYlBu_r"]
    for y, feature, j in zip(y_positions, order, order_idx):
        xs = shap_values[:, j]
        vals = X_scaled[:, j]
        ax.scatter(xs, beeswarm_y(xs, y), c=vals, cmap=cmap, vmin=0, vmax=1, s=18, edgecolor="none", alpha=0.90, rasterized=True)
        ax.axhline(y, color="#D6D6D6", linestyle=(0, (1.2, 3)), linewidth=0.8, zorder=0)
    ax.axvline(0, color="#303030", linewidth=1.5)
    ax.set_yticks(y_positions, [label_map[f] for f in order], fontsize=11, fontweight="bold")
    xlim = np.nanquantile(np.abs(shap_values[:, order_idx]), 0.995) * 1.15
    ax.set_xlim(-xlim, xlim)
    ax.set_ylim(-0.8, len(order) - 0.2)
    ax.set_xlabel("SHAP value (change in fitted prediction of w_sum)", fontsize=13, fontweight="bold")
    cbar = fig.colorbar(mpl.cm.ScalarMappable(cmap=cmap, norm=mpl.colors.Normalize(vmin=0, vmax=1)), ax=ax, fraction=0.035, pad=0.025)
    cbar.set_label("Scaled feature value", fontsize=12, fontweight="bold")
    cbar.set_ticks([0, 1], labels=["Low", "High"])
    ax.text(-0.08, 1.08, "a", transform=ax.transAxes, fontsize=24, fontweight="bold")
    ax2 = fig.add_subplot(gs[1])
    if srex_imp.empty:
        ax2.text(0.5, 0.5, "SREX M2 results not available", ha="center", va="center", transform=ax2.transAxes)
        ax2.axis("off")
    else:
        reg_order = [r for r in SREX_ORDER_ORIGINAL if r in set(srex_imp["srex_abbr"])]
        reg_order += sorted(set(srex_imp["srex_abbr"]) - set(reg_order))
        feature_order = M2_FEATURES
        ymap = {r: i for i, r in enumerate(reg_order[::-1])}
        xmap = {f: i for i, f in enumerate(feature_order)}
        plot_df = srex_imp[srex_imp["feature"].isin(feature_order) & srex_imp["srex_abbr"].isin(reg_order)].copy()
        plot_df["x"] = plot_df["feature"].map(xmap)
        plot_df["y"] = plot_df["srex_abbr"].map(ymap)
        sizes = 32 + plot_df["relative_importance"].clip(lower=0) * 1600
        sc = ax2.scatter(plot_df["x"], plot_df["y"], s=sizes, c=plot_df["shap_feature_corr"], cmap="RdYlBu_r", vmin=-0.9, vmax=0.9, edgecolor="black", linewidth=0.8, alpha=0.95)
        ax2.set_xticks(np.arange(len(feature_order)), [feature_label(f) for f in feature_order], fontsize=9.5, fontweight="bold", rotation=25, ha="right")
        ax2.set_yticks(np.arange(len(reg_order)), reg_order[::-1], fontsize=10.5, fontweight="bold")
        ax2.set_xlim(-0.45, len(feature_order) - 0.55)
        ax2.set_ylim(-0.75, len(reg_order) - 0.25)
        ax2.grid(which="major", color="#D0D0D0", linestyle="--", linewidth=0.8)
        ax2.set_axisbelow(True)
        for spine in ["left", "bottom", "top", "right"]:
            ax2.spines[spine].set_visible(False)
        ax2.tick_params(length=0)
        cax = fig.add_axes([0.84, 0.19, 0.025, 0.22])
        cb = fig.colorbar(sc, cax=cax)
        cb.set_label("Feature-SHAP\ncorrelation", fontsize=11, fontweight="bold")
    ax2.text(-0.08, 1.02, "b", transform=ax2.transAxes, fontsize=24, fontweight="bold")
    perf_map = dict(zip(perf["model"], perf["r2"]))
    fig.text(0.08, 0.018, f"Event-level M2 model. Target: w_sum. Test R2: M0={perf_map.get('M0', np.nan):.3f}, M1={perf_map.get('M1', np.nan):.3f}, M2={perf_map.get('M2', np.nan):.3f}. SHAP values are predictive diagnostics, not causal physical contributions.", fontsize=8.2)
    out = FIG_DIR / "Fig4_event_level_original_consistent_M2_SHAP"
    save_all(fig, out)
    plt.close(fig)
    return out.with_suffix(".png")


def plot_model_comparison(perf: pd.DataFrame, comp: pd.DataFrame) -> Path:
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.7), gridspec_kw={"width_ratios": [0.85, 1.15]})
    x = np.arange(len(perf))
    axes[0].bar(x, perf["r2"], color=[COLORS[m] for m in perf["model"]], width=0.62)
    axes[0].set_xticks(x, perf["model"])
    axes[0].set_ylabel("Test R2")
    axes[0].set_title("a  Nested model skill", loc="left", fontweight="bold")
    axes[0].grid(axis="y", color="#E6E6E6", linewidth=0.6)
    for i, v in enumerate(perf["r2"]):
        axes[0].text(i, v + 0.01, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
    bottom = np.zeros(len(comp))
    for col, label in [("relative_shap_meteorology", "Meteorology"), ("relative_shap_land_cover", "Land cover"), ("relative_shap_event_structure", "Event structure")]:
        vals = comp[col].fillna(0).to_numpy()
        axes[1].bar(x, vals, bottom=bottom, color=COLORS[label], width=0.62, label=label)
        bottom += vals
    axes[1].set_xticks(x, comp["model"])
    axes[1].set_ylim(0, 1)
    axes[1].set_ylabel("Relative mean |SHAP|")
    axes[1].set_title("b  Predictor-block importance", loc="left", fontweight="bold")
    axes[1].legend(frameon=False, fontsize=7.5, loc="upper right")
    axes[1].grid(axis="y", color="#E6E6E6", linewidth=0.6)
    out = FIG_DIR / "FigS_event_level_original_consistent_M0_M1_M2_comparison"
    save_all(fig, out)
    plt.close(fig)
    return out.with_suffix(".png")


def write_run_manifest(args: argparse.Namespace, sample_path: Path, fig4: Path, fig_comp: Path) -> None:
    meta = {
        "target": TARGET,
        "sample_path": str(sample_path),
        "max_model_rows": args.max_model_rows,
        "max_region_rows": args.max_region_rows,
        "n_estimators_global": args.n_estimators,
        "n_jobs_per_model": args.n_jobs_per_model,
        "shap_sample_size": args.shap_sample_size,
        "interpretation": "SHAP values are model-based predictive diagnostics for fitted w_sum predictions, not causal physical contributions.",
        "outputs": {"fig4": str(fig4), "model_comparison": str(fig_comp)},
    }
    (REPORT_DIR / "run_manifest.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
    shutil.copy2(Path(__file__), CODE_DIR / Path(__file__).name)


def run_main(args: argparse.Namespace) -> None:
    ensure_dirs()
    sample_path = sample_event_table(args.max_model_rows, args.seed)
    launch_global_workers(sample_path, args)
    perf, imp, comp = combine_global_results()
    srex_perf, srex_imp = fit_srex_summary(sample_path, args.min_region_rows, args.max_region_rows, args.seed, args.n_jobs_region, args.n_estimators_region, args.shap_sample_size)
    fig4 = plot_fig4(imp, srex_imp, perf)
    fig_comp = plot_model_comparison(perf, comp)
    write_run_manifest(args, sample_path, fig4, fig_comp)
    print(f"Wrote {fig4}")
    print(f"Wrote {fig_comp}")
    print(DATA_DIR)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--worker-model", choices=["M0", "M1", "M2"], default=None)
    parser.add_argument("--sample-path", type=str, default=None)
    parser.add_argument("--max-model-rows", type=int, default=200000)
    parser.add_argument("--max-region-rows", type=int, default=50000)
    parser.add_argument("--min-region-rows", type=int, default=500)
    parser.add_argument("--shap-sample-size", type=int, default=2000)
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--n-estimators", type=int, default=500)
    parser.add_argument("--n-estimators-region", type=int, default=300)
    parser.add_argument("--n-jobs-per-model", type=int, default=6)
    parser.add_argument("--n-jobs", type=int, default=6)
    parser.add_argument("--n-jobs-region", type=int, default=6)
    args = parser.parse_args()
    if args.shap_sample_size > 2000:
        raise ValueError("shap-sample-size must be <= 2000")
    return args


if __name__ == "__main__":
    parsed = parse_args()
    if parsed.worker_model:
        if parsed.sample_path is None:
            raise ValueError("--sample-path is required for worker mode")
        run_worker(parsed)
    else:
        run_main(parsed)


### redraw_fig4_rank_scaled_colors

Main Fig. 4 final rank-color SHAP redraw. Uses within-feature percentile colors for visual comparability.




In [ ]:
# Source file: D:\smq\paper2\modify\敏感区\code\redraw_fig4_rank_scaled_colors.py
# Staging copy used for notebook assembly: D:\software\codex\paper2\office_work\code_notebook_order_20260609\sensitive_code\redraw_fig4_rank_scaled_colors.py
# Detected encoding: utf-8-sig
# Methods-order section: Main Fig. 4 final rank-color SHAP redraw
# NOTE: This is the final modified sensitive-region/SHAP script, not legacy notebook code.

from __future__ import annotations

import shutil
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec


ROOT = Path(r"D:\Software\codex\sm_heat_drought")
BASE = ROOT / "outputs" / "20260522_shap_method_rebuild" / "event_level_original_consistent_m0_m1_m2_shap_parallel"
DATA_DIR = BASE / "data"
FIG_DIR = BASE / "figures"
PACKAGE_DIR = Path(r"E:\smq\aa\敏感区")
PACKAGE_FIG_DIR = PACKAGE_DIR / "figures"
PACKAGE_CODE_DIR = PACKAGE_DIR / "code"

M0_FEATURES = ["spei_sum", "tmax_sum", "precip_sum", "vpd_sum", "et0_sum"]
LAND_FEATURES = ["fracforest", "fracgrassland", "fracirrNonPaddy", "fracirrPaddy", "fracsealed", "fracwater"]
M2_FEATURES = M0_FEATURES + LAND_FEATURES + ["duration"]
SPARSE_LAND_FEATURES = {"fracirrNonPaddy", "fracirrPaddy", "fracsealed", "fracwater"}

FEATURE_LABEL = {
    "spei_sum": "SPEI",
    "tmax_sum": "Tmax",
    "precip_sum": "Pr",
    "vpd_sum": "VPD",
    "et0_sum": "ET0",
    "fracforest": "Forest",
    "fracgrassland": "Grass",
    "fracirrNonPaddy": "IrrNonPaddy",
    "fracirrPaddy": "IrrPaddy",
    "fracsealed": "Sealed",
    "fracwater": "Water",
    "duration": "Duration",
}

SREX_ORDER_ORIGINAL = [
    "WSA", "WNA", "WAF", "SSA", "SEA", "SAU", "SAS", "SAF", "NEU", "NEB",
    "NAU", "NAS", "MED", "ENA", "EAS", "EAF", "CNA", "CEU", "AMZ", "ALA",
    "CGI", "TIB", "CAS", "WAS", "SAH",
]


mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "sans-serif"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.linewidth": 0.8,
    "font.size": 8,
})


def feature_label(feature: str) -> str:
    return FEATURE_LABEL.get(feature, feature)


def empirical_percentile(x: np.ndarray) -> np.ndarray:
    """Return approximately uniform within-feature ranks in [0, 1]."""
    x = np.asarray(x, dtype=float)
    out = np.full(x.shape, np.nan, dtype=float)
    valid = np.isfinite(x)
    if valid.sum() <= 1:
        out[valid] = 0.5
        return out
    s = pd.Series(x[valid])
    out[valid] = s.rank(method="average", pct=True).to_numpy()
    return np.clip(out, 0, 1)


def color_values_for_feature(feature: str, raw: np.ndarray) -> np.ndarray:
    """Use rank colors; keep true zeros visible for sparse land-cover fractions."""
    raw = np.asarray(raw, dtype=float)
    if feature in SPARSE_LAND_FEATURES:
        out = np.zeros(raw.shape, dtype=float)
        nonzero = np.isfinite(raw) & (raw > 1e-6)
        if nonzero.sum() > 1:
            # Expand the non-zero tail without pretending zeros are comparable
            # to small but real land-cover fractions.
            out[nonzero] = 0.18 + 0.82 * empirical_percentile(raw[nonzero])
        elif nonzero.sum() == 1:
            out[nonzero] = 1.0
        out[~np.isfinite(raw)] = np.nan
        return out
    return empirical_percentile(raw)


def beeswarm_y(x: np.ndarray, base_y: float, max_width: float = 0.20, bins: int = 90) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    y = np.full_like(x, base_y, dtype=float)
    valid = np.isfinite(x)
    if valid.sum() == 0 or np.nanmin(x[valid]) == np.nanmax(x[valid]):
        return y
    edges = np.linspace(np.nanquantile(x[valid], 0.005), np.nanquantile(x[valid], 0.995), bins + 1)
    bin_id = np.digitize(x, edges) - 1
    for b in np.unique(bin_id[valid]):
        idx = np.where(valid & (bin_id == b))[0]
        n = len(idx)
        if n <= 1:
            continue
        # Density-style beeswarm: dense SHAP bins become thicker, sparse bins
        # remain close to the center line. This mimics the cumulative thickness
        # of standard SHAP beeswarm plots without changing x values.
        local_width = max_width * min(1.0, np.sqrt(n / 28.0))
        if n == 2:
            offsets = np.array([-0.35, 0.35]) * local_width
        else:
            step = min(local_width / max(1, np.ceil(n / 2)), 0.055)
            pattern = [0.0]
            k = 1
            while len(pattern) < n:
                pattern.extend([k * step, -k * step])
                k += 1
            offsets = np.array(pattern[:n])
            offsets = np.clip(offsets, -local_width, local_width)
        # Stable ordering avoids random-looking noise while keeping rows readable.
        idx_sorted = idx[np.argsort(x[idx], kind="mergesort")]
        offsets = offsets[np.argsort(np.abs(offsets), kind="mergesort")]
        y[idx_sorted] = base_y + offsets[:n]
    return y


def load_global_shap() -> tuple[np.ndarray, np.ndarray, list[str], list[str]]:
    z = np.load(DATA_DIR / "global_M2_shap_values.npz", allow_pickle=True)
    return (
        z["shap_values"],
        z["X_shap_raw"],
        [str(v) for v in z["feature_names"]],
        [str(v) for v in z["feature_labels"]],
    )


def draw() -> Path:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    PACKAGE_FIG_DIR.mkdir(parents=True, exist_ok=True)
    PACKAGE_CODE_DIR.mkdir(parents=True, exist_ok=True)

    global_imp = pd.read_csv(DATA_DIR / "event_level_global_shap_importance.csv")
    srex_imp = pd.read_csv(DATA_DIR / "event_level_srex_M2_shap_importance.csv")
    perf = pd.read_csv(DATA_DIR / "event_level_model_performance.csv")
    shap_values, x_raw, features, labels = load_global_shap()

    imp_m2 = global_imp[global_imp["model"] == "M2"].copy()
    order = imp_m2.sort_values("mean_abs_shap", ascending=False)["feature"].tolist()
    y_positions = np.arange(len(order))[::-1]
    order_idx = [features.index(f) for f in order]
    label_map = dict(zip(features, labels))
    imp_map = dict(zip(imp_m2["feature"], imp_m2["mean_abs_shap"]))

    fig = plt.figure(figsize=(13.4, 14.2))
    gs = GridSpec(2, 1, height_ratios=[1.28, 2.25], hspace=0.34, figure=fig)
    ax = fig.add_subplot(gs[0])
    ax_bar = ax.twiny()
    ax_bar.set_zorder(0)
    ax.set_zorder(2)
    ax.patch.set_alpha(0)

    max_imp = max(imp_map.values()) * 1.22 if imp_map else 1.0
    ax_bar.set_xlim(0, max_imp)
    ax_bar.barh(
        y_positions,
        [imp_map[f] for f in order],
        height=0.82,
        color="#D8D8D8",
        edgecolor="none",
        zorder=0,
    )
    ax_bar.set_yticks([])
    ax_bar.set_xlabel("")
    ax_bar.tick_params(axis="x", colors="#777777", labelsize=10, width=1.0, length=6)
    ax_bar.spines["top"].set_color("#999999")
    ax_bar.spines["bottom"].set_visible(False)
    ax_bar.spines["left"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)

    cmap = mpl.colormaps["RdYlBu_r"]
    for y, feature, j in zip(y_positions, order, order_idx):
        xs = shap_values[:, j]
        raw_vals = x_raw[:, j]
        color_vals = color_values_for_feature(feature, raw_vals)
        ax.scatter(
            xs,
            beeswarm_y(xs, y),
            c=color_vals,
            cmap=cmap,
            vmin=0,
            vmax=1,
            s=5.5,
            edgecolor="none",
            alpha=0.78,
            zorder=3,
            rasterized=True,
        )
        ax.axhline(y, color="#D6D6D6", linestyle=(0, (1.2, 3)), linewidth=0.8, zorder=1)

    ax.axvline(0, color="#303030", linewidth=1.5)
    ax.set_yticks(y_positions, [label_map[f] for f in order], fontsize=11, fontweight="bold")
    xlim = np.nanquantile(np.abs(shap_values[:, order_idx]), 0.995) * 1.15
    ax.set_xlim(-xlim, xlim)
    ax.set_ylim(-0.8, len(order) - 0.2)
    ax.set_xlabel("SHAP value (change in fitted prediction of w_sum)", fontsize=13, fontweight="bold")
    cbar = fig.colorbar(mpl.cm.ScalarMappable(cmap=cmap, norm=mpl.colors.Normalize(vmin=0, vmax=1)), ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Within-feature percentile value", fontsize=12, fontweight="bold")
    cbar.set_ticks([0, 1], labels=["Low", "High"])
    ax.text(-0.08, 1.08, "a", transform=ax.transAxes, fontsize=24, fontweight="bold")

    ax2 = fig.add_subplot(gs[1])
    reg_order = [r for r in SREX_ORDER_ORIGINAL if r in set(srex_imp["srex_abbr"])]
    reg_order += sorted(set(srex_imp["srex_abbr"]) - set(reg_order))
    feature_order = M2_FEATURES
    ymap = {r: i for i, r in enumerate(reg_order[::-1])}
    xmap = {f: i for i, f in enumerate(feature_order)}
    plot_df = srex_imp[srex_imp["feature"].isin(feature_order) & srex_imp["srex_abbr"].isin(reg_order)].copy()
    plot_df["x"] = plot_df["feature"].map(xmap)
    plot_df["y"] = plot_df["srex_abbr"].map(ymap)
    sizes = 32 + plot_df["relative_importance"].clip(lower=0) * 1600
    sc = ax2.scatter(
        plot_df["x"],
        plot_df["y"],
        s=sizes,
        c=plot_df["shap_feature_corr"],
        cmap="RdYlBu_r",
        vmin=-0.9,
        vmax=0.9,
        edgecolor="black",
        linewidth=0.8,
        alpha=0.95,
    )
    ax2.set_xticks(np.arange(len(feature_order)), [feature_label(f) for f in feature_order], fontsize=9.5, fontweight="bold", rotation=25, ha="right")
    ax2.set_yticks(np.arange(len(reg_order)), reg_order[::-1], fontsize=10.5, fontweight="bold")
    ax2.set_xlim(-0.45, len(feature_order) - 0.55)
    ax2.set_ylim(-0.75, len(reg_order) - 0.25)
    ax2.grid(which="major", color="#D0D0D0", linestyle="--", linewidth=0.8)
    ax2.set_axisbelow(True)
    for spine in ["left", "bottom", "top", "right"]:
        ax2.spines[spine].set_visible(False)
    ax2.tick_params(length=0)
    cax = fig.add_axes([0.875, 0.18, 0.022, 0.22])
    cb = fig.colorbar(sc, cax=cax)
    cb.set_label("Feature-SHAP\ncorrelation", fontsize=11, fontweight="bold")

    # Bubble-size legend for regional relative mean |SHAP| in panel b.
    # Use the same area scaling as the scatter plot: size = 32 + importance * 1600.
    size_levels = [0.05, 0.10, 0.20]
    handles = [
        ax2.scatter(
            [],
            [],
            s=32 + level * 1600,
            facecolor="#E8E8E8",
            edgecolor="black",
            linewidth=0.8,
        )
        for level in size_levels
    ]
    labels = [f"{level * 100:.0f}%" for level in size_levels]
    size_leg = ax2.legend(
        handles,
        labels,
        title="Relative\nmean |SHAP|",
        loc="upper left",
        bbox_to_anchor=(0.88, 0.47),
        bbox_transform=fig.transFigure,
        frameon=False,
        labelspacing=1.15,
        handletextpad=1.1,
        borderaxespad=0.0,
        scatterpoints=1,
        fontsize=8.5,
        title_fontsize=9.0,
    )
    ax2.add_artist(size_leg)
    ax2.text(-0.08, 1.02, "b", transform=ax2.transAxes, fontsize=24, fontweight="bold")

    perf_map = dict(zip(perf["model"], perf["r2"]))

    out = FIG_DIR / "Fig4_event_level_original_consistent_M2_SHAP_rankcolor"
    fig.savefig(out.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")
    fig.savefig(out.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")
    plt.close(fig)

    for ext in [".png", ".pdf"]:
        shutil.copy2(out.with_suffix(ext), PACKAGE_FIG_DIR / out.with_suffix(ext).name)
    shutil.copy2(Path(__file__), PACKAGE_CODE_DIR / Path(__file__).name)
    return out.with_suffix(".png")


if __name__ == "__main__":
    print(draw())


### draw_fig4_shap_dependence_plot

Supplementary D SHAP dependence diagnostics. Directionality and nonlinear fitted-response diagnostics.




In [ ]:
# Source file: D:\smq\paper2\modify\敏感区\code\draw_fig4_shap_dependence_plots.py
# Staging copy used for notebook assembly: D:\software\codex\paper2\office_work\code_notebook_order_20260609\sensitive_code\draw_fig4_shap_dependence_plots.py
# Detected encoding: utf-8-sig
# Methods-order section: Supplementary D SHAP dependence diagnostics
# NOTE: This is the final modified sensitive-region/SHAP script, not legacy notebook code.

from __future__ import annotations

import shutil
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


ROOT = Path(r"D:\Software\codex\sm_heat_drought")
BASE = ROOT / "outputs" / "20260522_shap_method_rebuild" / "event_level_original_consistent_m0_m1_m2_shap_parallel"
DATA_DIR = BASE / "data"
FIG_DIR = BASE / "figures"
PACKAGE_DIR = Path(r"E:\smq\aa\敏感区")
PACKAGE_FIG_DIR = PACKAGE_DIR / "figures"
PACKAGE_CODE_DIR = PACKAGE_DIR / "code"

FEATURE_LABEL = {
    "spei_sum": "SPEI",
    "tmax_sum": "Tmax",
    "precip_sum": "Pr",
    "vpd_sum": "VPD",
    "et0_sum": "ET0",
    "fracforest": "Forest",
    "fracgrassland": "Grass",
    "fracirrNonPaddy": "IrrNonPaddy",
    "fracirrPaddy": "IrrPaddy",
    "fracsealed": "Sealed",
    "fracwater": "Water",
    "duration": "Duration",
}

SPARSE_LAND_FEATURES = {"fracirrNonPaddy", "fracirrPaddy", "fracsealed", "fracwater"}


mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "sans-serif"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.linewidth": 0.7,
    "font.size": 7.5,
})


def empirical_percentile(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    out = np.full(x.shape, np.nan, dtype=float)
    valid = np.isfinite(x)
    if valid.sum() <= 1:
        out[valid] = 0.5
        return out
    out[valid] = pd.Series(x[valid]).rank(method="average", pct=True).to_numpy()
    return np.clip(out, 0, 1)


def display_percentile(feature: str, raw: np.ndarray) -> np.ndarray:
    raw = np.asarray(raw, dtype=float)
    if feature in SPARSE_LAND_FEATURES:
        out = np.zeros(raw.shape, dtype=float)
        nonzero = np.isfinite(raw) & (raw > 1e-6)
        if nonzero.sum() > 1:
            out[nonzero] = 0.18 + 0.82 * empirical_percentile(raw[nonzero])
        elif nonzero.sum() == 1:
            out[nonzero] = 1.0
        out[~np.isfinite(raw)] = np.nan
        return out
    return empirical_percentile(raw)


def binned_stats(x: np.ndarray, y: np.ndarray, bins: int = 12) -> pd.DataFrame:
    df = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if df.empty:
        return pd.DataFrame(columns=["x_mid", "median", "q25", "q75", "n"])
    edges = np.linspace(0, 1, bins + 1)
    df["bin"] = pd.cut(df["x"], edges, include_lowest=True, labels=False)
    rows = []
    for b, g in df.groupby("bin", observed=True):
        if len(g) < 10:
            continue
        rows.append({
            "x_mid": float(g["x"].median()),
            "median": float(g["y"].median()),
            "q25": float(g["y"].quantile(0.25)),
            "q75": float(g["y"].quantile(0.75)),
            "n": int(len(g)),
        })
    return pd.DataFrame(rows)


def raw_range_label(feature: str, raw: np.ndarray) -> str:
    raw = np.asarray(raw, dtype=float)
    raw = raw[np.isfinite(raw)]
    if raw.size == 0:
        return ""
    q05, q50, q95 = np.quantile(raw, [0.05, 0.50, 0.95])
    if feature.startswith("frac"):
        return f"raw q05/q50/q95: {q05:.2f}/{q50:.2f}/{q95:.2f}"
    if feature == "duration":
        return f"raw q05/q50/q95: {q05:.0f}/{q50:.0f}/{q95:.0f} d"
    return f"raw q05/q50/q95: {q05:.2f}/{q50:.2f}/{q95:.2f}"


def draw() -> Path:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    PACKAGE_FIG_DIR.mkdir(parents=True, exist_ok=True)
    PACKAGE_CODE_DIR.mkdir(parents=True, exist_ok=True)

    z = np.load(DATA_DIR / "global_M2_shap_values.npz", allow_pickle=True)
    shap_values = z["shap_values"]
    x_raw = z["X_shap_raw"]
    features = [str(v) for v in z["feature_names"]]

    imp = pd.read_csv(DATA_DIR / "event_level_global_shap_importance.csv")
    imp_m2 = imp[imp["model"].eq("M2")].sort_values("mean_abs_shap", ascending=False)
    selected = imp_m2["feature"].head(12).tolist()

    fig, axes = plt.subplots(3, 4, figsize=(10.8, 7.6), sharex=True)
    axes = axes.ravel()
    cmap = mpl.colormaps["RdYlBu_r"]

    for ax, feature in zip(axes, selected):
        j = features.index(feature)
        x = display_percentile(feature, x_raw[:, j])
        y = shap_values[:, j]
        color = x
        rng = np.random.default_rng(12)
        jitter = rng.normal(0, 0.006, size=len(x))
        ax.scatter(
            np.clip(x + jitter, 0, 1),
            y,
            c=color,
            cmap=cmap,
            vmin=0,
            vmax=1,
            s=10,
            alpha=0.48,
            edgecolor="none",
            rasterized=True,
        )
        stats = binned_stats(x, y, bins=12)
        if not stats.empty:
            ax.fill_between(stats["x_mid"], stats["q25"], stats["q75"], color="#4B4B4B", alpha=0.13, linewidth=0)
            ax.plot(stats["x_mid"], stats["median"], color="#222222", linewidth=1.4)
        ax.axhline(0, color="#888888", linewidth=0.7, linestyle="--")
        ax.set_title(FEATURE_LABEL.get(feature, feature), loc="left", fontsize=9.3, fontweight="bold", pad=2)
        ax.text(0.02, 0.97, raw_range_label(feature, x_raw[:, j]), transform=ax.transAxes, ha="left", va="top", fontsize=5.8, color="#666666")
        ax.grid(axis="y", color="#E5E5E5", linewidth=0.55)
        ax.set_xlim(-0.02, 1.02)

    for ax in axes[len(selected):]:
        ax.axis("off")
    for ax in axes[::4]:
        ax.set_ylabel("SHAP value")
    for ax in axes[-4:]:
        ax.set_xlabel("Within-feature percentile")

    cax = fig.add_axes([0.92, 0.24, 0.017, 0.52])
    cb = fig.colorbar(mpl.cm.ScalarMappable(cmap=cmap, norm=mpl.colors.Normalize(0, 1)), cax=cax)
    cb.set_label("Within-feature percentile", fontweight="bold")
    cb.set_ticks([0, 1], labels=["Low", "High"])
    fig.suptitle("M2 SHAP dependence diagnostics", x=0.05, y=0.985, ha="left", fontsize=13, fontweight="bold")
    fig.text(
        0.05,
        0.018,
        "Points are event-level SHAP samples. X-axis uses within-feature percentile scaling for visual comparability; "
        "zero-dominated land-cover fractions retain true zeros as the low category and expand non-zero values internally. "
        "Black lines show binned medians and shaded bands show IQR.",
        fontsize=7.2,
    )
    fig.subplots_adjust(left=0.07, right=0.895, top=0.92, bottom=0.095, hspace=0.36, wspace=0.23)

    out = FIG_DIR / "Fig4_event_level_original_consistent_M2_SHAP_dependence"
    fig.savefig(out.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")
    fig.savefig(out.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")
    plt.close(fig)

    for ext in [".png", ".pdf"]:
        shutil.copy2(out.with_suffix(ext), PACKAGE_FIG_DIR / out.with_suffix(ext).name)
    shutil.copy2(Path(__file__), PACKAGE_CODE_DIR / Path(__file__).name)

    # Save binned source data for manuscript audit.
    rows = []
    for feature in selected:
        j = features.index(feature)
        stats = binned_stats(display_percentile(feature, x_raw[:, j]), shap_values[:, j], bins=12)
        stats.insert(0, "feature", feature)
        stats.insert(1, "feature_label", FEATURE_LABEL.get(feature, feature))
        rows.append(stats)
    if rows:
        pd.concat(rows, ignore_index=True).to_csv(DATA_DIR / "event_level_M2_shap_dependence_binned.csv", index=False, encoding="utf-8-sig")
        shutil.copy2(DATA_DIR / "event_level_M2_shap_dependence_binned.csv", PACKAGE_DIR / "data" / "event_level_M2_shap_dependence_binned.csv")

    return out.with_suffix(".png")


if __name__ == "__main__":
    print(draw())


### draw_fig4_shap_dependence_lowess_style

Supplementary D LOWESS-style SHAP dependence diagnostics. Smoothed dependence-plot version.




In [ ]:
# Source file: D:\smq\paper2\modify\敏感区\code\draw_fig4_shap_dependence_lowess_style.py
# Staging copy used for notebook assembly: D:\software\codex\paper2\office_work\code_notebook_order_20260609\sensitive_code\draw_fig4_shap_dependence_lowess_style.py
# Detected encoding: utf-8-sig
# Methods-order section: Supplementary D LOWESS-style SHAP dependence diagnostics
# NOTE: This is the final modified sensitive-region/SHAP script, not legacy notebook code.

from __future__ import annotations

import shutil
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


ROOT = Path(r"D:\Software\codex\sm_heat_drought")
BASE = ROOT / "outputs" / "20260522_shap_method_rebuild" / "event_level_original_consistent_m0_m1_m2_shap_parallel"
DATA_DIR = BASE / "data"
FIG_DIR = BASE / "figures"
PACKAGE_DIR = Path(r"E:\smq\aa\敏感区")
PACKAGE_FIG_DIR = PACKAGE_DIR / "figures"
PACKAGE_CODE_DIR = PACKAGE_DIR / "code"

FEATURE_LABEL = {
    "spei_sum": "SPEI",
    "tmax_sum": "Tmax",
    "precip_sum": "Pr",
    "vpd_sum": "VPD",
    "et0_sum": "ET0",
    "fracforest": "Forest",
    "fracgrassland": "Grass",
    "fracirrNonPaddy": "IrrNonPaddy",
    "fracirrPaddy": "IrrPaddy",
    "fracsealed": "Sealed",
    "fracwater": "Water",
    "duration": "Duration",
}

FEATURE_UNIT = {
    "spei_sum": "SPEI sum",
    "tmax_sum": "Tmax sum",
    "precip_sum": "Pr sum",
    "vpd_sum": "VPD sum",
    "et0_sum": "ET0 sum",
    "fracforest": "Forest fraction",
    "fracgrassland": "Grassland fraction",
    "fracirrNonPaddy": "Irrigated non-paddy fraction",
    "fracirrPaddy": "Irrigated paddy fraction",
    "fracsealed": "Sealed fraction",
    "fracwater": "Water fraction",
    "duration": "Duration (days)",
}

SPARSE_LAND_FEATURES = {"fracirrNonPaddy", "fracirrPaddy", "fracsealed", "fracwater"}
POINT_COLOR = "#2F6F9F"
SMOOTH_COLOR = "#C65F2E"
BAND_COLOR = "#F0B27A"
ZERO_COLOR = "#8A8A8A"


mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "sans-serif"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.linewidth": 0.75,
    "font.size": 7.5,
})


def robust_display_xy(feature: str, x: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray, tuple[float, float]]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]
    if len(x) == 0:
        return x, y, (0.0, 1.0)
    if feature.startswith("frac"):
        lo, hi = 0.0, min(1.0, max(float(np.nanquantile(x, 0.995)), 0.05))
        if feature in SPARSE_LAND_FEATURES:
            hi = min(1.0, max(float(np.nanquantile(x, 0.995)), 0.12))
    else:
        lo, hi = np.nanquantile(x, [0.01, 0.99])
        if lo == hi:
            lo, hi = np.nanmin(x), np.nanmax(x)
    keep = (x >= lo) & (x <= hi)
    return x[keep], y[keep], (float(lo), float(hi))


def kernel_smooth_ci(x: np.ndarray, y: np.ndarray, xgrid: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Gaussian-kernel smooth with local IQR band; dependency-free LOWESS-like curve."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 20:
        nan = np.full_like(xgrid, np.nan, dtype=float)
        return nan, nan, nan, np.zeros_like(xgrid)
    span = max(np.nanquantile(x, 0.95) - np.nanquantile(x, 0.05), 1e-6)
    bw = span * 0.085
    med = np.full_like(xgrid, np.nan, dtype=float)
    q25 = np.full_like(xgrid, np.nan, dtype=float)
    q75 = np.full_like(xgrid, np.nan, dtype=float)
    n_eff = np.zeros_like(xgrid, dtype=float)
    for i, gx in enumerate(xgrid):
        d = (x - gx) / bw
        w = np.exp(-0.5 * d * d)
        local = w > 0.08
        if local.sum() < 18:
            continue
        yy = y[local]
        ww = w[local]
        # Weighted mean for smooth central tendency, local quantiles for uncertainty.
        med[i] = float(np.average(yy, weights=ww))
        q25[i] = float(np.quantile(yy, 0.25))
        q75[i] = float(np.quantile(yy, 0.75))
        n_eff[i] = float((ww.sum() ** 2) / np.sum(ww * ww))
    return med, q25, q75, n_eff


def jitter_x(feature: str, x: np.ndarray, xlim: tuple[float, float]) -> np.ndarray:
    rng = np.random.default_rng(18)
    width = xlim[1] - xlim[0]
    if width <= 0:
        return x
    if feature in SPARSE_LAND_FEATURES:
        return np.clip(x + rng.normal(0, width * 0.006, len(x)), xlim[0], xlim[1])
    if feature == "duration":
        return np.clip(x + rng.normal(0, 0.035, len(x)), xlim[0], xlim[1])
    return x


def draw() -> Path:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    PACKAGE_FIG_DIR.mkdir(parents=True, exist_ok=True)
    PACKAGE_CODE_DIR.mkdir(parents=True, exist_ok=True)

    z = np.load(DATA_DIR / "global_M2_shap_values.npz", allow_pickle=True)
    shap_values = z["shap_values"]
    x_raw = z["X_shap_raw"]
    features = [str(v) for v in z["feature_names"]]

    imp = pd.read_csv(DATA_DIR / "event_level_global_shap_importance.csv")
    selected = (
        imp[imp["model"].eq("M2")]
        .sort_values("mean_abs_shap", ascending=False)["feature"]
        .head(12)
        .tolist()
    )

    fig, axes = plt.subplots(3, 4, figsize=(10.8, 7.8))
    axes = axes.ravel()
    rows = []

    for idx, (ax, feature) in enumerate(zip(axes, selected), start=1):
        j = features.index(feature)
        x, y, xlim = robust_display_xy(feature, x_raw[:, j], shap_values[:, j])
        x_plot = jitter_x(feature, x, xlim)
        ax.scatter(x_plot, y, s=8, c=POINT_COLOR, alpha=0.56, edgecolor="none", rasterized=True)
        if len(x) >= 30 and xlim[1] > xlim[0]:
            xgrid = np.linspace(xlim[0], xlim[1], 120)
            smooth, q25, q75, n_eff = kernel_smooth_ci(x, y, xgrid)
            ok = np.isfinite(smooth)
            if ok.any():
                ax.fill_between(xgrid[ok], q25[ok], q75[ok], color=BAND_COLOR, alpha=0.30, linewidth=0)
                ax.plot(xgrid[ok], smooth[ok], color=SMOOTH_COLOR, linewidth=1.65)
                for gx, sm, lo, hi, ne in zip(xgrid[ok], smooth[ok], q25[ok], q75[ok], n_eff[ok]):
                    rows.append({
                        "feature": feature,
                        "feature_label": FEATURE_LABEL.get(feature, feature),
                        "x": float(gx),
                        "smooth": float(sm),
                        "q25": float(lo),
                        "q75": float(hi),
                        "n_eff": float(ne),
                    })
        ax.axhline(0, color=ZERO_COLOR, linewidth=0.75, linestyle="--")
        ax.set_xlim(*xlim)
        ax.grid(axis="y", color="#E9E9E9", linewidth=0.55)
        ax.set_title(f"{chr(96 + idx)}  {FEATURE_LABEL.get(feature, feature)}", loc="left", fontsize=9.5, fontweight="bold")
        ax.set_xlabel(FEATURE_UNIT.get(feature, feature), fontsize=7.5)
        ax.set_ylabel("SHAP value", fontsize=7.5)
        ax.tick_params(labelsize=7)

    for ax in axes[len(selected):]:
        ax.axis("off")

    fig.suptitle("M2 SHAP dependence plots", x=0.055, y=0.988, ha="left", fontsize=13, fontweight="bold")
    fig.text(
        0.055,
        0.018,
        "Blue points are event-level SHAP samples. Orange curves show kernel-smoothed central tendencies and light orange bands show local IQR. "
        "X-axes use raw feature values with robust display limits; sparse land-cover fractions are lightly jittered only for visibility.",
        fontsize=7.2,
    )
    fig.subplots_adjust(left=0.07, right=0.985, top=0.925, bottom=0.095, hspace=0.48, wspace=0.35)

    out = FIG_DIR / "Fig4_event_level_original_consistent_M2_SHAP_dependence_publication_style"
    fig.savefig(out.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")
    fig.savefig(out.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")
    plt.close(fig)

    for ext in [".png", ".pdf"]:
        shutil.copy2(out.with_suffix(ext), PACKAGE_FIG_DIR / out.with_suffix(ext).name)
    shutil.copy2(Path(__file__), PACKAGE_CODE_DIR / Path(__file__).name)
    if rows:
        smooth_csv = DATA_DIR / "event_level_M2_shap_dependence_lowess_style_smooth.csv"
        pd.DataFrame(rows).to_csv(smooth_csv, index=False, encoding="utf-8-sig")
        shutil.copy2(smooth_csv, PACKAGE_DIR / "data" / smooth_csv.name)
    return out.with_suffix(".png")


if __name__ == "__main__":
    print(draw())


### run_srex_landcover_shap_and_update_docs

Supplementary D SREX regional M2 models and document update. Regional M2 models and regional SHAP tables.




In [ ]:
# Source file: D:\smq\paper2\modify\敏感区\code\run_srex_landcover_shap_and_update_docs.py
# Staging copy used for notebook assembly: D:\software\codex\paper2\office_work\code_notebook_order_20260609\sensitive_code\run_srex_landcover_shap_and_update_docs.py
# Detected encoding: utf-8-sig
# Methods-order section: Supplementary D SREX regional M2 models and document update
# NOTE: This is the final modified sensitive-region/SHAP script, not legacy notebook code.

from __future__ import annotations

import json
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import regionmask
import geopandas as gpd
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import shap

from docx import Document
from docx.shared import Cm, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn


ROOT = Path(r"D:\Software\codex\sm_heat_drought")
BASE = ROOT / "outputs" / "20260522_shap_method_rebuild"
IN_DIR = BASE / "pixel_level_asw_landcover_shap"
OUT = BASE / "manuscript_supplement_shap_landcover_revision"
FIG_DIR = OUT / "figures"
DATA_DIR = OUT / "data"
DOC_DIR = OUT / "documents"
CODE_DIR = OUT / "code"
REPORT_DIR = OUT / "reports"
for d in [OUT, FIG_DIR, DATA_DIR, DOC_DIR, CODE_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MANUSCRIPT_SRC = Path(r"E:\smq\Manuscript(1).docx")
SUPPLEMENT_SRC = Path(r"E:\smq\Supplementary(1).docx")

FEATURES_M2 = [
    "spei_cdhw_mean",
    "tmax_cdhw_mean",
    "precip_anom_cdhw_mean",
    "vpd_anom_cdhw_mean",
    "etref_anom_cdhw_mean",
    "fracforest",
    "fracgrassland",
    "fracirrNonPaddy",
    "fracirrPaddy",
    "fracsealed",
    "fracwater",
    "cdhw_day_count",
]

FEATURE_LABEL = {
    "vpd_anom_cdhw_mean": "VPD",
    "tmax_cdhw_mean": "Tmax",
    "precip_anom_cdhw_mean": "Pr",
    "etref_anom_cdhw_mean": "ETref",
    "spei_cdhw_mean": "SPEI",
    "fracgrassland": "Grassland",
    "fracforest": "Forest",
    "fracirrNonPaddy": "Irrigated non-paddy",
    "fracirrPaddy": "Irrigated paddy",
    "fracsealed": "Sealed",
    "fracwater": "Water",
    "cdhw_day_count": "CDHW-day count",
}

BLOCK = {
    "spei_cdhw_mean": "Meteorology",
    "tmax_cdhw_mean": "Meteorology",
    "precip_anom_cdhw_mean": "Meteorology",
    "vpd_anom_cdhw_mean": "Meteorology",
    "etref_anom_cdhw_mean": "Meteorology",
    "fracforest": "Land cover",
    "fracgrassland": "Land cover",
    "fracirrNonPaddy": "Land cover",
    "fracirrPaddy": "Land cover",
    "fracsealed": "Land cover",
    "fracwater": "Land cover",
    "cdhw_day_count": "Sample support",
}

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans", "Microsoft YaHei", "SimHei"],
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "font.size": 7.2,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "axes.linewidth": 0.7,
        "savefig.transparent": False,
        "image.composite_image": False,
        "pdf.compression": 0,
    }
)

COLORS = {
    "Meteorology": "#4E79A7",
    "Land cover": "#59A14F",
    "Sample support": "#9C755F",
    "M0": "#7895B2",
    "M1": "#5FA777",
    "M2": "#C07A4A",
    "accent": "#B65F5D",
    "grey": "#D8D8D8",
}


def save_all(fig, stem: Path, dpi: int = 600):
    fig.savefig(stem.with_suffix(".png"), dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(stem.with_suffix(".tiff"), dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")
    fig.savefig(stem.with_suffix(".svg"), bbox_inches="tight", facecolor="white")


def feature_label(f: str) -> str:
    return FEATURE_LABEL.get(f, f)


def assign_srex(df: pd.DataFrame) -> pd.DataFrame:
    regions = regionmask.defined_regions.srex
    poly = regions.to_geodataframe().reset_index().rename(columns={"index": "srex_num"})
    poly = poly.rename(columns={"numbers": "srex_num", "abbrevs": "srex_abbr", "names": "srex_name"})
    poly = poly[["srex_num", "srex_abbr", "srex_name", "geometry"]]
    pts = gpd.GeoDataFrame(
        df[["lat", "lon"]].copy(),
        geometry=gpd.points_from_xy(df["lon"].astype(float), df["lat"].astype(float)),
        crs=poly.crs,
    )
    joined = gpd.sjoin(pts, poly, how="left", predicate="within")
    out = df.copy()
    out["srex_num"] = joined["srex_num"].to_numpy()
    out["srex_abbr"] = joined["srex_abbr"].to_numpy()
    out["srex_name"] = joined["srex_name"].to_numpy()
    return out


def model_region(df: pd.DataFrame, region: str, min_n: int = 350, shap_n: int = 350):
    sub = df[df["srex_abbr"] == region].copy()
    sub = sub.dropna(subset=["asw_cdhw_mean"] + FEATURES_M2)
    if len(sub) < min_n:
        return None, []
    X_raw = sub[FEATURES_M2].astype(float)
    y = sub["asw_cdhw_mean"].astype(float).values
    scaler = MinMaxScaler()
    X = pd.DataFrame(scaler.fit_transform(X_raw), columns=FEATURES_M2)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
    model = ExtraTreesRegressor(
        n_estimators=300,
        max_depth=22,
        max_features="sqrt",
        bootstrap=False,
        random_state=0,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    perf = {
        "srex_abbr": region,
        "srex_name": sub["srex_name"].iloc[0],
        "n": len(sub),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "r2": r2_score(y_test, pred),
        "rmse": mean_squared_error(y_test, pred) ** 0.5,
        "mae": mean_absolute_error(y_test, pred),
    }
    sample = X_train.sample(n=min(shap_n, len(X_train)), random_state=0)
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(sample)
    imp = []
    mean_abs = np.abs(sv).mean(axis=0)
    denom = mean_abs.sum()
    for f, v in zip(FEATURES_M2, mean_abs):
        imp.append(
            {
                "srex_abbr": region,
                "srex_name": sub["srex_name"].iloc[0],
                "feature": f,
                "feature_label": feature_label(f),
                "block": BLOCK[f],
                "mean_abs_shap": float(v),
                "relative_importance": float(v / denom) if denom else np.nan,
            }
        )
    return perf, imp


def compute_region_results(force: bool = False):
    perf_path = DATA_DIR / "srex_landcover_shap_performance.csv"
    imp_path = DATA_DIR / "srex_landcover_shap_importance.csv"
    table_path = DATA_DIR / "pixel_level_asw_landcover_driver_table_with_srex.csv.gz"
    if perf_path.exists() and imp_path.exists() and table_path.exists() and not force:
        return pd.read_csv(table_path), pd.read_csv(perf_path), pd.read_csv(imp_path)

    df = pd.read_csv(IN_DIR / "pixel_level_asw_landcover_driver_table.csv.gz")
    df = assign_srex(df)
    df.to_csv(table_path, index=False, compression="gzip")

    perfs = []
    imps = []
    for region in sorted(df["srex_abbr"].dropna().unique()):
        perf, imp = model_region(df, region)
        if perf is not None:
            perfs.append(perf)
            imps.extend(imp)
            print("region", region, "n", perf["n"], "r2", round(perf["r2"], 3))
        else:
            print("region", region, "skipped")
    perf_df = pd.DataFrame(perfs).sort_values("srex_abbr")
    imp_df = pd.DataFrame(imps)
    perf_df.to_csv(perf_path, index=False, encoding="utf-8-sig")
    imp_df.to_csv(imp_path, index=False, encoding="utf-8-sig")
    return df, perf_df, imp_df


def plot_main_figures():
    perf_global = pd.read_csv(IN_DIR / "pixel_level_model_performance.csv")
    imp_global = pd.read_csv(IN_DIR / "pixel_level_shap_importance.csv")
    imp_global["block"] = imp_global["feature"].map(BLOCK)
    imp_global["feature_label"] = imp_global["feature"].map(feature_label)
    block = (
        imp_global.groupby(["model", "block"], as_index=False)["relative_importance"]
        .sum()
        .pivot(index="model", columns="block", values="relative_importance")
        .fillna(0)
    )
    for col in ["Meteorology", "Land cover", "Sample support"]:
        if col not in block:
            block[col] = 0.0
    block = block[["Meteorology", "Land cover", "Sample support"]].reset_index()
    block.to_csv(DATA_DIR / "global_shap_block_relative_importance.csv", index=False, encoding="utf-8-sig")

    fig = plt.figure(figsize=(7.25, 5.25))
    gs = GridSpec(2, 2, figure=fig, height_ratios=[0.88, 1.24], width_ratios=[1.0, 1.35], hspace=0.45, wspace=0.38)
    ax = fig.add_subplot(gs[0, 0])
    x = np.arange(len(perf_global))
    ax.bar(x, perf_global["r2"], color=[COLORS["M0"], COLORS["M1"], COLORS["M2"]], width=0.64)
    for i, v in enumerate(perf_global["r2"]):
        ax.text(i, v + 0.012, f"{v:.3f}", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x, ["M0\nmeteorology", "M1\n+ land cover", "M2\n+ support"])
    ax.set_ylabel("Test $R^2$")
    ax.set_ylim(0, 0.72)
    ax.set_title("a  Nested model performance", loc="left", fontweight="bold")
    ax.grid(axis="y", color="#E6E6E6", linewidth=0.6)

    ax = fig.add_subplot(gs[0, 1])
    bottom = np.zeros(len(block))
    for col in ["Meteorology", "Land cover", "Sample support"]:
        vals = block[col].values
        ax.bar(x, vals, bottom=bottom, color=COLORS[col], width=0.64, label=col)
        for i, v in enumerate(vals):
            if v > 0.055:
                ax.text(i, bottom[i] + v / 2, f"{v*100:.0f}%", ha="center", va="center", fontsize=6.5, color="white")
        bottom += vals
    ax.set_xticks(x, ["M0\nmeteorology", "M1\n+ land cover", "M2\n+ support"])
    ax.set_ylim(0, 1)
    ax.set_ylabel("Relative mean |SHAP|")
    ax.set_title("b  Predictor-block importance", loc="left", fontweight="bold")
    ax.legend(ncol=3, loc="lower left", bbox_to_anchor=(0, 1.08), fontsize=6.5, handlelength=1.0)
    ax.grid(axis="y", color="#E6E6E6", linewidth=0.6)

    ax = fig.add_subplot(gs[1, 0])
    m2 = imp_global[imp_global["model"] == "M2_meteo_landcover_support"].sort_values("mean_abs_shap", ascending=True).tail(10)
    ax.barh(m2["feature_label"], m2["mean_abs_shap"], color=[COLORS[BLOCK[f]] for f in m2["feature"]])
    ax.set_xlabel("Mean |SHAP|")
    ax.set_title("c  Leading predictors in M2", loc="left", fontweight="bold")
    ax.grid(axis="x", color="#E6E6E6", linewidth=0.6)

    ax = fig.add_subplot(gs[1, 1])
    m0 = imp_global[imp_global["model"] == "M0_meteo"].set_index("feature")
    m1 = imp_global[imp_global["model"] == "M1_meteo_landcover"].set_index("feature")
    meteo = ["vpd_anom_cdhw_mean", "tmax_cdhw_mean", "precip_anom_cdhw_mean", "etref_anom_cdhw_mean", "spei_cdhw_mean"]
    y = np.arange(len(meteo))
    before = np.array([m0.loc[f, "mean_abs_shap"] for f in meteo])
    after = np.array([m1.loc[f, "mean_abs_shap"] for f in meteo])
    ax.hlines(y, before, after, color="#999999", linewidth=1)
    ax.scatter(before, y, color=COLORS["M0"], s=18, label="M0")
    ax.scatter(after, y, color=COLORS["M1"], s=18, label="M1")
    ax.set_yticks(y, [feature_label(f) for f in meteo])
    ax.invert_yaxis()
    ax.set_xlabel("Mean |SHAP|")
    ax.set_title("d  Meteorological importance after land cover", loc="left", fontweight="bold")
    ax.legend(loc="lower right", fontsize=6.5)
    ax.grid(axis="x", color="#E6E6E6", linewidth=0.6)
    fig.text(0.01, 0.004, "Target: pixel-level mean ASW during CDHW days. SHAP values are model-based predictive diagnostics.", fontsize=6.5)
    save_all(fig, FIG_DIR / "Fig4_rebuilt_SHAP_landcover_global")
    plt.close(fig)

    return perf_global, imp_global, block


def plot_region_figure(perf_region: pd.DataFrame, imp_region: pd.DataFrame):
    top_features = ["vpd_anom_cdhw_mean", "tmax_cdhw_mean", "fracgrassland", "fracforest", "precip_anom_cdhw_mean", "fracirrNonPaddy"]
    mat = (
        imp_region[imp_region["feature"].isin(top_features)]
        .pivot(index="srex_abbr", columns="feature", values="relative_importance")
        .reindex(columns=top_features)
    )
    order = perf_region.sort_values("r2", ascending=False)["srex_abbr"].tolist()
    mat = mat.reindex(order)

    fig = plt.figure(figsize=(7.25, 6.0))
    gs = GridSpec(2, 1, figure=fig, height_ratios=[0.9, 1.65], hspace=0.34)
    ax = fig.add_subplot(gs[0, 0])
    top = perf_region.sort_values("r2", ascending=False).head(12)
    ax.bar(np.arange(len(top)), top["r2"], color="#6F9BC2", width=0.65)
    ax.set_xticks(np.arange(len(top)), top["srex_abbr"], rotation=0)
    ax.set_ylabel("Regional test $R^2$")
    ax.set_title("a  SREX regions with higher M2 predictive skill", loc="left", fontweight="bold")
    ax.grid(axis="y", color="#E6E6E6", linewidth=0.6)
    for i, (_, row) in enumerate(top.iterrows()):
        ax.text(i, row["r2"] + 0.015, f"{row['r2']:.2f}", ha="center", va="bottom", fontsize=6.2)

    ax = fig.add_subplot(gs[1, 0])
    im = ax.imshow(mat.values, aspect="auto", cmap="YlGnBu", vmin=0, vmax=np.nanquantile(mat.values, 0.95))
    ax.set_yticks(np.arange(len(mat.index)), mat.index)
    ax.set_xticks(np.arange(len(top_features)), [feature_label(f) for f in top_features], rotation=35, ha="right")
    ax.set_title("b  Regional relative SHAP importance in the land-cover-augmented model", loc="left", fontweight="bold")
    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.012)
    cbar.set_label("Relative mean |SHAP|")
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat.values[i, j]
            if np.isfinite(v) and v >= 0.13:
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=5.4, color="black")
    fig.text(0.01, 0.004, "Only SREX regions with sufficient samples were modelled. Regional SHAP values describe within-model predictive importance.", fontsize=6.5)
    save_all(fig, FIG_DIR / "FigS_SHAP_landcover_SREX_regional")
    plt.close(fig)


def set_run_font(run, bold=False, size=10.5):
    run.bold = bold
    run.font.name = "Times New Roman"
    run.font.size = Pt(size)
    rpr = run._element.get_or_add_rPr()
    rfonts = rpr.rFonts
    if rfonts is None:
        rfonts = OxmlElement("w:rFonts")
        rpr.append(rfonts)
    rfonts.set(qn("w:ascii"), "Times New Roman")
    rfonts.set(qn("w:hAnsi"), "Times New Roman")
    rfonts.set(qn("w:cs"), "Times New Roman")
    rfonts.set(qn("w:eastAsia"), "宋体")


def clear_and_set(paragraph, text, bold=False, size=10.5, first_indent=True):
    for child in list(paragraph._p):
        paragraph._p.remove(child)
    paragraph.paragraph_format.first_line_indent = Pt(21) if first_indent else Pt(0)
    paragraph.paragraph_format.line_spacing = 1.15
    paragraph.paragraph_format.space_after = Pt(6)
    run = paragraph.add_run(text)
    set_run_font(run, bold=bold, size=size)


def add_para_after(paragraph, text, style=None, first_indent=True):
    new_p = OxmlElement("w:p")
    paragraph._p.addnext(new_p)
    p = paragraph._parent.add_paragraph()
    p._p.getparent().remove(p._p)
    new_p.getparent().replace(new_p, p._p)
    if style:
        p.style = style
    clear_and_set(p, text, first_indent=first_indent)
    return p


def insert_picture_after(paragraph, image_path: Path, caption: str, width_cm=15.5):
    p = add_para_after(paragraph, "", first_indent=False)
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = p.add_run()
    run.add_picture(str(image_path), width=Cm(width_cm))
    cap = add_para_after(p, caption, first_indent=False)
    cap.alignment = WD_ALIGN_PARAGRAPH.CENTER
    for r in cap.runs:
        set_run_font(r, size=9)
    return cap


def set_cell_text(cell, text, bold=False):
    cell.text = ""
    p = cell.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.first_line_indent = Pt(0)
    r = p.add_run(str(text))
    set_run_font(r, bold=bold, size=8.5)
    cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER


def shade_cell(cell, fill="F2F2F2"):
    tc_pr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), fill)
    tc_pr.append(shd)


def add_table_after(paragraph, df: pd.DataFrame, caption: str):
    cap = add_para_after(paragraph, caption, first_indent=False)
    cap.alignment = WD_ALIGN_PARAGRAPH.CENTER
    for r in cap.runs:
        set_run_font(r, bold=True, size=8.5)
    table = Document().add_table(rows=1, cols=len(df.columns))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.style = "Table Grid"
    for j, col in enumerate(df.columns):
        set_cell_text(table.rows[0].cells[j], col, bold=True)
        shade_cell(table.rows[0].cells[j])
    for _, row in df.iterrows():
        cells = table.add_row().cells
        for j, v in enumerate(row):
            if isinstance(v, float):
                text = f"{v:.3f}"
            else:
                text = str(v)
            set_cell_text(cells[j], text)
    cap._p.addnext(table._tbl)
    return table


def replace_range(doc: Document, start: int, end: int, texts: list[str]):
    clear_and_set(doc.paragraphs[start], texts[0], first_indent=(start != 33))
    for idx, text in zip(range(start + 1, end + 1), texts[1:]):
        clear_and_set(doc.paragraphs[idx], text)
    for idx in range(start + 1 + len(texts[1:]), end + 1):
        clear_and_set(doc.paragraphs[idx], "")


def update_manuscript(perf_global, imp_global, perf_region, imp_region):
    out_path = DOC_DIR / "Manuscript_SHAP_landcover_revision_v1.docx"
    shutil.copy2(MANUSCRIPT_SRC, out_path)
    doc = Document(out_path)
    # Results section: paragraphs 33-39.
    clear_and_set(doc.paragraphs[33], "Model-based predictor contributions to CDHW-day ASW conditions", bold=True, size=12, first_indent=False)
    result_texts = [
        (
            "Once CDHW events are established, spatial variation in available soil moisture (ASW) reflects not only meteorological forcing but also land-surface context. "
            "We therefore rebuilt the ExtraTrees-SHAP analysis as a model-based diagnostic of predictive information rather than a causal attribution analysis. "
            "Three nested pixel-level models were evaluated: a meteorology-only model (M0), a meteorology plus land-cover model (M1), and a model additionally including CDHW-day sample support (M2). "
            f"Adding land-cover fractions increased test R² from {perf_global.loc[perf_global.model=='M0_meteo','r2'].iloc[0]:.3f} in M0 to {perf_global.loc[perf_global.model=='M1_meteo_landcover','r2'].iloc[0]:.3f} in M1, while M2 reached R² = {perf_global.loc[perf_global.model=='M2_meteo_landcover_support','r2'].iloc[0]:.3f} (Fig. 4a)."
        ),
        (
            "In the land-cover-augmented model, VPD and Tmax remained the strongest meteorological predictors, but land-cover variables entered the leading predictor group. "
            "Grassland fraction, forest fraction and irrigated non-paddy fraction ranked among the major contributors to model predictions of CDHW-day ASW (Fig. 4b,c). "
            "The inclusion of land-cover variables also redistributed part of the meteorological SHAP importance, indicating that a meteorology-only model partially absorbs land-surface heterogeneity into meteorological predictors (Fig. 4d)."
        ),
        (
            "The SREX subcontinental analysis showed clear regional heterogeneity in model performance and predictor importance. "
            f"Across SREX regions with sufficient samples, the median regional R² was {perf_region['r2'].median():.3f}, with higher predictive skill in regions such as {', '.join(perf_region.sort_values('r2', ascending=False).head(4)['srex_abbr'].tolist())}. "
            "The leading predictors varied among regions, with VPD and Tmax generally retaining broad importance while land-cover fractions contributed substantially in several subcontinental regions (Supplementary Fig. Sx). "
            "These patterns are interpreted as predictive associations within the fitted ExtraTrees models."
        ),
        "",
        (
            "Fig. 4 Model-based ExtraTrees-SHAP diagnostics for CDHW-day available soil moisture (ASW). a, Test R² of nested models using meteorological predictors only (M0), meteorological plus land-cover predictors (M1), and meteorological, land-cover and sample-support predictors (M2). b, Predictor-block contributions based on relative mean absolute SHAP values. c, Leading predictors in M2. d, Changes in meteorological mean absolute SHAP values after adding land-cover fractions. SHAP values describe contributions to fitted model predictions and are not interpreted as causal physical contributions."
        ),
    ]
    for idx, text in zip(range(34, 39), result_texts):
        clear_and_set(doc.paragraphs[idx], text)
    insert_picture_after(doc.paragraphs[38], FIG_DIR / "Fig4_rebuilt_SHAP_landcover_global.png", "Updated Fig. 4. Rebuilt SHAP analysis with land-cover context.", width_cm=15.7)

    # Methods section: paragraphs 88-91.
    clear_and_set(doc.paragraphs[88], "Event-based factor processing and ExtraTrees-SHAP diagnostic analysis", bold=True, size=12, first_indent=False)
    method_texts = [
        (
            "To evaluate predictor information for CDHW-related soil-water conditions, we reconstructed the ExtraTrees-SHAP analysis as a predictive diagnostic framework. "
            "The response variable in this analysis was pixel-level mean ASW during CDHW days, and predictors were grouped into meteorological forcing, land-cover composition and sample-support descriptors. "
            "This formulation was designed to test whether land-surface context improved model performance beyond meteorological forcing alone."
        ),
        (
            "Meteorological predictors included VPD anomaly, precipitation anomaly, ETref anomaly, Tmax and SPEI. "
            "Land-cover predictors were taken from CWatM fractional land-cover fields and included forest, grassland, irrigated non-paddy, irrigated paddy, sealed and water fractions. "
            "We compared three nested models: M0 using meteorological variables only, M1 adding land-cover fractions, and M2 further adding CDHW-day count as a sample-support descriptor."
        ),
        (
            "ExtraTrees models were fitted using a 70%–30% train–test split. SHAP values were then calculated from the fitted tree ensembles and summarized using mean absolute SHAP values. "
            "For subcontinental diagnostics, the same M2 predictor set was fitted separately within IPCC SREX regions with sufficient samples. "
            "Because SHAP explains the fitted model rather than the physical system directly, we interpret the results as model-based predictive contributions."
        ),
    ]
    for idx, text in zip(range(89, 92), method_texts):
        clear_and_set(doc.paragraphs[idx], text)
    doc.save(out_path)
    return out_path


def add_supplement_section(doc: Document, perf_global, imp_global, perf_region, imp_region):
    # Replace old S5 block paragraphs 33-58.
    clear_and_set(doc.paragraphs[33], "S5. Rebuilt ExtraTrees-SHAP analysis with land-cover context", bold=True, size=12, first_indent=False)
    texts = [
        (
            "We rebuilt the ExtraTrees-SHAP analysis to address the role of land-surface heterogeneity in model predictions of CDHW-day available soil moisture (ASW). "
            "The analysis was treated as a model-based diagnostic of predictive information, not as a causal attribution framework."
        ),
        (
            "The response variable was pixel-level mean ASW during CDHW days. Meteorological predictors included SPEI, Tmax, precipitation anomaly, VPD anomaly and ETref anomaly. "
            "Land-cover predictors were CWatM fractional cover variables: forest, grassland, irrigated non-paddy, irrigated paddy, sealed and water fractions. "
            "CDHW-day count was included only in the support-augmented model to represent sample support."
        ),
        (
            "Three nested ExtraTrees models were fitted: M0 used meteorological predictors only; M1 added land-cover fractions; and M2 added land-cover fractions and CDHW-day count. "
            "The same 70%–30% train–test split was used for model comparison. SHAP values were calculated using TreeExplainer and summarized as mean absolute SHAP values."
        ),
        (
            "At the global scale, adding land-cover fractions increased model performance from R² = "
            f"{perf_global.loc[perf_global.model=='M0_meteo','r2'].iloc[0]:.3f} to R² = {perf_global.loc[perf_global.model=='M1_meteo_landcover','r2'].iloc[0]:.3f}; "
            f"the support-augmented model reached R² = {perf_global.loc[perf_global.model=='M2_meteo_landcover_support','r2'].iloc[0]:.3f}. "
            "The strongest M2 predictors were VPD, Tmax, grassland fraction, forest fraction, precipitation anomaly and irrigated non-paddy fraction."
        ),
        (
            "For the subcontinental analysis, M2 was refitted within IPCC SREX regions with sufficient sample size. "
            f"The median regional R² was {perf_region['r2'].median():.3f}, and regional predictor rankings showed clear heterogeneity. "
            "This regional analysis complements the global model by showing that the relative information carried by meteorological and land-cover variables differs among subcontinental regions."
        ),
    ]
    for idx, text in zip(range(34, 39), texts):
        clear_and_set(doc.paragraphs[idx], text)
    for idx in range(39, 59):
        clear_and_set(doc.paragraphs[idx], "")

    p = doc.paragraphs[38]
    # Add tables and figures immediately after S5 text.
    perf_tbl = perf_global.copy()
    perf_tbl["Model"] = perf_tbl["model"].map(
        {
            "M0_meteo": "M0 meteorology",
            "M1_meteo_landcover": "M1 + land cover",
            "M2_meteo_landcover_support": "M2 + land cover + support",
        }
    )
    perf_tbl = perf_tbl[["Model", "n", "n_train", "n_test", "r2", "rmse", "mae"]]
    perf_tbl.columns = ["Model", "n", "Train", "Test", "R²", "RMSE", "MAE"]
    add_table_after(p, perf_tbl, "Table Sx. Predictive performance of nested ExtraTrees-SHAP models.")

    top = imp_global[imp_global["model"] == "M2_meteo_landcover_support"].sort_values("relative_importance", ascending=False).head(8)
    top_tbl = top[["feature_label", "block", "mean_abs_shap", "relative_importance"]].copy()
    top_tbl.columns = ["Predictor", "Block", "Mean |SHAP|", "Relative importance"]
    add_table_after(p, top_tbl, "Table Sx. Leading predictors in the land-cover-augmented global model.")

    reg = perf_region.sort_values("r2", ascending=False).head(12)[["srex_abbr", "srex_name", "n", "r2", "rmse", "mae"]].copy()
    reg.columns = ["Region", "Name", "n", "R²", "RMSE", "MAE"]
    add_table_after(p, reg, "Table Sx. Highest-performing SREX regional M2 models.")

    insert_picture_after(p, FIG_DIR / "Fig4_rebuilt_SHAP_landcover_global.png", "Fig. Sx. Global SHAP diagnostics for nested models with land-cover context.", width_cm=15.7)
    insert_picture_after(p, FIG_DIR / "FigS_SHAP_landcover_SREX_regional.png", "Fig. Sx. SREX regional model performance and relative SHAP importance.", width_cm=15.7)


def update_supplement(perf_global, imp_global, perf_region, imp_region):
    out_path = DOC_DIR / "Supplementary_SHAP_landcover_revision_v1.docx"
    shutil.copy2(SUPPLEMENT_SRC, out_path)
    doc = Document(out_path)
    add_supplement_section(doc, perf_global, imp_global, perf_region, imp_region)
    doc.save(out_path)
    return out_path


def verify_docx(path: Path):
    doc = Document(path)
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
        media = sum(n.startswith("word/media/") for n in names)
        embeds = sum(n.startswith("word/embeddings/") for n in names)
        xml = z.read("word/document.xml").decode("utf-8", errors="ignore")
    return {
        "path": str(path),
        "paragraphs": len(doc.paragraphs),
        "tables": len(doc.tables),
        "inline_shapes": len(doc.inline_shapes),
        "media": media,
        "embeddings": embeds,
        "contains_R2_0639": "0.639" in xml,
        "contains_land_cover": "land-cover" in xml or "land cover" in xml,
    }


def copy_to_delivery(paths):
    target_base = Path(r"E:\smq\aa\敏感区")
    for p in paths:
        if p.suffix.lower() == ".docx":
            shutil.copy2(p, target_base / "documents" / p.name)
    for f in FIG_DIR.glob("Fig4_rebuilt_SHAP_landcover_global.*"):
        shutil.copy2(f, target_base / "figures" / f.name)
    for f in FIG_DIR.glob("FigS_SHAP_landcover_SREX_regional.*"):
        shutil.copy2(f, target_base / "figures" / f.name)
    for f in DATA_DIR.glob("*.csv"):
        shutil.copy2(f, target_base / "data" / f.name)
    shutil.copy2(Path(__file__), target_base / "code" / Path(__file__).name)
    shutil.copy2(REPORT_DIR / "shap_landcover_revision_validation.json", target_base / "reports" / "shap_landcover_revision_validation.json")


def main():
    df, perf_region, imp_region = compute_region_results()
    perf_global, imp_global, block = plot_main_figures()
    plot_region_figure(perf_region, imp_region)

    # Make clean labels in saved global importance table.
    imp_global["feature_label"] = imp_global["feature"].map(feature_label)
    imp_global["block"] = imp_global["feature"].map(BLOCK)
    imp_global.to_csv(DATA_DIR / "global_shap_importance_with_blocks.csv", index=False, encoding="utf-8-sig")
    perf_global.to_csv(DATA_DIR / "global_shap_model_performance.csv", index=False, encoding="utf-8-sig")

    manuscript = update_manuscript(perf_global, imp_global, perf_region, imp_region)
    supplement = update_supplement(perf_global, imp_global, perf_region, imp_region)
    validation = {
        "manuscript": verify_docx(manuscript),
        "supplement": verify_docx(supplement),
        "global_performance": perf_global.to_dict(orient="records"),
        "regional_models": {
            "n_regions": int(len(perf_region)),
            "median_r2": float(perf_region["r2"].median()),
            "top_regions": perf_region.sort_values("r2", ascending=False).head(8).to_dict(orient="records"),
        },
    }
    (REPORT_DIR / "shap_landcover_revision_validation.json").write_text(json.dumps(validation, indent=2, ensure_ascii=False), encoding="utf-8")
    shutil.copy2(Path(__file__), CODE_DIR / Path(__file__).name)
    copy_to_delivery([manuscript, supplement])
    print(json.dumps(validation, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()


### draw_original_style_shap_with_landcover
SHAP-landcover original-style figure diagnostic. Final-package diagnostic figure script.




In [ ]:
# Source file: D:\smq\paper2\modify\敏感区\code\draw_original_style_shap_with_landcover.py
# Staging copy used for notebook assembly: D:\software\codex\paper2\office_work\code_notebook_order_20260609\sensitive_code\draw_original_style_shap_with_landcover.py
# Detected encoding: utf-8-sig
# Methods-order section: SHAP-landcover original-style figure diagnostic
# NOTE: This is the final modified sensitive-region/SHAP script, not legacy notebook code.

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import shap


ROOT = Path(r"D:\Software\codex\sm_heat_drought")
BASE = ROOT / "outputs" / "20260522_shap_method_rebuild"
IN_DIR = BASE / "pixel_level_asw_landcover_shap"
REG_DIR = BASE / "manuscript_supplement_shap_landcover_revision" / "data"
OUT = BASE / "original_style_shap_landcover_figure"
FIG_DIR = OUT / "figures"
DATA_DIR = OUT / "data"
CODE_DIR = OUT / "code"
for d in [OUT, FIG_DIR, DATA_DIR, CODE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "spei_cdhw_mean",
    "tmax_cdhw_mean",
    "precip_anom_cdhw_mean",
    "vpd_anom_cdhw_mean",
    "etref_anom_cdhw_mean",
    "fracforest",
    "fracgrassland",
    "fracirrNonPaddy",
]

FEATURE_LABEL = {
    "spei_cdhw_mean": "SPEI",
    "tmax_cdhw_mean": "Tmax",
    "precip_anom_cdhw_mean": "Pr",
    "vpd_anom_cdhw_mean": "VPD",
    "etref_anom_cdhw_mean": "PET",
    "fracforest": "Forest",
    "fracgrassland": "Grass",
    "fracirrNonPaddy": "IrrNonPaddy",
}

REGION_ORDER_ORIGINAL = [
    "WSA", "WNA", "WAF", "SSA", "SEA", "SAU", "SAS", "SAF", "NEU", "NEB",
    "NAU", "NAS", "MED", "ENA", "EAS", "EAF", "CNA", "CEU", "CAM", "AMZ",
]

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.linewidth": 1.0,
    "font.size": 9,
    "savefig.transparent": False,
    "image.composite_image": False,
    "pdf.compression": 0,
})


def save_all(fig, stem: Path, dpi=600):
    fig.savefig(stem.with_suffix(".png"), dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(stem.with_suffix(".tiff"), dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")
    fig.savefig(stem.with_suffix(".svg"), bbox_inches="tight", facecolor="white")


def beeswarm_y(x, base_y, bin_count=55, step=0.045, max_spread=0.30):
    """Deterministic compact beeswarm-like y offsets."""
    x = np.asarray(x)
    bins = np.linspace(np.nanmin(x), np.nanmax(x), bin_count + 1)
    inds = np.digitize(x, bins) - 1
    y = np.empty_like(x, dtype=float)
    for b in np.unique(inds):
        loc = np.where(inds == b)[0]
        n = len(loc)
        if n == 0:
            continue
        offsets = []
        for k in range(n):
            if k == 0:
                offsets.append(0)
            else:
                mag = (k + 1) // 2
                sign = 1 if k % 2 else -1
                offsets.append(sign * mag * step)
        offsets = np.clip(offsets, -max_spread, max_spread)
        y[loc] = base_y + np.asarray(offsets)
    return y


def global_panel_data():
    z = np.load(IN_DIR / "M1_meteo_landcover_asw_cdhw_mean_shap_values.npz", allow_pickle=True)
    raw = [str(v) for v in z["raw_feature_names"]]
    X = z["X_shap"]
    S = z["shap_values"]
    rows = []
    for f in FEATURES:
        j = raw.index(f)
        rows.append({
            "feature": f,
            "label": FEATURE_LABEL[f],
            "mean_abs_shap": float(np.abs(S[:, j]).mean()),
            "corr": float(np.corrcoef(X[:, j], S[:, j])[0, 1]) if np.std(X[:, j]) and np.std(S[:, j]) else np.nan,
        })
    imp = pd.DataFrame(rows).sort_values("mean_abs_shap", ascending=False)
    imp.to_csv(DATA_DIR / "global_M1_original_style_feature_importance.csv", index=False, encoding="utf-8-sig")
    return z, raw, imp


def compute_regional_m1(force=False):
    out_perf = DATA_DIR / "srex_M1_landcover_original_style_performance.csv"
    out_imp = DATA_DIR / "srex_M1_landcover_original_style_importance_correlation.csv"
    if out_perf.exists() and out_imp.exists() and not force:
        return pd.read_csv(out_perf), pd.read_csv(out_imp)
    table_path = REG_DIR / "pixel_level_asw_landcover_driver_table_with_srex.csv.gz"
    if not table_path.exists():
        table_path = BASE / "manuscript_supplement_shap_landcover_revision" / "data" / "pixel_level_asw_landcover_driver_table_with_srex.csv.gz"
    df = pd.read_csv(table_path)
    perfs = []
    imps = []
    for region in REGION_ORDER_ORIGINAL:
        sub = df[df["srex_abbr"] == region].dropna(subset=["asw_cdhw_mean"] + FEATURES).copy()
        if len(sub) < 350:
            continue
        X_raw = sub[FEATURES].astype(float)
        y = sub["asw_cdhw_mean"].astype(float).values
        scaler = MinMaxScaler()
        X = pd.DataFrame(scaler.fit_transform(X_raw), columns=FEATURES)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
        model = ExtraTreesRegressor(
            n_estimators=300,
            max_depth=22,
            max_features="sqrt",
            bootstrap=False,
            random_state=0,
            n_jobs=-1,
        )
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        perfs.append({
            "srex_abbr": region,
            "srex_name": sub["srex_name"].iloc[0],
            "n": len(sub),
            "r2": r2_score(y_test, pred),
            "rmse": mean_squared_error(y_test, pred) ** 0.5,
            "mae": mean_absolute_error(y_test, pred),
        })
        sample = X_train.sample(n=min(500, len(X_train)), random_state=0)
        sv = shap.TreeExplainer(model).shap_values(sample)
        mean_abs = np.abs(sv).mean(axis=0)
        denom = mean_abs.sum()
        for j, f in enumerate(FEATURES):
            corr = np.corrcoef(sample[f].values, sv[:, j])[0, 1]
            imps.append({
                "srex_abbr": region,
                "srex_name": sub["srex_name"].iloc[0],
                "feature": f,
                "label": FEATURE_LABEL[f],
                "mean_abs_shap": float(mean_abs[j]),
                "relative_importance": float(mean_abs[j] / denom) if denom else np.nan,
                "shap_feature_corr": float(corr) if np.isfinite(corr) else np.nan,
            })
        print(region, len(sub), round(perfs[-1]["r2"], 3))
    perf = pd.DataFrame(perfs)
    imp = pd.DataFrame(imps)
    perf.to_csv(out_perf, index=False, encoding="utf-8-sig")
    imp.to_csv(out_imp, index=False, encoding="utf-8-sig")
    return perf, imp


def plot_original_style():
    z, raw, global_imp = global_panel_data()
    perf, reg_imp = compute_regional_m1()
    X = z["X_shap"]
    S = z["shap_values"]

    order = global_imp["feature"].tolist()
    labels = [FEATURE_LABEL[f] for f in order]
    nrows = len(order)

    fig = plt.figure(figsize=(8.8, 12.8))
    gs = GridSpec(2, 1, height_ratios=[0.88, 2.25], hspace=0.36, figure=fig)

    # Panel a
    ax = fig.add_subplot(gs[0])
    ax_bar = ax.twiny()
    ax.set_zorder(2)
    ax_bar.set_zorder(1)
    ax.patch.set_alpha(0)
    y_positions = np.arange(nrows)[::-1]
    imp_map = dict(zip(global_imp["feature"], global_imp["mean_abs_shap"]))
    max_imp = max(imp_map.values()) * 1.22
    ax_bar.set_xlim(0, max_imp)
    ax_bar.barh(y_positions, [imp_map[f] for f in order], height=0.78, color="#E9E9E9", edgecolor="none")
    ax_bar.set_yticks([])
    ax_bar.set_xlabel("Global importance (mean |SHAP|)", fontsize=12, fontweight="bold", labelpad=8, color="#777777")
    ax_bar.tick_params(axis="x", colors="#777777", labelsize=10, width=1.0, length=6)
    ticks = np.linspace(0, round(max_imp, 2), 5)
    ax_bar.set_xticks(ticks)
    ax_bar.spines["top"].set_color("#999999")
    ax_bar.spines["bottom"].set_visible(False)
    ax_bar.spines["left"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)

    cmap = mpl.cm.get_cmap("RdYlBu_r")
    for y, f in zip(y_positions, order):
        j = raw.index(f)
        xs = S[:, j]
        vals = X[:, j]
        yy = beeswarm_y(xs, y)
        ax.scatter(xs, yy, c=vals, cmap=cmap, vmin=0, vmax=1, s=18, edgecolor="none", alpha=0.90, rasterized=True)
        ax.axhline(y, color="#D6D6D6", linestyle=(0, (1.2, 3)), linewidth=0.8, zorder=0)
    ax.axvline(0, color="#303030", linewidth=1.5)
    ax.set_yticks(y_positions, labels, fontsize=12, fontweight="bold")
    xlim = np.nanquantile(np.abs(S[:, [raw.index(f) for f in order]]), 0.995) * 1.15
    ax.set_xlim(-xlim, xlim)
    ax.set_ylim(-0.8, nrows - 0.2)
    ax.set_xlabel("SHAP value (impact on CDHW-day ASW)", fontsize=13, fontweight="bold")
    ax.tick_params(axis="x", labelsize=10, width=1.0, length=5)
    ax.tick_params(axis="y", length=0)
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=mpl.colors.Normalize(vmin=0, vmax=1))
    cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.025)
    cbar.set_label("Feature value", fontsize=12, fontweight="bold")
    cbar.set_ticks([0, 1], labels=["Low", "High"])
    cbar.ax.tick_params(labelsize=10)
    ax.text(-0.08, 1.08, "a", transform=ax.transAxes, fontsize=24, fontweight="bold")

    # Panel b
    ax2 = fig.add_subplot(gs[1])
    reg_order = [r for r in REGION_ORDER_ORIGINAL if r in reg_imp["srex_abbr"].unique()]
    feature_order = ["spei_cdhw_mean", "tmax_cdhw_mean", "precip_anom_cdhw_mean", "vpd_anom_cdhw_mean",
                     "etref_anom_cdhw_mean", "fracforest", "fracgrassland", "fracirrNonPaddy"]
    ymap = {r: i for i, r in enumerate(reg_order[::-1])}
    xmap = {f: i for i, f in enumerate(feature_order)}
    plot_df = reg_imp[reg_imp["feature"].isin(feature_order) & reg_imp["srex_abbr"].isin(reg_order)].copy()
    plot_df["x"] = plot_df["feature"].map(xmap)
    plot_df["y"] = plot_df["srex_abbr"].map(ymap)
    # Normalize sizes to original-style legend range.
    size_scale = 1550
    sizes = 35 + plot_df["relative_importance"].clip(lower=0) * size_scale
    sc = ax2.scatter(
        plot_df["x"], plot_df["y"],
        s=sizes,
        c=plot_df["shap_feature_corr"],
        cmap="RdYlBu_r",
        vmin=-0.9, vmax=0.9,
        edgecolor="black",
        linewidth=0.9,
        alpha=0.95,
    )
    ax2.set_xticks(np.arange(len(feature_order)), [FEATURE_LABEL[f] for f in feature_order],
                   fontsize=11, fontweight="bold", rotation=0)
    ax2.set_yticks(np.arange(len(reg_order)), reg_order[::-1], fontsize=11, fontweight="bold")
    ax2.set_xlim(-0.45, len(feature_order) - 0.55)
    ax2.set_ylim(-0.75, len(reg_order) - 0.25)
    ax2.grid(which="major", color="#D0D0D0", linestyle="--", linewidth=0.8)
    ax2.set_axisbelow(True)
    for spine in ["left", "bottom", "top", "right"]:
        ax2.spines[spine].set_visible(False)
    ax2.tick_params(length=0)
    ax2.text(-0.08, 1.02, "b", transform=ax2.transAxes, fontsize=24, fontweight="bold")

    # Size legend
    handles = []
    labels_size = []
    for val in [0.05, 0.15, 0.30]:
        handles.append(ax2.scatter([], [], s=35 + val * size_scale, facecolor="white", edgecolor="black", linewidth=0.9))
        labels_size.append(f"{val:.2f}")
    leg = ax2.legend(handles, labels_size, title="Factor\nimportance", loc="upper left",
                     bbox_to_anchor=(1.015, 0.94), frameon=False, borderaxespad=0, labelspacing=1.2)
    plt.setp(leg.get_title(), fontsize=12, fontweight="bold")
    for t in leg.get_texts():
        t.set_fontsize(10)

    cax = fig.add_axes([0.83, 0.19, 0.025, 0.22])
    cb = fig.colorbar(sc, cax=cax)
    cb.set_label("SHAP-based\ncorrelation", fontsize=12, fontweight="bold")
    cb.ax.tick_params(labelsize=10)

    fig.text(0.08, 0.018,
             "Panels use the land-cover-augmented M1 model. Land-cover fractions are added to the original meteorological predictors; SHAP values describe model predictions, not causal contributions.",
             fontsize=8.5)
    save_all(fig, FIG_DIR / "Fig4_original_style_SHAP_with_landcover")
    plt.close(fig)

    summary = {
        "figure": str(FIG_DIR / "Fig4_original_style_SHAP_with_landcover.png"),
        "global_features": global_imp.to_dict(orient="records"),
        "regional_performance": perf.to_dict(orient="records"),
    }
    (OUT / "original_style_shap_landcover_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")


def main():
    plot_original_style()
    import shutil
    shutil.copy2(Path(__file__), CODE_DIR / Path(__file__).name)
    target = Path(r"E:\smq\aa\敏感区")
    for f in FIG_DIR.glob("Fig4_original_style_SHAP_with_landcover.*"):
        shutil.copy2(f, target / "figures" / f.name)
    for f in DATA_DIR.glob("*.csv"):
        shutil.copy2(f, target / "data" / f.name)
    shutil.copy2(Path(__file__), target / "code" / Path(__file__).name)
    shutil.copy2(OUT / "original_style_shap_landcover_summary.json", target / "reports" / "original_style_shap_landcover_summary.json")
    print("saved", FIG_DIR / "Fig4_original_style_SHAP_with_landcover.png")


if __name__ == "__main__":
    main()


### create_shap_landcover_main_text_package.py

SHAP-landcover main-text package builder. Packages SHAP/land-cover figures, tables and document material.



In [ ]:
# Source file: D:\smq\paper2\modify\敏感区\code\create_shap_landcover_main_text_package.py
# Staging copy used for notebook assembly: D:\software\codex\paper2\office_work\code_notebook_order_20260609\sensitive_code\create_shap_landcover_main_text_package.py
# Detected encoding: utf-8-sig
# Methods-order section: SHAP-landcover main-text package builder
# NOTE: This is the final modified sensitive-region/SHAP script, not legacy notebook code.

from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from docx import Document
from docx.shared import Cm, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn


ROOT = Path(r"D:\Software\codex\sm_heat_drought")
IN_DIR = ROOT / "outputs" / "20260522_shap_method_rebuild" / "pixel_level_asw_landcover_shap"
OUT_DIR = ROOT / "outputs" / "20260522_shap_method_rebuild" / "main_text_shap_landcover_section"
FIG_DIR = OUT_DIR / "figures"
DATA_DIR = OUT_DIR / "data"
DOC_DIR = OUT_DIR / "documents"
CODE_DIR = OUT_DIR / "code"
for d in [OUT_DIR, FIG_DIR, DATA_DIR, DOC_DIR, CODE_DIR]:
    d.mkdir(parents=True, exist_ok=True)


mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans", "Microsoft YaHei", "SimHei"],
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "font.size": 7.5,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "axes.linewidth": 0.7,
        "xtick.major.width": 0.7,
        "ytick.major.width": 0.7,
        "legend.frameon": False,
        "savefig.transparent": False,
        "image.composite_image": False,
        "pdf.compression": 0,
    }
)

FEATURE_LABEL = {
    "vpd_anom_cdhw_mean": "VPD anomaly",
    "tmax_cdhw_mean": "Tmax",
    "precip_anom_cdhw_mean": "Precip. anomaly",
    "etref_anom_cdhw_mean": "ETref anomaly",
    "spei_cdhw_mean": "SPEI",
    "fracgrassland": "Grassland fraction",
    "fracforest": "Forest fraction",
    "fracirrNonPaddy": "Irrigated non-paddy fraction",
    "fracirrPaddy": "Irrigated paddy fraction",
    "fracsealed": "Sealed fraction",
    "fracwater": "Water fraction",
    "cdhw_day_count": "CDHW-day count",
}

MODEL_LABEL = {
    "M0_meteo": "M0\nmeteorology",
    "M1_meteo_landcover": "M1\n+ land cover",
    "M2_meteo_landcover_support": "M2\n+ support",
}

BLOCK = {
    "spei_cdhw_mean": "Meteorology",
    "tmax_cdhw_mean": "Meteorology",
    "precip_anom_cdhw_mean": "Meteorology",
    "vpd_anom_cdhw_mean": "Meteorology",
    "etref_anom_cdhw_mean": "Meteorology",
    "fracforest": "Land cover",
    "fracgrassland": "Land cover",
    "fracirrNonPaddy": "Land cover",
    "fracirrPaddy": "Land cover",
    "fracsealed": "Land cover",
    "fracwater": "Land cover",
    "cdhw_day_count": "Sample support",
}

PALETTE = {
    "Meteorology": "#4E79A7",
    "Land cover": "#59A14F",
    "Sample support": "#9C755F",
    "M0": "#7895B2",
    "M1": "#5FA777",
    "M2": "#C07A4A",
    "bar": "#597FA7",
    "accent": "#B65F5D",
    "grey": "#D8D8D8",
    "dark": "#333333",
}


def label_feature(name: str) -> str:
    return FEATURE_LABEL.get(name, name.replace("_", " "))


def read_inputs():
    perf = pd.read_csv(IN_DIR / "pixel_level_model_performance.csv")
    imp = pd.read_csv(IN_DIR / "pixel_level_shap_importance.csv")
    imp["block"] = imp["feature"].map(BLOCK)
    imp["feature_label"] = imp["feature"].map(label_feature)
    return perf, imp


def save_all(fig, stem: Path, dpi=600):
    fig.savefig(stem.with_suffix(".png"), dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(stem.with_suffix(".tiff"), dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")
    fig.savefig(stem.with_suffix(".svg"), bbox_inches="tight", facecolor="white")


def make_block_table(imp):
    block = (
        imp.groupby(["model", "block"], as_index=False)["relative_importance"]
        .sum()
        .pivot(index="model", columns="block", values="relative_importance")
        .fillna(0)
    )
    for col in ["Meteorology", "Land cover", "Sample support"]:
        if col not in block:
            block[col] = 0.0
    block = block[["Meteorology", "Land cover", "Sample support"]].reset_index()
    block.to_csv(DATA_DIR / "shap_block_relative_importance.csv", index=False, encoding="utf-8-sig")
    return block


def plot_summary(perf, imp, block):
    fig = plt.figure(figsize=(7.25, 5.2))
    gs = GridSpec(2, 2, figure=fig, height_ratios=[0.95, 1.3], width_ratios=[1.0, 1.35], hspace=0.42, wspace=0.38)

    ax1 = fig.add_subplot(gs[0, 0])
    x = np.arange(len(perf))
    colors = [PALETTE["M0"], PALETTE["M1"], PALETTE["M2"]]
    ax1.bar(x, perf["r2"], color=colors, width=0.64)
    for i, v in enumerate(perf["r2"]):
        ax1.text(i, v + 0.012, f"{v:.3f}", ha="center", va="bottom", fontsize=7)
    ax1.set_xticks(x, [MODEL_LABEL[m] for m in perf["model"]])
    ax1.set_ylabel("Test $R^2$")
    ax1.set_ylim(0, 0.72)
    ax1.set_title("a  Predictive skill", loc="left", fontweight="bold")
    ax1.grid(axis="y", color="#E6E6E6", linewidth=0.6)

    ax2 = fig.add_subplot(gs[0, 1])
    bottom = np.zeros(len(block))
    for col in ["Meteorology", "Land cover", "Sample support"]:
        vals = block[col].values
        ax2.bar(x, vals, bottom=bottom, color=PALETTE[col], label=col, width=0.64)
        for i, v in enumerate(vals):
            if v > 0.055:
                ax2.text(i, bottom[i] + v / 2, f"{v*100:.0f}%", ha="center", va="center", fontsize=6.5, color="white")
        bottom += vals
    ax2.set_xticks(x, [MODEL_LABEL[m] for m in block["model"]])
    ax2.set_ylim(0, 1.0)
    ax2.set_ylabel("Relative mean |SHAP|")
    ax2.set_title("b  Importance by predictor block", loc="left", fontweight="bold")
    ax2.legend(ncol=3, bbox_to_anchor=(0, 1.12), loc="lower left", fontsize=6.5, handlelength=1.0)
    ax2.grid(axis="y", color="#E6E6E6", linewidth=0.6)

    ax3 = fig.add_subplot(gs[1, 0])
    m2 = imp[imp["model"] == "M2_meteo_landcover_support"].sort_values("mean_abs_shap", ascending=True).tail(10)
    bar_colors = [PALETTE[BLOCK[f]] for f in m2["feature"]]
    ax3.barh(m2["feature_label"], m2["mean_abs_shap"], color=bar_colors)
    ax3.set_xlabel("Mean |SHAP|")
    ax3.set_title("c  Top predictors in M2", loc="left", fontweight="bold")
    ax3.grid(axis="x", color="#E6E6E6", linewidth=0.6)

    ax4 = fig.add_subplot(gs[1, 1])
    # Land-cover gain is shown as the change in mean absolute SHAP after land-cover inclusion.
    m0 = imp[imp["model"] == "M0_meteo"].set_index("feature")
    m1 = imp[imp["model"] == "M1_meteo_landcover"].set_index("feature")
    meteo_features = ["vpd_anom_cdhw_mean", "tmax_cdhw_mean", "precip_anom_cdhw_mean", "etref_anom_cdhw_mean", "spei_cdhw_mean"]
    y = np.arange(len(meteo_features))
    before = np.array([m0.loc[f, "mean_abs_shap"] for f in meteo_features])
    after = np.array([m1.loc[f, "mean_abs_shap"] for f in meteo_features])
    ax4.hlines(y, before, after, color="#999999", linewidth=1.0)
    ax4.scatter(before, y, color=PALETTE["M0"], s=18, label="M0")
    ax4.scatter(after, y, color=PALETTE["M1"], s=18, label="M1")
    ax4.set_yticks(y, [label_feature(f) for f in meteo_features])
    ax4.invert_yaxis()
    ax4.set_xlabel("Mean |SHAP|")
    ax4.set_title("d  Meteorological importance after adding land cover", loc="left", fontweight="bold")
    ax4.legend(loc="lower right", fontsize=6.5)
    ax4.grid(axis="x", color="#E6E6E6", linewidth=0.6)

    fig.text(
        0.01,
        0.005,
        "Target: pixel-level mean ASW during CDHW days. SHAP values describe model-based predictive contributions, not causal effects.",
        ha="left",
        va="bottom",
        fontsize=6.5,
        color="#333333",
    )
    save_all(fig, FIG_DIR / "fig_shap_landcover_model_summary")
    plt.close(fig)


def binned_curve(x, y, bins=16):
    good = np.isfinite(x) & np.isfinite(y)
    x = np.asarray(x[good])
    y = np.asarray(y[good])
    if len(x) < bins * 5:
        return np.array([]), np.array([])
    qs = np.linspace(0, 1, bins + 1)
    edges = np.unique(np.quantile(x, qs))
    xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (x >= lo) & (x <= hi)
        if sel.sum() >= 15:
            xs.append(np.nanmedian(x[sel]))
            ys.append(np.nanmedian(y[sel]))
    return np.array(xs), np.array(ys)


def plot_dependence():
    z = np.load(IN_DIR / "M2_meteo_landcover_support_asw_cdhw_mean_shap_values.npz", allow_pickle=True)
    X = z["X_shap"]
    S = z["shap_values"]
    raw = [str(v) for v in z["raw_feature_names"]]
    top_features = [
        "vpd_anom_cdhw_mean",
        "tmax_cdhw_mean",
        "fracgrassland",
        "fracforest",
        "precip_anom_cdhw_mean",
        "fracirrNonPaddy",
    ]
    fig, axes = plt.subplots(2, 3, figsize=(7.25, 4.65), sharey=False)
    axes = axes.ravel()
    rng = np.random.default_rng(42)
    for ax, feat in zip(axes, top_features):
        j = raw.index(feat)
        x = X[:, j]
        y = S[:, j]
        idx = np.arange(len(x))
        if len(idx) > 1200:
            idx = rng.choice(idx, 1200, replace=False)
        ax.scatter(x[idx], y[idx], s=5, color="#B7B7B7", edgecolor="none", alpha=0.65)
        bx, by = binned_curve(x, y, bins=14)
        if len(bx):
            ax.plot(bx, by, color=PALETTE["accent"], linewidth=1.5)
            ax.scatter(bx, by, color=PALETTE["accent"], s=12, zorder=3)
        ax.axhline(0, color="#666666", linewidth=0.6)
        ax.set_title(label_feature(feat), fontsize=7.5, fontweight="bold")
        ax.set_xlabel("Standardized feature value")
        ax.set_ylabel("SHAP value")
        ax.grid(color="#E8E8E8", linewidth=0.5)
    fig.suptitle("SHAP dependence diagnostics for the land-cover-augmented model", fontsize=9, fontweight="bold", x=0.01, ha="left")
    fig.text(
        0.01,
        0.005,
        "Grey points are sampled pixels; red lines are quantile-bin medians. Dependence curves are diagnostic model responses.",
        ha="left",
        va="bottom",
        fontsize=6.5,
        color="#333333",
    )
    fig.subplots_adjust(left=0.08, right=0.98, top=0.88, bottom=0.14, wspace=0.32, hspace=0.42)
    save_all(fig, FIG_DIR / "fig_shap_landcover_dependence_diagnostics")
    plt.close(fig)


def set_cell_text(cell, text, bold=False):
    cell.text = ""
    p = cell.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.first_line_indent = Pt(0)
    run = p.add_run(str(text))
    set_run_font(run, bold=bold, size=9)
    cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER


def set_run_font(run, bold=False, size=10.5):
    run.bold = bold
    run.font.name = "Times New Roman"
    run.font.size = Pt(size)
    run.font.color.rgb = None
    rpr = run._element.get_or_add_rPr()
    rfonts = rpr.rFonts
    if rfonts is None:
        rfonts = OxmlElement("w:rFonts")
        rpr.append(rfonts)
    rfonts.set(qn("w:ascii"), "Times New Roman")
    rfonts.set(qn("w:hAnsi"), "Times New Roman")
    rfonts.set(qn("w:cs"), "Times New Roman")
    rfonts.set(qn("w:eastAsia"), "宋体")


def set_paragraph_font(paragraph, size=10.5, bold=False):
    for run in paragraph.runs:
        set_run_font(run, bold=bold, size=size)


def add_para(doc, text, style=None, first_indent=True):
    p = doc.add_paragraph(style=style)
    if first_indent:
        p.paragraph_format.first_line_indent = Pt(21)
    else:
        p.paragraph_format.first_line_indent = Pt(0)
    p.paragraph_format.line_spacing = 1.15
    p.paragraph_format.space_after = Pt(6)
    run = p.add_run(text)
    set_run_font(run, size=10.5)
    return p


def add_heading(doc, text, level=1):
    p = doc.add_heading(level=level)
    p.paragraph_format.first_line_indent = Pt(0)
    p.paragraph_format.space_before = Pt(10 if level == 1 else 6)
    p.paragraph_format.space_after = Pt(4)
    p.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = p.add_run(text)
    set_run_font(run, bold=True, size=14 if level == 1 else 12)
    return p


def add_caption(doc, text):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.first_line_indent = Pt(0)
    p.paragraph_format.space_after = Pt(6)
    run = p.add_run(text)
    set_run_font(run, size=9)
    return p


def shade_cell(cell, fill):
    tc_pr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), fill)
    tc_pr.append(shd)


def add_df_table(doc, df, caption=None):
    if caption:
        p = doc.add_paragraph()
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        p.paragraph_format.first_line_indent = Pt(0)
        run = p.add_run(caption)
        set_run_font(run, bold=True, size=9)
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.style = "Table Grid"
    hdr = table.rows[0].cells
    for j, col in enumerate(df.columns):
        set_cell_text(hdr[j], col, bold=True)
        shade_cell(hdr[j], "F2F2F2")
    for _, row in df.iterrows():
        cells = table.add_row().cells
        for j, val in enumerate(row):
            if isinstance(val, float):
                text = f"{val:.3f}"
            else:
                text = str(val)
            set_cell_text(cells[j], text)
    for row in table.rows:
        for cell in row.cells:
            for p in cell.paragraphs:
                p.paragraph_format.first_line_indent = Pt(0)
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    return table


def create_word(perf, imp, block):
    doc = Document()
    sec = doc.sections[0]
    sec.top_margin = Cm(2.2)
    sec.bottom_margin = Cm(2.0)
    sec.left_margin = Cm(2.3)
    sec.right_margin = Cm(2.3)
    styles = doc.styles
    styles["Normal"].font.name = "Times New Roman"
    styles["Normal"].font.size = Pt(10.5)
    styles["Normal"]._element.rPr.rFonts.set(qn("w:eastAsia"), "宋体")

    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    title.paragraph_format.first_line_indent = Pt(0)
    r = title.add_run("Rebuilt ExtraTrees-SHAP analysis with land-cover context")
    set_run_font(r, bold=True, size=15)
    subtitle = doc.add_paragraph()
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    subtitle.paragraph_format.first_line_indent = Pt(0)
    r = subtitle.add_run("正文式方法与结果说明")
    set_run_font(r, bold=True, size=12)

    add_heading(doc, "1. Purpose and method reconstruction", 1)
    add_para(
        doc,
        "This analysis reconstructs the SHAP workflow as a model-based diagnostic rather than a causal attribution framework. "
        "The response variable was retained within the ASW family, and the current pilot used pixel-level mean ASW during CDHW days as the prediction target. "
        "The model therefore evaluates which meteorological and land-surface context variables carry predictive information for spatial variation in CDHW-conditioned ASW state.",
    )
    add_para(
        doc,
        "Three nested ExtraTrees models were fitted. M0 used only meteorological predictors, including SPEI, Tmax, precipitation anomaly, VPD anomaly and ETref anomaly. "
        "M1 added CWatM land-cover fractions, including forest, grassland, irrigated non-paddy, irrigated paddy, sealed and water fractions. "
        "M2 further added CDHW-day count as a sample-support descriptor. SHAP values were calculated for the fitted models and summarized using mean absolute SHAP values.",
    )

    add_heading(doc, "2. Model performance", 1)
    add_para(
        doc,
        "Adding land-cover information improved the pixel-level prediction of CDHW-day mean ASW. "
        f"The meteorology-only model achieved R² = {perf.loc[perf.model == 'M0_meteo', 'r2'].iloc[0]:.3f}. "
        f"After adding land-cover fractions, R² increased to {perf.loc[perf.model == 'M1_meteo_landcover', 'r2'].iloc[0]:.3f}. "
        f"The support-augmented model reached R² = {perf.loc[perf.model == 'M2_meteo_landcover_support', 'r2'].iloc[0]:.3f}. "
        "This improvement indicates that land-cover composition contains information not fully represented by the meteorological predictors.",
    )
    perf_tbl = perf.copy()
    perf_tbl["Model"] = perf_tbl["model"].map(
        {
            "M0_meteo": "M0: meteorology",
            "M1_meteo_landcover": "M1: meteorology + land cover",
            "M2_meteo_landcover_support": "M2: meteorology + land cover + support",
        }
    )
    perf_tbl = perf_tbl[["Model", "n", "n_train", "n_test", "r2", "rmse", "mae"]]
    perf_tbl.columns = ["Model", "n", "Train", "Test", "R²", "RMSE", "MAE"]
    add_df_table(doc, perf_tbl, "Table 1. Predictive performance of nested SHAP models.")

    add_heading(doc, "3. SHAP importance after adding land cover", 1)
    m2 = imp[imp["model"] == "M2_meteo_landcover_support"].sort_values("relative_importance", ascending=False)
    top = m2.head(8).copy()
    top["Feature"] = top["feature_label"]
    top["Block"] = top["block"]
    top["Mean |SHAP|"] = top["mean_abs_shap"]
    top["Relative importance"] = top["relative_importance"]
    add_para(
        doc,
        "In the final model, VPD anomaly and Tmax remained the leading meteorological predictors. "
        "However, land-cover variables entered the leading predictor group, especially grassland fraction, forest fraction and irrigated non-paddy fraction. "
        "This result directly addresses the reviewer concern that land-surface heterogeneity should be considered in the SHAP analysis.",
    )
    add_df_table(
        doc,
        top[["Feature", "Block", "Mean |SHAP|", "Relative importance"]],
        "Table 2. Leading predictors in the land-cover-augmented model.",
    )

    add_heading(doc, "4. Figures", 1)
    for fig_name, cap in [
        (
            "fig_shap_landcover_model_summary.png",
            "Figure 1. Land-cover-augmented SHAP model summary. The figure compares predictive skill, predictor-block importance, final-model predictor ranking and changes in meteorological importance after adding land-cover variables.",
        ),
        (
            "fig_shap_landcover_dependence_diagnostics.png",
            "Figure 2. SHAP dependence diagnostics for the final model. Points show sampled pixels and red curves show quantile-bin medians. These curves describe the fitted model response, not physical causal effects.",
        ),
    ]:
        p = doc.add_paragraph()
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        p.paragraph_format.first_line_indent = Pt(0)
        run = p.add_run()
        run.add_picture(str(FIG_DIR / fig_name), width=Cm(15.5))
        add_caption(doc, cap)

    add_heading(doc, "5. Interpretation boundary", 1)
    add_para(
        doc,
        "The revised SHAP analysis should be interpreted as model-based predictive diagnostics. "
        "The land-cover fractions improve prediction and appear in the SHAP ranking, but this does not by itself demonstrate that a particular land-cover class has an intrinsic causal sensitivity to CDHWs. "
        "The result supports a narrower conclusion: land-surface composition carries additional predictive information for CDHW-conditioned ASW variation beyond meteorological forcing alone.",
    )
    add_para(
        doc,
        "The current version is a pixel-level pilot based on precomputed CDHW-day driver means. "
        "For the final manuscript, the strongest version would use event-level ASW loss, grouped temporal or spatial validation, repeated random seeds and sensitivity tests across alternative response targets. "
        "The present result is already useful for revising the method logic and for demonstrating that the omission of land-cover information affects model performance and predictor ranking.",
    )

    path = DOC_DIR / "SHAP_landcover_rebuilt_main_text_section_v1.docx"
    doc.save(path)
    return path


def make_readme(perf, imp, block):
    readme = f"""# SHAP land-cover rebuilt analysis package

This package contains a pixel-level pilot of the rebuilt ExtraTrees-SHAP workflow.

## Inputs

- Model performance: `{IN_DIR / "pixel_level_model_performance.csv"}`
- SHAP importance: `{IN_DIR / "pixel_level_shap_importance.csv"}`
- Driver table: `{IN_DIR / "pixel_level_asw_landcover_driver_table.csv.gz"}`
- SHAP samples: `{IN_DIR / "M0_meteo_asw_cdhw_mean_shap_values.npz"}`, `{IN_DIR / "M1_meteo_landcover_asw_cdhw_mean_shap_values.npz"}`, `{IN_DIR / "M2_meteo_landcover_support_asw_cdhw_mean_shap_values.npz"}`

## Main results

- M0 meteorology-only R² = {perf.loc[perf.model == "M0_meteo", "r2"].iloc[0]:.3f}
- M1 meteorology + land cover R² = {perf.loc[perf.model == "M1_meteo_landcover", "r2"].iloc[0]:.3f}
- M2 meteorology + land cover + support R² = {perf.loc[perf.model == "M2_meteo_landcover_support", "r2"].iloc[0]:.3f}
- Land-cover variables enter the leading SHAP predictors, especially grassland, forest and irrigated non-paddy fractions.

## Interpretation

These SHAP values are model-based predictive diagnostics. They should not be written as causal factor contributions.
"""
    (OUT_DIR / "README.md").write_text(readme, encoding="utf-8")


def copy_key_inputs():
    for name in [
        "pixel_level_model_performance.csv",
        "pixel_level_shap_importance.csv",
        "pixel_level_run_manifest.json",
    ]:
        shutil.copy2(IN_DIR / name, DATA_DIR / name)
    shutil.copy2(Path(__file__), CODE_DIR / Path(__file__).name)


def main():
    perf, imp = read_inputs()
    block = make_block_table(imp)
    plot_summary(perf, imp, block)
    plot_dependence()
    doc_path = create_word(perf, imp, block)
    make_readme(perf, imp, block)
    copy_key_inputs()
    summary = {
        "output_dir": str(OUT_DIR),
        "docx": str(doc_path),
        "figures": [
            str(FIG_DIR / "fig_shap_landcover_model_summary.png"),
            str(FIG_DIR / "fig_shap_landcover_dependence_diagnostics.png"),
        ],
        "performance": perf.to_dict(orient="records"),
        "top_m2_features": imp[imp["model"] == "M2_meteo_landcover_support"]
        .sort_values("relative_importance", ascending=False)
        .head(8)[["feature", "block", "mean_abs_shap", "relative_importance"]]
        .to_dict(orient="records"),
    }
    (OUT_DIR / "package_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(summary, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()


## 8. Supplementary A and B utilities

VPD, meteorological anomalies and SREX-region utilities. SPEI and trend/TCIE code appears in earlier sections.


### VPD calculation from huss, Tmax and surface pressure



In [ ]:
import xarray as xr
import numpy as np
from dask.distributed import Client, LocalCluster
import os

# ==================================
# 0️⃣ 启动 Dask
# ==================================
cluster = LocalCluster(
    n_workers=12,
    threads_per_worker=1,
    memory_limit="100GB"
)
client = Client(cluster)
print("Dashboard 地址:", client.dashboard_link)

# ==================================
# 1️⃣ 打开数据
# ==================================
fp_huss   = r"F:\smq\paper2\data\gswp3\gswp3-w5e5_obsclim_huss_global_daily_*.nc"
fp_tasmax = r"F:\smq\paper2\data\gswp3\gswp3-w5e5_obsclim_tasmax_global_daily_*.nc"
fp_ps     = r"F:\smq\paper2\data\gswp3\gswp3-w5e5_obsclim_ps_global_daily_*.nc"
fp_mask   = r"D:\smq\paper2\data\modified_mask.nc"

huss = xr.open_mfdataset(
    fp_huss, parallel=True, combine="by_coords", engine="h5netcdf"
)["huss"].chunk({"time": 365, "lat": 30, "lon": 30})

tasmax = xr.open_mfdataset(
    fp_tasmax, parallel=True, combine="by_coords", engine="h5netcdf"
)["tasmax"].chunk({"time": 365, "lat": 30, "lon": 30})

ps = xr.open_mfdataset(
    fp_ps, parallel=True, combine="by_coords", engine="h5netcdf"
)["ps"].chunk({"time": 365, "lat": 30, "lon": 30})

mask_data = xr.open_dataset(fp_mask)["mask_data"]


# ==================================
# 2️⃣ 掩膜，只保留 mask > 1 区域
# ==================================
valid_mask = (mask_data > 1)

tasmax = tasmax.where(valid_mask)
huss   = huss.where(valid_mask)
ps     = ps.where(valid_mask)


# ==================================
# 3️⃣ numpy 公式：VPD 计算
# ==================================
def vpd_np(t, q, psur):
    # 温度转换成摄氏度（tasmax 为 K）
    T = t - 273.15  

    # 饱和水汽压（kPa）
    es = 0.6108 * np.exp(17.27 * T / (T + 237.3))

    # 实际水汽压（kPa），psur 单位 Pa
    ea = (q * psur) / (0.622 + 0.378 * q) / 1000

    # VPD = es - ea
    return np.maximum(es - ea, 0)


# ==================================
# 4️⃣ 并行计算 VPD
# ==================================
vpd = xr.apply_ufunc(
    vpd_np,
    tasmax, huss, ps,
    input_core_dims=[[], [], []],
    output_core_dims=[[]],
    dask="parallelized",
    vectorize=True,
    output_dtypes=[np.float32],
)

vpd.name = "vpd"
vpd.attrs["units"] = "kPa"
vpd.attrs["long_name"] = "Vapor Pressure Deficit"

vpd_ds = vpd.to_dataset()


# ==================================
# 5️⃣ 输出 NetCDF（最稳定写法）
# ==================================
out_file = r"D:\smq\paper2\data\cdhw\contribution\vpd_global.nc"

encoding = {
    "vpd": {
        "zlib": True,
        "complevel": 4,
        "dtype": "float32",
        "chunksizes": (365, 30, 30),
    }
}

if os.path.exists(out_file):
    os.remove(out_file)

vpd_ds.to_netcdf(
    out_file,
    engine="h5netcdf",
    encoding=encoding,
    compute=True      # ❗ 不要用 compute=False
)

print("🎉 VPD + 掩膜 计算完成:", out_file)


In [ ]:
import xarray as xr
import numpy as np
from dask.distributed import Client, LocalCluster, as_completed
import os
# ======================
# 文件路径
# ======================
cluster = LocalCluster(
    n_workers=10,
    threads_per_worker=2,
    memory_limit="100GB"
)
client = Client(cluster)
print("Dashboard 地址:", client.dashboard_link)

fp_vpd = r"D:\smq\paper2\data\cdhw\contribution\vpd_global.nc"
fp_mask = r"D:\smq\paper2\data\cdhw\masks_cdhw_hw_di.nc"

# ======================
# 1. 打开数据
# ======================
ds_vpd = xr.open_dataset(fp_vpd, engine="h5netcdf", chunks={"time": -1, "lat": 50, "lon": 50})
ds_mask = xr.open_dataset(fp_mask, chunks={"time": -1, "lat": 50, "lon": 50})

# 查看掩膜变量
mask = ds_mask["cdhw_mask"]

# ======================
# 3. 掩膜 SPEI 数据
# ======================
# 假设 SPEI 变量名为 'SPEI'
vpd = ds_vpd['vpd']

# 应用掩膜：mask=1 保留，其他为NaN
vpd_masked = vpd.where(mask > 0)

# ======================
# 4. 可选：保存结果
# ======================
out_path = r"D:\smq\paper2\data\cdhw\vpd_cdhw_masked.nc"

if os.path.exists(out_path):
    os.remove(out_path)

# 关键：encoding 的 key 必须和变量名一致
encoding = {"vpd": {"zlib": True, "complevel": 4, "dtype": "float32"}}

delayed_obj = vpd_masked.rename("vpd").to_netcdf(
        out_path,
        engine="h5netcdf",
        encoding=encoding,
        compute=False)

import dask
dask.compute(delayed_obj)
print("✅ 已完成压缩保存：", out_path)



### Daily ASM anomaly and CDHW-masked ASM anomaly



In [ ]:
import xarray as xr
import numpy as np

fp = r"F:\smq\paper2\w\w_daily.nc"
ds = xr.open_dataset(fp, engine="h5netcdf").w
ds.load()
climatology = ds.groupby("time.month").mean("time")
w_anom = ds.groupby("time.month") - climatology

out_fp = r"D:\smq\paper2\data\cdhw\contribution\w_daily_anomaly_monthly.nc"
w_anom.to_netcdf(out_fp, engine="h5netcdf", compute=True)

mask = xr.open_dataset( r"D:\smq\paper2\data\cdhw\masks_cdhw_hw_di.nc")['cdhw_mask']
w_masked = w.where(mask > 0)

w_masked.to_netcdf(r"D:\smq\paper2\data\cdhw\contribution\w_daily_anomaly_monthly_masked.nc")

### ETRef anomaly calculation



In [ ]:
import xarray as xr
import numpy as np

fp = r"F:\smq\paper2\su\ETRef_daily.nc"
ds = xr.open_dataset(fp, engine="h5netcdf").ETRef

ds.load()

climatology = ds.groupby("time.month").mean("time")
w_anom = ds.groupby("time.month") - climatology

out_fp = r"D:\smq\paper2\data\cdhw\contribution\ETRef_daily_anomaly.nc"
w_anom.to_netcdf(out_fp, engine="h5netcdf", compute=True)


### CDHW-masked ETRef anomaly



In [ ]:
import xarray as xr
ETRef = xr.open_dataset( r"D:\smq\paper2\data\cdhw\contribution\ETRef_daily_anomaly.nc")['ETRef']
mask = xr.open_dataset( r"D:\smq\paper2\data\cdhw\masks_cdhw_hw_di.nc")['cdhw_mask']
w_masked = ETRef.where(mask > 0) 
w_masked.to_netcdf(r"D:\smq\paper2\data\cdhw\contribution\ETRef_daily_anomaly_monthly_masked.nc")

### Supplementary B Fig. S1 SREX region map



In [ ]:
import regionmask
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import cartopy.feature as cfeature

# 创建画布和子图
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

# 设置底图特征
ax.add_feature(cfeature.LAND, color='lightgrey', alpha=0.5)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='gray')

# 获取 SREX 区域
srex = regionmask.defined_regions.srex

# 定义文字与边界样式
text_kws = dict(
    bbox=dict(facecolor="none", edgecolor="none", pad=0),
    color='dimgrey',
    weight='bold',
    fontsize=12,
)
line_kws = dict(
    color='white',
    linewidth=2,
)

# 绘制区域边界与标签
srex.plot(
    ax=ax,
    label='abbrev',
    line_kws=line_kws,
    text_kws=text_kws
)

# 去掉所有边框和坐标轴
ax.set_axis_off()
ax.set_aspect(1.2)
# 调整布局
plt.tight_layout()

pdf_save_path = r"D:\smq\paper2\result\pie_basemap.pdf"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(pdf_save_path, format='pdf', bbox_inches='tight', dpi=300)

plt.show()


### SREX region map alternate layout



In [ ]:
import regionmask
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import cartopy.feature as cfeature

# 创建画布和子图
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

# 设置底图特征
ax.add_feature(cfeature.LAND, color='lightgrey', alpha=0.5)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='gray')

# 获取 SREX 区域
srex = regionmask.defined_regions.srex

# 定义文字与边界样式
text_kws = dict(
    bbox=dict(facecolor="none", edgecolor="none", pad=0),
    color='dimgrey',
    weight='bold',
    fontsize=12,
)
line_kws = dict(
    color='white',
    linewidth=2,
)

# 绘制区域边界与标签
srex.plot(
    ax=ax,
    label='abbrev',
    line_kws=line_kws,
    text_kws=text_kws
)

# 去掉所有边框和坐标轴
ax.set_axis_off()
ax.set_aspect(1.2)
# 调整布局
plt.tight_layout()

pdf_save_path = r"D:\smq\paper2\result\pie_basemap.png"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(pdf_save_path, format='png', bbox_inches='tight', dpi=300)

plt.show()


## 9. Supplementary E uncertainty and validation

CWatM-ERA5/ISMN soil moisture validation, SPEI-scPDSI comparison and TCIE threshold sensitivity diagnostics.


### CWatM soil moisture versus multi-model/monthly validation input processing



In [ ]:
import xarray as xr
import numpy as np
import os
from glob import glob
from dask.distributed import Client, LocalCluster
import pandas as pd
import xarray as xr

# =====================================================
# 1. 启动 Dask 并行
# =====================================================
cluster = LocalCluster(n_workers=8, threads_per_worker=1, memory_limit="20GB")
client = Client(cluster)
print("✅ Dask dashboard:", client.dashboard_link)

# =====================================================
# 2. 路径配置
# =====================================================
soil_dir = r"F:\smq\paper2\soilmoist"
pattern = os.path.join(soil_dir, "*_gswp3-w5e5_obsclim_histsoc_default_soilmoist_global_monthly_1901_2019.nc")
files = sorted(glob(pattern))

out_dir = os.path.join(soil_dir, "correlation_results")
os.makedirs(out_dir, exist_ok=True)

print(f"找到 {len(files)} 个 soilmoist 文件")


# =====================================================
# 3.读取 w 数据并手动创建时间轴
# =====================================================
w_fp = r"E:\smq\w\w_monthavg.nc"

# 1️⃣ 关闭时间解码，读取原始 numeric 时间
w_ds = xr.open_dataset(w_fp, engine="h5netcdf", decode_times=False, chunks={"time": -1, "lat": 30, "lon": 30})

# 2️⃣ 获取时间信息
time_var = w_ds["time"]
units = time_var.attrs.get("units", "")
print("时间单位:", units)

# 3️⃣ 假设第一个月是 1901-01，对应 "Months since 1901-01-01"
#     如果起始年份不同，请改成你的起点
base_year = 1915
base_month = 1

# 构造新的时间轴
new_time = pd.date_range(
    start=f"{base_year}-{base_month:02d}-01",
    periods=len(time_var),
    freq="MS"  # 每月开始
)

# 4️⃣ 替换时间坐标
w = w_ds["w_monthavg"].assign_coords(time=new_time)
print("✅ w 时间范围:", str(w.time.values[0])[:10], "→", str(w.time.values[-1])[:10])

# =====================================================
# 4. 定义相关性函数
# =====================================================
def corr_func(a, b):
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 3:
        return np.nan
    return np.corrcoef(a[mask], b[mask])[0, 1]

# =====================================================
# 5. 遍历所有模式逐一计算相关性
# =====================================================
for f in files:
    model_name = os.path.basename(f).split("_")[0]
    print(f"\n🔹 处理模型: {model_name}")

    out_fp = os.path.join(out_dir, f"{model_name}_soilmoist_w_corr.nc")
    if os.path.exists(out_fp):
        print(f"⏩ 已存在，跳过: {out_fp}")
        continue

    # ---- 读取 soilmoist ----
    ds = xr.open_dataset(f, engine="h5netcdf", chunks={"time": -1, "lat": 30, "lon": 30})
    da = ds["soilmoist"]

    # ---- 选择表层土壤水 ----
    if "depth" in da.dims:
        if da.sizes["depth"] > 0:
            da = da.isel(depth=0)
        else:
            da = da.squeeze("depth", drop=True)
    da = da.squeeze(drop=True)

    # ✅ 调整时间为月初（处理月中问题）

    if "time" in da.coords:
        time_vals = da["time"].values
    
        # 如果是 cftime 类型（非标准公历）
        if "cftime" in str(type(time_vals[0])):
            # 转为 datetime64（月初）
            new_time = np.array(
                [np.datetime64(f"{t.year}-{t.month:02d}-01") for t in time_vals]
            )
        else:
            # 常规 datetime64 情况
            new_time = pd.to_datetime(time_vals).to_period("M").to_timestamp()
    
        da = da.assign_coords(time=new_time)

    # ---- 时间对齐 ----
    da, w_align = xr.align(da, w, join="inner")

    # ✅ 检查时间重叠
    if da.sizes.get("time", 0) == 0 or w_align.sizes.get("time", 0) == 0:
        print(f"⚠️ 模型 {model_name} 与 w 时间无重叠（可能时间定义异常），跳过计算。")
        continue

    # ---- 计算逐格点相关性 ----
    corr = xr.apply_ufunc(
        corr_func,
        da,
        w_align,
        input_core_dims=[["time"], ["time"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
    )
    corr.name = f"{model_name}_corr"

    corr.to_netcdf(out_fp, engine="h5netcdf")
    print(f"✅ 已输出: {out_fp}")

print("\n🎯 全部模型相关性计算完成！结果保存在 correlation_results 文件夹。")


### Batch validation file reading and correlation calculation



In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import numpy as np
import os

# =========================================
# 1. 文件读取
# =========================================
folder = r"D:\smq\paper2\uncertainty\correlation_results"
files = sorted(glob.glob(os.path.join(folder, "*_soilmoist_w_corr.nc")))
print(f"共找到 {len(files)} 个文件")

datasets = []

for f in files:
    try:
        ds = xr.open_dataset(f, engine="h5netcdf")

        # 自动识别变量名
        var_name = list(ds.data_vars.keys())[0]
        da = ds[var_name]

        # --------- 维度标准化 ----------
        # 某些文件可能出现 ('lat','lat') 或 ('lon','lon') 的重复维度警告
        # 这里强制命名为 (lat, lon)
        if len(da.dims) == 2:
            da = da.rename({da.dims[0]: "lat", da.dims[1]: "lon"})
        elif "latitude" in da.dims:
            da = da.rename({"latitude": "lat"})
        elif "Longitude" in da.dims:
            da = da.rename({"Longitude": "lon"})

        # 强制 lat/lon 为坐标（防止某些文件 lat/lon 是变量）
        if "lat" not in da.coords and "lat" in ds.variables:
            da = da.assign_coords(lat=ds["lat"])
        if "lon" not in da.coords and "lon" in ds.variables:
            da = da.assign_coords(lon=ds["lon"])

        # 丢弃缺失坐标的文件
        if "lat" in da.coords and "lon" in da.coords:
            datasets.append(da)
        else:
            print(f"⚠️ {os.path.basename(f)} 缺失坐标，已跳过。")

    except Exception as e:
        print(f"❌ 读取 {f} 出错: {e}")

# =========================================
# 2. 拼接与求平均
# =========================================
if len(datasets) == 0:
    raise RuntimeError("没有成功读取任何可用数据，请检查文件内容。")

# 宽松拼接：允许 lat/lon 不完全一致
da_all = xr.concat(datasets, dim="model", join="outer", coords="minimal", compat="override")


# 求均值
da_mean = da_all.mean(dim="model", skipna=True)

# =========================================
# 3. 绘图
# =========================================
fig = plt.figure(figsize=(10, 5))
ax = plt.axes(projection=ccrs.Robinson())

im = da_mean.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    cbar_kwargs={"label": "Mean correlation across models"}
)
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_title("Mean Soil Moisture–W Correlation (All Models)", fontsize=12)
plt.tight_layout()
plt.show()

# =========================================
# 4. 保存结果
# =========================================
out_fp = os.path.join(r"D:\smq\paper2\uncertainty\mean_model_soilmoist_w_corr.nc")
da_mean.to_netcdf(out_fp)
print("✅ 已保存均值文件：", out_fp)


### CWatM ASM versus ERA5 soil moisture correlation



In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

# ======================
# 1. 文件路径
# ======================
fp_soil = r"F:\smq\paper2\soilmoist\sm_root_0.5degree.nc"
fp_w    = r"F:\smq\paper2\w\w_monthavg.nc"
out_fp  = r"D:\smq\paper2\uncertainty\result\sm_era5_w_corr.nc"

# ======================
# 2. soilmoist 数据读取
# ======================
sm = xr.open_dataset(fp_soil, engine="h5netcdf")
sm_var = list(sm.data_vars)[0]
sm_da = sm[sm_var]

# ======================
# 3. w 数据读取（关闭时间解码）
# ======================
w_ds = xr.open_dataset(fp_w, engine="h5netcdf", decode_times=False)
w_var = list(w_ds.data_vars)[0]
w_da = w_ds[w_var]

# 构建时间轴
base_year = 1915
base_month = 1
new_time = pd.date_range(
    start=f"{base_year}-{base_month:02d}-01",
    periods=len(w_da.time),
    freq="MS"
)
w_da = w_da.assign_coords(time=new_time)

# ======================
# 4. 时间对齐
# ======================
common_time = np.intersect1d(sm_da["time"].values, w_da["time"].values)
sm_da = sm_da.sel(time=common_time)
w_da  = w_da.sel(time=common_time)

# 纬度经度方向一致化
if np.any(np.diff(sm_da.lat) < 0):
    sm_da = sm_da.sortby("lat")
if np.any(np.diff(sm_da.lon) < 0):
    sm_da = sm_da.sortby("lon")
if np.any(np.diff(w_da.lat) < 0):
    w_da = w_da.sortby("lat")
if np.any(np.diff(w_da.lon) < 0):
    w_da = w_da.sortby("lon")

# 插值到统一网格
if not sm_da.lat.equals(w_da.lat) or not sm_da.lon.equals(w_da.lon):
    sm_da = sm_da.interp(lat=w_da.lat, lon=w_da.lon)

# ======================
# 5. 逐格点计算 r、p、R²
# ======================
def corr_r_p_r2(x, y):
    """x, y 是一维时间序列，返回 r, p, R2"""
    # 如果全是 NaN，返回 NaN
    if np.all(np.isnan(x)) or np.all(np.isnan(y)):
        return np.nan, np.nan, np.nan
    r, p = pearsonr(x, y)
    r2 = r**2
    return r, p, r2

# 使用 apply_ufunc
result = xr.apply_ufunc(
    corr_r_p_r2,
    sm_da, w_da,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[[], [], []],
    vectorize=True,
    dask="parallelized",  # 如果使用 dask，可并行
    output_dtypes=[float, float, float],
)

r_da, p_da, r2_da = result

# 设置名字和属性
r_da.name = "r"
p_da.name = "p_value"
r2_da.name = "R2"

for da in [r_da, p_da, r2_da]:
    da.attrs["description"] = "Pixel-wise correlation metric between soil moisture and w"

# ======================
# 6. 保存到 NetCDF
# ======================
xr.merge([r_da, p_da, r2_da]).to_netcdf(out_fp, engine="h5netcdf")
print(f"✅ 已保存相关性结果: {out_fp}")


### ERA5 validation R2 extraction diagnostic



In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xarray as xr
fig = plt.figure(figsize=(10, 5))
ax = plt.axes(projection=ccrs.Robinson())

r2_da=xr.open_dataset(r"D:\smq\paper2\uncertainty\result\sm_era5_w_corr.nc")['R2']
im = r2_da.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    cbar_kwargs={"label": "Mean correlation across models"}
)
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_title("Mean Soil Moisture–W Correlation (All Models)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import xarray as xr
r2_da=xr.open_dataset(r"D:\smq\paper2\uncertainty\result\sm_era5_w_corr.nc")['R2']
r2_da.mean()

### scPDSI-based CDHW daily mask construction



In [ ]:
import xarray as xr
import numpy as np
from dask.distributed import Client, LocalCluster
import os
import dask

# ============================================================
# 1. 启动 Dask 集群（根据设备配置修改）
# ============================================================
cluster = LocalCluster(
    n_workers=10,
    threads_per_worker=2,
    memory_limit="100GB"
)
client = Client(cluster)
print("✅ Dask dashboard:", client.dashboard_link)

# ============================================================
# 2. 参数设置
# ============================================================
min_days = 3          # 定义连续阈值
pdsi_thresh = -1      # 干旱阈值
start_date = "1920-01-01"
end_date = "2019-12-31"
out_fp = r"D:\smq\paper2\uncertainty\cdhw_mask_pdsi.nc"

# ============================================================
# 3. 数据读取
# ============================================================
print("📂 正在加载数据...")

# 热浪阈值 (tas95)
tas95 = (
    xr.open_dataset(
        r"E:\smq\tas95_1981_2010.nc",
        chunks={"lat": 30, "lon": 30}
    )["tas95"] - 273.15
)

# 日最高温 (tasmax)
tasmax = (
    xr.open_dataset(
        r"E:\smq\Tmax\TMax_daily.nc",
        engine="h5netcdf"
    )
    .sel(time=slice(start_date, end_date))
    .chunk({"time": -1, "lat": 30, "lon": 30})["TMax"]
)

# 月尺度 PDSI
pdsi = xr.open_dataset(
    r"D:\smq\paper2\uncertainty\scPDSI.cru_ts4.09early1.1901.2024.cal_1901_24.bams.2025.GLOBAL.IGBP.WHC.1901.2024.nc"
)

# 修正维度名
if "latitude" in pdsi.dims:
    pdsi = pdsi.rename({"latitude": "lat", "longitude": "lon"})

pdsi = pdsi['scpdsi'].sel(time=slice(start_date, end_date)).chunk({"time": -1, "lat": 30, "lon": 30})

print("✅ 数据加载完成：")
print(f"   - tas95: {tas95.shape}")
print(f"   - tasmax: {tasmax.shape}")
print(f"   - pdsi: {pdsi.shape}")

# ============================================================
# 4. 将月尺度 pdsi 转为日尺度
# ============================================================
print("🕒 将 PDSI 插值为日尺度...")
pdsi_daily = pdsi.resample(time="1D").ffill()

# ============================================================
# 5. 构建干旱与热浪标志
# ============================================================
print("🌡️ 构建热浪与干旱标志...")
drought_flag = pdsi_daily < pdsi_thresh
hot_flag = tasmax > tas95

# ============================================================
# 6. 连续事件检测函数
# ============================================================
def detect_events_1d(flag, min_days=3):
    """检测布尔序列中连续为 True 的事件"""
    mask = np.zeros_like(flag, dtype=bool)
    count = 0
    for i in range(len(flag)):
        if flag[i]:
            count += 1
        else:
            if count >= min_days:
                mask[i-count:i] = True
            count = 0
    if count >= min_days:
        mask[len(flag)-count:] = True
    return mask

# ============================================================
# 7. 并行热浪事件检测
# ============================================================
print("⚙️ 检测热浪事件（并行中）...")
hw_mask = xr.apply_ufunc(
    detect_events_1d,
    hot_flag,
    input_core_dims=[["time"]],
    output_core_dims=[["time"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[bool],
    kwargs={"min_days": min_days},
)

# ============================================================
# 8. 构建 CDHW 掩膜
# ============================================================
print("🔥 计算 CDHW 掩膜...")
cdhw_mask = hw_mask & drought_flag

# ============================================================
# 9. 保存结果（安全写出，防止锁冲突）
# ============================================================
print("💾 正在计算并保存结果...")
cdhw_mask_local = cdhw_mask.compute()  # 先完成计算（避免 Dask 并行写锁）

ds_out = xr.Dataset({"cdhw_mask": cdhw_mask_local.astype(np.int8)})

if os.path.exists(out_fp):
    os.remove(out_fp)

comp = dict(zlib=True, complevel=4)
encoding = {"cdhw_mask": comp}

ds_out.to_netcdf(out_fp, engine="h5netcdf", encoding=encoding)
print("✅ 已成功保存 CDHW 掩膜：", out_fp)

# ============================================================
# 10. 关闭 Dask 集群
# ============================================================
client.close()
cluster.close()
print("🏁 全部流程完成！")


### scPDSI mask parallel processing/report



In [ ]:
import xarray as xr
import numpy as np
from dask.distributed import Client, LocalCluster, performance_report
import dask
import os
import time

# ======================
# 配置 Dask
# ======================
cluster = LocalCluster(
    n_workers=10,
    threads_per_worker=1,
    memory_limit="100GB"
)
client = Client(cluster)
print("✅ Dashboard 地址:", client.dashboard_link)

# ======================
# 参数
# ======================
start_date, end_date = "1920-01-01", "2019-12-31"

# ======================
# 打开数据
# ======================
print("⏳ [1] 读取温度分位数据...")
tas25 = xr.open_dataset(r"D:\smq\paper2\data\tas25_1981_2010.nc")["tas25"] - 273.15
tas75 = xr.open_dataset(r"D:\smq\paper2\data\tas75_1981_2010.nc")["tas75"] - 273.15
tas95 = xr.open_dataset(r"E:\smq\tas95_1981_2010.nc")["tas95"] - 273.15

print("⏳ [2] 打开每日 Tmax 数据...")
tasmax = xr.open_dataset(
    r"E:\smq\Tmax\TMax_daily.nc", engine="h5netcdf"
).sel(time=slice(start_date, end_date)).chunk({"time": -1, "lat": 50, "lon": 50})["TMax"]

print("⏳ [3] 打开月尺度 scPDSI 数据...")
pdsi = xr.open_dataset(
    r"D:\smq\paper2\uncertainty\scPDSI.cru_ts4.09early1.1901.2024.cal_1901_24.bams.2025.GLOBAL.IGBP.WHC.1901.2024.nc"
)
if "latitude" in pdsi.dims:
    pdsi = pdsi.rename({"latitude": "lat", "longitude": "lon"})
pdsi = pdsi['scpdsi'].sel(time=slice(start_date, end_date)).chunk({"time": -1, "lat": 50, "lon": 50})

print("✅ [4] 转为日尺度...")
t0 = time.time()
scpdsi_daily = pdsi.resample(time="1D").ffill()
print(f"✅ 完成月→日插值 ({time.time()-t0:.2f}s)")

print("⏳ [5] 读取 CDHW 掩膜...")
cdhw_mask = xr.open_dataset(
    r"D:\smq\paper2\uncertainty\cdhw_mask_pdsi.nc", engine="h5netcdf"
)["cdhw_mask"].sel(time=slice(start_date, end_date)).chunk({"time": -1, "lat": 50, "lon": 50})

# 统一 chunk，防止 rechunk 卡顿
print("⏳ [6] 统一 chunk 大小...")
scpdsi_daily = scpdsi_daily.chunk({"time": -1, "lat": 50, "lon": 50})

print("⏳ [7] 构建 daily_severity 任务图...")
t0 = time.time()
daily_severity = xr.where(
    cdhw_mask,
    np.abs(scpdsi_daily) * (tasmax - tas25) / (tas75 - tas25),
    0
)
print(f"✅ daily_severity 构建完成 ({time.time()-t0:.2f}s)")
print(daily_severity)

# ======================
# 保存输出
# ======================
output_file = r"D:\smq\paper2\uncertainty\cdhw_daily_severity_pdsi.nc"
encoding = {"daily_severity": {"zlib": True, "complevel": 4, "dtype": "float32"}}

if os.path.exists(output_file):
    os.remove(output_file)

ds_out = xr.Dataset({"daily_severity": daily_severity})
delayed_obj = ds_out.to_netcdf(
    output_file,
    engine="h5netcdf",
    encoding=encoding,
    compute=False
)


### scPDSI event aggregation diagnostic



In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from dask.distributed import Client, LocalCluster
import os
import dask  # 提前导入dask，避免后续计算时导入延迟

# ======================
# 配置 Dask
# ======================
cluster = LocalCluster(
    n_workers=10,
    threads_per_worker=1,
    memory_limit="100GB"
)
client = Client(cluster)
print("Dashboard 地址:", client.dashboard_link)

# ======================
# 读数据（修复分块警告）
# ======================
mask_fp = r"D:\smq\paper2\data\cdhw\masks_cdhw_hw_di.nc"
ds_mask = xr.open_dataset(
    mask_fp,
    engine="h5netcdf"
).chunk({"time": -1, "lat": 50, "lon": 50})
cdhw_mask = ds_mask["cdhw_mask"]

severity_fp = r"D:\smq\paper2\uncertainty\cdhw_daily_severity_pdsi.nc"
ds_sev = xr.open_dataset(
    severity_fp,
    engine="h5netcdf"
).chunk({"time": -1, "lat": 50, "lon": 50})
daily_severity = ds_sev["daily_severity"]

def process_point_intensity(mask_1d, time_vals):

    # 转换时间为 pandas DatetimeIndex
    times = pd.to_datetime(time_vals)

    # 定义年份范围
    years_all = np.arange(times[0].year, times[-1].year + 1)
    c = np.zeros(len(years_all), dtype=np.float32)  # 存储每年的结果

    # Step1: 提取事件（连续非零天为一个事件）
    in_event = False
    event_id = np.zeros(len(mask_1d), dtype=int)
    eid = 0

    for i, val in enumerate(mask_1d):
        if val > 0 and not in_event:
            eid += 1
            in_event = True
        elif val == 0:
            in_event = False
        if in_event:
            event_id[i] = eid

    # Step2: 遍历每个事件，计算其平均强度，并记录起始年份
    yearly_intensities = {year: [] for year in years_all}  # 每年收集事件的平均强度

    for e in np.unique(event_id):
        if e == 0:
            continue  # 跳过无事件区域

        indices = np.where(event_id == e)[0]
        start_idx = indices[0]
        event_values = mask_1d[indices]

        avg_intensity = np.mean(event_values)  # 该事件的平均强度
        year = times[start_idx].year

        if year in yearly_intensities:
            yearly_intensities[year].append(avg_intensity)

    # Step3: 对每一年，取该年所有事件平均强度的平均值
    for i, year in enumerate(years_all):
        intensities = yearly_intensities[year]
        if len(intensities) > 0:
            c[i] = np.mean(intensities)  # 可改为 np.sum 如果需要总和
        else:
            c[i] = 0.0  # 无事件则为0

    return c  # shape=(nyears,)
# ======================
# 计算完整年份范围（用于输出维度）
# ======================
years_all = np.arange(
    int(cdhw_mask.time.dt.year.min()),
    int(cdhw_mask.time.dt.year.max()) + 1
)

# ======================
# 并行调用函数
# ======================
annual_strength = xr.apply_ufunc(
    process_point_intensity,
    daily_severity,
    input_core_dims=[["time"]],
    output_core_dims=[["year"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[np.float32],
    dask_gufunc_kwargs={"output_sizes": {"year": len(years_all)}},
    kwargs={"time_vals": daily_severity.time.values},
)


annual_strength = annual_strength.assign_coords(year=years_all)
annual_strength.name = "annual_severity"

# ======================
# 创建Dataset
# ======================
ds_out = xr.Dataset(
    data_vars={"annual_severity": annual_strength},
    coords={
        "year": years_all,
        "lat": cdhw_mask.lat,
        "lon": cdhw_mask.lon
    }
)

# ======================
# 保存结果
# ======================
output_file = r"D:\smq\paper2\uncertainty\cdhw_meanevent_yearly_severity.nc"
encoding = {
    "annual_severity": {"zlib": True, "complevel": 4, "dtype": "float32"}
}

if os.path.exists(output_file):
    os.remove(output_file)

print("保存结果...")
delayed = ds_out.to_netcdf(
    output_file,
    engine="h5netcdf",
    encoding=encoding,
    compute=False
)
dask.compute(delayed)

print(f"[Success] 文件已写入: {output_file}")
client.close()


### SPEI monthly-daily mask construction



In [ ]:
import xarray as xr
import numpy as np
from dask.distributed import Client, LocalCluster
import os
import dask

# ============================================================
# 1. 启动 Dask 集群（根据设备配置修改）
# ============================================================
cluster = LocalCluster(
    n_workers=10,
    threads_per_worker=2,
    memory_limit="100GB"
)
client = Client(cluster)
print("✅ Dask dashboard:", client.dashboard_link)

# ============================================================
# 2. 参数设置
# ============================================================
min_days = 3          # 定义连续阈值
pdsi_thresh = -1      # 干旱阈值
start_date = "1920-01-01"
end_date = "2019-12-31"
out_fp = r"D:\smq\paper2\uncertainty\cdhw_mask_spei.nc"

# ============================================================
# 3. 数据读取
# ============================================================
print("📂 正在加载数据...")

# 热浪阈值 (tas95)
tas95 = (
    xr.open_dataset(
        r"E:\smq\tas95_1981_2010.nc",
        chunks={"lat": 30, "lon": 30}
    )["tas95"] - 273.15
)

# 日最高温 (tasmax)
tasmax = (
    xr.open_dataset(
        r"E:\smq\Tmax\TMax_daily.nc",
        engine="h5netcdf"
    )
    .sel(time=slice(start_date, end_date))
    .chunk({"time": -1, "lat": 30, "lon": 30})["TMax"]
)

# 月尺度 PDSI
pdsi = xr.open_dataset(
    r"D:\smq\paper2\uncertainty\monthly_spei.nc"
)

# 修正维度名
pdsi = pdsi['spei'].sel(time=slice(start_date, end_date)).chunk({"time": -1, "lat": 30, "lon": 30})

print("✅ 数据加载完成：")
print(f"   - tas95: {tas95.shape}")
print(f"   - tasmax: {tasmax.shape}")
print(f"   - pdsi: {pdsi.shape}")

# ============================================================
# 4. 将月尺度 pdsi 转为日尺度
# ============================================================
print("🕒 将 PDSI 插值为日尺度...")
pdsi_daily = pdsi.resample(time="1D").ffill()

# ============================================================
# 5. 构建干旱与热浪标志
# ============================================================
print("🌡️ 构建热浪与干旱标志...")
drought_flag = pdsi_daily < pdsi_thresh
hot_flag = tasmax > tas95

# ============================================================
# 6. 连续事件检测函数
# ============================================================
def detect_events_1d(flag, min_days=3):
    """检测布尔序列中连续为 True 的事件"""
    mask = np.zeros_like(flag, dtype=bool)
    count = 0
    for i in range(len(flag)):
        if flag[i]:
            count += 1
        else:
            if count >= min_days:
                mask[i-count:i] = True
            count = 0
    if count >= min_days:
        mask[len(flag)-count:] = True
    return mask

# ============================================================
# 7. 并行热浪事件检测
# ============================================================
print("⚙️ 检测热浪事件（并行中）...")
hw_mask = xr.apply_ufunc(
    detect_events_1d,
    hot_flag,
    input_core_dims=[["time"]],
    output_core_dims=[["time"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[bool],
    kwargs={"min_days": min_days},
)

# ============================================================
# 8. 构建 CDHW 掩膜
# ============================================================
print("🔥 计算 CDHW 掩膜...")
cdhw_mask = hw_mask & drought_flag

# ============================================================
# 9. 保存结果（安全写出，防止锁冲突）
# ============================================================
print("💾 正在计算并保存结果...")
cdhw_mask_local = cdhw_mask.compute()  # 先完成计算（避免 Dask 并行写锁）

ds_out = xr.Dataset({"cdhw_mask": cdhw_mask_local.astype(np.int8)})

if os.path.exists(out_fp):
    os.remove(out_fp)

comp = dict(zlib=True, complevel=4)
encoding = {"cdhw_mask": comp}

ds_out.to_netcdf(out_fp, engine="h5netcdf", encoding=encoding)
print("✅ 已成功保存 CDHW 掩膜：", out_fp)

# ============================================================
# 10. 关闭 Dask 集群
# ============================================================
client.close()
cluster.close()
print("🏁 全部流程完成！")


### SPEI-scPDSI CDHW mask correlation



In [ ]:
# -*- coding: utf-8 -*-
import xarray as xr
import numpy as np
from scipy.stats import linregress

# ====================
# 路径设置
# ====================
file_spei = r"D:\smq\paper2\uncertainty\cdhw_mask_spei.nc"
file_pdsi = r"D:\smq\paper2\uncertainty\cdhw_mask_pdsi.nc"
out_file  = r"D:\smq\paper2\uncertainty\result\spei-pdsi_cdhw_mask_corr.nc"
mask = xr.open_dataset(r"D:\smq\paper2\data\modified_mask.nc").mask_data
mask = mask > 1

# ====================
# 读取数据
# ====================
ds_spei = xr.open_dataset(file_spei)
ds_pdsi = xr.open_dataset(file_pdsi)

var_spei = list(ds_spei.data_vars)[0]
var_pdsi = list(ds_pdsi.data_vars)[0]

da_spei = ds_spei[var_spei]
da_pdsi = ds_pdsi[var_pdsi]

# ====================
# 年聚合（按年份求和或平均）
# ====================
da_spei_yearly = da_spei.groupby('time.year').mean()
da_pdsi_yearly = da_pdsi.groupby('time.year').mean()

# ====================
# 定义逐格点函数
# ====================
def corr_func(x, y):
    """返回 r, p, r²"""
    mask_valid = np.isfinite(x) & np.isfinite(y)
    x = x[mask_valid]
    y = y[mask_valid]
    if len(x) < 2 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan, np.nan
    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    return r_value, p_value, r_value ** 2

# ====================
# 逐格点并行计算
# ====================
res = xr.apply_ufunc(
    corr_func,
    da_spei_yearly,
    da_pdsi_yearly,
    input_core_dims=[["year"], ["year"]],
    output_core_dims=[[], [], []],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float, float, float],
)

r, p, r2 = res

# ====================
# 构建输出 Dataset
# ====================
ds_out = xr.Dataset(
    {
        "r": (("lat", "lon"), r.where(mask).data),
        "p_value": (("lat", "lon"), p.where(mask).data),
        "R2": (("lat", "lon"), r2.where(mask).data),
    },
    coords={
        "lat": r.lat,
        "lon": r.lon,
    }
)


# ====================
# 保存结果
# ====================
ds_out.to_netcdf(out_file)
print(f"✅ 已保存空间相关结果，包括 r, p, r² 到: {out_file}")


### SPEI-scPDSI optional visualization



In [ ]:
# ====================
# 可视化（可选）
# ====================
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
r.plot(cmap='RdBu_r', vmin=-1, vmax=1)
plt.title("Spatial correlation between CDHW-SPEI and CDHW-PDSI")
plt.show()

In [ ]:
import xarray as xr
r2_da=xr.open_dataset(r"D:\smq\paper2\uncertainty\result\spei-pdsi_cdhw_mask_corr.nc")['R2']
r2_da.mean()

### ISMN station soil moisture point extraction



In [ ]:
import os
import pandas as pd
import xarray as xr
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
from tqdm import tqdm  # 可选，用于显示进度

# -----------------------------
# 1. ISMN 数据根目录
# -----------------------------
root_dir = r"F:\smq\paper2\soilmoist\ismn\Data_separate_files_header_19500101_20191231_12896_nFVG_20251215 (1)"

# -----------------------------
# 2. 读取模型土壤水数据
# -----------------------------
ds_w = xr.open_dataset(r"F:\smq\paper2\w\w_daily.nc")
w_time = pd.to_datetime(ds_w["time"].values)
w_lat = ds_w["lat"].values
w_lon = ds_w["lon"].values

# -----------------------------
# 3. 循环每个网络/站点
# -----------------------------
results = []

networks = [n for n in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, n))]

for network in networks:
    network_dir = os.path.join(root_dir, network)
    stations = [s for s in os.listdir(network_dir) if os.path.isdir(os.path.join(network_dir, s))]

    for station in tqdm(stations, desc=f"Processing {network}"):
        sta_dir = os.path.join(network_dir, station)
        stm_files = [f for f in os.listdir(sta_dir) if f.endswith(".stm")]

        records = []
        lat = lon = None

        for stm_file in stm_files:
            stm_path = os.path.join(sta_dir, stm_file)
            try:
                with open(stm_path, "r") as f:
                    lines = f.readlines()
                    if len(lines) < 2:
                        continue

                    # 头信息
                    header = lines[0].strip().split()
                    if len(header) < 8:
                        continue

                    if lat is None:  # 取第一个文件的 lat/lon
                        lat = float(header[3])
                        lon = float(header[4])

                    # 数据记录
                    for line in lines[1:]:
                        parts = line.strip().split()
                        if len(parts) < 4:
                            continue
                        date = parts[0] + " " + parts[1]
                        value = parts[2]
                        try:
                            value = float(value)
                        except:
                            value = None
                        records.append({"datetime": date, "value": value})
            except Exception as e:
                print(f"FAILED: {stm_path}\n{e}")

        if not records or lat is None:
            continue  # 无数据跳过

        # 转为 DataFrame 并按天聚合
        df_sta = pd.DataFrame(records)
        df_sta["datetime"] = pd.to_datetime(df_sta["datetime"])
        df_sta["date"] = df_sta["datetime"].dt.date
        df_sta_daily = df_sta.groupby("date")["value"].mean().reset_index()

        # 剔除不足一年的点
        if len(df_sta_daily) < 365:
            continue

        # 提取最近网格点
        w_sel = ds_w.sel(lat=lat, lon=lon, method="nearest")["w"]
        df_w = pd.DataFrame({
            "date": w_time.date,
            "w_value": w_sel.values
        })

        # 合并并筛选有效值
        df_merge = pd.merge(df_sta_daily, df_w, on="date", how="inner")
        valid = df_merge["value"].notna() & df_merge["w_value"].notna()
        if valid.sum() < 365:
            continue

        x = df_merge.loc[valid, "value"]
        y = df_merge.loc[valid, "w_value"]

        r, p_value = pearsonr(x, y)
        r2 = r2_score(x, y)

        results.append({
            "network": network,
            "station": station,
            "lat": lat,
            "lon": lon,
            "r": r,
            "r2": r2,
            "p_value": p_value,
            "n_samples": valid.sum()
        })

# -----------------------------
# 4. 保存结果
# -----------------------------
df_results = pd.DataFrame(results)
df_results.to_csv(r"F:\smq\paper2\ismn_vs_model_r_results.csv", index=False)
print("完成！共有站点:", len(df_results))
print(df_results.head())


### Fig. S9 ASM versus ERA5 spatial correlation



In [ ]:
# -*- coding: utf-8 -*-
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import cartopy.feature as cfeature
import os

# ======================
# 数据读取
# ======================
ds = xr.open_dataset(r"D:\smq\paper2\uncertainty\result\sm_era5_w_corr.nc", decode_times=False)
r = ds["r"]
p = ds["p_value"]

# 新增掩膜
eff = xr.open_dataset(r"D:\smq\paper2\data\modified_mask.nc")["mask_data"]
mask = eff > 1

# 只保留掩膜内区域
r = r.where(mask)
p = p.where(mask)

# ======================
# 显著性筛选与灰色背景
# ======================
r_sig = r.where(p < 0.05)  # 显著区域保留
r_nsig = xr.where((p >= 0.05) & mask, 1, np.nan)  # 不显著区域标记灰色

# # ======================
# # 自定义截取色带
# # ======================
# base_cmap = plt.cm.YlGn
# new_cmap = LinearSegmentedColormap.from_list(
#     "truncated_YlGn",
#     base_cmap(np.linspace(0.2, 0.6, 256))
# )

# # ======================
# # 自定义截取中间色带
# # ======================
# base_cmap = plt.cm.RdYlBu
# new_cmap = LinearSegmentedColormap.from_list(
#     "truncated_Blues",
#     base_cmap(np.linspace(0.2, 0.8, 256))  # 截取 0.3–0.7 区间
# )

# ======================
# 绘图
# ======================
fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

# 绘制不显著区域（灰色）
ax.pcolormesh(
    r.lon,
    r.lat,
    r_nsig,
    transform=ccrs.PlateCarree(),
    cmap="Greys",
    vmin=0,
    vmax=1,
    alpha=0.3
)

# 绘制显著区域
im = r_sig.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap='RdYlBu_r',
    # vmax=1,
    # vmin=0,
    add_colorbar=True,
    cbar_kwargs={
        "label": "R² (p < 0.05)",
        "orientation": "horizontal",
        "pad": 0.05,
        "shrink": 0.6,
        "aspect": 40
    }
)

# 修改 colorbar 样式
cbar = im.colorbar
cbar.ax.tick_params(labelsize=10)
# cbar.set_label("R² (p < 0.05)", fontsize=16, fontweight='bold')
cbar.set_label("Correlation with ERA5 SM", fontsize=16, fontweight='bold')

# 添加地理要素
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='black', alpha=0.8)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, alpha=0.8)

# 去掉边框与标题
plt.title("")
plt.box(False)
plt.tight_layout()

pdf_save_path = r"D:\smq\paper2\result\era5-r.png"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(pdf_save_path, format='png', bbox_inches='tight')
plt.show()


### Combined ERA5 and SPEI-scPDSI uncertainty map layout



In [ ]:
# -*- coding: utf-8 -*-
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import cartopy.feature as cfeature
import os

# ======================
# 数据路径设置
# ======================
file1 = r"D:\smq\paper2\uncertainty\result\sm_era5_w_corr.nc"
file2 = r"D:\smq\paper2\uncertainty\result\spei-pdsi_cdhw_mask_corr.nc"
mask_file = r"D:\smq\paper2\data\modified_mask.nc"

# ======================
# 掩膜加载
# ======================
eff = xr.open_dataset(mask_file)["mask_data"]
mask = eff > 1

# ======================
# 自定义色带
# ======================
base_cmap = plt.cm.YlGn
new_cmap = LinearSegmentedColormap.from_list(
    "truncated_YlGn",
    base_cmap(np.linspace(0.2, 0.7, 256))
)

# ======================
# 绘图准备函数
# ======================
def process_dataset(path, mask):
    ds = xr.open_dataset(path, decode_times=False)
    r = ds["R2"].where(mask)
    p = ds["p_value"].where(mask)
    r_sig = r.where(p < 0.05)
    r_nsig = xr.where((p >= 0.05) & mask, 1, np.nan)
    return r, p, r_sig, r_nsig

# 读取两个文件
r1, p1, r_sig1, r_nsig1 = process_dataset(file1, mask)
r2, p2, r_sig2, r_nsig2 = process_dataset(file2, mask)

# ======================
# 绘图（并排，无标题、无边框）
# ======================
fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

datasets = [(r1, r_sig1, r_nsig1), (r2, r_sig2, r_nsig2)]

for ax, (r, r_sig, r_nsig) in zip(axes, datasets):
    ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

    # 不显著区域灰色
    ax.pcolormesh(
        r.lon, r.lat, r_nsig,
        transform=ccrs.PlateCarree(),
        cmap="Greys", vmin=0, vmax=1, alpha=0.3
    )

    # 显著区域
    im = r_sig.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=new_cmap,
        vmax=1, vmin=0,
        add_colorbar=False
    )

    # 添加地理要素
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor='black', alpha=0.8)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, alpha=0.8)

    # 去除边框与标题
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title("")
    ax.set_frame_on(False)


# ======================
# 共享 colorbar
# ======================
cbar = fig.colorbar(
    im,
    ax=axes,
    orientation='horizontal',
    fraction=0.035,
    pad=0.22,   # ✅ 向下移动色带（原为0.15）
    shrink=0.4,
    aspect=25
)
cbar.set_label("R² (p < 0.05)", fontsize=16, fontweight='bold', labelpad=10)
cbar.ax.tick_params(labelsize=14, width=0.8)

cbar.set_label("R² (p < 0.05)", fontsize=16, fontweight='bold', labelpad=10)
cbar.ax.tick_params(labelsize=14, width=0.8)

plt.subplots_adjust(bottom=0.18, wspace=0.05)  # 手动微调间距
pdf_save_path = r"D:\smq\paper2\result\uncertainty.pdf"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(pdf_save_path, format='pdf', bbox_inches='tight', transparent=True)
plt.show()


### Fig. S11 SPEI versus scPDSI CDHW correlation



In [ ]:
# -*- coding: utf-8 -*-
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import cartopy.feature as cfeature
import os

# ======================
# 数据读取
# ======================
ds = xr.open_dataset(r"D:\smq\paper2\uncertainty\result\spei-pdsi_cdhw_mask_corr.nc", decode_times=False)
r = ds["r"]
p = ds["p_value"]

# 新增掩膜
eff = xr.open_dataset(r"D:\smq\paper2\data\modified_mask.nc")["mask_data"]
mask = eff > 1

# 只保留掩膜内区域
r = r.where(mask)
p = p.where(mask)

# ======================
# 显著性筛选与灰色背景
# ======================
r_sig = r.where(p < 0.05)  # 显著区域保留
r_nsig = xr.where((p >= 0.05) & mask, 1, np.nan)  # 不显著区域标记灰色

# # ======================
# # 自定义截取色带
# # ======================
# base_cmap = plt.cm.YlGn
# new_cmap = LinearSegmentedColormap.from_list(
#     "truncated_YlGn",
#     base_cmap(np.linspace(0.2, 0.6, 256))
# )

# # ======================
# # 自定义截取中间色带
# # ======================
# base_cmap = plt.cm.RdYlBu
# new_cmap = LinearSegmentedColormap.from_list(
#     "truncated_Blues",
#     base_cmap(np.linspace(0.2, 0.8, 256))  # 截取 0.3–0.7 区间
# )

# ======================
# 绘图
# ======================
fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

# 绘制不显著区域（灰色）
ax.pcolormesh(
    r.lon,
    r.lat,
    r_nsig,
    transform=ccrs.PlateCarree(),
    cmap="Greys",
    vmin=0,
    vmax=1,
    alpha=0.3
)

# 绘制显著区域
im = r_sig.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap='RdYlBu_r',
    # vmax=1,
    # vmin=0,
    add_colorbar=True,
    cbar_kwargs={
        "label": "R² (p < 0.05)",
        "orientation": "horizontal",
        "pad": 0.05,
        "shrink": 0.6,
        "aspect": 40
    }
)

# 修改 colorbar 样式
cbar = im.colorbar
cbar.ax.tick_params(labelsize=10)
# cbar.set_label("R² (p < 0.05)", fontsize=16, fontweight='bold')
cbar.set_label("Correlation of Drought Indicators", fontsize=16, fontweight='bold')

# 添加地理要素
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='black', alpha=0.8)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, alpha=0.8)

# 去掉边框与标题
plt.title("")
plt.box(False)
plt.tight_layout()

pdf_save_path = r"D:\smq\paper2\result\spei-pdsi-r.png"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(pdf_save_path, format='png', bbox_inches='tight')
plt.show()


### Fig. S10 ISMN station validation map



In [ ]:
import os
import pandas as pd
import geopandas as gpd
import regionmask
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import cartopy.feature as cfeature
import matplotlib as mpl
import numpy as np

# =============================
# 1. 读取合并后的站点数据
# =============================
folder = r"F:\smq\paper2\ismn_validation_weighted_sum"
all_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".csv")]
df_all = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)

# 剔除 p_value < 0.05
# df_all = df_all[df_all["p_value"] >= 0.05].copy()
print(f"Total stations: {len(df_all)}")

# 转 GeoDataFrame
gdf = gpd.GeoDataFrame(
    df_all,
    geometry=gpd.points_from_xy(df_all["lon"], df_all["lat"]),
    crs="EPSG:4326"
)

# =============================
# 2. 创建底图
# =============================
fig = plt.figure(figsize=(7, 4))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

# 底图
ax.add_feature(cfeature.LAND, color='lightgrey', alpha=0.5)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='gray')

# =============================
# 3. 绘制站点 r
# =============================
vmin, vmax = -1, 1
cmap = plt.get_cmap('RdYlBu_r')

scatter = ax.scatter(
    gdf['lon'], gdf['lat'],
    c=gdf['r'],
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    s=25,
    transform=ccrs.PlateCarree()
)

# =============================
# 4. 短色带，只保留指定刻度
# =============================
ticks = [-1, -0.5, 0, 0.5, 1]
cbar_ax = fig.add_axes([0.35, 0.05, 0.3, 0.02])  # [left, bottom, width, height] 调短
cbar = plt.colorbar(scatter, cax=cbar_ax, orientation='horizontal', ticks=ticks)
cbar.set_label('Correlation with ISMN SM (Site Data)', fontsize=12, fontweight='bold')
cbar.ax.tick_params(labelsize=10)

# 去掉坐标轴
ax.set_axis_off()
# ax.set_aspect(1.2)

# =============================
# 5. 保存
# =============================
pdf_save_path = r"D:\smq\paper2\result\ismn_r_sm.png"
os.makedirs(os.path.dirname(pdf_save_path), exist_ok=True)
plt.savefig(pdf_save_path, format='png', bbox_inches='tight', dpi=300)

plt.show()
